# Cross-Lingual Memory in LLM Agents: `main_r1`, Part 3 of 4: reader Llama-3.1-8B

Settings: **GPU T4 x2**, **Internet on**, secret **`HF_TOKEN`** attached (Add-ons → Secrets). Inputs needed: the output of `main_r1_1_prep`
(right panel → **Add Input** → *Your Work* → the notebook → its latest version).

Run it unattended with **Save Version → Save & Run All (Commit)**. Rough time: ~2-4 h (estimate). Every stage caches what it
has finished, so if the session dies, commit again and finished work is skipped. Blocks: `main` (165 items,
20 sessions) and the ablations `short`, `q3`, `digits` (ENDSEM_PLAN.md §1).

## 1. Install and check the GPUs

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!pip install -q vllm==0.30.0 sentence-transformers 2>&1 | tail -2
# vLLM upgrades PyTorch; Kaggle's preinstalled torchaudio then no longer matches and breaks
# `import transformers`. We never use audio, so remove it.
!pip uninstall -y -q torchaudio
import subprocess, os, glob, shutil
CHECK = ("import torch, vllm; from vllm import LLM; import transformers; "
         "print('torch', torch.__version__, '| cuda', torch.version.cuda, '| GPUs visible:', torch.cuda.device_count(), "
         "'| vllm', vllm.__version__, '| transformers', transformers.__version__); "
         "assert torch.cuda.device_count() == 2, 'PyTorch cannot see both T4 GPUs'")
r = subprocess.run(["python", "-c", CHECK], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip()[-1500:])
if r.returncode != 0:
    raise RuntimeError("STOP: environment check failed (see message above). Copy it to Claude.")
os.environ["CLM_BACKEND"] = "vllm"
print("Environment OK -> backend vllm")

## 2. Hugging Face token (from Kaggle Secrets; never printed)

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
print("HF token loaded:", bool(os.environ["HF_TOKEN"]))

## 3. Write pipeline code and data (frozen fillers included)

In [ ]:
!mkdir -p /kaggle/working/clm/src/clm /kaggle/working/clm/data/items /kaggle/working/clm/data/haystacks /kaggle/working/clm/results/main_r1

In [ ]:
%%writefile /kaggle/working/clm/src/clm/__init__.py
"""Cross-Lingual Memory in LLM Agents: experiment pipeline (design_spec.md v1.0)."""


In [ ]:
%%writefile /kaggle/working/clm/src/clm/config.py
"""Fixed experimental settings. Values trace to phase2/design_spec.md (section in brackets)."""
import os
from pathlib import Path

# Data root: the project folder locally, /kaggle/working/clm on Kaggle (set CLM_ROOT).
ROOT = Path(os.environ.get("CLM_ROOT", Path(__file__).resolve().parents[2]))
DATA = ROOT / "data"
LANGS = ("en", "hi", "hg")                       # [2.1]
ALL_SETUPS = ("FC", "RM-S1", "RM-S2", "RM-S3", "OR-S1", "OR-S2", "OR-S3")   # [4.1]
TOP_K = 5                                        # [4.2]
RECALL_KS = (1, 3, 5, 10)                        # [4.2]
MAX_SESSIONS = 20                                # fillers are drawn for this length; shorter histories use a prefix
FILLER_POOL_SIZE = 40                            # [3.5]
# Evidence session slot for each position class, as a function of history length n.
# n=10 gives 2 / 5 / 9, exactly the pilot's slots; n=20 gives 4 / 10 / 19.
EVIDENCE_SLOT = {"early": lambda n: round(0.2 * n), "middle": lambda n: n // 2, "late": lambda n: n - 1}

# Experiment blocks [7] + ENDSEM_PLAN §1. Each block is run separately (CLM_BLOCK) and writes its own results
# folder; all blocks of one run share the English-translation cache. Item membership comes from flags set by
# src/merge_items.py. The pilot hit the ceiling rule (§8), so Long (20 sessions) is the main length and Short
# (10) is ablation A (DEVIATIONS #8).
BLOCKS = {
    "main":   {"sessions": 20, "qvs": ("q1", "q2"), "setups": ALL_SETUPS, "stores": LANGS, "flag": None, "digits": False},
    "short":  {"sessions": 10, "qvs": ("q1",), "setups": ("FC", "RM-S1", "RM-S3"), "stores": LANGS,
               "flag": "abl_short", "digits": False},
    "q3":     {"sessions": 20, "qvs": ("q3",), "setups": ("FC", "RM-S1", "RM-S2", "RM-S3"), "stores": LANGS,
               "flag": "abl_q3", "digits": False},
    "digits": {"sessions": 20, "qvs": ("q1",), "setups": ("FC", "RM-S1", "RM-S2", "RM-S3"), "stores": ("hi",),
               "flag": "abl_digits", "digits": True},   # Hindi histories written with Devanagari digits
}
BLOCK = os.environ.get("CLM_BLOCK", "main")
if BLOCK not in BLOCKS:
    raise ValueError(f"CLM_BLOCK must be one of {list(BLOCKS)}, got {BLOCK!r}")
_B = BLOCKS[BLOCK]

RUN_DIR = ROOT / "results" / os.environ.get("CLM_RUN", "pilot")
RESULTS = RUN_DIR if BLOCK == "main" else RUN_DIR / f"abl_{BLOCK}"
TRANSLATION_CACHE = RUN_DIR / "en_translations.json"     # shared by all blocks of a run

QUERY_VARIANTS = _B["qvs"]                       # [1.2] q1/q2 paraphrases; q3 = indirect (DEVIATIONS #7)
SETUPS = _B["setups"]
STORE_LANGS = _B["stores"]                       # storage languages built for this block (queries: all LANGS)
ITEM_FLAG = _B["flag"]
DEV_DIGITS = _B["digits"]
# Override with CLM_SESSIONS (the pilot r2 run used 10).
N_SESSIONS = int(os.environ.get("CLM_SESSIONS", _B["sessions"]))

MODELS = {                                       # [5.1], [4.3], [6.2]
    "qwen3-8b": "Qwen/Qwen3-8B",
    "llama3.1-8b": "meta-llama/Llama-3.1-8B-Instruct",
    "judge": "Qwen/Qwen3-14B-AWQ",
}
TRANSLATOR = "qwen3-8b"                          # [4.3] fixed translator for both readers
EMBEDDER = "BAAI/bge-m3"                         # [4.2]
EMBEDDER_2 = "intfloat/multilingual-e5-large"    # [4.2] robustness, recall only

READER_MAX_TOKENS = 96                           # [5.1]
# Context window given to vLLM. Readers need room for 20-session Hindi histories (~22k Qwen tokens, see
# DEVIATIONS #10); 32,768 is Qwen3-8B's native context. Judge and translator prompts are short.
MAX_MODEL_LEN = 32768                            # readers; verified per prompt by tokens.py
SHORT_MODEL_LEN = 8192                           # judge, translator
MODEL_LEN = {"qwen3-8b": MAX_MODEL_LEN, "llama3.1-8b": MAX_MODEL_LEN, "judge": SHORT_MODEL_LEN}


In [ ]:
%%writefile /kaggle/working/clm/src/clm/generate.py
"""Stage 3 (GPU, one reader per process): answer every query under every setup [design_spec 4.1, 7].

Request id = item|store|query|qv|setup. Outputs are cached in generations_<model>.jsonl; reruns only
fill in missing ids, so a crashed Kaggle session can simply be restarted.
"""
import json

from . import config, haystack, prompts


def memory_text(unit_text, strategy, store, en):
    """How a stored memory is shown to the reader under each strategy. English memories need no translation."""
    if store == "en" or strategy == "S1":
        return unit_text
    if strategy == "S2":
        return en.get(unit_text, unit_text)
    return f"{unit_text} (English: {en.get(unit_text, unit_text)})"   # S3


def build_requests(items, setups=config.SETUPS):
    hist = {(h["item_id"], h["lang"]): h for h in haystack.load_jsonl(config.RESULTS / "histories.jsonl")}
    ret = {}
    if any(s.startswith("RM") for s in setups):
        ret = {(r["item_id"], r["store"], r["query"], r["qv"], r["strategy"]): r["ranked"]
               for r in haystack.load_jsonl(config.RESULTS / "retrieval.jsonl") if r["retriever"] == "bge-m3"}
    en = json.loads(config.TRANSLATION_CACHE.read_text(encoding="utf-8"))
    reqs = []
    for it in items:
        for s in config.STORE_LANGS:
            h = hist[(it["item_id"], s)]
            unit_text = dict(haystack.units(h))
            order = [u for u, _ in haystack.units(h)]
            for q in config.LANGS:
                for qv in config.QUERY_VARIANTS:
                    query = it["text"][q][qv]
                    for setup in setups:
                        rid = f"{it['item_id']}|{s}|{q}|{qv}|{setup}"
                        if setup == "FC":
                            sessions = [[{"role": t["role"], "content": t["content"]} for t in sess]
                                        for sess in h["sessions"]]
                            msgs = prompts.reader_fc(sessions, query)
                        else:
                            kind, strat = setup.split("-")
                            if kind == "RM":
                                chosen = ret[(it["item_id"], s, q, qv, strat)][:config.TOP_K]
                            else:  # oracle: gold (+ distractor) in chronological order
                                chosen = sorted([u for u in (h["gold_unit"], h["distractor_unit"]) if u],
                                                key=order.index)
                            mems = [memory_text(unit_text[u], strat, s, en) for u in chosen]
                            msgs = prompts.reader_memories(mems, query)
                        reqs.append({"id": rid, "messages": msgs})
    # Full-context prompts that share a history go first and back to back, so vLLM's prefix cache can reuse the
    # (up to ~23k-token) history across the 3 query languages x query variants. Python's sort is stable.
    reqs.sort(key=lambda r: r["id"].split("|")[4] != "FC")
    return reqs


def run(items_path, model_key, backend=None, setups=config.SETUPS):
    from .llm import Generator
    items = haystack.load_items(items_path)
    reqs = build_requests(items, setups)
    out_path = config.RESULTS / f"generations_{model_key}.jsonl"
    done = {r["id"] for r in haystack.load_jsonl(out_path)} if out_path.exists() else set()
    todo = [r for r in reqs if r["id"] not in done]
    print(f"{model_key}: {len(reqs)} requests, {len(todo)} to run")
    if not todo:
        return
    gen = Generator(model_key, backend=backend)
    chunk = 2000   # write to disk regularly so a crash loses little
    for i in range(0, len(todo), chunk):
        part = todo[i:i + chunk]
        outs = gen.chat([r["messages"] for r in part], max_tokens=config.READER_MAX_TOKENS)
        with open(out_path, "a", encoding="utf-8") as f:
            for r, o in zip(part, outs):
                f.write(json.dumps({"id": r["id"], "model": model_key, "response": o}, ensure_ascii=False) + "\n")
        print(f"  wrote {i + len(part)}/{len(todo)}")


In [ ]:
%%writefile /kaggle/working/clm/src/clm/haystack.py
"""Build conversation histories [design_spec 3.5].

Every item gets an n-session history per storage language (config.N_SESSIONS: 20 in the main run, 10 for
the Short ablation and the pilot). Filler choice, evidence position and
distractor placement depend only on the item's seed, so the three language versions of a history
differ in language and nothing else. Unanswerable controls reuse their base item's history.
"""
import json
import random

from . import config


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def load_items(path):
    """Items of the current block (config.ITEM_FLAG, set by src/merge_items.py); the main block keeps everything."""
    items = load_jsonl(path)
    return [i for i in items if i.get(config.ITEM_FLAG)] if config.ITEM_FLAG else items


ASCII_TO_DEV = str.maketrans("0123456789", "०१२३४५६७८९")


def to_devanagari_digits(text):
    """Ablation C: write every number in a Hindi history with Devanagari digits (15 -> १५)."""
    return text.translate(ASCII_TO_DEV)


def filler_path(lang):
    """EN pool lives in data/; HI/HG pools come from this run's translate stage, unless frozen copies exist in data/."""
    frozen = config.DATA / "haystacks" / f"filler_pool_{lang}.jsonl"
    return frozen if lang == "en" or frozen.exists() else config.RESULTS / f"filler_pool_{lang}.jsonl"


def load_fillers():
    pools = {}
    for lang in config.LANGS:
        p = filler_path(lang)
        if p.exists():
            pools[lang] = {s["filler_id"]: s for s in load_jsonl(p)}
    return pools


def evidence_slot(base, n):
    """Session slot of the evidence: from the position class (main study) or the fixed pilot slot (old items)."""
    if "evidence_class" in base:
        return config.EVIDENCE_SLOT[base["evidence_class"]](n)
    return base["evidence_position"]


def layout(base, n=None):
    """Deterministic layout shared by all languages: which filler goes in which slot, where the distractor is.

    Fillers are always drawn for MAX_SESSIONS and a shorter history takes the first n-1 of them, so the
    Short history is a subset of the Long one. The distractor's host is drawn from its own seeded stream.
    """
    n = n or config.N_SESSIONS
    rng = random.Random(base["filler_seed"])
    banned = set(base.get("banned_fillers", []))   # fillers mentioning the gold value [3.5]; set by merge_items.py
    pool = [i for i in range(1, config.FILLER_POOL_SIZE + 1) if f"FL{i:02d}" not in banned]
    filler_ids = [f"FL{i:02d}" for i in rng.sample(pool, config.MAX_SESSIONS - 1)][: n - 1]
    ev = evidence_slot(base, n)
    slots, it = {}, iter(filler_ids)
    for k in range(1, n + 1):
        slots[k] = "EVIDENCE" if k == ev else next(it)
    host = None
    if base.get("has_distractor"):
        host = random.Random(f"{base['filler_seed']}-dis-{n}").choice(
            [k for k in slots if slots[k] != "EVIDENCE" and abs(k - ev) >= 2])
    return slots, host


def build(base, lang, fillers, n=None):
    """Return {'sessions': [[turn,...],...], 'gold_unit', 'distractor_unit'}; user turns carry a unit_id."""
    n = n or config.N_SESSIONS
    slots, host = layout(base, n)
    t = base["text"][lang]
    sessions, gold_unit, dis_unit = [], None, None
    for k in range(1, n + 1):
        if slots[k] == "EVIDENCE":
            turns = [{"role": "user", "content": t["memory_turn"]},
                     {"role": "assistant", "content": t["assistant_reply"]},
                     {"role": "user", "content": t["followup_turn"]},
                     {"role": "assistant", "content": t["followup_reply"]}]
        else:
            turns = [dict(x) for x in fillers[lang][slots[k]]["turns"]]
            if k == host:
                turns[0]["content"] = turns[0]["content"].rstrip() + " " + t["distractor_sentence"]
        u = 0
        for turn in turns:
            if turn["role"] == "user":
                u += 1
                turn["unit_id"] = f"s{k}u{u}"
        if slots[k] == "EVIDENCE":
            gold_unit = f"s{k}u1"
        if k == host:
            dis_unit = f"s{k}u1"
        if config.DEV_DIGITS and lang == "hi":
            for turn in turns:
                turn["content"] = to_devanagari_digits(turn["content"])
        sessions.append(turns)
    return {"sessions": sessions, "gold_unit": gold_unit, "distractor_unit": dis_unit,
            "filler_ids": [slots[k] for k in sorted(slots)]}


def build_all(items, fillers, n=None):
    n = n or config.N_SESSIONS
    by_id = {i["item_id"]: i for i in items}
    out = []
    for it in items:
        base = it if it.get("is_answerable", True) else by_id[it["base_item"]]
        for lang in config.STORE_LANGS:
            h = build(base, lang, fillers, n)
            out.append({"item_id": it["item_id"], "lang": lang, "n_sessions": n, **h})
    return out


def units(history):
    """User turns as memory units, in chronological order."""
    return [(t["unit_id"], t["content"]) for s in history["sessions"] for t in s if t["role"] == "user"]


In [ ]:
%%writefile /kaggle/working/clm/src/clm/judge.py
"""Stage 4 (GPU): LLM judge over every generation [design_spec 6.2]. Cached in judgments.jsonl."""
import json
import re

from . import config, haystack, prompts

LABEL = re.compile(r"\b(CORRECT|INCORRECT|NO_ANSWER|ANSWERED)\b")


def parse(text):
    m = LABEL.search(text.upper())
    return m.group(1) if m else "UNPARSED"


def run(items_path, backend=None):
    from .llm import Generator
    items = {i["item_id"]: i for i in haystack.load_jsonl(items_path)}
    gens = []
    for p in sorted(config.RESULTS.glob("generations_*.jsonl")):
        gens += haystack.load_jsonl(p)
    out_path = config.RESULTS / "judgments.jsonl"
    done = {(r["id"], r["model"]) for r in haystack.load_jsonl(out_path)} if out_path.exists() else set()
    todo = [g for g in gens if (g["id"], g["model"]) not in done]
    print(f"judge: {len(gens)} generations, {len(todo)} to judge")
    if not todo:
        return
    msgs = []
    for g in todo:
        iid, s, q, qv, setup = g["id"].split("|")
        it = items[iid]
        query = it["text"][q][qv]
        if it.get("is_answerable", True):
            aliases = [a for sl in it["gold"]["slots"] for a in sl["aliases"]]
            msgs.append(prompts.judge(query, it["gold"]["answer_en"], aliases, g["response"]))
        else:
            msgs.append(prompts.judge_abstention(query, g["response"]))
    gen = Generator("judge", backend=backend)
    outs = gen.chat(msgs, max_tokens=8)
    with open(out_path, "a", encoding="utf-8") as f:
        for g, o in zip(todo, outs):
            f.write(json.dumps({"id": g["id"], "model": g["model"], "judge_raw": o, "judge": parse(o)},
                               ensure_ascii=False) + "\n")


In [ ]:
%%writefile /kaggle/working/clm/src/clm/llm.py
"""Chat-generation wrapper: vLLM (default), Hugging Face transformers (fallback), or a mock (local tests).

All backends use greedy decoding (temperature 0) and fp16 [design_spec 5.1]. Qwen3 thinking mode is
disabled through the chat template [5.1].
"""
import os
import re

from . import config

THINK = re.compile(r"<think>.*?</think>", re.S)


def _clean(text):
    return THINK.sub("", text).strip()


class Generator:
    def __init__(self, model_key, backend=None, max_model_len=None):
        self.model_key = model_key
        self.model_id = config.MODELS[model_key]
        self.backend = backend or os.environ.get("CLM_BACKEND", "vllm")
        self.max_model_len = max_model_len or config.MODEL_LEN.get(model_key, config.MAX_MODEL_LEN)
        if self.backend == "vllm":
            self._init_vllm()
        elif self.backend == "hf":
            self._init_hf()
        elif self.backend != "mock":
            raise ValueError(f"unknown backend {self.backend}")

    # ---------------- vLLM ----------------
    def _init_vllm(self):
        import torch
        from vllm import LLM
        self.llm = LLM(model=self.model_id, dtype="half", tensor_parallel_size=max(1, torch.cuda.device_count()),
                       max_model_len=self.max_model_len, gpu_memory_utilization=0.88,
                       enable_prefix_caching=True, seed=0)

    def _gen_vllm(self, conversations, max_tokens):
        from vllm import SamplingParams
        sp = SamplingParams(temperature=0.0, max_tokens=max_tokens)
        outs = self.llm.chat(conversations, sp, use_tqdm=True,
                             chat_template_kwargs={"enable_thinking": False})
        return [_clean(o.outputs[0].text) for o in outs]

    # ---------------- transformers ----------------
    def _init_hf(self):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.tok = AutoTokenizer.from_pretrained(self.model_id, padding_side="left")
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(self.model_id, torch_dtype=torch.float16, device_map="auto")
        self.model.eval()

    def _gen_hf(self, conversations, max_tokens, batch_size=4):
        import torch
        out = []
        for i in range(0, len(conversations), batch_size):
            batch = conversations[i:i + batch_size]
            texts = [self.tok.apply_chat_template(c, tokenize=False, add_generation_prompt=True,
                                                  enable_thinking=False) for c in batch]
            enc = self.tok(texts, return_tensors="pt", padding=True).to(self.model.device)
            with torch.no_grad():
                gen = self.model.generate(**enc, max_new_tokens=max_tokens, do_sample=False,
                                          pad_token_id=self.tok.pad_token_id)
            for row in gen[:, enc["input_ids"].shape[1]:]:
                out.append(_clean(self.tok.decode(row, skip_special_tokens=True)))
        return out

    # ---------------- mock (CPU tests) ----------------
    def _gen_mock(self, conversations, max_tokens):
        """Echoes the first memory line for readers, copies the text for translators, says CORRECT for judges."""
        out = []
        for c in conversations:
            msg = c[-1]["content"]
            if "Reply with exactly one word" in msg:
                out.append("NO_ANSWER" if "ANSWERED -" in msg else "CORRECT")
            elif msg.startswith(("Translate", "Rewrite", "Write the following")):
                out.append(msg.split("Text:", 1)[-1].strip())
            else:
                m = re.search(r"\n1\. (.+)", msg) or re.search(r"User: (.+)", msg)
                out.append(m.group(1)[:200] if m else "I don't know.")
        return out

    def chat(self, conversations, max_tokens):
        if not conversations:
            return []
        return getattr(self, f"_gen_{self.backend}")(conversations, max_tokens)


In [ ]:
%%writefile /kaggle/working/clm/src/clm/prompts.py
"""Prompt templates, verbatim from design_spec.md Appendix A (plus the filler-translation prompts [3.5])."""

READER_SYSTEM = (
    "You are a personal assistant with memory of your past conversations with this user.\n"
    "Answer the user's question using only what the user told you in past conversations.\n"
    "Answer in one short sentence, in the same language as the question.\n"
    "If the information is not in the past conversations, say that you don't know."
)


def reader_fc(sessions, query):
    blocks = []
    for k, sess in enumerate(sessions, 1):
        lines = [f"[Session {k}]"] + [f"{'User' if t['role'] == 'user' else 'Assistant'}: {t['content']}" for t in sess]
        blocks.append("\n".join(lines))
    user = "Past conversations:\n" + "\n\n".join(blocks) + f"\n\nCurrent question: {query}"
    return [{"role": "system", "content": READER_SYSTEM}, {"role": "user", "content": user}]


def reader_memories(memories, query):
    lines = "\n".join(f"{i}. {m}" for i, m in enumerate(memories, 1))
    user = ("Relevant memories from past conversations (may be incomplete or irrelevant):\n"
            f"{lines}\n\nCurrent question: {query}")
    return [{"role": "system", "content": READER_SYSTEM}, {"role": "user", "content": user}]


def judge(query, gold_en, aliases, response):
    user = (
        "You are grading an assistant's answer about a user's personal information.\n"
        f"Question: {query}\n"
        f"Gold answer: {gold_en}   Accepted forms: {'; '.join(aliases) if aliases else '(none)'}\n"
        f"Assistant answer: {response}\n"
        "The answer may be in English, Hindi (Devanagari) or Hinglish (romanized). Judge meaning, not language.\n"
        "Reply with exactly one word:\n"
        "CORRECT - states the gold information without contradiction\n"
        "INCORRECT - wrong, incomplete, lists several candidates, or contradicts the gold\n"
        "NO_ANSWER - says it does not know / has no information"
    )
    return [{"role": "user", "content": user}]


def judge_abstention(query, response):
    """Unanswerable controls: the only correct behaviour is to say it does not know."""
    user = (
        "You are grading an assistant's answer about a user's personal information.\n"
        f"Question: {query}\n"
        "Gold answer: the user never told the assistant this, so the assistant should say it does not know.\n"
        f"Assistant answer: {response}\n"
        "The answer may be in English, Hindi (Devanagari) or Hinglish (romanized). Judge meaning, not language.\n"
        "Reply with exactly one word:\n"
        "ANSWERED - gives some specific answer to the question\n"
        "NO_ANSWER - says it does not know / has no information"
    )
    return [{"role": "user", "content": user}]


def to_english(text):
    return [{"role": "user", "content":
             "Translate the following text to English. It may be Hindi, Hinglish (romanized Hindi) or English.\n"
             "Keep names, numbers and dates exactly. Output only the translation.\n"
             f"Text: {text}"}]


def to_hindi(text):
    return [{"role": "user", "content":
             "Translate the following text into natural, conversational Hindi written in Devanagari script.\n"
             "Keep brand and product names in Latin script, write person and place names in Devanagari, "
             "and keep digits as 0-9. Output only the translation.\n"
             f"Text: {text}"}]


HINGLISH_INSTRUCTION = (
    "Rewrite the following text in Hinglish: the casual Hindi-English mix used in Indian chat, "
    "with Hindi grammar, common English words mixed in, written ONLY in Roman (Latin) script. "
    "Never use Devanagari. Keep names, numbers and brand names unchanged. Output only the rewritten text.\n"
)

# Few-shot pairs (EN -> HG) from batch01 items F001 (team-verified), F002, P009. Showing the target
# style fixes the first pilot's failure, where Qwen3-8B answered in Devanagari for 77/160 filler turns.
HINGLISH_SHOTS = [
    ("I've been listening to audiobooks during my daily commute, which takes 45 minutes each way. "
     "Can you recommend some good fiction audiobooks?",
     "Main aajkal daily office aate-jaate audiobooks sunti hoon, ek side ka travel 45 minute ka hota hai. "
     "Kya aap kuch acchi fiction audiobooks suggest kar sakte hain?"),
    ("Congratulations on the new role! A simple system helps: scan receipts the same day, keep one folder "
     "per month, and log every expense in a spreadsheet with date, amount and purpose.",
     "Nayi job ki congratulations! Ek simple system follow kijiye: receipts usi din scan karein, har mahine ka "
     "alag folder rakhein, aur har expense ko date, amount aur purpose ke saath ek spreadsheet mein likhein."),
    ("You could use a little less honey and add cinnamon or cardamom for sweetness without sugar.",
     "Honey thoda kam karke sweetness ke liye cinnamon ya elaichi daal sakti hain."),
]


def to_hinglish(text):
    msgs = []
    for en, hg in HINGLISH_SHOTS:
        msgs += [{"role": "user", "content": HINGLISH_INSTRUCTION + f"Text: {en}"},
                 {"role": "assistant", "content": hg}]
    return msgs + [{"role": "user", "content": HINGLISH_INSTRUCTION + f"Text: {text}"}]


def romanize_hindi(hindi_text):
    """Fallback: turn an existing Devanagari Hindi translation into Roman-script Hinglish."""
    return [{"role": "user", "content":
             "Write the following Hindi text in Roman (Latin) script the way Indians type Hindi in chat "
             "(e.g. 'Main kal office gaya tha'). Write common English loanwords in English spelling. "
             "Do not output any Devanagari characters. Output only the romanized text.\n"
             f"Text: {hindi_text}"}]


In [ ]:
%%writefile /kaggle/working/clm/src/clm/report.py
"""Stage 5 (CPU): score every generation and write the analysis tables [design_spec 1, 6].

Outputs in results/<run>/:
  scored.csv                    one row per (model, item, store, query, qv, setup)
  table_accuracy_3x3.csv        accuracy per model x setup x store x query
  table_gap.csv                 H1: monolingual vs cross-lingual gap (paired by item, bootstrap CI)
  table_mirror.csv              H2: mirrored directions, exact McNemar
  table_recall.csv              recall@k per retriever x strategy x store x query
  table_stages.csv              where failures happen: retrieval miss / reading error / oracle error
  table_noise_floor.csv         paraphrase disagreement on monolingual cells
  table_abstention.csv          hallucination rate on unanswerable controls
  table_response_language.csv   answer language by query language
  judge_validation_sample.csv   200 stratified responses for human labelling
  summary.md                    human-readable digest
"""
import csv
import json
import random
from collections import defaultdict

from . import config, haystack
from .textutil import any_match, is_abstention, response_language, slot_match

CROSS = [(s, q) for s in config.LANGS for q in config.LANGS if s != q]
DIAG = [(l, l) for l in config.LANGS]


def mcnemar_exact(b, c):
    """Two-sided exact McNemar p-value from discordant counts b, c."""
    from math import comb
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    p = sum(comb(n, i) for i in range(k + 1)) / 2 ** n
    return min(1.0, 2 * p)


def score_rows(items):
    judg = {(j["id"], j["model"]): j["judge"] for j in haystack.load_jsonl(config.RESULTS / "judgments.jsonl")} \
        if (config.RESULTS / "judgments.jsonl").exists() else {}
    ret = {(r["item_id"], r["store"], r["query"], r["qv"], r["strategy"]): r
           for r in haystack.load_jsonl(config.RESULTS / "retrieval.jsonl") if r["retriever"] == "bge-m3"}
    rows = []
    for p in sorted(config.RESULTS.glob("generations_*.jsonl")):
        for g in haystack.load_jsonl(p):
            iid, s, q, qv, setup = g["id"].split("|")
            it = items[iid]
            ans = it.get("is_answerable", True)
            resp = g["response"]
            j = judg.get((g["id"], g["model"]))
            if ans:
                slots = it["gold"]["slots"]
                correct = (j == "CORRECT") if j else slot_match(resp, slots)
                sm = slot_match(resp, slots)
            else:
                correct = (j == "NO_ANSWER") if j else is_abstention(resp)
                sm = None
            gold_rank = ret[(iid, s, q, qv, setup.split("-")[1])]["gold_rank"] if setup.startswith("RM") and ans else None
            rows.append({
                "model": g["model"], "item_id": iid, "memory_type": it["memory_type"], "answerable": ans,
                "store": s, "query": q, "qv": qv, "setup": setup, "judge": j or "", "correct": int(bool(correct)),
                "slot_match": "" if sm is None else int(sm),
                "distractor_capture": int(any_match(resp, it.get("distractor", {}).get("value_aliases"))) if ans else "",
                "near_miss_capture": int(any_match(resp, it.get("near_miss_aliases"))) if ans else "",
                "abstained": int(is_abstention(resp)), "resp_lang": response_language(resp),
                "gold_rank": gold_rank if gold_rank is not None else "", "response": resp})
    return rows


def write_csv(name, rows):
    if not rows:
        return
    # utf-8-sig adds the BOM that Excel needs to display Hindi correctly (plain utf-8 shows garbage on Mac Excel).
    with open(config.RESULTS / name, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)


def mean(xs):
    xs = list(xs)
    return sum(xs) / len(xs) if xs else float("nan")


def item_level(rows, model, setup, cells):
    """Per item: mean correctness over the given cells (both query variants)."""
    acc = defaultdict(list)
    for r in rows:
        if r["model"] == model and r["setup"] == setup and r["answerable"] and (r["store"], r["query"]) in cells:
            acc[r["item_id"]].append(r["correct"])
    return {k: mean(v) for k, v in acc.items()}


def bootstrap_ci(diffs, n=10000, seed=0):
    rng = random.Random(seed)
    if not diffs:
        return float("nan"), float("nan")
    means = sorted(mean(rng.choices(diffs, k=len(diffs))) for _ in range(n))
    return means[int(0.025 * n)], means[int(0.975 * n)]


def run(items_path):
    items = {i["item_id"]: i for i in haystack.load_items(items_path)}
    rows = score_rows(items)
    write_csv("scored.csv", rows)
    models = sorted({r["model"] for r in rows})
    setups = [s for s in config.SETUPS if any(r["setup"] == s for r in rows)]
    ans = [r for r in rows if r["answerable"]]
    md = [f"# Results summary ({config.RESULTS.name})", "",
          f"Generations scored: {len(rows)} | answerable: {len(ans)} | models: {', '.join(models)}", ""]

    # --- 3x3 accuracy ---
    t3 = []
    for m in models:
        for st in setups:
            for s in config.LANGS:
                for q in config.LANGS:
                    xs = [r["correct"] for r in ans if (r["model"], r["setup"], r["store"], r["query"]) == (m, st, s, q)]
                    t3.append({"model": m, "setup": st, "store": s, "query": q, "n": len(xs), "accuracy": round(mean(xs), 4)})
    write_csv("table_accuracy_3x3.csv", t3)
    md += ["## Accuracy (rows = storage language, columns = query language)", ""]
    for m in models:
        for st in setups:
            md += [f"**{m} / {st}**", "", "| store \\ query | EN | HI | HG |", "|---|---|---|---|"]
            for s in config.LANGS:
                cells = [next(x for x in t3 if (x["model"], x["setup"], x["store"], x["query"]) == (m, st, s, q))["accuracy"]
                         for q in config.LANGS]
                md.append(f"| {s.upper()} | " + " | ".join(f"{c:.0%}" for c in cells) + " |")
            md.append("")

    # --- H1 gap (paired by item) ---
    gap = []
    for m in models:
        for st in setups:
            d, c = item_level(ans, m, st, DIAG), item_level(ans, m, st, CROSS)
            diffs = [d[k] - c[k] for k in d if k in c]
            lo, hi = bootstrap_ci(diffs)
            gap.append({"model": m, "setup": st, "n_items": len(diffs), "mono_acc": round(mean(d.values()), 4),
                        "cross_acc": round(mean(c.values()), 4), "gap": round(mean(diffs), 4),
                        "ci95_low": round(lo, 4), "ci95_high": round(hi, 4)})
    write_csv("table_gap.csv", gap)
    md += ["## H1: monolingual minus cross-lingual accuracy (item-paired, 95% bootstrap CI)", "",
           "| model | setup | mono | cross | gap | 95% CI |", "|---|---|---|---|---|---|"]
    md += [f"| {g['model']} | {g['setup']} | {g['mono_acc']:.0%} | {g['cross_acc']:.0%} | {g['gap']*100:+.1f}pp | "
           f"[{g['ci95_low']*100:+.1f}, {g['ci95_high']*100:+.1f}] |" for g in gap] + [""]

    # --- H2 mirrored directions ---
    mir = []
    key = {(r["model"], r["setup"], r["item_id"], r["store"], r["query"], r["qv"]): r["correct"] for r in ans}
    for m in models:
        for st in setups:
            for a, b in (("hi", "en"), ("hg", "en"), ("hg", "hi")):
                # Unit = item: the two wordings of a question are not independent, so average them per item
                # and count items where one direction scores higher (exact sign / McNemar-type test).
                a_better = b_better = n = 0
                diffs = []
                for iid in items:
                    xs = [key.get((m, st, iid, a, b, qv)) for qv in config.QUERY_VARIANTS]
                    ys = [key.get((m, st, iid, b, a, qv)) for qv in config.QUERY_VARIANTS]
                    if None in xs or None in ys:
                        continue
                    n += 1
                    d = mean(xs) - mean(ys)
                    diffs.append(d)
                    a_better += d > 0
                    b_better += d < 0
                lo, hi = bootstrap_ci(diffs)
                mir.append({"model": m, "setup": st, "direction_A": f"{a}->{b}", "direction_B": f"{b}->{a}",
                            "items": n, "A_better_items": a_better, "B_better_items": b_better,
                            "diff_pp": round(100 * mean(diffs), 1) if diffs else "",
                            "ci95_low_pp": round(100 * lo, 1), "ci95_high_pp": round(100 * hi, 1),
                            "p_exact": round(mcnemar_exact(a_better, b_better), 4)})
    # Holm correction across ALL mirror tests (models x setups x pairs): the honest family size.
    order = sorted(range(len(mir)), key=lambda i: mir[i]["p_exact"])
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (len(mir) - rank) * mir[i]["p_exact"]))
        mir[i]["p_holm_all"] = round(running, 4)
    write_csv("table_mirror.csv", mir)
    md += [f"## H2: mirrored directions (item-level exact test; Holm over all {len(mir)} tests)", "",
           "| model | setup | A vs B | items A better / B better | diff (pp) [95% CI] | p | p (Holm) |",
           "|---|---|---|---|---|---|---|"]
    md += [f"| {r['model']} | {r['setup']} | {r['direction_A']} vs {r['direction_B']} | {r['A_better_items']} / "
           f"{r['B_better_items']} | {r['diff_pp']} [{r['ci95_low_pp']}, {r['ci95_high_pp']}] | {r['p_exact']} | "
           f"{r['p_holm_all']} |" for r in sorted(mir, key=lambda r: r["p_exact"])[:6]]
    md += ["(6 smallest p-values shown; full table in table_mirror.csv)", ""]

    # --- Recall@k ---
    rec = []
    rr = [r for r in haystack.load_jsonl(config.RESULTS / "retrieval.jsonl") if r["gold_rank"]]
    for name in sorted({r["retriever"] for r in rr}):
        for strat in ("S1", "S2", "S3"):
            for s in config.LANGS:
                for q in config.LANGS:
                    ranks = [r["gold_rank"] for r in rr if (r["retriever"], r["strategy"], r["store"], r["query"]) == (name, strat, s, q)]
                    rec.append({"retriever": name, "strategy": strat, "store": s, "query": q, "n": len(ranks),
                                **{f"R@{k}": round(mean(int(x <= k) for x in ranks), 4) for k in config.RECALL_KS}})
    write_csv("table_recall.csv", rec)
    md += ["## Recall@5 (bge-m3), mono vs cross", "", "| strategy | mono R@5 | cross R@5 |", "|---|---|---|"]
    for strat in ("S1", "S2", "S3"):
        sub = [x for x in rec if x["retriever"] == "bge-m3" and x["strategy"] == strat]
        md.append(f"| {strat} | {mean(x['R@5'] for x in sub if x['store'] == x['query']):.0%} | "
                  f"{mean(x['R@5'] for x in sub if x['store'] != x['query']):.0%} |")
    md.append("")

    # --- Stage decomposition (RM vs OR) ---
    stg = []
    for m in models:
        for strat in ("S1", "S2", "S3"):
            for s in config.LANGS:
                for q in config.LANGS:
                    rm = [r for r in ans if (r["model"], r["setup"], r["store"], r["query"]) == (m, f"RM-{strat}", s, q)]
                    orc = [r for r in ans if (r["model"], r["setup"], r["store"], r["query"]) == (m, f"OR-{strat}", s, q)]
                    if not rm:
                        continue
                    hit = [r for r in rm if r["gold_rank"] != "" and int(r["gold_rank"]) <= config.TOP_K]
                    stg.append({"model": m, "strategy": strat, "store": s, "query": q, "n": len(rm),
                                "retrieval_miss": round(1 - len(hit) / len(rm), 4),
                                "reading_error_given_hit": round(1 - mean(r["correct"] for r in hit), 4) if hit else "",
                                "oracle_error": round(1 - mean(r["correct"] for r in orc), 4) if orc else "",
                                "end_to_end_error": round(1 - mean(r["correct"] for r in rm), 4)})
    write_csv("table_stages.csv", stg)

    # --- Noise floor (paraphrase disagreement on monolingual cells) ---
    nf = []
    for m in models:
        for st in setups:
            dis = [int(key[(m, st, iid, l, l, "q1")] != key[(m, st, iid, l, l, "q2")])
                   for iid in items for l in config.LANGS
                   if (m, st, iid, l, l, "q1") in key and (m, st, iid, l, l, "q2") in key]
            nf.append({"model": m, "setup": st, "pairs": len(dis), "paraphrase_disagreement": round(mean(dis), 4)})
    write_csv("table_noise_floor.csv", nf)

    # --- Abstention controls ---
    ab = [r for r in rows if not r["answerable"]]
    abt = []
    for m in models:
        for st in setups:
            for s, q in DIAG + CROSS:
                xs = [1 - r["correct"] for r in ab if (r["model"], r["setup"], r["store"], r["query"]) == (m, st, s, q)]
                if xs:
                    abt.append({"model": m, "setup": st, "store": s, "query": q, "n": len(xs), "hallucination_rate": round(mean(xs), 4)})
    write_csv("table_abstention.csv", abt)

    # --- Response language ---
    rl = []
    for m in models:
        for q in config.LANGS:
            sub = [r for r in rows if r["model"] == m and r["query"] == q]
            rl.append({"model": m, "query_lang": q, "n": len(sub),
                       **{f"answered_{l}": round(mean(int(r["resp_lang"] == l) for r in sub), 4) for l in ("en", "hi", "hg", "other")}})
    write_csv("table_response_language.csv", rl)

    # --- Judge sanity + human-validation sample ---
    judged = [r for r in ans if r["judge"]]
    if judged:
        agree = mean(int((r["judge"] == "CORRECT") == bool(r["slot_match"])) for r in judged)
        md += [f"Judge vs slot-match agreement (answerable): {agree:.1%} over {len(judged)} responses "
               "(disagreements are what the human validation must resolve).", ""]
    rng = random.Random(7)
    strata = defaultdict(list)
    for r in rows:
        strata[(r["model"], r["store"], r["query"])].append(r)
    sample = []
    for k in sorted(strata):
        sample += rng.sample(strata[k], min(len(strata[k]), -(-200 // len(strata))))
    write_csv("judge_validation_sample.csv",
              [{"model": r["model"], "item_id": r["item_id"], "store": r["store"], "query": r["query"], "qv": r["qv"],
                "setup": r["setup"], "question": items[r["item_id"]]["text"][r["query"]][r["qv"]],
                "gold": items[r["item_id"]].get("gold", {}).get("answer_en", ""), "response": r["response"],
                "judge": r["judge"], "human_label (CORRECT/INCORRECT/NO_ANSWER)": ""} for r in sample[:200]])
    (config.RESULTS / "summary.md").write_text("\n".join(md), encoding="utf-8")
    print("\n".join(md[:60]))


In [ ]:
%%writefile /kaggle/working/clm/src/clm/retrieve.py
"""Stage 2 (GPU or CPU): rank memory units for every query [design_spec 4.1-4.2].

Strategies (what is indexed / what the query is):
  S1 original : original units            / original query
  S2 pivot    : English translations       / English translation of the query
  S3 dual     : original + English (merged by unit, max score) / original query
Retrievers: bge-m3 (feeds the readers), multilingual-e5 and BM25 (recall only).
"""
import json
import math
import re
from collections import Counter

import numpy as np

from . import config, haystack

TOKEN = re.compile(r"[^\s.,!?;:'\"()\[\]\-–—/।]+")


class DenseEmbedder:
    def __init__(self, model_id, backend="st"):
        self.model_id, self.backend = model_id, backend
        self.e5 = "e5" in model_id
        if backend == "st":
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(model_id, device="cuda" if _has_cuda() else "cpu")
        self.cache = {}

    def _mock(self, text):
        v = np.zeros(512)
        t = text.lower()
        for i in range(len(t) - 2):
            v[hash(t[i:i + 3]) % 512] += 1
        return v

    def embed(self, texts, is_query):
        prefix = ("query: " if is_query else "passage: ") if self.e5 else ""
        need = [t for t in dict.fromkeys(texts) if (prefix + t) not in self.cache]
        if need:
            if self.backend == "st":
                vecs = self.model.encode([prefix + t for t in need], batch_size=32, normalize_embeddings=True,
                                         show_progress_bar=False)
            else:
                vecs = [self._mock(t) for t in need]
                vecs = [v / (np.linalg.norm(v) or 1) for v in vecs]
            for t, v in zip(need, vecs):
                self.cache[prefix + t] = np.asarray(v, dtype=np.float32)
        return np.stack([self.cache[prefix + t] for t in texts])

    def scores(self, query, docs):
        return self.embed(docs, False) @ self.embed([query], True)[0]


class BM25:
    k1, b = 1.5, 0.75

    def scores(self, query, docs):
        toks = [TOKEN.findall(d.lower()) for d in docs]
        avg = sum(map(len, toks)) / max(1, len(toks))
        df = Counter(w for t in toks for w in set(t))
        n = len(docs)
        out = []
        for t in toks:
            tf = Counter(t)
            s = 0.0
            for w in TOKEN.findall(query.lower()):
                if w in tf:
                    idf = math.log(1 + (n - df[w] + 0.5) / (df[w] + 0.5))
                    s += idf * tf[w] * (self.k1 + 1) / (tf[w] + self.k1 * (1 - self.b + self.b * len(t) / avg))
            out.append(s)
        return np.array(out)


def _has_cuda():
    try:
        import torch
        return torch.cuda.is_available()
    except ImportError:
        return False


def rank(retriever, query, units, strategy, en):
    """Return unit ids ranked best-first."""
    ids = [u for u, _ in units]
    if strategy == "S1":
        sc = retriever.scores(query, [x for _, x in units])
    elif strategy == "S2":
        sc = retriever.scores(en.get(query, query), [en.get(x, x) for _, x in units])
    else:  # S3 dual index: score both representations, keep the max per unit
        a = retriever.scores(query, [x for _, x in units])
        b = retriever.scores(query, [en.get(x, x) for _, x in units])
        sc = np.maximum(a, b)
    order = np.argsort(-np.asarray(sc), kind="stable")
    return [ids[i] for i in order]


def run(items_path, backend=None):
    backend = backend or ("mock" if __import__("os").environ.get("CLM_BACKEND") == "mock" else "st")
    items = haystack.load_items(items_path)
    hist = {(h["item_id"], h["lang"]): h for h in haystack.load_jsonl(config.RESULTS / "histories.jsonl")}
    en = json.loads(config.TRANSLATION_CACHE.read_text(encoding="utf-8"))
    retrievers = {"bge-m3": DenseEmbedder(config.EMBEDDER, backend),
                  "me5": DenseEmbedder(config.EMBEDDER_2, backend),
                  "bm25": BM25()}
    rows = []
    for it in items:
        for s in config.STORE_LANGS:
            h = hist[(it["item_id"], s)]
            units = haystack.units(h)
            for q in config.LANGS:
                for qv in config.QUERY_VARIANTS:
                    query = it["text"][q][qv]
                    for strat in ("S1", "S2", "S3"):
                        for name, r in retrievers.items():
                            ranked = rank(r, query, units, strat, en)
                            gold = h["gold_unit"] if it.get("is_answerable", True) else None
                            rows.append({"item_id": it["item_id"], "store": s, "query": q, "qv": qv,
                                         "strategy": strat, "retriever": name, "ranked": ranked[:10],
                                         "gold_rank": ranked.index(gold) + 1 if gold else None,
                                         "distractor_rank": (ranked.index(h["distractor_unit"]) + 1
                                                             if h["distractor_unit"] else None)})
    with open(config.RESULTS / "retrieval.jsonl", "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"retrieval stage: {len(rows)} rankings")


In [ ]:
%%writefile /kaggle/working/clm/src/clm/stages.py
"""Command-line entry point. Each GPU stage runs in its own process so GPU memory is fully released.

  python -m clm.stages translate --items data/items/items_batch01_pilot.jsonl
  python -m clm.stages retrieve  --items ...
  python -m clm.stages tokens    --items ...
  python -m clm.stages generate  --items ... --model qwen3-8b
  python -m clm.stages generate  --items ... --model llama3.1-8b
  python -m clm.stages judge     --items ...
  python -m clm.stages report    --items ...
Set CLM_BACKEND=mock to test the whole pipeline on a CPU; CLM_RUN names the results folder.
"""
import argparse

from . import config


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("stage", choices=["translate", "retrieve", "tokens", "generate", "judge", "report"])
    ap.add_argument("--items", required=True)
    ap.add_argument("--model", default=None)
    ap.add_argument("--backend", default=None, help="vllm | hf | mock (default: env CLM_BACKEND or vllm)")
    ap.add_argument("--setups", default=",".join(config.SETUPS))
    a = ap.parse_args()
    config.RESULTS.mkdir(parents=True, exist_ok=True)
    if a.stage == "translate":
        from . import translate
        translate.run(a.items, a.backend)
    elif a.stage == "retrieve":
        from . import retrieve
        retrieve.run(a.items)
    elif a.stage == "tokens":
        from . import tokens
        tokens.run(a.items)
    elif a.stage == "generate":
        from . import generate
        generate.run(a.items, a.model, a.backend, tuple(a.setups.split(",")))
    elif a.stage == "judge":
        from . import judge
        judge.run(a.items, a.backend)
    elif a.stage == "report":
        from . import report
        report.run(a.items)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/clm/src/clm/textutil.py
"""Normalization and deterministic checks used in scoring [design_spec 6.1]."""
import re
import unicodedata

DEV_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")
NUKTA = "़"
DEVANAGARI = re.compile(r"[ऀ-ॿ]")
LATIN = re.compile(r"[A-Za-z]")

# Common Hindi function words in Roman script: their presence marks Hinglish rather than English.
HG_MARKERS = {"hai", "hain", "tha", "thi", "nahi", "nahin", "kya", "mera", "meri", "mere", "mujhe", "aap", "aapka",
              "aapki", "ka", "ki", "ke", "ko", "se", "mein", "aur", "ya", "toh", "hoon", "kar", "karti", "karta",
              "liya", "gaya", "gayi", "pe", "par", "wala", "wali", "kab", "kahan", "kitna", "kitne", "pata"}

ABSTAIN = re.compile(
    r"(don't know|do not know|not sure|no information|not (been )?mentioned|didn't (mention|tell|say)|"
    r"haven't (mentioned|told)|not in (our|the|my) (past )?conversations?|no record|cannot (find|determine)|"
    r"can't (find|determine)|pata nahi|nahi pata|nahin pata|jaankari nahi|maloom nahi|"
    r"नहीं पता|पता नहीं|जानकारी नहीं|उल्लेख नहीं|मालूम नहीं|नहीं बताया)", re.I)


def norm(s):
    s = unicodedata.normalize("NFC", s or "").translate(DEV_DIGITS).replace(NUKTA, "")
    s = s.replace("’", "'").replace("–", "-").replace("—", "-")
    return s.casefold()


def contains_exact(text, alias):
    """Word-bounded match for Latin aliases, substring match for Devanagari (after normalization).

    Used to build data (banned fillers, QA), where matching must not change once items are frozen.
    """
    t, a = norm(text), norm(alias)
    if not a:
        return False
    if re.search(r"[a-z0-9]", a):
        return re.search(rf"(?<![a-z0-9]){re.escape(a)}(?![a-z0-9])", t) is not None
    return a in t


# ---------------- lenient matching for scoring (ENDSEM_PLAN §3 change 4) ----------------
# The pilot's slot checker missed 237 answers the judge accepted (188 in Hindi): spelling variants (नाइके/नाइकी,
# लुना/लूना), gender/number endings (पीला/पीली/पीले), Hinglish month names ("10 janvar") and number words before
# units ("do hafta"). canon() maps all of these onto one form; contains() tries the exact match first.
_DEV_FOLD = str.maketrans({"ी": "ि", "ू": "ु", "ई": "इ", "ऊ": "उ", "ँ": "ं", "ॉ": "ा", "ऑ": "आ", "ॅ": "े", "ऍ": "ए"})
_DEV_FINAL = ("ं", "ि", "े", "ो")               # word-final endings stripped; "ा" kept: नाइका (Nykaa) must not match नाइकी (Nike)
_TOKEN = re.compile(r"[ऀ-ॿ]+|[a-z0-9]+(?:[:.%][0-9]+%?|%)?")
_MONTHS_HG = {
    "january": ("janvari", "janvary", "janvar", "janwari", "jan"), "february": ("farvari", "farwari", "faravari", "feb"),
    "march": ("maarch",), "april": ("aprail", "apral", "apr"), "may": ("mai",), "june": ("joon", "jun"),
    "july": ("julai", "jul"), "august": ("agast", "aug"), "september": ("sitambar", "sitmbar", "sep", "sept"),
    "october": ("aktubar", "aktoobar", "oct"), "november": ("navambar", "nov"), "december": ("disambar", "dec")}
_MONTH_OF = {v: k for k, vs in _MONTHS_HG.items() for v in vs}
_UNIT_HG = {"hafta": "hafte", "hafton": "hafte", "hafe": "hafte", "hafee": "hafte", "haftay": "hafte", "ghanta": "ghante", "ghanton": "ghante", "mahina": "mahine",
            "mahino": "mahine", "mahinon": "mahine", "minat": "minute", "mint": "minute", "minutes": "minute"}
_UNITS = {"hafte", "ghante", "mahine", "din", "saal", "minute", "weeks", "week", "hours", "hour", "months", "month",
          "days", "day", "years", "year", "skeins", "bikes", "shirts", "bass", "comedians",
          "हफ्त", "घंट", "महिन", "दिन", "साल", "मिनट"}
_NUM_WORDS = {w: str(i) for i, ws in enumerate((
    (), ("ek", "one", "एक"), ("do", "two", "दो"), ("teen", "three", "तिन"), ("char", "chaar", "four", "चार"),
    ("paanch", "panch", "five", "पांच"), ("chhe", "chhah", "che", "six", "छह", "छः", "छ"), ("saat", "seven", "सात"),
    ("aath", "eight", "आठ"), ("nau", "nine", "नौ"), ("das", "ten", "दस"), ("gyarah", "eleven", "ग्यारह"),
    ("barah", "baarah", "twelve", "बारह"))) for w in ws}


def _dev_stem(tok):
    tok = tok.translate(_DEV_FOLD)
    for _ in range(2):                           # e.g. जूतों -> जुतो -> जुत
        if len(tok) > 2 and tok.endswith(_DEV_FINAL):
            tok = tok[:-1]
    return tok


def canon(s):
    """Space-joined canonical tokens: Devanagari folded and de-inflected, Hinglish months and units unified,
    number words before a unit turned into digits."""
    toks = _TOKEN.findall(norm(s))
    out = []
    for t in toks:
        if re.match(r"[ऀ-ॿ]", t):
            out.append(_dev_stem(t))
        else:
            t = _MONTH_OF.get(t, t)
            out.append(_UNIT_HG.get(t, t))
    for i in range(len(out) - 1):
        if out[i + 1] in _UNITS and out[i] in _NUM_WORDS:
            out[i] = _NUM_WORDS[out[i]]
    return " ".join(out)


def _drop_aa(tok):
    return tok[:-1] if len(tok) > 2 and tok.endswith("ा") else tok


def contains(text, alias):
    """Scoring match: exact (contains_exact) or, failing that, a token-window match of canonical forms.

    A final "ा" is also ignored, but only for alias words that are NOT the last word: agreement changes
    the modifier before a noun (पीला/पीली ड्रेस), while names usually stand alone or come last (नाइका ≠ नाइकी).
    """
    if contains_exact(text, alias):
        return True
    a, t = canon(alias).split(), canon(text).split()
    if not a:
        return False
    for i in range(len(t) - len(a) + 1):
        if all(x == y or (k < len(a) - 1 and _drop_aa(x) == _drop_aa(y))
               for k, (x, y) in enumerate(zip(a, t[i:i + len(a)]))):
            return True
    return False


def slot_match(response, slots):
    """All gold slots present (any alias, any script)."""
    return bool(slots) and all(any(contains(response, a) for a in s["aliases"]) for s in slots)


def any_match(response, aliases):
    return any(contains(response, a) for a in aliases or [])


def is_abstention(response):
    return bool(ABSTAIN.search(response or ""))


def response_language(text):
    """'hi' (Devanagari), 'hg' (romanized Hindi), 'en', or 'other'."""
    dev, lat = len(DEVANAGARI.findall(text or "")), len(LATIN.findall(text or ""))
    if dev + lat == 0:
        return "other"
    if dev / (dev + lat) > 0.3:
        return "hi"
    words = re.findall(r"[a-z]+", (text or "").lower())
    hg = sum(w in HG_MARKERS for w in words)
    return "hg" if words and (hg >= 2 or hg / len(words) > 0.15) else "en"


def devanagari_share(text):
    dev, lat = len(DEVANAGARI.findall(text or "")), len(LATIN.findall(text or ""))
    return dev / (dev + lat) if dev + lat else 0.0


In [ ]:
%%writefile /kaggle/working/clm/src/clm/tokens.py
"""Token counts of full-context prompts per storage language and tokenizer [design_spec 3.5 CHECK IN PHASE 4].

Hindi/Hinglish need more tokens than English for the same content; we report this and confirm every
prompt fits MAX_MODEL_LEN. Writes table_tokens.csv.
"""
import csv

from . import config, generate, haystack


def run(items_path):
    from transformers import AutoTokenizer
    items = haystack.load_items(items_path)
    reqs = [r for r in generate.build_requests(items, setups=("FC",)) if r["id"].split("|")[3] == config.QUERY_VARIANTS[0]]
    rows = []
    for key in ("qwen3-8b", "llama3.1-8b"):
        tok = AutoTokenizer.from_pretrained(config.MODELS[key])
        by_lang = {}
        for r in reqs:
            # Render to text, then tokenize: in transformers 5.x apply_chat_template(tokenize=True) returns a
            # dict, and len() of it was 2 in the first pilot. The template already holds special tokens.
            text = tok.apply_chat_template(r["messages"], tokenize=False, add_generation_prompt=True)
            n = len(tok(text, add_special_tokens=False)["input_ids"])
            by_lang.setdefault(r["id"].split("|")[1], []).append(n)
        for lang, ns in sorted(by_lang.items()):
            rows.append({"tokenizer": key, "store_lang": lang, "prompts": len(ns), "mean_tokens": round(sum(ns) / len(ns)),
                         "max_tokens": max(ns), "fits_context": max(ns) + config.READER_MAX_TOKENS <= config.MAX_MODEL_LEN})
    with open(config.RESULTS / "table_tokens.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    for r in rows:
        print(r)
    if not all(r["fits_context"] for r in rows):
        raise RuntimeError("Some full-context prompts exceed MAX_MODEL_LEN - see table_tokens.csv")


In [ ]:
%%writefile /kaggle/working/clm/src/clm/translate.py
"""Stage 1 (GPU, fixed translator Qwen3-8B) [design_spec 3.5, 4.3].

1. Translate the 40 English filler sessions into HI and HG (all turns).
2. Build every history (item x storage language).
3. Translate every HI/HG memory unit and query into English (for S2/S3), cached in en_translations.json.
"""
import json

from . import config, haystack, prompts
from .textutil import HG_MARKERS, devanagari_share


def _write_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def translate_fillers(gen, log):
    pool_en = haystack.load_jsonl(config.DATA / "haystacks" / "filler_pool_en.jsonl")
    texts = [t["content"] for s in pool_en for t in s["turns"]]
    hi = gen.chat([prompts.to_hindi(x) for x in texts], max_tokens=400)
    hg = gen.chat([prompts.to_hinglish(x) for x in texts], max_tokens=400)
    # Hinglish must be Roman script. Fallback for turns where Devanagari leaked in: romanize the
    # Hindi translation of the same turn (an easier task than translating from English).
    bad = [i for i, x in enumerate(hg) if devanagari_share(x) > 0.1]
    if bad:
        retry = gen.chat([prompts.romanize_hindi(hi[i]) for i in bad], max_tokens=400)
        for i, x in zip(bad, retry):
            hg[i] = x
    # Second failure mode seen in the r2 spot check: the model returns the English text unchanged. Catch long turns
    # with almost no Hindi function words and romanize the Hindi version instead.
    english_leak = [i for i, x in enumerate(hg) if i not in bad and len(x.split()) >= 12
                    and sum(w in HG_MARKERS for w in x.lower().split()) / len(x.split()) < 0.05]
    if english_leak:
        retry = gen.chat([prompts.romanize_hindi(hi[i]) for i in english_leak], max_tokens=400)
        for i, x in zip(english_leak, retry):
            hg[i] = x
    log["hinglish_english_leak_fixed"] = len(english_leak)
    still = [i for i, x in enumerate(hg) if devanagari_share(x) > 0.1]
    log["filler_turns"] = len(texts)
    log["hinglish_fewshot_failures"] = len(bad)
    log["hinglish_still_devanagari"] = len(still)
    log["hindi_low_devanagari"] = sum(devanagari_share(x) < 0.5 for x in hi)
    print("filler translation:", log)
    # Fail loudly rather than silently building half-Devanagari "Hinglish" histories [design_spec 3.5].
    # (Skipped for the mock backend, whose "translations" are English copies by design.)
    if gen.backend != "mock" and (len(still) > 0.05 * len(texts) or log["hindi_low_devanagari"] > 0.05 * len(texts)):
        raise RuntimeError(f"Filler translation quality check failed: {log}. Hinglish/Hindi fillers not saved.")
    for lang, outs in (("hi", hi), ("hg", hg)):
        it = iter(outs)
        rows = [{**s, "turns": [{"role": t["role"], "content": next(it)} for t in s["turns"]]} for s in pool_en]
        _write_jsonl(haystack.filler_path(lang), rows)


def run(items_path, backend=None):
    from .llm import Generator
    gen = None

    def translator():   # loaded only if something needs translating (ablation blocks reuse the main cache)
        nonlocal gen
        if gen is None:
            gen = Generator(config.TRANSLATOR, backend=backend, max_model_len=config.SHORT_MODEL_LEN)
        return gen

    log = {}
    if not (haystack.filler_path("hi").exists() and haystack.filler_path("hg").exists()):
        translate_fillers(translator(), log)
    items = haystack.load_items(items_path)
    hist = haystack.build_all(items, haystack.load_fillers())
    _write_jsonl(config.RESULTS / "histories.jsonl", hist)

    cache_path = config.TRANSLATION_CACHE
    cache = json.loads(cache_path.read_text(encoding="utf-8")) if cache_path.exists() else {}
    todo = set()
    for h in hist:
        if h["lang"] != "en":
            todo.update(text for _, text in haystack.units(h))
    for it in items:
        for lang in ("hi", "hg"):
            todo.update(it["text"][lang][q] for q in config.QUERY_VARIANTS)
    todo = sorted(x for x in todo if x not in cache)
    if todo:
        for src, out in zip(todo, translator().chat([prompts.to_english(x) for x in todo], max_tokens=300)):
            cache[src] = out
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache_path.write_text(json.dumps(cache, ensure_ascii=False, indent=0), encoding="utf-8")
    log["en_translations"] = len(cache)
    log["newly_translated"] = len(todo)
    log["block"], log["n_sessions"] = config.BLOCK, config.N_SESSIONS
    (config.RESULTS / "log_translate.json").write_text(json.dumps(log, indent=2))
    print("translate stage:", log)


In [ ]:
%%writefile /kaggle/working/clm/data/items/items_main.jsonl
{"item_id": "F001", "cand_id": "FC106", "memory_type": "fact", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I've been listening to audiobooks during my daily commute, which takes 45 minutes each way. Can you recommend some good fiction audiobooks?", "assistant_reply": "That's a great way to use your travel time! A few popular picks: The Nightingale by Kristin Hannah, The Silent Patient by Alex Michaelides, and Project Hail Mary by Andy Weir. Do you prefer mysteries or historical fiction?", "followup_turn": "I'm already reading Gone Girl on my Kindle, but The Nightingale sounds interesting. What is it about?", "followup_reply": "It's a historical novel set in France during World War II, following two sisters who resist the occupation in very different ways.", "q1": "How long does my commute to work take, one way?", "q2": "One way, how much time does it take me to get to work?", "distractor_sentence": "By the way, my sister's commute is only 20 minutes each way, lucky her!"}, "hi": {"memory_turn": "मैं आजकल रोज़ ऑफिस आते-जाते ऑडियोबुक्स सुनती हूँ, एक तरफ़ का सफ़र 45 मिनट का होता है। क्या आप कुछ अच्छी फिक्शन ऑडियोबुक्स बता सकते हैं?", "assistant_reply": "सफ़र के समय का यह बढ़िया इस्तेमाल है! कुछ लोकप्रिय विकल्प: क्रिस्टिन हैना की The Nightingale, एलेक्स माइकलिडीज़ की The Silent Patient और एंडी वियर की Project Hail Mary। आपको मिस्ट्री पसंद है या ऐतिहासिक कहानियाँ?", "followup_turn": "मैं Gone Girl पहले से अपने Kindle पर पढ़ रही हूँ, पर The Nightingale दिलचस्प लग रही है। यह किस बारे में है?", "followup_reply": "यह दूसरे विश्व युद्ध के दौरान फ़्रांस में बसी एक ऐतिहासिक कहानी है, जिसमें दो बहनें अलग-अलग तरीक़ों से कब्ज़े का विरोध करती हैं।", "q1": "मेरे ऑफिस जाने में एक तरफ़ कितना समय लगता है?", "q2": "एक तरफ़ से, मुझे काम पर पहुँचने में कितनी देर लगती है?", "distractor_sentence": "वैसे, मेरी बहन का ऑफिस तो बस एक तरफ़ 20 मिनट दूर है, किस्मत वाली है!"}, "hg": {"memory_turn": "Main aajkal daily office aate-jaate audiobooks sunti hoon, ek side ka travel 45 minute ka hota hai. Kya aap kuch acchi fiction audiobooks suggest kar sakte hain?", "assistant_reply": "Travel time ka yeh bahut accha use hai! Kuch popular options: Kristin Hannah ki The Nightingale, Alex Michaelides ki The Silent Patient, aur Andy Weir ki Project Hail Mary. Aapko mystery pasand hai ya historical fiction?", "followup_turn": "Main Gone Girl already apne Kindle pe padh rahi hoon, par The Nightingale interesting lag rahi hai. Yeh kis baare mein hai?", "followup_reply": "Yeh World War II ke time France mein set ek historical novel hai, jismein do behnein alag-alag tareeke se occupation ka virodh karti hain.", "q1": "Mera office jaane mein ek side kitna time lagta hai?", "q2": "One way, mujhe kaam pe pahunchne mein kitni der lagti hai?", "distractor_sentence": "Waise, meri behen ka office toh bas ek side 20 minute door hai, lucky hai woh!"}}, "gold": {"answer_en": "45 minutes (each way)", "slots": [{"name": "duration", "aliases": ["45 minutes", "45 minute", "45 mins", "45 min", "45 मिनट", "पौन घंटा", "paun ghanta", "three quarters of an hour"]}]}, "distractor": {"value_aliases": ["20 minutes", "20 minute", "20 min", "20 मिनट"]}, "source": {"dataset": "LongMemEval", "question_id": "118b2229", "lme_type": "single-session-user"}, "topic": "work/study", "is_answerable": true, "evidence_position": 2, "filler_seed": 1599741519, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "early", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F002", "cand_id": "FC105", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I graduated with a degree in Business Administration, and it has really helped in my new role. Any advice on staying organized with paperwork, especially tracking expenses and reimbursements?", "assistant_reply": "Congratulations on the new role! A simple system helps: scan receipts the same day, keep one folder per month, and log every expense in a spreadsheet with date, amount and purpose.", "followup_turn": "A spreadsheet sounds good. Should I use Excel or Google Sheets for this?", "followup_reply": "Either works. Google Sheets is handy if you want to update it from your phone and share it with your manager; Excel is better for heavier analysis offline.", "q1": "What degree did I graduate with?", "q2": "What subject is my degree in?"}, "hi": {"memory_turn": "मैंने बिज़नेस एडमिनिस्ट्रेशन में डिग्री ली है, और इससे मुझे अपनी नई नौकरी में बहुत मदद मिली है। कागज़ात सँभालकर रखने के लिए कोई सलाह देंगे, ख़ासकर ख़र्चों और रीइम्बर्समेंट का हिसाब रखने के लिए?", "assistant_reply": "नई नौकरी की बधाई! एक आसान तरीक़ा अपनाइए: रसीदें उसी दिन स्कैन करें, हर महीने का एक अलग फ़ोल्डर रखें, और हर ख़र्च को तारीख़, रक़म और मक़सद के साथ एक स्प्रेडशीट में लिखें।", "followup_turn": "स्प्रेडशीट अच्छा आइडिया है। इसके लिए Excel ठीक रहेगा या Google Sheets?", "followup_reply": "दोनों चलेंगे। अगर आप फ़ोन से अपडेट करना और मैनेजर के साथ शेयर करना चाहते हैं तो Google Sheets आसान है; ऑफ़लाइन ज़्यादा विश्लेषण के लिए Excel बेहतर है।", "q1": "मैंने किस विषय में डिग्री ली है?", "q2": "मेरी ग्रेजुएशन की डिग्री किस विषय में है?"}, "hg": {"memory_turn": "Maine Business Administration mein degree li hai, aur isse meri nayi job mein bahut help mili hai. Paperwork organized rakhne ke liye koi advice denge, especially expenses aur reimbursements track karne ke liye?", "assistant_reply": "Nayi job ki congratulations! Ek simple system follow kijiye: receipts usi din scan karein, har mahine ka alag folder rakhein, aur har expense ko date, amount aur purpose ke saath ek spreadsheet mein likhein.", "followup_turn": "Spreadsheet accha idea hai. Iske liye Excel theek rahega ya Google Sheets?", "followup_reply": "Dono chalenge. Agar aap phone se update karna aur manager ke saath share karna chahte hain toh Google Sheets easy hai; offline zyada analysis ke liye Excel better hai.", "q1": "Maine kis subject mein degree li hai?", "q2": "Meri graduation ki degree kis subject mein hai?"}}, "gold": {"answer_en": "Business Administration", "slots": [{"name": "degree", "aliases": ["Business Administration", "BBA", "बिज़नेस एडमिनिस्ट्रेशन", "बिजनेस एडमिनिस्ट्रेशन", "व्यवसाय प्रशासन"]}]}, "source": {"dataset": "LongMemEval", "question_id": "e47becba", "lme_type": "single-session-user"}, "topic": "work/study", "is_answerable": true, "evidence_position": 5, "filler_seed": 42481310, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "late", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F003", "cand_id": "FC110", "memory_type": "fact", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I need to update my address with my health insurance provider. Also, I recently changed my last name and I'm still getting used to it. My old surname was Johnson, but now it's Winters. Does that need a separate update?", "assistant_reply": "Yes, a name change is usually a separate request. Insurers typically ask for a copy of the legal document, such as a marriage certificate or court order, along with your policy number.", "followup_turn": "Okay. Can I change the address online, or do I have to call them?", "followup_reply": "Most insurers let you change your address online through the member portal or app. Calling is only needed if the portal does not allow edits for your plan.", "q1": "What was my last name before I changed it?", "q2": "What was my old surname?"}, "hi": {"memory_turn": "मुझे अपनी हेल्थ इंश्योरेंस कंपनी में अपना पता अपडेट करवाना है। साथ ही, मैंने हाल ही में अपना सरनेम बदला है और अभी उसकी आदत डाल रही हूँ। मेरा पुराना सरनेम जॉनसन था, अब विंटर्स है। क्या इसके लिए अलग से अपडेट करना होगा?", "assistant_reply": "हाँ, नाम बदलने का अनुरोध आमतौर पर अलग से होता है। इंश्योरेंस कंपनियाँ अक्सर क़ानूनी दस्तावेज़ की कॉपी माँगती हैं, जैसे विवाह प्रमाणपत्र या कोर्ट का आदेश, साथ में आपका पॉलिसी नंबर।", "followup_turn": "ठीक है। क्या पता ऑनलाइन बदल सकती हूँ, या उन्हें फ़ोन करना पड़ेगा?", "followup_reply": "ज़्यादातर कंपनियाँ मेंबर पोर्टल या ऐप से ऑनलाइन पता बदलने देती हैं। फ़ोन तभी करना पड़ता है जब आपके प्लान में पोर्टल पर बदलाव की सुविधा न हो।", "q1": "सरनेम बदलने से पहले मेरा सरनेम क्या था?", "q2": "मेरा पुराना सरनेम क्या था?"}, "hg": {"memory_turn": "Mujhe apni health insurance company mein apna address update karwana hai. Saath hi, maine recently apna surname change kiya hai aur abhi uski aadat daal rahi hoon. Mera purana surname Johnson tha, ab Winters hai. Kya iske liye alag se update karna hoga?", "assistant_reply": "Haan, name change ki request usually alag hoti hai. Insurance companies aksar legal document ki copy maangti hain, jaise marriage certificate ya court order, saath mein aapka policy number.", "followup_turn": "Theek hai. Kya address online change kar sakti hoon, ya unhe call karna padega?", "followup_reply": "Zyadatar companies member portal ya app se online address change karne deti hain. Call tabhi karna padta hai jab aapke plan mein portal pe changes ki facility na ho.", "q1": "Surname change karne se pehle mera surname kya tha?", "q2": "Mera purana surname kya tha?"}}, "gold": {"answer_en": "Johnson", "slots": [{"name": "old_surname", "aliases": ["Johnson", "जॉनसन", "जानसन"]}]}, "near_miss_aliases": ["Winters", "विंटर्स"], "source": {"dataset": "LongMemEval", "question_id": "c5e8278d", "lme_type": "single-session-user"}, "topic": "fitness/health", "is_answerable": true, "evidence_position": 9, "filler_seed": 4209568809, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F004", "cand_id": "FC118", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I've been to the Great Ocean Road before. I went there with friends during my study abroad program at the University of Melbourne, and we spent the whole day hiking along the coast. Any other must-see places in Australia?", "assistant_reply": "Lucky you, that drive is stunning! If you go back, consider the Blue Mountains near Sydney, the Great Barrier Reef in Queensland, and Uluru in the Northern Territory.", "followup_turn": "Uluru sounds amazing. What's the best season to visit it?", "followup_reply": "Roughly May to September is best. Days are mild then, while summer temperatures there often go above 40 degrees Celsius.", "q1": "At which university did I do my study abroad program?", "q2": "Where did I study during my study abroad program?"}, "hi": {"memory_turn": "मैं पहले ग्रेट ओशन रोड जा चुका हूँ। अपने स्टडी अब्रॉड प्रोग्राम के दौरान, जो यूनिवर्सिटी ऑफ़ मेलबर्न में था, मैं दोस्तों के साथ वहाँ गया था और हमने पूरे दिन तट के किनारे हाइकिंग की। ऑस्ट्रेलिया में और कौन-सी जगहें ज़रूर देखनी चाहिए?", "assistant_reply": "आप ख़ुशक़िस्मत हैं, वह रास्ता बेहद ख़ूबसूरत है! अगर दोबारा जाएँ तो सिडनी के पास ब्लू माउंटेंस, क्वींसलैंड में ग्रेट बैरियर रीफ़ और नॉर्दर्न टेरिटरी में उलुरु देखिए।", "followup_turn": "उलुरु कमाल का लगता है। वहाँ जाने का सबसे अच्छा मौसम कौन-सा है?", "followup_reply": "मई से सितंबर के बीच का समय सबसे अच्छा है। तब दिन सुहावने रहते हैं, जबकि गर्मियों में वहाँ तापमान अक्सर 40 डिग्री सेल्सियस से ऊपर चला जाता है।", "q1": "मैंने अपना स्टडी अब्रॉड प्रोग्राम किस यूनिवर्सिटी में किया था?", "q2": "स्टडी अब्रॉड के दौरान मैं कहाँ पढ़ा था?"}, "hg": {"memory_turn": "Main pehle Great Ocean Road ja chuka hoon. Apne study abroad program ke time, jo University of Melbourne mein tha, main dosto ke saath wahan gaya tha aur humne poore din coast ke kinare hiking ki. Australia mein aur kaunsi jagah zaroor dekhni chahiye?", "assistant_reply": "Aap lucky hain, woh drive bahut khoobsurat hai! Agar dobara jaayein toh Sydney ke paas Blue Mountains, Queensland mein Great Barrier Reef aur Northern Territory mein Uluru dekhiye.", "followup_turn": "Uluru kamaal ka lagta hai. Wahan jaane ka best season kaunsa hai?", "followup_reply": "May se September ke beech ka time sabse accha hai. Tab din pleasant rehte hain, jabki garmiyon mein wahan temperature aksar 40 degree Celsius se upar chala jaata hai.", "q1": "Maine apna study abroad program kis university mein kiya tha?", "q2": "Study abroad ke time main kahan padha tha?"}}, "gold": {"answer_en": "University of Melbourne", "slots": [{"name": "university", "aliases": ["University of Melbourne", "Melbourne University", "Melbourne", "यूनिवर्सिटी ऑफ़ मेलबर्न", "यूनिवर्सिटी ऑफ मेलबर्न", "मेलबर्न यूनिवर्सिटी", "मेलबर्न"]}]}, "source": {"dataset": "LongMemEval", "question_id": "3b6f954b", "lme_type": "single-session-user"}, "topic": "work/study", "is_answerable": true, "evidence_position": 2, "filler_seed": 2679826000, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "middle", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F005", "cand_id": "FC121", "memory_type": "fact", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "For my sister's birthday, I got her a yellow dress and matching earrings. Now I want to plan a small dinner for her at home. Any ideas for a cozy menu?", "assistant_reply": "How thoughtful! Try a simple starter like tomato soup, a main such as pasta or a paneer dish, and finish with a homemade dessert like chocolate mousse.", "followup_turn": "She loves Italian food, so pasta sounds perfect. Which sauce is easy but impressive?", "followup_reply": "A creamy garlic and mushroom sauce is quick and feels special. Finish it with fresh parsley and grated parmesan just before serving.", "q1": "What did I buy for my sister's birthday?", "q2": "What gift did I get my sister for her birthday?"}, "hi": {"memory_turn": "अपनी बहन के जन्मदिन पर मैंने उसके लिए एक पीली ड्रेस और उसके साथ मैचिंग इयररिंग्स लिए। अब मैं उसके लिए घर पर एक छोटा-सा डिनर रखना चाहती हूँ। कोई अच्छा मेन्यू सुझाएँगे?", "assistant_reply": "कितना प्यारा ख़याल है! शुरुआत में टमाटर सूप, मेन कोर्स में पास्ता या पनीर की कोई डिश, और आख़िर में घर पर बना चॉकलेट मूस रख सकती हैं।", "followup_turn": "उसे इटैलियन खाना बहुत पसंद है, तो पास्ता बढ़िया रहेगा। कौन-सी सॉस आसान भी हो और ख़ास भी लगे?", "followup_reply": "क्रीमी गार्लिक मशरूम सॉस जल्दी बनती है और ख़ास भी लगती है। परोसने से ठीक पहले ऊपर से ताज़ा पार्सले और कद्दूकस किया पार्मेज़ान डालें।", "q1": "मैंने अपनी बहन के जन्मदिन पर उसके लिए क्या ख़रीदा था?", "q2": "बहन के बर्थडे पर मैंने उसे क्या गिफ़्ट दिया था?"}, "hg": {"memory_turn": "Apni behen ke birthday pe maine uske liye ek yellow dress aur uske saath matching earrings li. Ab main uske liye ghar pe ek chhota sa dinner rakhna chahti hoon. Koi accha menu suggest karenge?", "assistant_reply": "Kitna pyaara idea hai! Starter mein tomato soup, main course mein pasta ya koi paneer dish, aur end mein ghar ka bana chocolate mousse rakh sakti hain.", "followup_turn": "Use Italian food bahut pasand hai, toh pasta perfect rahega. Kaunsi sauce easy bhi ho aur special bhi lage?", "followup_reply": "Creamy garlic mushroom sauce jaldi banti hai aur special bhi lagti hai. Serve karne se theek pehle upar se fresh parsley aur grated parmesan daalein.", "q1": "Maine apni behen ke birthday pe uske liye kya khareeda tha?", "q2": "Behen ke birthday pe maine use kya gift diya tha?"}}, "gold": {"answer_en": "a yellow dress (and matching earrings)", "slots": [{"name": "gift", "aliases": ["yellow dress", "पीली ड्रेस", "पीली पोशाक", "peeli dress", "pili dress", "पीला ड्रेस", "पीले रंग की ड्रेस"]}]}, "source": {"dataset": "LongMemEval", "question_id": "66f24dbb", "lme_type": "single-session-user"}, "topic": "family/social", "is_answerable": true, "evidence_position": 5, "filler_seed": 547367346, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "middle", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F006", "cand_id": "FC128", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I'm thinking of getting Max a new collar with a name tag. Do you have any recommendations for a good collar for a Golden Retriever like him?", "assistant_reply": "For a medium-to-large dog, a padded nylon or leather collar with a sturdy metal buckle works well. Make sure you can fit two fingers under it once it is on.", "followup_turn": "Should the name tag have my phone number on it?", "followup_reply": "Yes, your phone number is the most useful detail. Many owners also add a short line saying the dog is microchipped, so whoever finds him knows to get it scanned.", "q1": "What breed is my dog?", "q2": "Which breed is Max?"}, "hi": {"memory_turn": "मैं मैक्स के लिए नेम टैग वाला एक नया कॉलर लेने की सोच रहा हूँ। उसके जैसे गोल्डन रिट्रीवर के लिए कोई अच्छा कॉलर सुझाएँगे?", "assistant_reply": "मध्यम से बड़े कुत्ते के लिए गद्देदार नायलॉन या चमड़े का कॉलर, मज़बूत धातु के बकल के साथ, अच्छा रहता है। पहनाने के बाद ध्यान रखें कि उसके नीचे दो उँगलियाँ आ सकें।", "followup_turn": "क्या नेम टैग पर मेरा फ़ोन नंबर होना चाहिए?", "followup_reply": "हाँ, फ़ोन नंबर सबसे काम की जानकारी है। कई लोग एक छोटी-सी लाइन भी लिखते हैं कि कुत्ते में माइक्रोचिप है, ताकि जिसे वह मिले वह उसे स्कैन करवा सके।", "q1": "मेरा कुत्ता किस नस्ल का है?", "q2": "मैक्स किस नस्ल का है?"}, "hg": {"memory_turn": "Main Max ke liye name tag wala ek naya collar lene ki soch raha hoon. Uske jaise Golden Retriever ke liye koi accha collar suggest karenge?", "assistant_reply": "Medium se bade dog ke liye padded nylon ya leather collar, strong metal buckle ke saath, accha rehta hai. Pehnane ke baad dhyan rakhein ki uske neeche do ungliyan aa sakein.", "followup_turn": "Kya name tag pe mera phone number hona chahiye?", "followup_reply": "Haan, phone number sabse kaam ki detail hai. Kai log ek chhoti si line bhi likhte hain ki dog microchipped hai, taaki jise woh mile woh use scan karwa sake.", "q1": "Mera dog kis breed ka hai?", "q2": "Max kis breed ka hai?"}}, "gold": {"answer_en": "Golden Retriever", "slots": [{"name": "breed", "aliases": ["Golden Retriever", "गोल्डन रिट्रीवर", "golden retriver"]}]}, "source": {"dataset": "LongMemEval", "question_id": "75499fd8", "lme_type": "single-session-user"}, "topic": "pets", "is_answerable": true, "evidence_position": 9, "filler_seed": 2344873239, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "middle", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F007", "cand_id": "FC131", "memory_type": "fact", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "We've made a lot of changes at home recently, and my cat Luna has been such a sweetie through all of it. How can I tell if a cat is stressed by changes in her environment?", "assistant_reply": "Common signs include hiding more than usual, changes in appetite, over-grooming, and going outside the litter box. Keeping her feeding and play routine steady helps a lot.", "followup_turn": "She has been hiding under the bed a bit more. Should I try to pull her out?", "followup_reply": "No, let her come out on her own. Leave a soft blanket and some treats nearby, and spend quiet time in the room so she learns the new setup is safe.", "q1": "What is my cat's name?", "q2": "What's my cat called?", "distractor_sentence": "By the way, my neighbour's cat Coco keeps visiting our balcony."}, "hi": {"memory_turn": "हाल ही में हमने घर में काफ़ी बदलाव किए हैं, और मेरी बिल्ली लूना इस पूरे दौरान बहुत प्यारी बनी रही। कैसे पता करूँ कि माहौल बदलने से बिल्ली तनाव में है?", "assistant_reply": "आम संकेत हैं: सामान्य से ज़्यादा छिपना, भूख में बदलाव, बार-बार ख़ुद को चाटना, और लिटर बॉक्स के बाहर गंदगी करना। खाने और खेलने का रूटीन एक जैसा रखने से काफ़ी मदद मिलती है।", "followup_turn": "वह आजकल बेड के नीचे थोड़ा ज़्यादा छिप रही है। क्या उसे बाहर निकालूँ?", "followup_reply": "नहीं, उसे ख़ुद बाहर आने दीजिए। पास में एक नरम कंबल और कुछ ट्रीट्स रखें, और उस कमरे में शांति से समय बिताएँ ताकि उसे लगे कि नया माहौल सुरक्षित है।", "q1": "मेरी बिल्ली का नाम क्या है?", "q2": "मैंने अपनी बिल्ली का क्या नाम रखा है?", "distractor_sentence": "वैसे, पड़ोसियों की बिल्ली कोको बार-बार हमारी बालकनी में आ जाती है।"}, "hg": {"memory_turn": "Recently humne ghar mein kaafi changes kiye hain, aur meri cat Luna is poore time bahut sweet rahi. Kaise pata karoon ki environment badalne se cat stressed hai?", "assistant_reply": "Common signs hain: normal se zyada chhupna, appetite mein change, baar-baar khud ko chaatna, aur litter box ke bahar gandagi karna. Khaane aur khelne ka routine same rakhne se kaafi help milti hai.", "followup_turn": "Woh aajkal bed ke neeche thoda zyada chhup rahi hai. Kya use bahar nikaalun?", "followup_reply": "Nahi, use khud bahar aane dijiye. Paas mein ek soft blanket aur kuch treats rakhein, aur us room mein shaanti se time spend karein taaki use lage ki naya setup safe hai.", "q1": "Meri cat ka naam kya hai?", "q2": "Maine apni cat ka kya naam rakha hai?", "distractor_sentence": "Waise, padosiyon ki cat Coco baar-baar humari balcony mein aa jaati hai."}}, "gold": {"answer_en": "Luna", "slots": [{"name": "cat_name", "aliases": ["Luna", "लूना"]}]}, "distractor": {"value_aliases": ["Coco", "कोको"]}, "source": {"dataset": "LongMemEval", "question_id": "0862e8bf", "lme_type": "single-session-user"}, "topic": "pets", "is_answerable": true, "evidence_position": 2, "filler_seed": 3986903086, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F008", "cand_id": "FC151", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I visited Fushimi Inari Shrine when I was in Japan a few months ago. I spent two weeks travelling solo around the country. Can you recommend some budget-friendly places to stay in Kyoto?", "assistant_reply": "Kyoto has great budget options: guesthouses near Kyoto Station, capsule hotels downtown, and small ryokan-style hostels in the Gion area that still feel traditional.", "followup_turn": "Is it easy to get around Kyoto without a car?", "followup_reply": "Very easy. City buses reach most temples, the subway covers the centre, and renting a bicycle for a day is a popular and cheap way to explore.", "q1": "How long was I in Japan?", "q2": "How much time did I spend travelling in Japan?"}, "hi": {"memory_turn": "कुछ महीने पहले जब मैं जापान में था, तब मैं फ़ुशिमी इनारी श्राइन गया था। मैंने दो हफ़्ते अकेले पूरे देश में घूमते हुए बिताए। क्योटो में रुकने के लिए कुछ किफ़ायती जगहें बताएँगे?", "assistant_reply": "क्योटो में बजट के अच्छे विकल्प हैं: क्योटो स्टेशन के पास गेस्टहाउस, शहर के बीच कैप्सूल होटल, और गियोन इलाक़े में छोटे रयोकान-स्टाइल हॉस्टल, जिनमें पारंपरिक माहौल भी मिलता है।", "followup_turn": "क्या क्योटो में बिना कार के घूमना आसान है?", "followup_reply": "बहुत आसान है। शहर की बसें ज़्यादातर मंदिरों तक जाती हैं, सबवे शहर के बीच के हिस्से को कवर करता है, और एक दिन के लिए साइकिल किराए पर लेना घूमने का लोकप्रिय और सस्ता तरीक़ा है।", "q1": "मैं जापान में कितने समय तक रहा था?", "q2": "मैंने जापान घूमने में कितना समय बिताया?"}, "hg": {"memory_turn": "Kuch mahine pehle jab main Japan mein tha, tab main Fushimi Inari Shrine gaya tha. Maine do hafte akele poore country mein ghoomte hue bitaaye. Kyoto mein rukne ke liye kuch budget-friendly jagah batayenge?", "assistant_reply": "Kyoto mein budget ke acche options hain: Kyoto Station ke paas guesthouses, downtown mein capsule hotels, aur Gion area mein chhote ryokan-style hostels, jinmein traditional feel bhi milti hai.", "followup_turn": "Kya Kyoto mein bina car ke ghoomna easy hai?", "followup_reply": "Bahut easy hai. City buses zyadatar temples tak jaati hain, subway centre ko cover karta hai, aur ek din ke liye cycle rent karna ghoomne ka popular aur sasta tareeka hai.", "q1": "Main Japan mein kitne time tak raha tha?", "q2": "Maine Japan ghoomne mein kitna time bitaya?"}}, "gold": {"answer_en": "two weeks", "slots": [{"name": "duration", "aliases": ["two weeks", "2 weeks", "14 days", "fortnight", "दो हफ़्ते", "दो हफ्ते", "2 हफ़्ते", "दो सप्ताह", "14 दिन", "do hafte", "2 hafte", "14 din"]}]}, "source": {"dataset": "LongMemEval", "question_id": "19b5f2b3", "lme_type": "single-session-user"}, "topic": "travel", "is_answerable": true, "evidence_position": 5, "filler_seed": 4108670730, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "late", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F009", "cand_id": "FC155", "memory_type": "fact", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I just set up my new Samsung 55-inch 4K smart TV on Saturday and spent ages hiding the cables behind it. Can you recommend some good streaming services?", "assistant_reply": "Nice upgrade! Popular choices are Netflix for series, Amazon Prime Video for a big film collection, and JioHotstar for sports and Indian content.", "followup_turn": "Can I watch all of them through the TV's built-in apps?", "followup_reply": "Yes, Samsung smart TVs offer all three through the Tizen app store. You just need to download each app once and sign in with your account.", "q1": "What size is my new Samsung TV?", "q2": "How big is the new TV I set up?", "distractor_sentence": "My parents still use their old 43-inch TV in the living room."}, "hi": {"memory_turn": "मैंने शनिवार को अपना नया Samsung 55 इंच का 4K स्मार्ट टीवी सेट किया और उसके पीछे केबल छिपाने में काफ़ी समय लग गया। कोई अच्छी स्ट्रीमिंग सर्विस बताएँगे?", "assistant_reply": "बढ़िया अपग्रेड! सीरीज़ के लिए Netflix, फ़िल्मों के बड़े कलेक्शन के लिए Amazon Prime Video, और खेल व भारतीय कंटेंट के लिए JioHotstar लोकप्रिय विकल्प हैं।", "followup_turn": "क्या ये सब टीवी के बिल्ट-इन ऐप्स से चल जाएँगे?", "followup_reply": "हाँ, Samsung स्मार्ट टीवी पर ये तीनों Tizen ऐप स्टोर से मिल जाते हैं। बस हर ऐप एक बार डाउनलोड करके अपने अकाउंट से लॉग इन करना होगा।", "q1": "मेरा नया Samsung टीवी कितने इंच का है?", "q2": "मैंने जो नया टीवी लगाया है, वह कितना बड़ा है?", "distractor_sentence": "मेरे मम्मी-पापा अब भी लिविंग रूम में अपना पुराना 43 इंच वाला टीवी चलाते हैं।"}, "hg": {"memory_turn": "Maine Saturday ko apna naya Samsung 55-inch 4K smart TV set kiya aur uske peeche cables chhupane mein kaafi time lag gaya. Koi acchi streaming service batayenge?", "assistant_reply": "Badhiya upgrade! Series ke liye Netflix, movies ke bade collection ke liye Amazon Prime Video, aur sports aur Indian content ke liye JioHotstar popular options hain.", "followup_turn": "Kya yeh sab TV ke built-in apps se chal jayenge?", "followup_reply": "Haan, Samsung smart TV pe yeh teeno Tizen app store se mil jaate hain. Bas har app ek baar download karke apne account se login karna hoga.", "q1": "Mera naya Samsung TV kitne inch ka hai?", "q2": "Maine jo naya TV lagaya hai, woh kitna bada hai?", "distractor_sentence": "Mere mummy-papa ab bhi living room mein apna purana 43-inch wala TV chalate hain."}}, "gold": {"answer_en": "55 inches", "slots": [{"name": "size", "aliases": ["55-inch", "55 inch", "55 inches", "55 इंच", "55-इंच"]}]}, "distractor": {"value_aliases": ["43-inch", "43 inch", "43 inches", "43 इंच"]}, "source": {"dataset": "LongMemEval", "question_id": "76d63226", "lme_type": "single-session-user"}, "topic": "shopping/home", "is_answerable": true, "evidence_position": 9, "filler_seed": 1487936767, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "early", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F010", "cand_id": "FC161", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I've been practising guitar for 30 minutes every day and it's helping me progress nicely. Can you recommend some online resources for learning music theory and fingerpicking?", "assistant_reply": "Good habit! For theory, musictheory.net has free interactive lessons. For fingerpicking, JustinGuitar has a structured free course, and many teachers post step-by-step tutorials on YouTube.", "followup_turn": "Should I learn theory first or fingerpicking first?", "followup_reply": "Do both in parallel. Spend a few minutes on theory like scales and chord building, then apply it right away in a fingerpicking pattern so it sticks.", "q1": "How much time do I practise guitar every day?", "q2": "How long is my daily guitar practice?"}, "hi": {"memory_turn": "मैं रोज़ 30 मिनट गिटार की प्रैक्टिस कर रहा हूँ और इससे मेरी अच्छी प्रगति हो रही है। म्यूज़िक थ्योरी और फ़िंगरपिकिंग सीखने के लिए कुछ ऑनलाइन रिसोर्स बताएँगे?", "assistant_reply": "अच्छी आदत है! थ्योरी के लिए musictheory.net पर मुफ़्त इंटरैक्टिव लेसन हैं। फ़िंगरपिकिंग के लिए JustinGuitar पर एक व्यवस्थित मुफ़्त कोर्स है, और कई टीचर YouTube पर स्टेप-बाय-स्टेप ट्यूटोरियल डालते हैं।", "followup_turn": "पहले थ्योरी सीखूँ या फ़िंगरपिकिंग?", "followup_reply": "दोनों साथ-साथ सीखिए। कुछ मिनट स्केल और कॉर्ड बनाने जैसी थ्योरी पर लगाइए, फिर उसे तुरंत किसी फ़िंगरपिकिंग पैटर्न में आज़माइए ताकि वह याद रहे।", "q1": "मैं रोज़ कितनी देर गिटार की प्रैक्टिस करता हूँ?", "q2": "मेरी रोज़ की गिटार प्रैक्टिस कितने समय की होती है?"}, "hg": {"memory_turn": "Main roz 30 minute guitar practice kar raha hoon aur isse meri acchi progress ho rahi hai. Music theory aur fingerpicking seekhne ke liye kuch online resources batayenge?", "assistant_reply": "Acchi habit hai! Theory ke liye musictheory.net pe free interactive lessons hain. Fingerpicking ke liye JustinGuitar pe ek structured free course hai, aur kai teachers YouTube pe step-by-step tutorials daalte hain.", "followup_turn": "Pehle theory seekhun ya fingerpicking?", "followup_reply": "Dono saath-saath seekhiye. Kuch minute scales aur chord building jaisi theory pe lagaiye, phir use turant kisi fingerpicking pattern mein try kijiye taaki woh yaad rahe.", "q1": "Main roz kitni der guitar practice karta hoon?", "q2": "Meri roz ki guitar practice kitne time ki hoti hai?"}}, "gold": {"answer_en": "30 minutes", "slots": [{"name": "duration", "aliases": ["30 minutes", "30 minute", "30 mins", "30 min", "half an hour", "30 मिनट", "आधा घंटा", "aadha ghanta", "adha ghanta"]}]}, "source": {"dataset": "LongMemEval", "question_id": "29f2956b", "lme_type": "single-session-user"}, "topic": "hobbies", "is_answerable": true, "evidence_position": 2, "filler_seed": 2542195589, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "late", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": ["FL03"]}
{"item_id": "P001", "cand_id": "FC125", "memory_type": "preference", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "Nike has been my favourite brand for running shoes so far. I'm looking for a new pair of the same model as my current ones. Where can I find them, or are there similar models?", "assistant_reply": "If the exact model is discontinued, the brand's website and big sports retailers often list a newer version under the same name with a new number. A running store can also match the fit.", "followup_turn": "How often should running shoes be replaced?", "followup_reply": "A common guideline is every 500 to 800 kilometres. Replace them sooner if the cushioning feels flat or the soles show uneven wear.", "q1": "Which brand of running shoes is my favourite?", "q2": "What's my favourite running shoe brand?", "distractor_sentence": "My brother swears by Asics for his runs, but I've never tried them."}, "hi": {"memory_turn": "अब तक रनिंग शूज़ के लिए Nike मेरा पसंदीदा ब्रांड रहा है। मैं अपने अभी वाले जूतों का ही मॉडल फिर से लेना चाहती हूँ। वे कहाँ मिलेंगे, या उनसे मिलते-जुलते मॉडल कौन-से हैं?", "assistant_reply": "अगर वही मॉडल बंद हो गया है, तो ब्रांड की वेबसाइट और बड़े स्पोर्ट्स स्टोर अक्सर उसी नाम से नए नंबर वाला वर्ज़न रखते हैं। रनिंग स्टोर पर फ़िट मिलाकर भी देख सकती हैं।", "followup_turn": "रनिंग शूज़ कितने समय बाद बदलने चाहिए?", "followup_reply": "आम तौर पर हर 500 से 800 किलोमीटर के बाद। अगर कुशनिंग दबी हुई लगे या तले असमान रूप से घिसने लगें, तो पहले भी बदल दें।", "q1": "रनिंग शूज़ का मेरा पसंदीदा ब्रांड कौन-सा है?", "q2": "दौड़ने वाले जूतों में मुझे कौन-सा ब्रांड सबसे ज़्यादा पसंद है?", "distractor_sentence": "मेरा भाई दौड़ने के लिए सिर्फ़ Asics पर भरोसा करता है, पर मैंने कभी नहीं पहने।"}, "hg": {"memory_turn": "Ab tak running shoes ke liye Nike mera favourite brand raha hai. Main apne abhi wale shoes ka hi model phir se lena chahti hoon. Woh kahan milenge, ya unse milte-julte models kaunse hain?", "assistant_reply": "Agar wahi model band ho gaya hai, toh brand ki website aur bade sports stores aksar usi naam se naye number wala version rakhte hain. Running store pe fit match karke bhi dekh sakti hain.", "followup_turn": "Running shoes kitne time baad badalne chahiye?", "followup_reply": "Usually har 500 se 800 kilometre ke baad. Agar cushioning dabi hui lage ya soles uneven ghisne lagein, toh pehle bhi badal dein.", "q1": "Running shoes ka mera favourite brand kaunsa hai?", "q2": "Daudne wale shoes mein mujhe kaunsa brand sabse zyada pasand hai?", "distractor_sentence": "Mera bhai running ke liye sirf Asics pe bharosa karta hai, par maine kabhi nahi pehne."}}, "gold": {"answer_en": "Nike", "slots": [{"name": "brand", "aliases": ["Nike", "नाइकी"]}]}, "distractor": {"value_aliases": ["Asics", "ASICS", "एसिक्स"]}, "source": {"dataset": "LongMemEval", "question_id": "c8c3f81d", "lme_type": "single-session-user"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 3743301637, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "early", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P002", "cand_id": "FC140", "memory_type": "preference", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I'm planning to make some Japanese-style dishes with my favourite rice, Japanese short-grain rice. Any simple recipes that pair well with it, maybe with grilled chicken or vegetables?", "assistant_reply": "Try chicken teriyaki rice bowls, a vegetable donburi with a soy and mirin glaze, or onigiri rice balls with a simple filling. All three take under 40 minutes.", "followup_turn": "How do I keep the rice from getting too sticky?", "followup_reply": "Rinse it several times until the water runs almost clear, use slightly less water than usual, and let it rest covered for ten minutes after cooking.", "q1": "What type of rice is my favourite?", "q2": "Which rice do I like best?"}, "hi": {"memory_turn": "मैं अपने पसंदीदा जापानी शॉर्ट-ग्रेन चावल से कुछ जापानी स्टाइल की डिशेज़ बनाने की सोच रहा हूँ। उसके साथ अच्छी लगने वाली कोई आसान रेसिपी बताएँगे, शायद ग्रिल्ड चिकन या सब्ज़ियों के साथ?", "assistant_reply": "चिकन तेरियाकी राइस बाउल, सोया और मिरिन ग्लेज़ वाला वेजिटेबल डोनबुरी, या आसान फ़िलिंग वाले ओनिगिरी राइस बॉल्स आज़माइए। तीनों 40 मिनट से कम में बन जाते हैं।", "followup_turn": "चावल को ज़्यादा चिपचिपा होने से कैसे बचाऊँ?", "followup_reply": "पानी लगभग साफ़ होने तक उसे कई बार धोइए, सामान्य से थोड़ा कम पानी डालिए, और पकने के बाद दस मिनट ढककर रहने दीजिए।", "q1": "मुझे कौन-सा चावल सबसे ज़्यादा पसंद है?", "q2": "मेरा पसंदीदा चावल किस तरह का है?"}, "hg": {"memory_turn": "Main apne favourite Japanese short-grain rice se kuch Japanese style dishes banane ki soch raha hoon. Uske saath acchi lagne wali koi easy recipe batayenge, shayad grilled chicken ya veggies ke saath?", "assistant_reply": "Chicken teriyaki rice bowl, soy aur mirin glaze wala vegetable donburi, ya simple filling wale onigiri rice balls try kijiye. Teeno 40 minute se kam mein ban jaate hain.", "followup_turn": "Rice ko zyada sticky hone se kaise bachaun?", "followup_reply": "Paani lagbhag saaf hone tak use kai baar dhoiye, normal se thoda kam paani daaliye, aur pakne ke baad das minute dhak kar rehne dijiye.", "q1": "Mujhe kaunsa rice sabse zyada pasand hai?", "q2": "Mera favourite rice kis type ka hai?"}}, "gold": {"answer_en": "Japanese short-grain rice", "slots": [{"name": "rice", "aliases": ["Japanese short-grain rice", "Japanese short grain", "short-grain", "short grain", "Japanese rice", "जापानी शॉर्ट-ग्रेन", "शॉर्ट-ग्रेन", "जापानी चावल"]}]}, "source": {"dataset": "LongMemEval", "question_id": "60d45044", "lme_type": "single-session-user"}, "topic": "food/cooking", "is_answerable": true, "evidence_position": 5, "filler_seed": 4003791473, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "late", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P003", "cand_id": "PC075", "memory_type": "preference", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I'm trying to learn some advanced settings for video editing in Adobe Premiere Pro, which I really enjoy using. Can you give me some tips on where to start?", "assistant_reply": "Start with three areas: proxy workflows for smooth playback, the colour grading panels, and customising keyboard shortcuts. Mastering these speeds up almost every project.", "followup_turn": "What is a proxy workflow exactly?", "followup_reply": "You create lightweight copies of your high-resolution footage, edit with those for smooth playback, and the software switches back to the original files when you export.", "q1": "Which video editing software do I enjoy using?", "q2": "What's my favourite tool for editing videos?"}, "hi": {"memory_turn": "मैं Adobe Premiere Pro में वीडियो एडिटिंग की कुछ एडवांस्ड सेटिंग्स सीखने की कोशिश कर रही हूँ, मुझे इसे इस्तेमाल करना बहुत पसंद है। शुरुआत कहाँ से करूँ, कुछ टिप्स देंगे?", "assistant_reply": "तीन चीज़ों से शुरू कीजिए: स्मूद प्लेबैक के लिए प्रॉक्सी वर्कफ़्लो, कलर ग्रेडिंग पैनल, और कीबोर्ड शॉर्टकट को अपने हिसाब से सेट करना। इनमें माहिर होने से लगभग हर प्रोजेक्ट तेज़ हो जाता है।", "followup_turn": "प्रॉक्सी वर्कफ़्लो असल में क्या होता है?", "followup_reply": "आप अपनी हाई-रिज़ॉल्यूशन फ़ुटेज की हल्की कॉपियाँ बनाते हैं, स्मूद प्लेबैक के लिए उन्हीं से एडिट करते हैं, और एक्सपोर्ट के समय सॉफ़्टवेयर फिर से ओरिजिनल फ़ाइलें इस्तेमाल करता है।", "q1": "मुझे वीडियो एडिट करने के लिए कौन-सा सॉफ़्टवेयर इस्तेमाल करना पसंद है?", "q2": "वीडियो एडिटिंग के लिए मेरा पसंदीदा टूल कौन-सा है?"}, "hg": {"memory_turn": "Main Adobe Premiere Pro mein video editing ki kuch advanced settings seekhne ki koshish kar rahi hoon, mujhe ise use karna bahut pasand hai. Start kahan se karoon, kuch tips denge?", "assistant_reply": "Teen cheezon se start kijiye: smooth playback ke liye proxy workflow, colour grading panels, aur keyboard shortcuts ko apne hisaab se set karna. Inmein expert hone se lagbhag har project fast ho jaata hai.", "followup_turn": "Proxy workflow actually kya hota hai?", "followup_reply": "Aap apni high-resolution footage ki light copies banate hain, smooth playback ke liye unhi se edit karte hain, aur export ke time software phir se original files use karta hai.", "q1": "Mujhe video edit karne ke liye kaunsa software use karna pasand hai?", "q2": "Video editing ke liye mera favourite tool kaunsa hai?"}}, "gold": {"answer_en": "Adobe Premiere Pro", "slots": [{"name": "software", "aliases": ["Adobe Premiere Pro", "Premiere Pro", "Premiere", "एडोबी प्रीमियर प्रो", "प्रीमियर प्रो"]}]}, "source": {"dataset": "LongMemEval", "question_id": "8a2466db", "lme_type": "single-session-preference"}, "topic": "other", "is_answerable": true, "evidence_position": 9, "filler_seed": 302390493, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "early", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P004", "cand_id": "PC076", "memory_type": "preference", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I'm looking to upgrade my camera flash. Can you recommend some good options that are compatible with my Sony A7R IV?", "assistant_reply": "Good options include the Godox V1 for its round head and fast recycling, the Profoto A10 if budget allows, and the Godox TT685 II as a reliable value pick. Make sure you buy the version for your mount.", "followup_turn": "Do I need TTL support, or is manual flash enough?", "followup_reply": "TTL is very helpful at events where light changes quickly, because the flash meters automatically. For studio or planned shots, manual control gives more consistent results.", "q1": "Which camera brand do I use?", "q2": "What brand is my camera?"}, "hi": {"memory_turn": "मैं अपना कैमरा फ़्लैश अपग्रेड करना चाहता हूँ। मेरे Sony A7R IV के साथ चलने वाले कुछ अच्छे विकल्प बताएँगे?", "assistant_reply": "अच्छे विकल्प हैं: गोल हेड और तेज़ रीसाइक्लिंग के लिए Godox V1, बजट हो तो Profoto A10, और भरोसेमंद किफ़ायती विकल्प के तौर पर Godox TT685 II। ख़रीदते समय अपने माउंट वाला वर्ज़न ज़रूर देखें।", "followup_turn": "क्या मुझे TTL सपोर्ट चाहिए, या मैनुअल फ़्लैश काफ़ी है?", "followup_reply": "जहाँ रोशनी जल्दी-जल्दी बदलती है, जैसे इवेंट्स में, वहाँ TTL बहुत काम आता है क्योंकि फ़्लैश ख़ुद रोशनी नापता है। स्टूडियो या प्लान किए गए शॉट्स में मैनुअल कंट्रोल से ज़्यादा एक जैसे नतीजे मिलते हैं।", "q1": "मैं किस ब्रांड का कैमरा इस्तेमाल करता हूँ?", "q2": "मेरा कैमरा किस कंपनी का है?"}, "hg": {"memory_turn": "Main apna camera flash upgrade karna chahta hoon. Mere Sony A7R IV ke saath chalne wale kuch acche options batayenge?", "assistant_reply": "Acche options hain: round head aur fast recycling ke liye Godox V1, budget ho toh Profoto A10, aur reliable value pick ke taur pe Godox TT685 II. Khareedte time apne mount wala version zaroor dekhein.", "followup_turn": "Kya mujhe TTL support chahiye, ya manual flash kaafi hai?", "followup_reply": "Jahan light jaldi-jaldi badalti hai, jaise events mein, wahan TTL bahut kaam aata hai kyunki flash khud light measure karta hai. Studio ya planned shots mein manual control se zyada consistent results milte hain.", "q1": "Main kis brand ka camera use karta hoon?", "q2": "Mera camera kis company ka hai?"}}, "gold": {"answer_en": "Sony (A7R IV)", "slots": [{"name": "brand", "aliases": ["Sony", "सोनी"]}]}, "source": {"dataset": "LongMemEval", "question_id": "06878be2", "lme_type": "single-session-preference"}, "topic": "hobbies", "is_answerable": true, "evidence_position": 2, "filler_seed": 12467665, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P005", "cand_id": "PC081", "memory_type": "preference", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "What else can I do in the later part of the day? I prefer winding down by 9:30 pm so I can get a good night's sleep.", "assistant_reply": "Some calm options: a short walk after dinner, light stretching, journaling, or reading a paper book. Try to keep screens away for the last half hour before you wind down.", "followup_turn": "Does herbal tea actually help with sleep?", "followup_reply": "For many people, yes. Caffeine-free teas like chamomile can feel relaxing as part of a routine, though the habit itself often matters as much as the tea.", "q1": "By what time do I prefer to wind down at night?", "q2": "What time do I like to start winding down for sleep?"}, "hi": {"memory_turn": "दिन के आख़िरी हिस्से में मैं और क्या कर सकती हूँ? मुझे रात 9:30 बजे तक आराम के मूड में आ जाना पसंद है, ताकि अच्छी नींद आ सके।", "assistant_reply": "कुछ शांत विकल्प: खाने के बाद थोड़ी सैर, हल्की स्ट्रेचिंग, डायरी लिखना, या कोई किताब पढ़ना। सोने की तैयारी से पहले के आधे घंटे में स्क्रीन से दूर रहने की कोशिश करें।", "followup_turn": "क्या हर्बल चाय सच में नींद में मदद करती है?", "followup_reply": "कई लोगों के लिए हाँ। कैमोमाइल जैसी बिना कैफ़ीन वाली चाय रूटीन का हिस्सा बनकर सुकून देती है, हालाँकि अक्सर आदत उतनी ही मायने रखती है जितनी चाय।", "q1": "रात को मैं कितने बजे तक आराम के मूड में आ जाना पसंद करती हूँ?", "q2": "सोने से पहले मुझे किस समय तक रिलैक्स होना शुरू कर देना पसंद है?"}, "hg": {"memory_turn": "Din ke last part mein main aur kya kar sakti hoon? Mujhe raat 9:30 baje tak wind down kar lena pasand hai, taaki acchi neend aa sake.", "assistant_reply": "Kuch calm options: dinner ke baad thodi walk, light stretching, journaling, ya koi book padhna. Wind down se pehle ke aadhe ghante mein screens se door rehne ki koshish karein.", "followup_turn": "Kya herbal tea sach mein neend mein help karti hai?", "followup_reply": "Kai logon ke liye haan. Chamomile jaisi caffeine-free tea routine ka hissa bankar relax feel karati hai, halaanki aksar habit utni hi matter karti hai jitni tea.", "q1": "Raat ko main kitne baje tak wind down karna pasand karti hoon?", "q2": "Sone se pehle mujhe kis time tak relax hona shuru kar dena pasand hai?"}}, "gold": {"answer_en": "9:30 pm", "slots": [{"name": "time", "aliases": ["9:30", "9.30", "half past nine", "साढ़े नौ", "sadhe nau", "saade nau"]}]}, "source": {"dataset": "LongMemEval", "question_id": "195a1a1b", "lme_type": "single-session-preference"}, "topic": "other", "is_answerable": true, "evidence_position": 5, "filler_seed": 3815814950, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "late", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P006", "cand_id": "PC085", "memory_type": "preference", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I've been using basil and mint in my cooking a lot lately, and I've even harvested some cherry tomatoes from my garden. Which companion plants could help my tomatoes grow better?", "assistant_reply": "Marigolds are a classic companion because they help deter some pests. Chives and garlic planted nearby can also help, while borage attracts pollinators to the tomato flowers.", "followup_turn": "How often should I water the tomatoes in hot weather?", "followup_reply": "Deep watering every two to three days is usually better than a little every day. Water at the base in the morning so the leaves stay dry.", "q1": "Which herbs have I been using in my cooking lately?", "q2": "What herbs do I like to cook with these days?", "distractor_sentence": "My mom, on the other hand, mostly cooks with coriander and curry leaves."}, "hi": {"memory_turn": "आजकल मैं अपने खाने में बेसिल और पुदीना बहुत इस्तेमाल कर रहा हूँ, और मैंने अपने बग़ीचे से कुछ चेरी टमाटर भी तोड़े हैं। टमाटर बेहतर उगें, इसके लिए साथ में कौन-से पौधे लगाऊँ?", "assistant_reply": "गेंदा एक पुराना और अच्छा साथी पौधा है, क्योंकि यह कुछ कीड़ों को दूर रखता है। पास में चाइव्स और लहसुन लगाने से भी फ़ायदा होता है, और बोरेज टमाटर के फूलों तक परागण करने वाले कीड़ों को खींचता है।", "followup_turn": "गर्मी में टमाटर को कितनी बार पानी देना चाहिए?", "followup_reply": "रोज़ थोड़ा-थोड़ा देने के बजाय हर दो-तीन दिन में अच्छी तरह गहरा पानी देना बेहतर है। सुबह जड़ के पास पानी दें ताकि पत्तियाँ सूखी रहें।", "q1": "आजकल मैं अपने खाने में कौन-सी जड़ी-बूटियाँ इस्तेमाल कर रहा हूँ?", "q2": "इन दिनों मुझे किन हर्ब्स के साथ खाना बनाना पसंद है?", "distractor_sentence": "वहीं मेरी मम्मी ज़्यादातर धनिया और करी पत्ते से खाना बनाती हैं।"}, "hg": {"memory_turn": "Aajkal main apne khane mein basil aur pudina bahut use kar raha hoon, aur maine apne garden se kuch cherry tomatoes bhi tode hain. Tomatoes better ugein, iske liye saath mein kaunse plants lagaun?", "assistant_reply": "Marigold ek classic companion plant hai, kyunki yeh kuch keedon ko door rakhta hai. Paas mein chives aur lehsun lagane se bhi fayda hota hai, aur borage tomato ke phoolon tak pollinators ko kheenchta hai.", "followup_turn": "Garmi mein tomatoes ko kitni baar paani dena chahiye?", "followup_reply": "Roz thoda-thoda dene ke bajaye har do-teen din mein acche se deep watering better hai. Subah jadh ke paas paani dein taaki patte sookhe rahein.", "q1": "Aajkal main apne khane mein kaunse herbs use kar raha hoon?", "q2": "In dino mujhe kin herbs ke saath khana banana pasand hai?", "distractor_sentence": "Wahin meri mummy zyadatar dhaniya aur kadi patta se khana banati hain."}}, "gold": {"answer_en": "basil and mint", "slots": [{"name": "herb_1", "aliases": ["basil", "बेसिल", "तुलसी"]}, {"name": "herb_2", "aliases": ["mint", "पुदीना", "pudina", "मिंट"]}]}, "distractor": {"value_aliases": ["coriander", "धनिया", "dhaniya", "curry leaves", "करी पत्ते", "करी पत्ता", "kadi patta"]}, "source": {"dataset": "LongMemEval", "question_id": "06f04340", "lme_type": "single-session-preference"}, "topic": "hobbies", "is_answerable": true, "evidence_position": 9, "filler_seed": 2107664876, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "early", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P007", "cand_id": "PC091", "memory_type": "preference", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I'm looking for some mid-century modern design inspiration for a new bedroom dresser. Do you have any recommendations for websites or designers I should check out?", "assistant_reply": "Browse interior design websites and Pinterest boards, look through furniture stores' catalogues, and check second-hand marketplaces for original vintage pieces at good prices.", "followup_turn": "Would walnut or teak look better in a small bedroom?", "followup_reply": "Teak is slightly lighter and warmer, so it can make a small room feel more open. Walnut is darker and richer, which works well if your walls are light.", "q1": "Which furniture design style do I like?", "q2": "What design style am I looking at for my bedroom furniture?", "distractor_sentence": "My roommate is all about Scandinavian minimalism, though."}, "hi": {"memory_turn": "मैं अपने बेडरूम के नए ड्रेसर के लिए मिड-सेंचुरी मॉडर्न डिज़ाइन से कुछ प्रेरणा ढूँढ रही हूँ। कोई वेबसाइट या डिज़ाइनर बताएँगे जिन्हें मैं देखूँ?", "assistant_reply": "इंटीरियर डिज़ाइन की वेबसाइट और Pinterest बोर्ड देखिए, फ़र्नीचर दुकानों के कैटलॉग पलटिए, और अच्छे दाम पर पुराने विंटेज पीस के लिए सेकंड-हैंड मार्केटप्लेस भी देखिए।", "followup_turn": "छोटे बेडरूम में अखरोट की लकड़ी अच्छी लगेगी या सागौन?", "followup_reply": "सागौन थोड़ा हल्का और गर्म रंग का होता है, इसलिए छोटा कमरा ज़्यादा खुला लगता है। अखरोट गहरा और भरपूर रंग का होता है, जो हल्की दीवारों के साथ अच्छा लगता है।", "q1": "मुझे फ़र्नीचर का कौन-सा डिज़ाइन स्टाइल पसंद है?", "q2": "अपने बेडरूम के फ़र्नीचर के लिए मैं कौन-सा डिज़ाइन स्टाइल देख रही हूँ?", "distractor_sentence": "हालाँकि मेरी रूममेट को स्कैंडिनेवियन मिनिमलिज़्म ही पसंद है।"}, "hg": {"memory_turn": "Main apne bedroom ke naye dresser ke liye mid-century modern design se kuch inspiration dhoondh rahi hoon. Koi websites ya designers batayenge jinhe main dekhoon?", "assistant_reply": "Interior design websites aur Pinterest boards dekhiye, furniture stores ke catalogues check kijiye, aur acche price pe original vintage pieces ke liye second-hand marketplaces bhi dekhiye.", "followup_turn": "Chhote bedroom mein walnut accha lagega ya teak?", "followup_reply": "Teak thoda light aur warm hota hai, isliye chhota room zyada open lagta hai. Walnut dark aur rich hota hai, jo light walls ke saath accha lagta hai.", "q1": "Mujhe furniture ka kaunsa design style pasand hai?", "q2": "Apne bedroom furniture ke liye main kaunsa design style dekh rahi hoon?", "distractor_sentence": "Halaanki meri roommate ko Scandinavian minimalism hi pasand hai."}}, "gold": {"answer_en": "mid-century modern", "slots": [{"name": "style", "aliases": ["mid-century modern", "mid century modern", "mid-century", "मिड-सेंचुरी मॉडर्न", "मिड सेंचुरी मॉडर्न", "मिड-सेंचुरी"]}]}, "distractor": {"value_aliases": ["Scandinavian", "minimalism", "स्कैंडिनेवियन", "मिनिमलिज़्म"]}, "source": {"dataset": "LongMemEval", "question_id": "57f827a0", "lme_type": "single-session-preference"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 1949295071, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "middle", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P008", "cand_id": "PC092", "memory_type": "preference", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I'm considering upgrading from my Fender Stratocaster to a Gibson Les Paul. Can you tell me the main differences between these two guitars?", "assistant_reply": "The biggest differences are sound and feel. One has single-coil pickups with a bright, clear tone and a lighter body; the other has humbuckers with a thicker, warmer sound and a heavier body.", "followup_turn": "Is the heavier weight a problem for long gigs?", "followup_reply": "It can be. Many players use a wide padded strap to spread the load, and some models have chambered bodies that weigh noticeably less.", "q1": "Which guitar am I thinking of upgrading to?", "q2": "What guitar do I want to switch to?"}, "hi": {"memory_turn": "मैं अपने Fender Stratocaster से Gibson Les Paul पर अपग्रेड करने की सोच रहा हूँ। इन दोनों गिटार में मुख्य फ़र्क़ क्या हैं?", "assistant_reply": "सबसे बड़ा फ़र्क़ आवाज़ और पकड़ में है। एक में सिंगल-कॉइल पिकअप होते हैं, जिनकी आवाज़ तीखी और साफ़ होती है और बॉडी हल्की होती है; दूसरे में हम्बकर होते हैं, जिनकी आवाज़ भारी और गर्म होती है और बॉडी वज़नदार होती है।", "followup_turn": "क्या लंबे शो में ज़्यादा वज़न परेशानी बनता है?", "followup_reply": "बन सकता है। कई गिटारिस्ट वज़न बाँटने के लिए चौड़ा गद्देदार स्ट्रैप लगाते हैं, और कुछ मॉडल की बॉडी अंदर से खोखली होती है जिससे वे काफ़ी हल्के होते हैं।", "q1": "मैं किस गिटार पर अपग्रेड करने की सोच रहा हूँ?", "q2": "मैं अब कौन-सा गिटार लेना चाहता हूँ?"}, "hg": {"memory_turn": "Main apne Fender Stratocaster se Gibson Les Paul pe upgrade karne ki soch raha hoon. In dono guitars mein main differences kya hain?", "assistant_reply": "Sabse bada difference sound aur feel mein hai. Ek mein single-coil pickups hote hain, jinki sound bright aur clear hoti hai aur body light hoti hai; doosre mein humbuckers hote hain, jinki sound thick aur warm hoti hai aur body heavy hoti hai.", "followup_turn": "Kya lambe gigs mein zyada weight problem banta hai?", "followup_reply": "Ban sakta hai. Kai players weight distribute karne ke liye wide padded strap use karte hain, aur kuch models ki body andar se chambered hoti hai jisse woh kaafi light hote hain.", "q1": "Main kis guitar pe upgrade karne ki soch raha hoon?", "q2": "Main ab kaunsa guitar lena chahta hoon?"}}, "gold": {"answer_en": "Gibson Les Paul", "slots": [{"name": "guitar", "aliases": ["Les Paul", "लेस पॉल"]}]}, "near_miss_aliases": ["Stratocaster", "Strat", "स्ट्रैटोकास्टर"], "source": {"dataset": "LongMemEval", "question_id": "95228167", "lme_type": "single-session-preference"}, "topic": "other", "is_answerable": true, "evidence_position": 5, "filler_seed": 3781988423, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P009", "cand_id": "PC093", "memory_type": "preference", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I'm trying to cut down on sugar and save money, so I've started making my own flavoured creamer with almond milk, vanilla extract and honey. Any tips to make it even healthier?", "assistant_reply": "You could use a little less honey and add cinnamon or cardamom for sweetness without sugar. A pinch of salt also boosts flavour, and a glass jar keeps it fresh for about five days.", "followup_turn": "Can I make a chocolate version too?", "followup_reply": "Yes, whisk in a spoonful of unsweetened cocoa powder while it is slightly warm. Shake the jar before each use because cocoa tends to settle.", "q1": "Which milk do I use for my homemade creamer?", "q2": "What kind of milk goes into the creamer I make at home?"}, "hi": {"memory_turn": "मैं चीनी कम करना और पैसे बचाना चाहती हूँ, इसलिए मैंने बादाम के दूध, वनीला एक्सट्रैक्ट और शहद से अपना फ़्लेवर्ड क्रीमर ख़ुद बनाना शुरू किया है। इसे और हेल्दी बनाने के लिए कुछ टिप्स देंगे?", "assistant_reply": "शहद थोड़ा कम करके मिठास के लिए दालचीनी या इलायची डाल सकती हैं। एक चुटकी नमक से स्वाद बढ़ता है, और काँच के जार में रखने से यह लगभग पाँच दिन ताज़ा रहता है।", "followup_turn": "क्या मैं इसका चॉकलेट वाला वर्ज़न भी बना सकती हूँ?", "followup_reply": "हाँ, हल्का गुनगुना रहते हुए एक चम्मच बिना चीनी वाला कोको पाउडर फेंट दीजिए। हर बार इस्तेमाल से पहले जार हिला लें, क्योंकि कोको नीचे बैठ जाता है।", "q1": "अपना घर का क्रीमर बनाने के लिए मैं कौन-सा दूध इस्तेमाल करती हूँ?", "q2": "घर पर जो क्रीमर मैं बनाती हूँ, उसमें किस तरह का दूध डलता है?"}, "hg": {"memory_turn": "Main sugar kam karna aur paise bachana chahti hoon, isliye maine almond milk, vanilla extract aur honey se apna flavoured creamer khud banana shuru kiya hai. Ise aur healthy banane ke liye kuch tips denge?", "assistant_reply": "Honey thoda kam karke sweetness ke liye cinnamon ya elaichi daal sakti hain. Ek chutki namak se taste badhta hai, aur glass jar mein rakhne se yeh lagbhag paanch din fresh rehta hai.", "followup_turn": "Kya main iska chocolate wala version bhi bana sakti hoon?", "followup_reply": "Haan, halka garam rehte hue ek chammach unsweetened cocoa powder mila dijiye. Har baar use karne se pehle jar hila lein, kyunki cocoa neeche baith jaata hai.", "q1": "Apna ghar ka creamer banane ke liye main kaunsa milk use karti hoon?", "q2": "Ghar pe jo creamer main banati hoon, usmein kis type ka milk jaata hai?"}}, "gold": {"answer_en": "almond milk", "slots": [{"name": "milk", "aliases": ["almond milk", "almond", "बादाम के दूध", "बादाम का दूध", "बादाम दूध", "आमंड मिल्क", "badam ka doodh", "badam milk", "अल्मंड मिल्क", "बादाम"]}]}, "source": {"dataset": "LongMemEval", "question_id": "505af2f5", "lme_type": "single-session-preference"}, "topic": "other", "is_answerable": true, "evidence_position": 9, "filler_seed": 683709250, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P010", "cand_id": "PC100", "memory_type": "preference", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I've been watching a lot of documentaries lately, mostly on Netflix. Can you recommend more documentary series like Our Planet and Free Solo, which I just finished?", "assistant_reply": "If you liked those, try Planet Earth II for stunning nature footage, Meru for another climbing story, and Blue Planet II for ocean life.", "followup_turn": "Is Meru as intense as Free Solo?", "followup_reply": "In a different way. Meru follows a team on a dangerous climb over several years, so it is more about persistence and friendship than one single climb.", "q1": "Where do I usually watch documentaries?", "q2": "Which streaming service do I mostly use for documentaries?"}, "hi": {"memory_turn": "आजकल मैं बहुत सारी डॉक्यूमेंट्री देख रहा हूँ, ज़्यादातर Netflix पर। Our Planet और Free Solo जैसी कुछ और डॉक्यूमेंट्री सीरीज़ बताएँगे? ये दोनों मैंने अभी-अभी ख़त्म की हैं।", "assistant_reply": "अगर वे पसंद आईं, तो शानदार प्रकृति के दृश्यों के लिए Planet Earth II, चढ़ाई पर बनी फ़िल्म Meru, और समुद्री जीवन के लिए Blue Planet II देखिए।", "followup_turn": "क्या Meru भी Free Solo जितनी रोमांचक है?", "followup_reply": "अलग तरह से। Meru में कुछ पर्वतारोहियों की कई साल लंबी ख़तरनाक चढ़ाई दिखाई गई है, इसलिए यह एक चढ़ाई से ज़्यादा लगन और दोस्ती की कहानी है।", "q1": "मैं आमतौर पर डॉक्यूमेंट्री कहाँ देखता हूँ?", "q2": "डॉक्यूमेंट्री देखने के लिए मैं ज़्यादातर कौन-सी स्ट्रीमिंग सर्विस इस्तेमाल करता हूँ?"}, "hg": {"memory_turn": "Aajkal main bahut saari documentaries dekh raha hoon, zyadatar Netflix pe. Our Planet aur Free Solo jaisi kuch aur documentary series batayenge? Yeh dono maine abhi-abhi khatam ki hain.", "assistant_reply": "Agar woh pasand aayi, toh amazing nature visuals ke liye Planet Earth II, climbing pe bani film Meru, aur ocean life ke liye Blue Planet II dekhiye.", "followup_turn": "Kya Meru bhi Free Solo jitni thrilling hai?", "followup_reply": "Alag tarah se. Meru mein kuch climbers ki kai saal lambi khatarnaak chadhai dikhayi gayi hai, isliye yeh ek climb se zyada persistence aur dosti ki kahani hai.", "q1": "Main usually documentaries kahan dekhta hoon?", "q2": "Documentaries dekhne ke liye main zyadatar kaunsi streaming service use karta hoon?"}}, "gold": {"answer_en": "Netflix", "slots": [{"name": "service", "aliases": ["Netflix", "नेटफ़्लिक्स", "नेटफ्लिक्स"]}]}, "source": {"dataset": "LongMemEval", "question_id": "b0479f84", "lme_type": "single-session-preference"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 580460840, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "late", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "E001", "cand_id": "EC001", "memory_type": "event", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I'm thinking of getting my car detailed soon. Any tips on choosing a good detailer? By the way, I got my car serviced for the first time on March 15th, and it was a great experience.", "assistant_reply": "Glad the service went well! For detailing, check recent reviews with photos, ask exactly what each package includes, and confirm they use pH-neutral products that are safe for your paint.", "followup_turn": "Is ceramic coating worth paying extra for?", "followup_reply": "It can be if you keep the car for several years. It makes washing easier and protects the paint, but it is not scratch-proof and needs proper preparation to last.", "q1": "When did I get my car serviced for the first time?", "q2": "On what date was my car's first service?", "distractor_sentence": "My dad got his car serviced on April 2nd, and it took the whole day."}, "hi": {"memory_turn": "मैं जल्द ही अपनी कार की डिटेलिंग करवाने की सोच रही हूँ। अच्छा डिटेलर चुनने के लिए कोई टिप्स? वैसे, मैंने 15 मार्च को पहली बार अपनी कार की सर्विस करवाई थी, और अनुभव बहुत अच्छा रहा।", "assistant_reply": "अच्छा लगा कि सर्विस ठीक रही! डिटेलिंग के लिए फ़ोटो वाले हाल के रिव्यू देखिए, हर पैकेज में ठीक-ठीक क्या शामिल है यह पूछिए, और पक्का कीजिए कि वे पेंट के लिए सुरक्षित pH-न्यूट्रल प्रोडक्ट इस्तेमाल करते हैं।", "followup_turn": "क्या सिरेमिक कोटिंग के लिए ज़्यादा पैसे देना ठीक है?", "followup_reply": "अगर आप कार कई साल रखेंगी तो हाँ। इससे धोना आसान होता है और पेंट सुरक्षित रहता है, लेकिन यह स्क्रैच-प्रूफ़ नहीं है और टिकाऊ होने के लिए सही तैयारी ज़रूरी है।", "q1": "मैंने पहली बार अपनी कार की सर्विस कब करवाई थी?", "q2": "मेरी कार की पहली सर्विस किस तारीख़ को हुई थी?", "distractor_sentence": "मेरे पापा ने 2 अप्रैल को अपनी कार की सर्विस करवाई थी, और पूरा दिन लग गया था।"}, "hg": {"memory_turn": "Main jaldi hi apni car ki detailing karwane ki soch rahi hoon. Accha detailer choose karne ke liye koi tips? Waise, maine 15 March ko pehli baar apni car ki service karwayi thi, aur experience bahut accha raha.", "assistant_reply": "Accha laga ki service theek rahi! Detailing ke liye photos wale recent reviews dekhiye, har package mein exactly kya include hai yeh poochhiye, aur confirm kijiye ki woh paint ke liye safe pH-neutral products use karte hain.", "followup_turn": "Kya ceramic coating ke liye extra paise dena worth it hai?", "followup_reply": "Agar aap car kai saal rakhengi toh haan. Isse washing easy hoti hai aur paint protected rehta hai, lekin yeh scratch-proof nahi hai aur tikne ke liye sahi preparation zaroori hai.", "q1": "Maine pehli baar apni car ki service kab karwayi thi?", "q2": "Meri car ki pehli service kis date ko hui thi?", "distractor_sentence": "Mere papa ne 2 April ko apni car ki service karwayi thi, aur poora din lag gaya tha."}}, "gold": {"answer_en": "March 15", "date": {"day": 15, "month": 3}, "slots": [{"name": "date", "aliases": ["March 15", "March 15th", "15 March", "15th March", "15th of March", "Mar 15", "15 Mar", "15 मार्च"]}]}, "distractor": {"date": {"day": 2, "month": 4}, "value_aliases": ["April 2", "April 2nd", "2 April", "2nd April", "2nd of April", "Apr 2", "2 Apr", "2 अप्रैल"]}, "source": {"dataset": "LongMemEval", "question_id": "gpt4_2655b836", "lme_type": "temporal-reasoning"}, "topic": "vehicles", "is_answerable": true, "evidence_position": 2, "filler_seed": 2791523688, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "middle", "abl_short": true, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E002", "cand_id": "EC003", "memory_type": "event", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I got a new Samsung Galaxy S22 from the electronics store at the mall on February 20th, and I'm loving it so far. What are some must-have settings I should turn on?", "assistant_reply": "A few useful ones: enable the adaptive refresh rate for smoother scrolling, set up Find My Mobile in case it gets lost, and turn on battery protection to limit charging to 85 percent.", "followup_turn": "Should I get a case with a built-in screen protector?", "followup_reply": "It is a good choice if you want full protection in one product. If you prefer a slimmer phone, a separate tempered glass protector with a thin case works just as well.", "q1": "When did I get my Samsung Galaxy S22?", "q2": "On what date did I buy my new Galaxy S22?"}, "hi": {"memory_turn": "मैंने 20 फ़रवरी को मॉल के इलेक्ट्रॉनिक्स स्टोर से नया Samsung Galaxy S22 लिया, और अब तक मुझे यह बहुत पसंद आ रहा है। कौन-सी सेटिंग्स ज़रूर ऑन करनी चाहिए?", "assistant_reply": "कुछ काम की सेटिंग्स: स्मूद स्क्रॉलिंग के लिए अडैप्टिव रिफ़्रेश रेट ऑन करें, खो जाने की स्थिति के लिए Find My Mobile सेट करें, और चार्जिंग को 85 प्रतिशत तक सीमित करने के लिए बैटरी प्रोटेक्शन ऑन करें।", "followup_turn": "क्या बिल्ट-इन स्क्रीन प्रोटेक्टर वाला कवर लेना चाहिए?", "followup_reply": "अगर एक ही चीज़ में पूरी सुरक्षा चाहिए तो यह अच्छा विकल्प है। अगर फ़ोन पतला रखना पसंद है, तो अलग टेम्पर्ड ग्लास के साथ पतला कवर भी उतना ही अच्छा काम करता है।", "q1": "मैंने अपना Samsung Galaxy S22 कब लिया था?", "q2": "मैंने नया Galaxy S22 किस तारीख़ को ख़रीदा था?"}, "hg": {"memory_turn": "Maine 20 February ko mall ke electronics store se naya Samsung Galaxy S22 liya, aur ab tak mujhe yeh bahut pasand aa raha hai. Kaunsi settings zaroor on karni chahiye?", "assistant_reply": "Kuch kaam ki settings: smooth scrolling ke liye adaptive refresh rate on karein, kho jaane ki situation ke liye Find My Mobile set karein, aur charging ko 85 percent tak limit karne ke liye battery protection on karein.", "followup_turn": "Kya built-in screen protector wala cover lena chahiye?", "followup_reply": "Agar ek hi cheez mein full protection chahiye toh yeh accha option hai. Agar phone slim rakhna pasand hai, toh alag tempered glass ke saath patla cover bhi utna hi accha kaam karta hai.", "q1": "Maine apna Samsung Galaxy S22 kab liya tha?", "q2": "Maine naya Galaxy S22 kis date ko khareeda tha?"}}, "gold": {"answer_en": "February 20", "date": {"day": 20, "month": 2}, "slots": [{"name": "date", "aliases": ["February 20", "February 20th", "20 February", "20th February", "20th of February", "Feb 20", "20 Feb", "20 फ़रवरी", "20 फरवरी"]}]}, "source": {"dataset": "LongMemEval", "question_id": "gpt4_2312f94c", "lme_type": "temporal-reasoning"}, "topic": "shopping/home", "is_answerable": true, "evidence_position": 5, "filler_seed": 1908762337, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "late", "abl_short": false, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E003", "cand_id": "EC005", "memory_type": "event", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I attended a workshop on Effective Communication in the Workplace on January 10th, and it really helped my active listening and conflict resolution skills. How can I keep practising these at work?", "assistant_reply": "Try summarising what a colleague said before you respond, ask one clarifying question in each meeting, and when a disagreement starts, name the shared goal before discussing solutions.", "followup_turn": "What if a colleague keeps interrupting me in meetings?", "followup_reply": "Stay calm and say something like: let me finish this point, then I would love to hear yours. If it continues, raise it with them privately rather than in front of the team.", "q1": "When did I attend the workshop on Effective Communication in the Workplace?", "q2": "On what date was the communication workshop I went to?"}, "hi": {"memory_turn": "मैंने 10 जनवरी को 'कार्यस्थल पर प्रभावी संवाद' पर एक वर्कशॉप में हिस्सा लिया, और इससे ध्यान से सुनने और विवाद सुलझाने की मेरी स्किल्स में बहुत मदद मिली। ऑफ़िस में इनकी प्रैक्टिस कैसे जारी रखूँ?", "assistant_reply": "जवाब देने से पहले सहकर्मी की बात को संक्षेप में दोहराइए, हर मीटिंग में कम से कम एक स्पष्ट करने वाला सवाल पूछिए, और जब असहमति शुरू हो तो समाधान पर बात करने से पहले साझा लक्ष्य बताइए।", "followup_turn": "अगर कोई सहकर्मी मीटिंग में बार-बार मुझे बीच में टोके तो क्या करूँ?", "followup_reply": "शांत रहिए और कुछ ऐसा कहिए: मुझे यह बात पूरी करने दीजिए, फिर मैं आपकी बात ज़रूर सुनना चाहूँगी। अगर यह जारी रहे, तो टीम के सामने नहीं बल्कि अकेले में उनसे बात कीजिए।", "q1": "मैं 'कार्यस्थल पर प्रभावी संवाद' वाली वर्कशॉप में कब गई थी?", "q2": "जिस कम्युनिकेशन वर्कशॉप में मैं गई थी, वह किस तारीख़ को थी?"}, "hg": {"memory_turn": "Maine 10 January ko 'Effective Communication in the Workplace' pe ek workshop attend ki, aur isse meri active listening aur conflict resolution skills mein bahut help mili. Office mein inki practice kaise continue karoon?", "assistant_reply": "Jawab dene se pehle colleague ki baat ko short mein repeat kijiye, har meeting mein kam se kam ek clarifying question poochhiye, aur jab disagreement shuru ho toh solutions pe baat karne se pehle common goal bataiye.", "followup_turn": "Agar koi colleague meeting mein baar-baar mujhe beech mein toke toh kya karoon?", "followup_reply": "Calm rahiye aur kuch aisa kahiye: mujhe yeh point poora karne dijiye, phir main aapki baat zaroor sunna chahungi. Agar yeh continue ho, toh team ke saamne nahi balki akele mein unse baat kijiye.", "q1": "Main 'Effective Communication in the Workplace' wali workshop mein kab gayi thi?", "q2": "Jis communication workshop mein main gayi thi, woh kis date ko thi?"}}, "gold": {"answer_en": "January 10", "date": {"day": 10, "month": 1}, "slots": [{"name": "date", "aliases": ["January 10", "January 10th", "10 January", "10th January", "10th of January", "Jan 10", "10 Jan", "10 जनवरी"]}]}, "source": {"dataset": "LongMemEval", "question_id": "0bb5a684", "lme_type": "temporal-reasoning"}, "topic": "other", "is_answerable": true, "evidence_position": 9, "filler_seed": 4232278006, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "late", "abl_short": false, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E004", "cand_id": "EC012", "memory_type": "event", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I attended the Holi celebration at my local temple on February 26th and had a blast throwing colours and learning about its significance. Can you tell me more about the story behind Holi?", "assistant_reply": "Holi is linked to the legend of Prahlad and Holika, which celebrates devotion and the victory of good over evil. It also marks the arrival of spring, which is why colours are such a big part of it.", "followup_turn": "How do I get the colour stains off my skin?", "followup_reply": "Wash with lukewarm water and a mild soap, then gently rub in some coconut oil or a besan and curd paste. Avoid scrubbing hard; the remaining colour fades in a day or two.", "q1": "When did I attend the Holi celebration at my temple?", "q2": "On what date did I go to the Holi celebration at the local temple?", "distractor_sentence": "My cousin went to a Holi party at her college on March 1st."}, "hi": {"memory_turn": "मैं 26 फ़रवरी को अपने यहाँ के मंदिर में होली के उत्सव में गया था, रंग खेलने और इसका महत्व जानने में बहुत मज़ा आया। होली के पीछे की कहानी के बारे में और बताएँगे?", "assistant_reply": "होली प्रह्लाद और होलिका की कथा से जुड़ी है, जो भक्ति और बुराई पर अच्छाई की जीत का उत्सव है। यह वसंत के आने का भी प्रतीक है, इसीलिए इसमें रंगों का इतना महत्व है।", "followup_turn": "त्वचा से रंग के दाग़ कैसे छुड़ाऊँ?", "followup_reply": "गुनगुने पानी और हल्के साबुन से धोइए, फिर धीरे-धीरे नारियल तेल या बेसन और दही का लेप लगाइए। ज़ोर से मत रगड़िए; बचा हुआ रंग एक-दो दिन में हल्का हो जाएगा।", "q1": "मैं अपने यहाँ के मंदिर में होली के उत्सव में कब गया था?", "q2": "मंदिर में होली के कार्यक्रम में मैं किस तारीख़ को गया था?", "distractor_sentence": "मेरी कज़िन 1 मार्च को अपने कॉलेज की होली पार्टी में गई थी।"}, "hg": {"memory_turn": "Main 26 February ko apne local temple mein Holi celebration mein gaya tha, rang khelne aur iski significance jaanne mein bahut maza aaya. Holi ke peeche ki kahani ke baare mein aur batayenge?", "assistant_reply": "Holi Prahlad aur Holika ki katha se judi hai, jo bhakti aur buraai pe acchai ki jeet ka celebration hai. Yeh spring ke aane ka bhi symbol hai, isiliye ismein rangon ka itna importance hai.", "followup_turn": "Skin se rang ke daag kaise chhudaun?", "followup_reply": "Lukewarm paani aur mild soap se dhoiye, phir dheere-dheere coconut oil ya besan aur dahi ka lep lagaiye. Zor se mat ragdiye; bacha hua rang ek-do din mein halka ho jayega.", "q1": "Main apne local temple mein Holi celebration mein kab gaya tha?", "q2": "Temple mein Holi ke function mein main kis date ko gaya tha?", "distractor_sentence": "Meri cousin 1 March ko apne college ki Holi party mein gayi thi."}}, "gold": {"answer_en": "February 26", "date": {"day": 26, "month": 2}, "slots": [{"name": "date", "aliases": ["February 26", "February 26th", "26 February", "26th February", "26th of February", "Feb 26", "26 Feb", "26 फ़रवरी", "26 फरवरी"]}]}, "distractor": {"date": {"day": 1, "month": 3}, "value_aliases": ["March 1", "March 1st", "1 March", "1st March", "1st of March", "Mar 1", "1 Mar", "1 मार्च"]}, "source": {"dataset": "LongMemEval", "question_id": "2a1811e2", "lme_type": "temporal-reasoning"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 450491531, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "late", "abl_short": false, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E005", "cand_id": "EC017", "memory_type": "event", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I ran 5 kilometres in the Run for the Cure event on October 15th and raised money for breast cancer research. It was an amazing experience! How can I build up to a 10K?", "assistant_reply": "Increase your weekly distance by about ten percent at a time, add one longer slow run each week, and include a rest day after hard sessions. Most runners can go from 5K to 10K in eight to ten weeks.", "followup_turn": "Should I run every day while training?", "followup_reply": "Not necessarily. Three or four runs a week with cross-training like cycling or yoga on other days lowers injury risk and still builds endurance.", "q1": "When did I run in the Run for the Cure event?", "q2": "On what date did I do the Run for the Cure 5K?"}, "hi": {"memory_turn": "मैंने 15 अक्टूबर को 'रन फ़ॉर द क्योर' इवेंट में 5 किलोमीटर दौड़ लगाई और ब्रेस्ट कैंसर रिसर्च के लिए पैसे जुटाए। बहुत शानदार अनुभव था! 10 किलोमीटर तक कैसे पहुँचूँ?", "assistant_reply": "हर हफ़्ते की दूरी एक बार में लगभग दस प्रतिशत बढ़ाइए, हफ़्ते में एक लंबी धीमी दौड़ जोड़िए, और कठिन अभ्यास के बाद एक दिन आराम कीजिए। ज़्यादातर लोग आठ से दस हफ़्तों में 5K से 10K तक पहुँच जाते हैं।", "followup_turn": "क्या ट्रेनिंग के दौरान रोज़ दौड़ना चाहिए?", "followup_reply": "ज़रूरी नहीं। हफ़्ते में तीन-चार बार दौड़ें और बाक़ी दिनों में साइकिलिंग या योग जैसी क्रॉस-ट्रेनिंग करें, इससे चोट का ख़तरा कम होता है और स्टैमिना भी बढ़ता है।", "q1": "मैंने 'रन फ़ॉर द क्योर' इवेंट में कब दौड़ लगाई थी?", "q2": "'रन फ़ॉर द क्योर' की 5K दौड़ मैंने किस तारीख़ को की थी?"}, "hg": {"memory_turn": "Maine 15 October ko 'Run for the Cure' event mein 5 kilometre run kiya aur breast cancer research ke liye paise collect kiye. Bahut amazing experience tha! 10K tak kaise pahunchoon?", "assistant_reply": "Har hafte ki distance ek baar mein lagbhag das percent badhaiye, hafte mein ek long slow run add kijiye, aur hard sessions ke baad ek din rest kijiye. Zyadatar runners aath se das hafton mein 5K se 10K tak pahunch jaate hain.", "followup_turn": "Kya training ke time roz run karna chahiye?", "followup_reply": "Zaroori nahi. Hafte mein teen-chaar runs aur baaki dino mein cycling ya yoga jaisi cross-training karein, isse injury ka risk kam hota hai aur stamina bhi badhta hai.", "q1": "Maine 'Run for the Cure' event mein kab run kiya tha?", "q2": "'Run for the Cure' ka 5K maine kis date ko kiya tha?"}}, "gold": {"answer_en": "October 15", "date": {"day": 15, "month": 10}, "slots": [{"name": "date", "aliases": ["October 15", "October 15th", "15 October", "15th October", "15th of October", "Oct 15", "15 Oct", "15 अक्टूबर", "15 अक्तूबर"]}]}, "source": {"dataset": "LongMemEval", "question_id": "a3838d2b", "lme_type": "temporal-reasoning"}, "topic": "fitness/health", "is_answerable": true, "evidence_position": 5, "filler_seed": 1940062016, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "late", "abl_short": false, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E006", "cand_id": "EC030", "memory_type": "event", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I had a great time celebrating my best friend's 30th birthday party on the 22nd of April. She loved the personalized photo album I got her. Now I want to make a scrapbook for my parents. Any ideas?", "assistant_reply": "Organise it by decades or by family trips, add handwritten captions, and leave a few blank pages at the end so your parents can add their own memories later.", "followup_turn": "What paper should I use so the photos don't fade?", "followup_reply": "Use acid-free, lignin-free paper and photo-safe glue or corners. Keep the scrapbook away from direct sunlight to protect the colours.", "q1": "When was my best friend's 30th birthday party?", "q2": "On what date did I celebrate my best friend turning 30?"}, "hi": {"memory_turn": "22 अप्रैल को मैंने अपनी सबसे अच्छी दोस्त का 30वाँ जन्मदिन मनाया, बहुत मज़ा आया। उसे मेरा दिया हुआ पर्सनलाइज़्ड फ़ोटो एल्बम बहुत पसंद आया। अब मैं अपने मम्मी-पापा के लिए एक स्क्रैपबुक बनाना चाहता हूँ। कोई आइडिया?", "assistant_reply": "इसे दशकों या परिवार की यात्राओं के हिसाब से सजाइए, हाथ से लिखे कैप्शन जोड़िए, और आख़िर में कुछ ख़ाली पन्ने छोड़िए ताकि मम्मी-पापा बाद में अपनी यादें जोड़ सकें।", "followup_turn": "किस काग़ज़ का इस्तेमाल करूँ ताकि फ़ोटो फीकी न पड़ें?", "followup_reply": "एसिड-फ़्री और लिग्निन-फ़्री काग़ज़ और फ़ोटो के लिए सुरक्षित गोंद या कॉर्नर इस्तेमाल कीजिए। रंग बचाने के लिए स्क्रैपबुक को सीधी धूप से दूर रखिए।", "q1": "मेरी सबसे अच्छी दोस्त की 30वें जन्मदिन की पार्टी कब थी?", "q2": "मैंने अपनी बेस्ट फ़्रेंड का 30वाँ जन्मदिन किस तारीख़ को मनाया था?"}, "hg": {"memory_turn": "22 April ko maine apni best friend ka 30th birthday celebrate kiya, bahut maza aaya. Use mera diya hua personalized photo album bahut pasand aaya. Ab main apne mummy-papa ke liye ek scrapbook banana chahta hoon. Koi ideas?", "assistant_reply": "Ise decades ya family trips ke hisaab se arrange kijiye, haath se likhe captions add kijiye, aur end mein kuch blank pages chhodiye taaki mummy-papa baad mein apni yaadein add kar sakein.", "followup_turn": "Kaunsa paper use karoon taaki photos fade na hon?", "followup_reply": "Acid-free aur lignin-free paper aur photo-safe glue ya corners use kijiye. Colours bachane ke liye scrapbook ko direct sunlight se door rakhiye.", "q1": "Meri best friend ki 30th birthday party kab thi?", "q2": "Maine apni best friend ka 30th birthday kis date ko celebrate kiya tha?"}}, "gold": {"answer_en": "April 22", "date": {"day": 22, "month": 4}, "slots": [{"name": "date", "aliases": ["April 22", "April 22nd", "22 April", "22nd April", "22nd of April", "Apr 22", "22 Apr", "22 अप्रैल"]}]}, "source": {"dataset": "LongMemEval", "question_id": "a3045048", "lme_type": "temporal-reasoning"}, "topic": "media/books/games", "is_answerable": true, "evidence_position": 9, "filler_seed": 1586390893, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "early", "abl_short": true, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E007", "cand_id": "EC039", "memory_type": "event", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "My bike's gears were getting stuck, so I took it in for a tune-up on April 20th. Since then it rides much better. How often should I get a tune-up?", "assistant_reply": "For regular riders, a basic tune-up every six months is a good rule, with a full service once a year. Clean and lube the chain yourself every few weeks in between.", "followup_turn": "What's the easiest way to clean the chain at home?", "followup_reply": "Wipe it with a rag and degreaser while turning the pedals backwards, let it dry, then put a drop of chain lube on each link and wipe off the excess.", "q1": "When did I take my bike in for a tune-up?", "q2": "On what date did my bike get its tune-up?"}, "hi": {"memory_turn": "मेरी साइकिल के गियर अटक रहे थे, इसलिए मैं 20 अप्रैल को उसे ट्यून-अप के लिए ले गई थी। तब से वह बहुत अच्छी चल रही है। ट्यून-अप कितने समय बाद करवाना चाहिए?", "assistant_reply": "जो लोग नियमित रूप से चलाते हैं, उनके लिए हर छह महीने में बेसिक ट्यून-अप और साल में एक बार पूरी सर्विस ठीक रहती है। बीच-बीच में हर कुछ हफ़्तों में चेन ख़ुद साफ़ करके तेल डालिए।", "followup_turn": "घर पर चेन साफ़ करने का सबसे आसान तरीक़ा क्या है?", "followup_reply": "पैडल पीछे की ओर घुमाते हुए कपड़े और डीग्रीज़र से चेन पोंछिए, सूखने दीजिए, फिर हर कड़ी पर चेन ल्यूब की एक बूँद डालकर अतिरिक्त पोंछ दीजिए।", "q1": "मैं अपनी साइकिल ट्यून-अप के लिए कब ले गई थी?", "q2": "मेरी साइकिल का ट्यून-अप किस तारीख़ को हुआ था?"}, "hg": {"memory_turn": "Meri cycle ke gears atak rahe the, isliye main 20 April ko use tune-up ke liye le gayi thi. Tab se woh bahut acchi chal rahi hai. Tune-up kitne time baad karwana chahiye?", "assistant_reply": "Regular riders ke liye har chhe mahine mein basic tune-up aur saal mein ek baar full service theek rehti hai. Beech-beech mein har kuch hafton mein chain khud saaf karke lube daaliye.", "followup_turn": "Ghar pe chain saaf karne ka sabse easy tareeka kya hai?", "followup_reply": "Pedals ko peeche ki taraf ghumaate hue kapde aur degreaser se chain ponchhiye, sookhne dijiye, phir har link pe chain lube ki ek boond daal kar extra ponchh dijiye.", "q1": "Main apni cycle tune-up ke liye kab le gayi thi?", "q2": "Meri cycle ka tune-up kis date ko hua tha?"}}, "gold": {"answer_en": "April 20", "date": {"day": 20, "month": 4}, "slots": [{"name": "date", "aliases": ["April 20", "April 20th", "20 April", "20th April", "20th of April", "Apr 20", "20 Apr", "20 अप्रैल"]}]}, "source": {"dataset": "LongMemEval", "question_id": "gpt4_d84a3211", "lme_type": "multi-session"}, "topic": "vehicles", "is_answerable": true, "evidence_position": 2, "filler_seed": 4104915379, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "late", "abl_short": true, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E008", "cand_id": "EC045", "memory_type": "event", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I went on a guided tour at the History Museum on February 24th, and it really sparked my interest in ancient history and art. Can you suggest some beginner-friendly books on ancient civilisations?", "assistant_reply": "Good starting points are 'A Little History of the World' by E. H. Gombrich, 'Sapiens' by Yuval Noah Harari for a broad overview, and 'The Story of Art' by Gombrich for art history.", "followup_turn": "Which one should I read first?", "followup_reply": "Start with 'A Little History of the World'. It is short, easy to read, and gives you a timeline that makes the other two books easier to follow.", "q1": "When did I go on the guided tour at the History Museum?", "q2": "On what date did I visit the History Museum for a guided tour?"}, "hi": {"memory_turn": "मैं 24 फ़रवरी को हिस्ट्री म्यूज़ियम में एक गाइडेड टूर पर गया था, और इससे प्राचीन इतिहास और कला में मेरी रुचि जाग गई। प्राचीन सभ्यताओं पर शुरुआती लोगों के लिए कुछ किताबें सुझाएँगे?", "assistant_reply": "शुरुआत के लिए अच्छी किताबें हैं: ई. एच. गोम्ब्रिच की 'A Little History of the World', व्यापक नज़रिए के लिए युवाल नोआ हरारी की 'Sapiens', और कला इतिहास के लिए गोम्ब्रिच की 'The Story of Art'।", "followup_turn": "इनमें से पहले कौन-सी पढ़ूँ?", "followup_reply": "'A Little History of the World' से शुरू कीजिए। यह छोटी और आसान है, और इससे एक टाइमलाइन बन जाती है जिससे बाक़ी दोनों किताबें समझना आसान हो जाता है।", "q1": "मैं हिस्ट्री म्यूज़ियम में गाइडेड टूर पर कब गया था?", "q2": "मैं गाइडेड टूर के लिए हिस्ट्री म्यूज़ियम किस तारीख़ को गया था?"}, "hg": {"memory_turn": "Main 24 February ko History Museum mein ek guided tour pe gaya tha, aur isse ancient history aur art mein meri interest jaag gayi. Ancient civilisations pe beginners ke liye kuch books suggest karenge?", "assistant_reply": "Start ke liye acchi books hain: E. H. Gombrich ki 'A Little History of the World', broad overview ke liye Yuval Noah Harari ki 'Sapiens', aur art history ke liye Gombrich ki 'The Story of Art'.", "followup_turn": "Inmein se pehle kaunsi padhoon?", "followup_reply": "'A Little History of the World' se start kijiye. Yeh chhoti aur easy hai, aur isse ek timeline ban jaati hai jisse baaki dono books samajhna easy ho jaata hai.", "q1": "Main History Museum mein guided tour pe kab gaya tha?", "q2": "Main guided tour ke liye History Museum kis date ko gaya tha?"}}, "gold": {"answer_en": "February 24", "date": {"day": 24, "month": 2}, "slots": [{"name": "date", "aliases": ["February 24", "February 24th", "24 February", "24th February", "24th of February", "Feb 24", "24 Feb", "24 फ़रवरी", "24 फरवरी"]}]}, "source": {"dataset": "LongMemEval", "question_id": "2ce6a0f2", "lme_type": "multi-session"}, "topic": "other", "is_answerable": true, "evidence_position": 5, "filler_seed": 2034785533, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "late", "abl_short": false, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E009", "cand_id": "EC049", "memory_type": "event", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I just turned 32 on February 12th, so I'm feeling more motivated to take care of myself. What are some simple health habits I could start this year?", "assistant_reply": "Start small: a 20-minute walk most days, a regular sleep time, more vegetables at lunch, and a yearly health check-up. Small habits you can keep beat big plans you drop.", "followup_turn": "What tests should a yearly check-up include at my age?", "followup_reply": "Commonly a blood count, blood sugar, cholesterol, thyroid and vitamin D, plus blood pressure. Your doctor may add others based on your family history.", "q1": "When did I turn 32?", "q2": "On what date is my birthday?", "distractor_sentence": "My brother turns 29 on August 8th, so we're already planning his party."}, "hi": {"memory_turn": "12 फ़रवरी को मैं 32 साल की हो गई, इसलिए अब अपनी सेहत का ध्यान रखने का मन ज़्यादा कर रहा है। इस साल कौन-सी आसान हेल्दी आदतें शुरू कर सकती हूँ?", "assistant_reply": "छोटे से शुरू कीजिए: ज़्यादातर दिनों में 20 मिनट की सैर, सोने का एक तय समय, दोपहर के खाने में ज़्यादा सब्ज़ियाँ, और साल में एक बार हेल्थ चेक-अप। जो छोटी आदतें टिक सकें, वे बड़े अधूरे प्लान से बेहतर हैं।", "followup_turn": "मेरी उम्र में सालाना चेक-अप में कौन-से टेस्ट होने चाहिए?", "followup_reply": "आमतौर पर ब्लड काउंट, ब्लड शुगर, कोलेस्ट्रॉल, थायरॉइड और विटामिन D, साथ में ब्लड प्रेशर। आपके परिवार के इतिहास के हिसाब से डॉक्टर कुछ और टेस्ट जोड़ सकते हैं।", "q1": "मैं 32 साल की कब हुई?", "q2": "मेरा जन्मदिन किस तारीख़ को है?", "distractor_sentence": "मेरा भाई 8 अगस्त को 29 साल का होगा, इसलिए हम अभी से उसकी पार्टी की तैयारी कर रहे हैं।"}, "hg": {"memory_turn": "12 February ko main 32 saal ki ho gayi, isliye ab apni health ka dhyan rakhne ka mann zyada kar raha hai. Is saal kaunsi easy healthy habits shuru kar sakti hoon?", "assistant_reply": "Chhote se shuru kijiye: zyadatar dino mein 20 minute ki walk, sone ka ek fixed time, lunch mein zyada sabziyan, aur saal mein ek baar health check-up. Jo chhoti habits tik sakein, woh bade adhoore plans se better hain.", "followup_turn": "Meri age mein yearly check-up mein kaunse tests hone chahiye?", "followup_reply": "Usually blood count, blood sugar, cholesterol, thyroid aur vitamin D, saath mein blood pressure. Aapki family history ke hisaab se doctor kuch aur tests add kar sakte hain.", "q1": "Main 32 saal ki kab hui?", "q2": "Mera birthday kis date ko hai?", "distractor_sentence": "Mera bhai 8 August ko 29 saal ka hoga, isliye hum abhi se uski party ki planning kar rahe hain."}}, "gold": {"answer_en": "February 12", "date": {"day": 12, "month": 2}, "slots": [{"name": "date", "aliases": ["February 12", "February 12th", "12 February", "12th February", "12th of February", "Feb 12", "12 Feb", "12 फ़रवरी", "12 फरवरी"]}]}, "distractor": {"date": {"day": 8, "month": 8}, "value_aliases": ["August 8", "August 8th", "8 August", "8th August", "8th of August", "Aug 8", "8 Aug", "8 अगस्त"]}, "source": {"dataset": "LongMemEval", "question_id": "gpt4_d12ceb0e", "lme_type": "multi-session"}, "topic": "other", "is_answerable": true, "evidence_position": 9, "filler_seed": 3481914820, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "middle", "abl_short": true, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E010", "cand_id": "EC176", "memory_type": "event", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I completed my first full marathon on April 10th and I'm feeling a bit sore. What should I do to recover properly?", "assistant_reply": "Congratulations! Rest for a few days, walk gently to keep blood flowing, eat enough protein and carbohydrates, and drink plenty of fluids. Wait about two weeks before any hard running.", "followup_turn": "Is a massage a good idea right now?", "followup_reply": "A light massage after two or three days can ease tightness. Avoid deep tissue work in the first days, when your muscles are still repairing.", "q1": "When did I complete my first full marathon?", "q2": "On what date did I run my first marathon?"}, "hi": {"memory_turn": "मैंने 10 अप्रैल को अपनी पहली फ़ुल मैराथन पूरी की और अब शरीर में थोड़ा दर्द है। ठीक से रिकवर होने के लिए क्या करूँ?", "assistant_reply": "बधाई हो! कुछ दिन आराम कीजिए, ख़ून का संचार बनाए रखने के लिए हल्का टहलिए, पर्याप्त प्रोटीन और कार्बोहाइड्रेट खाइए, और ख़ूब पानी पीजिए। किसी भी तेज़ दौड़ से पहले लगभग दो हफ़्ते रुकिए।", "followup_turn": "क्या अभी मसाज करवाना ठीक रहेगा?", "followup_reply": "दो-तीन दिन बाद हल्की मसाज से जकड़न कम हो सकती है। शुरुआती दिनों में गहरी मसाज से बचिए, जब मांसपेशियाँ अभी ठीक हो रही होती हैं।", "q1": "मैंने अपनी पहली फ़ुल मैराथन कब पूरी की थी?", "q2": "मैंने अपनी पहली मैराथन किस तारीख़ को दौड़ी थी?"}, "hg": {"memory_turn": "Maine 10 April ko apni pehli full marathon complete ki aur ab body mein thoda dard hai. Theek se recover hone ke liye kya karoon?", "assistant_reply": "Congratulations! Kuch din rest kijiye, blood flow bana rehne ke liye halka tehliye, kaafi protein aur carbs khaiye, aur khoob paani peejiye. Kisi bhi hard run se pehle lagbhag do hafte rukiye.", "followup_turn": "Kya abhi massage karwana theek rahega?", "followup_reply": "Do-teen din baad light massage se stiffness kam ho sakti hai. Shuru ke dino mein deep tissue massage se bachiye, jab muscles abhi repair ho rahi hoti hain.", "q1": "Maine apni pehli full marathon kab complete ki thi?", "q2": "Maine apni pehli marathon kis date ko daudi thi?"}}, "gold": {"answer_en": "April 10", "date": {"day": 10, "month": 4}, "slots": [{"name": "date", "aliases": ["April 10", "April 10th", "10 April", "10th April", "10th of April", "Apr 10", "10 Apr", "10 अप्रैल"]}]}, "source": {"dataset": "LongMemEval", "question_id": "21d02d0d", "lme_type": "multi-session"}, "topic": "fitness/health", "is_answerable": true, "evidence_position": 2, "filler_seed": 471514393, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "early", "abl_short": false, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "F001-ABS", "base_item": "F001", "memory_type": "fact", "is_answerable": false, "text": {"en": {"q1": "Which audiobook app do I use on my commute?", "q2": "What's the name of the app I listen to audiobooks on?"}, "hi": {"q1": "मैं सफ़र में ऑडियोबुक्स सुनने के लिए कौन-सा ऐप इस्तेमाल करती हूँ?", "q2": "जिस ऐप पर मैं ऑडियोबुक्स सुनती हूँ, उसका नाम क्या है?"}, "hg": {"q1": "Main commute mein audiobooks sunne ke liye kaunsa app use karti hoon?", "q2": "Jis app pe main audiobooks sunti hoon, uska naam kya hai?"}}, "gold": {"answer_en": "NO_ANSWER (never mentioned; Kindle is mentioned only for reading a book)"}, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "early", "banned_fillers": [], "abl_short": false, "abl_q3": false, "abl_digits": false}
{"item_id": "P003-ABS", "base_item": "P003", "memory_type": "preference", "is_answerable": false, "text": {"en": {"q1": "Which laptop do I edit my videos on?", "q2": "What laptop do I have for video editing?"}, "hi": {"q1": "मैं अपने वीडियो किस लैपटॉप पर एडिट करती हूँ?", "q2": "वीडियो एडिटिंग के लिए मेरे पास कौन-सा लैपटॉप है?"}, "hg": {"q1": "Main apne videos kis laptop pe edit karti hoon?", "q2": "Video editing ke liye mere paas kaunsa laptop hai?"}}, "gold": {"answer_en": "NO_ANSWER (laptop never mentioned)"}, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "early", "banned_fillers": [], "abl_short": false, "abl_q3": false, "abl_digits": false}
{"item_id": "E001-ABS", "base_item": "E001", "memory_type": "event", "is_answerable": false, "text": {"en": {"q1": "When did I get my car detailed?", "q2": "On what date was my car's detailing done?"}, "hi": {"q1": "मैंने अपनी कार की डिटेलिंग कब करवाई थी?", "q2": "मेरी कार की डिटेलिंग किस तारीख़ को हुई थी?"}, "hg": {"q1": "Maine apni car ki detailing kab karwayi thi?", "q2": "Meri car ki detailing kis date ko hui thi?"}}, "gold": {"answer_en": "NO_ANSWER (detailing was only planned, never done; the service date is a trap)"}, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026", "evidence_class": "middle", "banned_fillers": [], "abl_short": false, "abl_q3": false, "abl_digits": false}
{"item_id": "F011", "cand_id": "FC108", "memory_type": "fact", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I went to the local community theater last weekend, and the play I saw was a production of The Glass Menagerie. My niece Emily loved it too. Do you think she should audition for a role next season?", "assistant_reply": "Community theater is a friendly place to start. If she enjoys performing, an audition is great practice even if she doesn't get a big role. Encourage her to prepare a short monologue.", "followup_turn": "What should she wear to the audition?", "followup_reply": "Simple, comfortable clothes she can move in, with no distracting logos. Directors want to see her expressions and hear her voice, not the outfit.", "q1": "Which play did I see at the local community theater?", "q2": "What was the name of the play I watched at the community theater?", "q3": "My friend is asking what I watched on stage recently. What should I tell her?", "distractor_sentence": "My cousin went to see Death of a Salesman at the same theater last month."}, "hi": {"memory_turn": "मैं पिछले वीकेंड यहाँ के कम्युनिटी थिएटर गई थी, और वहाँ मैंने The Glass Menagerie का नाटक देखा। मेरी भतीजी एमिली को भी यह बहुत पसंद आया। क्या उसे अगले सीज़न में किसी रोल के लिए ऑडिशन देना चाहिए?", "assistant_reply": "कम्युनिटी थिएटर शुरुआत के लिए अच्छी जगह है। अगर उसे अभिनय पसंद है, तो ऑडिशन अच्छा अभ्यास है, चाहे बड़ा रोल न भी मिले। उसे एक छोटा मोनोलॉग तैयार करने को कहिए।", "followup_turn": "ऑडिशन में उसे क्या पहनना चाहिए?", "followup_reply": "सादे, आरामदायक कपड़े जिनमें वह आसानी से हिल-डुल सके, बिना बड़े लोगो के। डायरेक्टर उसकी आवाज़ और भाव देखना चाहते हैं, कपड़े नहीं।", "q1": "मैंने कम्युनिटी थिएटर में कौन-सा नाटक देखा था?", "q2": "कम्युनिटी थिएटर में जो नाटक मैंने देखा, उसका नाम क्या था?", "q3": "मेरी दोस्त पूछ रही है कि मैंने हाल में स्टेज पर क्या देखा। उसे क्या बताऊँ?", "distractor_sentence": "मेरे कज़िन ने पिछले महीने उसी थिएटर में Death of a Salesman देखा था।"}, "hg": {"memory_turn": "Main pichhle weekend local community theatre gayi thi, aur wahan maine The Glass Menagerie ka play dekha. Meri bhatiji Emily ko bhi yeh bahut pasand aaya. Kya use next season mein kisi role ke liye audition dena chahiye?", "assistant_reply": "Community theatre shuru karne ke liye acchi jagah hai. Agar use acting pasand hai, toh audition accha practice hai, chahe bada role na mile. Use ek chhota monologue taiyaar karne ko kahiye.", "followup_turn": "Audition mein use kya pehenna chahiye?", "followup_reply": "Simple, comfortable kapde jinmein woh aasani se move kar sake, bina bade logo ke. Director uski awaaz aur expressions dekhna chahte hain, kapde nahi.", "q1": "Maine community theatre mein kaunsa play dekha tha?", "q2": "Community theatre mein jo play maine dekha, uska naam kya tha?", "q3": "Meri friend pooch rahi hai ki maine recently stage pe kya dekha. Use kya bataun?", "distractor_sentence": "Mere cousin ne pichhle mahine usi theatre mein Death of a Salesman dekha tha."}}, "gold": {"answer_en": "The Glass Menagerie", "slots": [{"name": "play", "aliases": ["The Glass Menagerie", "Glass Menagerie", "द ग्लास मेनेजरी", "ग्लास मेनेजरी"]}]}, "distractor": {"value_aliases": ["Death of a Salesman", "डेथ ऑफ़ अ सेल्समैन"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "58bf7951", "lme_type": "single-session-user"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 403979681, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": true, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "F012", "cand_id": "FC109", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I've been listening to a playlist I created on Spotify called Summer Vibes. It's full of chill tracks for relaxing or working out. Can you suggest a service that helps me discover new ambient and lo-fi artists?", "assistant_reply": "Spotify's own Discover Weekly is a good start, and Bandcamp is great for independent ambient and lo-fi artists. Last.fm can also suggest similar artists based on what you already play.", "followup_turn": "Is Bandcamp free to use?", "followup_reply": "Yes, browsing and streaming most tracks is free. You pay only if you want to buy and download music, and much of that money goes directly to the artists.", "q1": "What is the name of the playlist I made on Spotify?", "q2": "What did I call my own Spotify playlist?"}, "hi": {"memory_turn": "मैं Spotify पर अपनी बनाई हुई एक प्लेलिस्ट सुन रहा हूँ, जिसका नाम Summer Vibes है। इसमें आराम करने या वर्कआउट के लिए शांत गाने हैं। क्या आप कोई ऐसी सर्विस बता सकते हैं जिससे मैं नए ambient और lo-fi आर्टिस्ट खोज सकूँ?", "assistant_reply": "Spotify का अपना Discover Weekly अच्छी शुरुआत है, और स्वतंत्र ambient और lo-fi आर्टिस्ट के लिए Bandcamp बढ़िया है। Last.fm भी आपकी पसंद के हिसाब से मिलते-जुलते आर्टिस्ट सुझा सकता है।", "followup_turn": "क्या Bandcamp इस्तेमाल करना मुफ़्त है?", "followup_reply": "हाँ, ज़्यादातर गाने देखना और सुनना मुफ़्त है। पैसे सिर्फ़ तब लगते हैं जब आप संगीत ख़रीदकर डाउनलोड करना चाहें, और उसका बड़ा हिस्सा सीधे आर्टिस्ट को जाता है।", "q1": "Spotify पर मैंने जो प्लेलिस्ट बनाई है, उसका नाम क्या है?", "q2": "मैंने अपनी Spotify प्लेलिस्ट का क्या नाम रखा है?"}, "hg": {"memory_turn": "Main Spotify pe apni banayi hui ek playlist sun raha hoon, jiska naam Summer Vibes hai. Ismein relax karne ya workout ke liye chill songs hain. Kya aap koi aisi service bata sakte hain jisse main naye ambient aur lo-fi artists discover kar sakoon?", "assistant_reply": "Spotify ka apna Discover Weekly accha start hai, aur independent ambient aur lo-fi artists ke liye Bandcamp badhiya hai. Last.fm bhi aapki pasand ke hisaab se milte-julte artists suggest kar sakta hai.", "followup_turn": "Kya Bandcamp use karna free hai?", "followup_reply": "Haan, zyaadatar tracks browse karna aur sunna free hai. Paise sirf tab lagte hain jab aap music khareed ke download karna chahein, aur uska bada hissa seedha artists ko jaata hai.", "q1": "Spotify pe maine jo playlist banayi hai, uska naam kya hai?", "q2": "Maine apni Spotify playlist ka kya naam rakha hai?"}}, "gold": {"answer_en": "Summer Vibes", "slots": [{"name": "playlist", "aliases": ["Summer Vibes", "समर वाइब्स"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "1e043500", "lme_type": "single-session-user"}, "topic": "vehicles", "is_answerable": true, "evidence_position": 5, "filler_seed": 2030015734, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F013", "cand_id": "FC111", "memory_type": "fact", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I've been using the Down Dog app for my home practice, which helps on days when I can't make it to my classes at Serenity Yoga. Can you suggest a short evening routine for better sleep?", "assistant_reply": "Try ten minutes of gentle stretches: child's pose, a seated forward fold, legs up the wall and a reclined twist. Hold each for about a minute and breathe slowly through your nose.", "followup_turn": "Should I avoid any poses right before bed?", "followup_reply": "Skip strong backbends and fast flows, since they can be energising. Keep the session slow, quiet and close to the floor.", "q1": "Where do I take my yoga classes?", "q2": "Which studio do I go to for yoga classes?", "distractor_sentence": "My roommate goes to Lotus Flow Studio for her yoga, but she says it's always crowded."}, "hi": {"memory_turn": "घर पर प्रैक्टिस के लिए मैं Down Dog ऐप इस्तेमाल कर रही हूँ, जिससे उन दिनों मदद मिलती है जब मैं Serenity Yoga की क्लास में नहीं जा पाती। क्या आप अच्छी नींद के लिए शाम का कोई छोटा रूटीन बता सकते हैं?", "assistant_reply": "दस मिनट के हल्के स्ट्रेच आज़माइए: चाइल्ड पोज़, बैठकर आगे झुकना, दीवार पर पैर टिकाना और लेटकर ट्विस्ट। हर एक को करीब एक मिनट रोकें और नाक से धीरे-धीरे साँस लें।", "followup_turn": "क्या सोने से ठीक पहले कुछ आसन नहीं करने चाहिए?", "followup_reply": "तेज़ बैकबेंड और तेज़ फ़्लो से बचें, क्योंकि ये शरीर में ऊर्जा भर देते हैं। सेशन धीमा, शांत और ज़मीन के पास रखें।", "q1": "मैं योगा क्लास कहाँ लेती हूँ?", "q2": "योगा क्लास के लिए मैं किस स्टूडियो में जाती हूँ?", "distractor_sentence": "मेरी रूममेट योगा के लिए Lotus Flow Studio जाती है, पर उसका कहना है कि वहाँ हमेशा भीड़ रहती है।"}, "hg": {"memory_turn": "Ghar pe practice ke liye main Down Dog app use kar rahi hoon, jisse un dinon help milti hai jab main Serenity Yoga ki class mein nahi ja paati. Kya aap acchi neend ke liye shaam ka koi chhota routine bata sakte hain?", "assistant_reply": "Das minute ke halke stretches try kijiye: child's pose, baith ke aage jhukna, deewar pe pair tikana aur let ke twist. Har ek ko kareeb ek minute hold karein aur naak se dheere-dheere saans lein.", "followup_turn": "Kya sone se theek pehle kuch poses avoid karne chahiye?", "followup_reply": "Strong backbends aur fast flows avoid karein, kyunki yeh energy badha dete hain. Session slow, shaant aur zameen ke paas rakhein.", "q1": "Main yoga classes kahan leti hoon?", "q2": "Yoga class ke liye main kaunse studio jaati hoon?", "distractor_sentence": "Meri roommate yoga ke liye Lotus Flow Studio jaati hai, par uska kehna hai ki wahan hamesha bheed rehti hai."}}, "gold": {"answer_en": "Serenity Yoga", "slots": [{"name": "studio", "aliases": ["Serenity Yoga", "Serenity", "सेरेनिटी योगा", "सेरेनिटी"]}]}, "distractor": {"value_aliases": ["Lotus Flow", "लोटस फ़्लो"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "6ade9755", "lme_type": "single-session-user"}, "topic": "pets", "is_answerable": true, "evidence_position": 9, "filler_seed": 5507157, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F014", "cand_id": "FC112", "memory_type": "fact", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I've been redecorating and recently repainted my bedroom walls a lighter shade of gray, which makes the room feel much brighter. Now I want some plants for it. Is a ZZ plant easy to care for?", "assistant_reply": "Yes, ZZ plants are very forgiving. Water only when the soil is completely dry, usually every two to three weeks, and feed it lightly once a month in spring and summer.", "followup_turn": "Can it survive in a room with only one small window?", "followup_reply": "It can. ZZ plants tolerate low light well, though they grow more slowly there. Just keep it away from cold drafts in winter.", "q1": "What colour did I repaint my bedroom walls?", "q2": "Which colour are my bedroom walls now, after I repainted them?", "q3": "I'm choosing curtains for the room where I sleep. What colour are the walls there?", "distractor_sentence": "My brother painted his living room beige last year and now regrets it."}, "hi": {"memory_turn": "मैं घर की सजावट बदल रहा हूँ और हाल ही में मैंने बेडरूम की दीवारों पर हल्का ग्रे रंग करवाया है, जिससे कमरा काफ़ी रोशन लगता है। अब मुझे उसके लिए कुछ पौधे चाहिए। क्या ZZ प्लांट की देखभाल आसान है?", "assistant_reply": "हाँ, ZZ प्लांट बहुत सहनशील होता है। मिट्टी पूरी सूख जाने पर ही पानी दें, आमतौर पर दो-तीन हफ़्ते में एक बार, और वसंत व गर्मियों में महीने में एक बार हल्की खाद दें।", "followup_turn": "क्या यह सिर्फ़ एक छोटी खिड़की वाले कमरे में रह पाएगा?", "followup_reply": "हाँ। ZZ प्लांट कम रोशनी में भी ठीक रहता है, बस वहाँ धीरे बढ़ता है। सर्दियों में इसे ठंडी हवा से दूर रखें।", "q1": "मैंने अपने बेडरूम की दीवारों पर कौन-सा रंग करवाया?", "q2": "दोबारा पेंट करवाने के बाद मेरे बेडरूम की दीवारें अब किस रंग की हैं?", "q3": "मैं जिस कमरे में सोता हूँ उसके लिए पर्दे चुन रहा हूँ। वहाँ की दीवारें किस रंग की हैं?", "distractor_sentence": "मेरे भाई ने पिछले साल अपना लिविंग रूम बेज रंग में पेंट करवाया और अब पछता रहा है।"}, "hg": {"memory_turn": "Main ghar ka decor badal raha hoon aur haal hi mein maine bedroom ki deewaron pe light grey colour karwaya hai, jisse room kaafi bright lagta hai. Ab mujhe uske liye kuch plants chahiye. Kya ZZ plant ki care easy hai?", "assistant_reply": "Haan, ZZ plant bahut forgiving hota hai. Mitti poori sookh jaane pe hi paani dein, usually do-teen hafte mein ek baar, aur spring aur garmiyon mein mahine mein ek baar halki khaad dein.", "followup_turn": "Kya yeh sirf ek chhoti khidki wale room mein survive kar payega?", "followup_reply": "Haan. ZZ plant kam roshni mein bhi theek rehta hai, bas wahan dheere badhta hai. Sardiyon mein ise thandi hawa se door rakhein.", "q1": "Maine apne bedroom ki deewaron pe kaunsa colour karwaya?", "q2": "Repaint ke baad mere bedroom ki deewarein ab kis colour ki hain?", "q3": "Main jis room mein sota hoon uske liye curtains choose kar raha hoon. Wahan ki deewarein kis colour ki hain?", "distractor_sentence": "Mere bhai ne pichhle saal apna living room beige paint karwaya aur ab pachhta raha hai."}}, "gold": {"answer_en": "a lighter shade of gray", "slots": [{"name": "colour", "aliases": ["gray", "grey", "ग्रे", "सलेटी"]}]}, "distractor": {"value_aliases": ["beige", "बेज"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "6f9b354f", "lme_type": "single-session-user"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 2651568786, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "F015", "cand_id": "FC114", "memory_type": "fact", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I'm really happy with my new tennis racket, which I got from a sports store downtown. I feel like it has improved my game. Any tips on how to clean and maintain it?", "assistant_reply": "Wipe the frame with a damp cloth after play, keep it out of hot cars, and store it in a cover. Replace the grip when it gets slippery, and restring it regularly if you play often.", "followup_turn": "How do I know when the strings need replacing?", "followup_reply": "If they keep moving out of place, feel dead, or the ball stops responding the way it used to, it's time. Visible fraying is another clear sign.", "q1": "Where did I buy my new tennis racket?", "q2": "From which shop did I get my new tennis racket?"}, "hi": {"memory_turn": "मैं अपने नए टेनिस रैकेट से बहुत ख़ुश हूँ, जो मैंने शहर के मेन मार्केट के एक स्पोर्ट्स स्टोर से लिया था। मुझे लगता है इससे मेरा खेल बेहतर हुआ है। इसकी सफ़ाई और देखभाल के लिए कोई टिप्स?", "assistant_reply": "खेलने के बाद फ़्रेम को गीले कपड़े से पोंछें, इसे धूप में खड़ी गाड़ी में न छोड़ें और कवर में रखें। ग्रिप फिसलने लगे तो बदल दें, और अगर आप अक्सर खेलती हैं तो नियमित रूप से तारें बदलवाएँ।", "followup_turn": "कैसे पता चलेगा कि तारें बदलने का समय आ गया है?", "followup_reply": "अगर तारें बार-बार अपनी जगह से खिसकती हैं, बेजान लगती हैं या गेंद पहले जैसी प्रतिक्रिया नहीं देती, तो समय आ गया है। तारों का घिसना भी साफ़ संकेत है।", "q1": "मैंने अपना नया टेनिस रैकेट कहाँ से ख़रीदा?", "q2": "मैंने अपना नया टेनिस रैकेट किस दुकान से लिया था?"}, "hg": {"memory_turn": "Main apne naye tennis racket se bahut khush hoon, jo maine downtown ke ek sports store se liya tha. Mujhe lagta hai isse mera game better hua hai. Iski cleaning aur maintenance ke liye koi tips?", "assistant_reply": "Khelne ke baad frame ko geele kapde se pochhein, ise dhoop mein khadi gaadi mein na chhodein aur cover mein rakhein. Grip slippery ho jaaye toh badal dein, aur agar aap aksar khelti hain toh regularly strings badlwayein.", "followup_turn": "Kaise pata chalega ki strings badalne ka time aa gaya hai?", "followup_reply": "Agar strings baar-baar apni jagah se khisakti hain, dead lagti hain ya ball pehle jaisa response nahi deti, toh time aa gaya hai. Strings ka ghisna bhi clear sign hai.", "q1": "Maine apna naya tennis racket kahan se khareeda?", "q2": "Mera naya tennis racket kis shop se liya tha?"}}, "gold": {"answer_en": "a sports store downtown", "slots": [{"name": "shop", "aliases": ["sports store", "sports shop", "स्पोर्ट्स स्टोर", "स्पोर्ट्स की दुकान"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "f8c5f88b", "lme_type": "single-session-user"}, "topic": "media/books/games", "is_answerable": true, "evidence_position": 5, "filler_seed": 4221724610, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F016", "cand_id": "FC115", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "In my previous role as a marketing specialist at a small startup, I used Trello every day. Now I'm exploring other tools. How does ClickUp differ from Trello?", "assistant_reply": "Trello is simple and board-based, which suits small teams. ClickUp packs in more: lists, timelines, docs, goals and time tracking in one place, but it takes longer to learn.", "followup_turn": "Which one would be better for managing a team of five?", "followup_reply": "For five people with straightforward projects, Trello is usually enough. Choose ClickUp if you need dependencies, reporting or several project views.", "q1": "What was my previous job?", "q2": "What did I work as before my current role?"}, "hi": {"memory_turn": "अपनी पिछली नौकरी में मैं एक छोटे स्टार्टअप में मार्केटिंग स्पेशलिस्ट था, और रोज़ Trello इस्तेमाल करता था। अब मैं दूसरे टूल देख रहा हूँ। ClickUp, Trello से कैसे अलग है?", "assistant_reply": "Trello सरल है और बोर्ड पर आधारित है, जो छोटी टीमों के लिए ठीक है। ClickUp में ज़्यादा चीज़ें हैं: लिस्ट, टाइमलाइन, डॉक्स, लक्ष्य और टाइम ट्रैकिंग एक ही जगह, पर इसे सीखने में ज़्यादा समय लगता है।", "followup_turn": "पाँच लोगों की टीम संभालने के लिए कौन-सा बेहतर रहेगा?", "followup_reply": "सीधे-सादे प्रोजेक्ट वाले पाँच लोगों के लिए Trello आमतौर पर काफ़ी है। अगर आपको डिपेंडेंसी, रिपोर्टिंग या कई प्रोजेक्ट व्यू चाहिए, तो ClickUp चुनें।", "q1": "मेरी पिछली नौकरी क्या थी?", "q2": "अभी के काम से पहले मैं क्या काम करता था?"}, "hg": {"memory_turn": "Apni pichhli job mein main ek chhote startup mein marketing specialist tha, aur roz Trello use karta tha. Ab main doosre tools dekh raha hoon. ClickUp, Trello se kaise alag hai?", "assistant_reply": "Trello simple hai aur board-based hai, jo chhoti teams ke liye theek hai. ClickUp mein zyaada cheezein hain: lists, timelines, docs, goals aur time tracking ek hi jagah, par ise seekhne mein zyaada time lagta hai.", "followup_turn": "Paanch logon ki team manage karne ke liye kaunsa better rahega?", "followup_reply": "Seedhe-saade projects wale paanch logon ke liye Trello usually kaafi hai. Agar aapko dependencies, reporting ya kai project views chahiye, toh ClickUp chunein.", "q1": "Meri pichhli job kya thi?", "q2": "Abhi ke kaam se pehle main kya kaam karta tha?"}}, "gold": {"answer_en": "marketing specialist at a small startup", "slots": [{"name": "job", "aliases": ["marketing specialist", "marketing", "मार्केटिंग"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "5d3d2817", "lme_type": "single-session-user"}, "topic": "work/study", "is_answerable": true, "evidence_position": 9, "filler_seed": 38396520, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F017", "cand_id": "FC116", "memory_type": "fact", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I want to start tracking my expenses. I've been using my credit card a lot, and last month I bought a designer handbag for $800. Can you recommend a good budgeting app?", "assistant_reply": "YNAB is great if you want to give every dollar a job in advance, while Wallet or Monefy are simpler for logging daily spending. Linking your card makes tracking automatic.", "followup_turn": "How much of my income should go to fun spending?", "followup_reply": "A common rule is 50/30/20: half for needs, 30 percent for wants and 20 percent for savings. Adjust it if you have debt to pay off first.", "q1": "How much did I spend on the designer handbag?", "q2": "What did the designer handbag I bought cost me?", "q3": "I'm adding up last month's big purchases for my budget. What amount should I write down for my bag?", "distractor_sentence": "My sister spent $450 on a pair of boots last week."}, "hi": {"memory_turn": "मैं अपने ख़र्चों का हिसाब रखना शुरू करना चाहती हूँ। मैं क्रेडिट कार्ड बहुत इस्तेमाल कर रही हूँ, और पिछले महीने मैंने $800 का एक डिज़ाइनर हैंडबैग ख़रीदा। क्या आप कोई अच्छा बजटिंग ऐप बता सकते हैं?", "assistant_reply": "अगर आप हर डॉलर की योजना पहले से बनाना चाहती हैं तो YNAB बढ़िया है, जबकि रोज़ का ख़र्च लिखने के लिए Wallet या Monefy ज़्यादा आसान हैं। कार्ड लिंक करने से हिसाब अपने-आप बनता रहता है।", "followup_turn": "मेरी आमदनी का कितना हिस्सा मौज-मस्ती पर ख़र्च होना चाहिए?", "followup_reply": "एक आम नियम 50/30/20 है: आधा ज़रूरतों के लिए, 30 प्रतिशत इच्छाओं के लिए और 20 प्रतिशत बचत के लिए। अगर पहले कोई क़र्ज़ चुकाना है तो इसे बदल लें।", "q1": "डिज़ाइनर हैंडबैग पर मैंने कितने पैसे ख़र्च किए?", "q2": "जो डिज़ाइनर हैंडबैग मैंने ख़रीदा, वह मुझे कितने का पड़ा?", "q3": "मैं बजट के लिए पिछले महीने की बड़ी ख़रीदारियाँ जोड़ रही हूँ। अपने बैग के लिए कितनी रक़म लिखूँ?", "distractor_sentence": "मेरी बहन ने पिछले हफ़्ते एक जोड़ी बूट्स पर $450 ख़र्च कर दिए।"}, "hg": {"memory_turn": "Main apne kharchon ka hisaab rakhna shuru karna chahti hoon. Main credit card bahut use kar rahi hoon, aur pichhle mahine maine $800 ka ek designer handbag khareeda. Kya aap koi accha budgeting app bata sakte hain?", "assistant_reply": "Agar aap har dollar ka plan pehle se banana chahti hain toh YNAB badhiya hai, jabki roz ka kharcha likhne ke liye Wallet ya Monefy zyaada easy hain. Card link karne se tracking automatic ho jaati hai.", "followup_turn": "Meri income ka kitna hissa fun spending pe jaana chahiye?", "followup_reply": "Ek common rule 50/30/20 hai: aadha zarooraton ke liye, 30 percent wants ke liye aur 20 percent savings ke liye. Agar pehle koi loan chukana hai toh ise adjust kar lein.", "q1": "Designer handbag pe maine kitne paise kharch kiye?", "q2": "Jo designer handbag maine khareeda, woh mujhe kitne ka pada?", "q3": "Main budget ke liye pichhle mahine ki badi shopping jod rahi hoon. Apne bag ke liye kitna amount likhun?", "distractor_sentence": "Meri behen ne pichhle hafte ek pair boots pe $450 kharch kar diye."}}, "gold": {"answer_en": "$800", "slots": [{"name": "amount", "aliases": ["$800", "800 dollars", "800 dollar", "800", "800 डॉलर"]}]}, "distractor": {"value_aliases": ["$450", "450"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "7527f7e2", "lme_type": "single-session-user"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 3974829076, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "F018", "cand_id": "FC119", "memory_type": "fact", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "Speaking of first purchases, I got a 10% discount on my first order from a new clothing brand I found on Instagram. I want to promote my writing services there too. Any tips for getting started?", "assistant_reply": "Pick one clear niche, use a simple bio that says what you write and for whom, and post samples or short tips three times a week. Engaging with other writers' posts helps people find you.", "followup_turn": "Should I use a personal account or a business account?", "followup_reply": "A business or creator account is better, because it shows insights about who sees your posts and lets you add a contact button.", "q1": "What discount did I get on my first purchase from the new clothing brand?", "q2": "How much off did the new clothing brand give me on my first order?", "q3": "My friend wants to try that clothing brand I found on Instagram. How much will she save on her first order if she gets the same deal I did?", "distractor_sentence": "My friend got 25% off at a shoe website that same week."}, "hi": {"memory_turn": "पहली ख़रीदारी की बात चली तो, Instagram पर मिले एक नए कपड़ों के ब्रांड से मेरे पहले ऑर्डर पर मुझे 10% की छूट मिली थी। मैं वहाँ अपनी राइटिंग सर्विस का प्रचार भी करना चाहता हूँ। शुरुआत के लिए कोई टिप्स?", "assistant_reply": "एक साफ़ विषय चुनें, बायो में सरल शब्दों में लिखें कि आप क्या और किसके लिए लिखते हैं, और हफ़्ते में तीन बार सैंपल या छोटे टिप्स पोस्ट करें। दूसरे लेखकों की पोस्ट पर जुड़ने से लोग आपको ढूँढ पाते हैं।", "followup_turn": "पर्सनल अकाउंट इस्तेमाल करूँ या बिज़नेस अकाउंट?", "followup_reply": "बिज़नेस या क्रिएटर अकाउंट बेहतर है, क्योंकि इसमें दिखता है कि आपकी पोस्ट कौन देख रहा है और आप कॉन्टैक्ट बटन जोड़ सकते हैं।", "q1": "नए कपड़ों के ब्रांड से पहली ख़रीदारी पर मुझे कितनी छूट मिली थी?", "q2": "नए कपड़ों के ब्रांड ने मेरे पहले ऑर्डर पर कितना डिस्काउंट दिया था?", "q3": "मेरी दोस्त Instagram वाले उस कपड़ों के ब्रांड से ख़रीदना चाहती है। अगर उसे भी मेरे जैसा ऑफ़र मिले, तो पहले ऑर्डर पर कितनी बचत होगी?", "distractor_sentence": "मेरे दोस्त को उसी हफ़्ते एक जूतों की वेबसाइट पर 25% की छूट मिली थी।"}, "hg": {"memory_turn": "First purchase ki baat chali toh, Instagram pe mile ek naye clothing brand se mere pehle order pe mujhe 10% discount mila tha. Main wahan apni writing services promote bhi karna chahta hoon. Shuru karne ke liye koi tips?", "assistant_reply": "Ek clear niche chunein, bio mein simple words mein likhein ki aap kya aur kiske liye likhte hain, aur hafte mein teen baar samples ya chhote tips post karein. Doosre writers ki posts pe engage karne se log aapko dhoondh paate hain.", "followup_turn": "Personal account use karoon ya business account?", "followup_reply": "Business ya creator account better hai, kyunki ismein dikhta hai ki aapki posts kaun dekh raha hai aur aap contact button add kar sakte hain.", "q1": "Naye clothing brand se first purchase pe mujhe kitna discount mila tha?", "q2": "Naye clothing brand ne mere pehle order pe kitna off diya tha?", "q3": "Meri friend Instagram wale us clothing brand se khareedna chahti hai. Agar use bhi mere jaisa offer mile, toh pehle order pe kitni saving hogi?", "distractor_sentence": "Mere dost ko usi hafte ek shoe website pe 25% discount mila tha."}}, "gold": {"answer_en": "10%", "slots": [{"name": "discount", "aliases": ["10%", "10 percent", "10 per cent", "10 प्रतिशत", "10 फ़ीसदी", "10 pratishat"]}]}, "distractor": {"value_aliases": ["25%", "25 percent", "25 प्रतिशत"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "726462e0", "lme_type": "single-session-user"}, "topic": "other", "is_answerable": true, "evidence_position": 5, "filler_seed": 2638530421, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "F019", "cand_id": "FC120", "memory_type": "fact", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I just assembled an IKEA bookshelf, and it took me 4 hours, which wasn't too bad. Now I'm choosing a coffee table. Would a glass top scratch more easily than wood?", "assistant_reply": "Glass resists scratches better than most wood finishes, but it shows fingerprints, dust and water marks quickly. Wood hides daily marks well, though it can dent and needs coasters.", "followup_turn": "Which is easier to keep clean with a cat at home?", "followup_reply": "Wood is usually easier, since paw prints and hair are less visible. A matte or textured finish hides marks best.", "q1": "How long did it take me to assemble the IKEA bookshelf?", "q2": "How much time did putting together my IKEA bookshelf take?"}, "hi": {"memory_turn": "मैंने अभी एक IKEA बुकशेल्फ़ जोड़ी, और इसमें मुझे 4 घंटे लगे, जो बहुत बुरा नहीं था। अब मैं एक कॉफ़ी टेबल चुन रही हूँ। क्या काँच का टॉप लकड़ी से जल्दी खरोंच जाता है?", "assistant_reply": "काँच ज़्यादातर लकड़ी की फ़िनिश से कम खरोंचता है, पर उस पर उँगलियों के निशान, धूल और पानी के धब्बे जल्दी दिखते हैं। लकड़ी रोज़ के निशान छिपा लेती है, हालाँकि उसमें गड्ढे पड़ सकते हैं और कोस्टर चाहिए।", "followup_turn": "घर में बिल्ली हो तो किसे साफ़ रखना आसान है?", "followup_reply": "आमतौर पर लकड़ी, क्योंकि पंजों के निशान और बाल कम दिखते हैं। मैट या टेक्सचर वाली फ़िनिश निशान सबसे अच्छे से छिपाती है।", "q1": "IKEA बुकशेल्फ़ जोड़ने में मुझे कितना समय लगा?", "q2": "मेरी IKEA बुकशेल्फ़ को असेंबल करने में कितनी देर लगी?"}, "hg": {"memory_turn": "Maine abhi ek IKEA bookshelf assemble ki, aur ismein mujhe 4 ghante lage, jo bahut bura nahi tha. Ab main ek coffee table choose kar rahi hoon. Kya glass top wood se jaldi scratch hota hai?", "assistant_reply": "Glass zyaadatar wood finish se kam scratch hota hai, par uspe fingerprints, dhool aur paani ke daag jaldi dikhte hain. Wood roz ke nishaan chhupa leti hai, halanki usmein dents pad sakte hain aur coasters chahiye.", "followup_turn": "Ghar mein billi ho toh kise saaf rakhna easy hai?", "followup_reply": "Usually wood, kyunki panjon ke nishaan aur baal kam dikhte hain. Matte ya textured finish nishaan sabse acche se chhupati hai.", "q1": "IKEA bookshelf assemble karne mein mujhe kitna time laga?", "q2": "Meri IKEA bookshelf jodne mein kitni der lagi?"}}, "gold": {"answer_en": "4 hours", "slots": [{"name": "duration", "aliases": ["4 hours", "four hours", "4 hrs", "4 hour", "4 घंटे", "चार घंटे", "4 ghante", "char ghante", "chaar ghante"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "94f70d80", "lme_type": "single-session-user"}, "topic": "shopping/home", "is_answerable": true, "evidence_position": 9, "filler_seed": 3371395026, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F020", "cand_id": "FC122", "memory_type": "fact", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "My internet has been really good lately, especially when streaming on Netflix. I upgraded my plan to 500 Mbps about three weeks ago. But my laptop battery drains fast. Could the battery just be old?", "assistant_reply": "It could be. Batteries lose capacity after a few hundred charge cycles. Check the battery health report in your system settings; if it shows less than about 80 percent of the original capacity, a replacement will help.", "followup_turn": "Can I do anything to make the current battery last longer?", "followup_reply": "Lower the screen brightness, close background apps, and avoid keeping it at full charge all the time. Many laptops have a setting that stops charging at around 80 percent.", "q1": "What speed is my new internet plan?", "q2": "How fast is the internet plan I upgraded to?", "distractor_sentence": "My neighbour only has a 100 Mbps plan and complains about it every day."}, "hi": {"memory_turn": "पिछले कुछ दिनों से मेरा इंटरनेट बहुत अच्छा चल रहा है, ख़ासकर Netflix पर। करीब तीन हफ़्ते पहले मैंने अपना प्लान 500 Mbps पर अपग्रेड किया था। पर मेरे लैपटॉप की बैटरी जल्दी ख़त्म हो जाती है। क्या बैटरी बस पुरानी हो गई है?", "assistant_reply": "हो सकता है। कुछ सौ बार चार्ज होने के बाद बैटरी की क्षमता घटने लगती है। सिस्टम सेटिंग्स में बैटरी हेल्थ रिपोर्ट देखें; अगर वह असली क्षमता के लगभग 80 प्रतिशत से कम है, तो बदलने से फ़ायदा होगा।", "followup_turn": "क्या मौजूदा बैटरी को ज़्यादा चलाने के लिए मैं कुछ कर सकता हूँ?", "followup_reply": "स्क्रीन की ब्राइटनेस कम करें, बैकग्राउंड ऐप बंद करें और इसे हर समय पूरा चार्ज न रखें। कई लैपटॉप में ऐसी सेटिंग होती है जो लगभग 80 प्रतिशत पर चार्जिंग रोक देती है।", "q1": "मेरे नए इंटरनेट प्लान की स्पीड कितनी है?", "q2": "मैंने जिस इंटरनेट प्लान पर अपग्रेड किया, वह कितना तेज़ है?", "distractor_sentence": "मेरे पड़ोसी के पास तो बस 100 Mbps का प्लान है और वह हर दिन शिकायत करता है।"}, "hg": {"memory_turn": "Pichhle kuch dinon se mera internet bahut accha chal raha hai, especially Netflix pe. Kareeb teen hafte pehle maine apna plan 500 Mbps pe upgrade kiya tha. Par mere laptop ki battery jaldi khatam ho jaati hai. Kya battery bas purani ho gayi hai?", "assistant_reply": "Ho sakta hai. Kuch sau charge cycles ke baad battery ki capacity kam hone lagti hai. System settings mein battery health report dekhiye; agar woh original capacity ke lagbhag 80 percent se kam hai, toh replace karne se fayda hoga.", "followup_turn": "Kya current battery ko zyaada chalane ke liye main kuch kar sakta hoon?", "followup_reply": "Screen brightness kam karein, background apps band karein aur ise har time full charge na rakhein. Kai laptops mein aisi setting hoti hai jo lagbhag 80 percent pe charging rok deti hai.", "q1": "Mere naye internet plan ki speed kitni hai?", "q2": "Maine jis internet plan pe upgrade kiya, woh kitna fast hai?", "distractor_sentence": "Mere padosi ke paas toh bas 100 Mbps ka plan hai aur woh roz complain karta hai."}}, "gold": {"answer_en": "500 Mbps", "slots": [{"name": "speed", "aliases": ["500 Mbps", "500 megabits", "500 एमबीपीएस", "500"]}]}, "distractor": {"value_aliases": ["100 Mbps", "100 एमबीपीएस", "100"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "ad7109d1", "lme_type": "single-session-user"}, "topic": "shopping/home", "is_answerable": true, "evidence_position": 2, "filler_seed": 1234839494, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P011", "cand_id": "PC078", "memory_type": "preference", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "Apart from a great view, I love hotels with a rooftop pool and a hot tub on the balcony. I'm planning a trip to Miami next month. Which area should I stay in?", "assistant_reply": "South Beach is lively and close to the ocean, Brickell is modern and good for restaurants, and Mid-Beach is quieter with large resort hotels. Choose based on whether you want nightlife or rest.", "followup_turn": "When is the beach least crowded?", "followup_reply": "Early mornings on weekdays are the calmest. By late morning, especially on weekends, the popular stretches fill up quickly.", "q1": "Apart from views, what hotel features do I like?", "q2": "Besides a nice view, which amenities do I look for in a hotel?", "q3": "I'm filtering hotel search results for my Miami trip. Which two amenity filters should I tick for myself?", "distractor_sentence": "My friend always picks hotels with a free breakfast buffet."}, "hi": {"memory_turn": "अच्छे नज़ारे के अलावा, मुझे ऐसे होटल बहुत पसंद हैं जिनमें रूफ़टॉप पूल हो और बालकनी में हॉट टब हो। मैं अगले महीने मियामी घूमने का प्लान बना रही हूँ। मुझे किस इलाक़े में रुकना चाहिए?", "assistant_reply": "साउथ बीच जीवंत है और समंदर के पास है, ब्रिकेल आधुनिक है और रेस्टोरेंट के लिए अच्छा है, और मिड-बीच शांत है जहाँ बड़े रिज़ॉर्ट होटल हैं। चुनें कि आपको नाइटलाइफ़ चाहिए या आराम।", "followup_turn": "बीच पर सबसे कम भीड़ कब होती है?", "followup_reply": "हफ़्ते के दिनों में सुबह-सुबह सबसे शांत रहता है। देर सुबह तक, ख़ासकर वीकेंड पर, मशहूर हिस्से जल्दी भर जाते हैं।", "q1": "नज़ारों के अलावा, मुझे होटल की कौन-सी सुविधाएँ पसंद हैं?", "q2": "अच्छे व्यू के अलावा मैं होटल में कौन-सी सुविधाएँ ढूँढती हूँ?", "q3": "मियामी ट्रिप के लिए मैं होटल सर्च में फ़िल्टर लगा रही हूँ। अपने लिए कौन-से दो सुविधा वाले फ़िल्टर चुनूँ?", "distractor_sentence": "मेरी दोस्त हमेशा वही होटल चुनती है जिसमें मुफ़्त ब्रेकफ़ास्ट बुफ़े हो।"}, "hg": {"memory_turn": "Acche view ke alawa, mujhe aise hotels bahut pasand hain jinmein rooftop pool ho aur balcony mein hot tub ho. Main next month Miami ghoomne ka plan bana rahi hoon. Mujhe kis area mein rukna chahiye?", "assistant_reply": "South Beach lively hai aur samundar ke paas hai, Brickell modern hai aur restaurants ke liye accha hai, aur Mid-Beach shaant hai jahan bade resort hotels hain. Chunein ki aapko nightlife chahiye ya aaraam.", "followup_turn": "Beach pe sabse kam bheed kab hoti hai?", "followup_reply": "Weekdays mein subah-subah sabse shaant rehta hai. Late morning tak, especially weekends pe, popular hisse jaldi bhar jaate hain.", "q1": "Views ke alawa, mujhe hotel ke kaunse features pasand hain?", "q2": "Acche view ke alawa main hotel mein kaunsi amenities dhoondhti hoon?", "q3": "Miami trip ke liye main hotel search mein filters laga rahi hoon. Apne liye kaunse do amenity filters select karoon?", "distractor_sentence": "Meri friend hamesha wahi hotel chunti hai jismein free breakfast buffet ho."}}, "gold": {"answer_en": "a rooftop pool and a hot tub on the balcony", "slots": [{"name": "pool", "aliases": ["rooftop pool", "rooftop swimming pool", "रूफ़टॉप पूल", "रूफटॉप पूल", "छत पर पूल"]}, {"name": "hot_tub", "aliases": ["hot tub", "jacuzzi", "हॉट टब", "जकूज़ी", "जकूजी"]}]}, "distractor": {"value_aliases": ["breakfast buffet", "ब्रेकफ़ास्ट बुफ़े"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "0edc2aef", "lme_type": "single-session-preference"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 2623323494, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "P012", "cand_id": "PC080", "memory_type": "preference", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I love stand-up specials with great storytelling, and John Mulaney's Kid Gorgeous is my favourite for that. Can you recommend something similar to watch tonight?", "assistant_reply": "Try Mike Birbiglia's The New One or Hasan Minhaj's Homecoming King. Both are built around long, personal stories with strong callbacks rather than quick one-liners.", "followup_turn": "Which of those is better to watch with my parents?", "followup_reply": "Homecoming King is a good pick for family viewing. It's about growing up in an immigrant family and has warm, relatable moments.", "q1": "Which comedy special do I like for its storytelling?", "q2": "What is my favourite stand-up special when it comes to storytelling?", "distractor_sentence": "My roommate keeps rewatching Nanette, but it's not really my thing."}, "hi": {"memory_turn": "मुझे ऐसे स्टैंड-अप स्पेशल पसंद हैं जिनमें कहानी कहने का अंदाज़ बढ़िया हो, और इसके लिए जॉन मुलेनी का Kid Gorgeous मेरा सबसे पसंदीदा है। क्या आप आज रात देखने के लिए कुछ ऐसा ही सुझा सकते हैं?", "assistant_reply": "माइक बिर्बिग्लिया का The New One या हसन मिन्हाज का Homecoming King देखिए। दोनों छोटे-छोटे जोक्स की बजाय लंबी, निजी कहानियों पर बने हैं।", "followup_turn": "इनमें से कौन-सा मम्मी-पापा के साथ देखना बेहतर रहेगा?", "followup_reply": "परिवार के साथ देखने के लिए Homecoming King अच्छा है। यह एक प्रवासी परिवार में बड़े होने के बारे में है और इसमें कई अपनापन भरे पल हैं।", "q1": "कहानी कहने के अंदाज़ के लिए मुझे कौन-सा कॉमेडी स्पेशल पसंद है?", "q2": "स्टोरीटेलिंग के मामले में मेरा पसंदीदा स्टैंड-अप स्पेशल कौन-सा है?", "distractor_sentence": "मेरा रूममेट बार-बार Nanette देखता रहता है, पर मुझे वह ख़ास पसंद नहीं।"}, "hg": {"memory_turn": "Mujhe aise stand-up specials pasand hain jinmein storytelling badhiya ho, aur iske liye John Mulaney ka Kid Gorgeous mera sabse favourite hai. Kya aap aaj raat dekhne ke liye kuch aisa hi suggest kar sakte hain?", "assistant_reply": "Mike Birbiglia ka The New One ya Hasan Minhaj ka Homecoming King dekhiye. Dono chhote-chhote jokes ki bajaye lambi, personal kahaniyon pe bane hain.", "followup_turn": "Inmein se kaunsa mummy-papa ke saath dekhna better rahega?", "followup_reply": "Family ke saath dekhne ke liye Homecoming King accha hai. Yeh ek immigrant family mein bade hone ke baare mein hai aur ismein kai relatable moments hain.", "q1": "Storytelling ke liye mujhe kaunsa comedy special pasand hai?", "q2": "Storytelling ke maamle mein mera favourite stand-up special kaunsa hai?", "distractor_sentence": "Mera roommate baar-baar Nanette dekhta rehta hai, par mujhe woh khaas pasand nahi."}}, "gold": {"answer_en": "John Mulaney's Kid Gorgeous", "slots": [{"name": "special", "aliases": ["Kid Gorgeous", "किड गॉर्जियस"]}]}, "distractor": {"value_aliases": ["Nanette", "नैनेट"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "32260d93", "lme_type": "single-session-preference"}, "topic": "hobbies", "is_answerable": true, "evidence_position": 5, "filler_seed": 2182144218, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P013", "cand_id": "PC089", "memory_type": "preference", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I've been baking chocolate chip cookies a lot, and I find turbinado sugar gives them a richer flavour than regular white sugar. They still feel like they need something extra, though. Any ideas?", "assistant_reply": "Brown the butter first for a nutty depth, add a pinch of flaky sea salt on top, and rest the dough in the fridge for 24 hours. Chopped dark chocolate instead of chips also melts into nicer pools.", "followup_turn": "Does resting the dough really make a difference?", "followup_reply": "Yes. The flour absorbs more moisture and the flavours deepen, so the cookies bake up chewier and more caramel-like.", "q1": "Which sugar do I find gives a richer flavour?", "q2": "What kind of sugar do I think makes my cookies taste richer?"}, "hi": {"memory_turn": "मैं आजकल बहुत चॉकलेट चिप कुकीज़ बना रही हूँ, और मुझे लगता है कि सादी सफ़ेद चीनी से ज़्यादा टर्बिनाडो चीनी उनका स्वाद गहरा बनाती है। फिर भी उनमें कुछ कमी-सी लगती है। कोई आइडिया?", "assistant_reply": "पहले मक्खन को हल्का भूरा कर लें ताकि उसमें मेवे जैसा स्वाद आए, ऊपर थोड़ा फ़्लेकी नमक डालें और आटे को 24 घंटे फ़्रिज में रखें। चिप्स की जगह कटी हुई डार्क चॉकलेट भी बेहतर पिघलती है।", "followup_turn": "क्या आटे को रखने से सच में फ़र्क पड़ता है?", "followup_reply": "हाँ। मैदा ज़्यादा नमी सोख लेता है और स्वाद गहरा हो जाता है, जिससे कुकीज़ ज़्यादा चबाने वाली और कैरामेल जैसी बनती हैं।", "q1": "मुझे किस चीनी से ज़्यादा गहरा स्वाद मिलता है?", "q2": "मेरे हिसाब से कौन-सी चीनी मेरी कुकीज़ को ज़्यादा स्वादिष्ट बनाती है?"}, "hg": {"memory_turn": "Main aajkal bahut chocolate chip cookies bana rahi hoon, aur mujhe lagta hai ki normal white sugar se zyaada turbinado sugar unka flavour rich banati hai. Phir bhi unmein kuch kami-si lagti hai. Koi idea?", "assistant_reply": "Pehle butter ko halka brown kar lein taaki usmein nutty flavour aaye, upar thoda flaky salt daalein aur dough ko 24 ghante fridge mein rakhein. Chips ki jagah kati hui dark chocolate bhi better pighalti hai.", "followup_turn": "Kya dough ko rest karne se sach mein fark padta hai?", "followup_reply": "Haan. Maida zyaada moisture sokh leta hai aur flavour gehra ho jaata hai, jisse cookies zyaada chewy aur caramel jaisi banti hain.", "q1": "Mujhe kis sugar se zyaada rich flavour milta hai?", "q2": "Mere hisaab se kaunsi sugar meri cookies ko zyaada tasty banati hai?"}}, "gold": {"answer_en": "turbinado sugar", "slots": [{"name": "sugar", "aliases": ["turbinado", "टर्बिनाडो"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "38146c39", "lme_type": "single-session-preference"}, "topic": "other", "is_answerable": true, "evidence_position": 9, "filler_seed": 3357306071, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P014", "cand_id": "PC090", "memory_type": "preference", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I love baking lemon cakes; they're my favourite thing to bake. I'm inviting my colleagues over for a small gathering this Saturday. What else could I make that's easy to serve?", "assistant_reply": "Mini savoury tarts, a big tray of roasted vegetables with dip, and some cookies all work well. Pick things you can make the day before so you can enjoy the evening with your guests.", "followup_turn": "How many cookies should I make for eight people?", "followup_reply": "Plan on about three per person, so around two dozen. That leaves a few extra for anyone who wants seconds.", "q1": "What flavour of cake do I like baking?", "q2": "Which cake flavour is my favourite to bake?", "q3": "My colleague wants to bring a cake that matches my baking taste. Which flavour should she get?", "distractor_sentence": "My sister always bakes chocolate cakes for every party."}, "hi": {"memory_turn": "मुझे लेमन केक बनाना बहुत पसंद है, यह मेरी सबसे पसंदीदा बेकिंग है। इस शनिवार मैं अपने साथियों को घर पर एक छोटी पार्टी के लिए बुला रहा हूँ। और क्या बना सकता हूँ जो परोसने में आसान हो?", "assistant_reply": "छोटे नमकीन टार्ट, डिप के साथ भुनी सब्ज़ियों की एक बड़ी ट्रे और कुछ कुकीज़ अच्छे रहेंगे। ऐसी चीज़ें चुनें जो एक दिन पहले बन सकें, ताकि आप मेहमानों के साथ शाम का मज़ा ले सकें।", "followup_turn": "आठ लोगों के लिए कितनी कुकीज़ बनाऊँ?", "followup_reply": "हर व्यक्ति के लिए लगभग तीन, यानी करीब दो दर्जन। इससे जो दोबारा लेना चाहे, उसके लिए भी कुछ बच जाएँगी।", "q1": "मुझे किस फ़्लेवर का केक बनाना पसंद है?", "q2": "बेक करने के लिए मेरा पसंदीदा केक फ़्लेवर कौन-सा है?", "q3": "मेरी एक साथी मेरी बेकिंग की पसंद से मिलता हुआ केक लाना चाहती है। उसे कौन-सा फ़्लेवर लेना चाहिए?", "distractor_sentence": "मेरी बहन हर पार्टी के लिए हमेशा चॉकलेट केक बनाती है।"}, "hg": {"memory_turn": "Mujhe lemon cake bake karna bahut pasand hai, yeh meri sabse favourite baking hai. Is Saturday main apne colleagues ko ghar pe ek chhoti party ke liye bula raha hoon. Aur kya bana sakta hoon jo serve karne mein easy ho?", "assistant_reply": "Chhote savoury tarts, dip ke saath roasted veggies ki ek badi tray aur kuch cookies acche rahenge. Aisi cheezein chunein jo ek din pehle ban sakein, taaki aap mehmaanon ke saath shaam enjoy kar sakein.", "followup_turn": "Aath logon ke liye kitni cookies banaoon?", "followup_reply": "Har person ke liye lagbhag teen, yaani kareeb do dozen. Isse jo dobara lena chahe, uske liye bhi kuch bach jayengi.", "q1": "Mujhe kis flavour ka cake bake karna pasand hai?", "q2": "Bake karne ke liye mera favourite cake flavour kaunsa hai?", "q3": "Meri ek colleague meri baking pasand se match karta hua cake laana chahti hai. Use kaunsa flavour lena chahiye?", "distractor_sentence": "Meri behen har party ke liye hamesha chocolate cake banati hai."}}, "gold": {"answer_en": "lemon", "slots": [{"name": "flavour", "aliases": ["lemon", "लेमन", "नींबू", "nimbu", "nimboo"]}]}, "distractor": {"value_aliases": ["chocolate", "चॉकलेट"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "d24813b1", "lme_type": "single-session-preference"}, "topic": "food/cooking", "is_answerable": true, "evidence_position": 2, "filler_seed": 3755859386, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "P015", "cand_id": "PC098", "memory_type": "preference", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "For my weekly meal prep I like to pair my roasted vegetables with quinoa. I'm getting bored of the same recipes, though. Any new healthy ideas for next week?", "assistant_reply": "Try a chickpea and spinach curry, sheet-pan chicken with peppers, or a lentil and sweet potato stew. All three keep well in the fridge for four days and reheat nicely.", "followup_turn": "Can I freeze the lentil stew?", "followup_reply": "Yes, it freezes very well for up to three months. Cool it completely, store it in portions, and thaw it overnight in the fridge.", "q1": "What do I like to pair my meal-prep vegetables with?", "q2": "Which grain do I usually eat with my meal-prep vegetables?"}, "hi": {"memory_turn": "हर हफ़्ते के मील प्रेप में मुझे भुनी हुई सब्ज़ियों के साथ क्विनोआ खाना पसंद है। पर अब वही रेसिपी खाकर बोर हो गई हूँ। अगले हफ़्ते के लिए कोई नए हेल्दी आइडिया?", "assistant_reply": "छोले और पालक की करी, शिमला मिर्च के साथ ओवन में बना चिकन, या मसूर और शकरकंद का स्ट्यू आज़माइए। तीनों फ़्रिज में चार दिन ठीक रहते हैं और गर्म करने पर अच्छे लगते हैं।", "followup_turn": "क्या मसूर का स्ट्यू फ़्रीज़ कर सकती हूँ?", "followup_reply": "हाँ, यह तीन महीने तक अच्छे से फ़्रीज़ होता है। पूरा ठंडा करें, हिस्सों में रखें और रात भर फ़्रिज में पिघलाएँ।", "q1": "मील प्रेप की सब्ज़ियों के साथ मुझे क्या खाना पसंद है?", "q2": "मील प्रेप वाली सब्ज़ियों के साथ मैं आमतौर पर कौन-सा अनाज खाती हूँ?"}, "hg": {"memory_turn": "Har hafte ke meal prep mein mujhe roasted veggies ke saath quinoa khaana pasand hai. Par ab wahi recipes kha ke bore ho gayi hoon. Next week ke liye koi naye healthy ideas?", "assistant_reply": "Chhole aur palak ki curry, shimla mirch ke saath oven wala chicken, ya masoor aur shakarkand ka stew try kijiye. Teeno fridge mein chaar din theek rehte hain aur garam karne pe acche lagte hain.", "followup_turn": "Kya masoor ka stew freeze kar sakti hoon?", "followup_reply": "Haan, yeh teen mahine tak acche se freeze hota hai. Poora thanda karein, portions mein rakhein aur raat bhar fridge mein thaw karein.", "q1": "Meal prep ki veggies ke saath mujhe kya khaana pasand hai?", "q2": "Meal prep wali sabziyon ke saath main usually kaunsa grain khaati hoon?"}}, "gold": {"answer_en": "quinoa", "slots": [{"name": "grain", "aliases": ["quinoa", "क्विनोआ", "kinwa"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "b6025781", "lme_type": "single-session-preference"}, "topic": "other", "is_answerable": true, "evidence_position": 5, "filler_seed": 3414476156, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P016", "cand_id": "PC099", "memory_type": "preference", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I absolutely love Denver's music scene, and I'm planning a trip there soon. What should I do while I'm in town?", "assistant_reply": "Catch a show at Red Rocks Amphitheatre if it's concert season, explore the record shops on South Broadway, and take a day trip into the mountains for a hike.", "followup_turn": "Do I need to book Red Rocks tickets early?", "followup_reply": "Yes, popular shows often sell out weeks in advance. Buy from the official venue site and check the weather, since it's an open-air venue.", "q1": "Which city's music scene do I love?", "q2": "The music scene of which city do I really enjoy?", "q3": "I'm making a list of places to see live gigs. Which city should go at the top of my list?", "distractor_sentence": "My cousin keeps telling me Austin has the best live music, but I'm not convinced."}, "hi": {"memory_turn": "मुझे डेनवर का म्यूज़िक सीन बेहद पसंद है, और मैं जल्द ही वहाँ घूमने जाने का प्लान बना रहा हूँ। वहाँ रहते हुए मुझे क्या-क्या करना चाहिए?", "assistant_reply": "अगर कॉन्सर्ट का सीज़न हो तो Red Rocks Amphitheatre में कोई शो ज़रूर देखें, साउथ ब्रॉडवे की रिकॉर्ड शॉप घूमें, और पहाड़ों में एक दिन की हाइकिंग पर जाएँ।", "followup_turn": "क्या Red Rocks के टिकट पहले से बुक करने पड़ते हैं?", "followup_reply": "हाँ, मशहूर शो अक्सर हफ़्तों पहले बिक जाते हैं। ऑफ़िशियल वेबसाइट से ख़रीदें और मौसम देख लें, क्योंकि यह खुले आसमान के नीचे है।", "q1": "मुझे किस शहर का म्यूज़िक सीन पसंद है?", "q2": "मैं किस शहर के संगीत माहौल का बहुत आनंद लेता हूँ?", "q3": "मैं लाइव गिग देखने की जगहों की लिस्ट बना रहा हूँ। मेरी लिस्ट में सबसे ऊपर कौन-सा शहर होना चाहिए?", "distractor_sentence": "मेरा कज़िन कहता रहता है कि सबसे अच्छा लाइव म्यूज़िक ऑस्टिन में है, पर मैं नहीं मानता।"}, "hg": {"memory_turn": "Mujhe Denver ka music scene bahut pasand hai, aur main jaldi hi wahan ghoomne jaane ka plan bana raha hoon. Wahan rehte hue mujhe kya-kya karna chahiye?", "assistant_reply": "Agar concert season ho toh Red Rocks Amphitheatre mein koi show zaroor dekhein, South Broadway ki record shops ghoomein, aur pahadon mein ek din ki hiking pe jaayein.", "followup_turn": "Kya Red Rocks ke tickets pehle se book karne padte hain?", "followup_reply": "Haan, popular shows aksar hafton pehle sold out ho jaate hain. Official website se khareedein aur mausam check kar lein, kyunki yeh open-air venue hai.", "q1": "Mujhe kis city ka music scene pasand hai?", "q2": "Main kis sheher ke music scene ko bahut enjoy karta hoon?", "q3": "Main live gigs dekhne ki jagahon ki list bana raha hoon. Meri list mein sabse upar kaunsa city hona chahiye?", "distractor_sentence": "Mera cousin kehta rehta hai ki best live music Austin mein hai, par main nahi maanta."}}, "gold": {"answer_en": "Denver", "slots": [{"name": "city", "aliases": ["Denver", "डेनवर"]}]}, "distractor": {"value_aliases": ["Austin", "ऑस्टिन"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "a89d7624", "lme_type": "single-session-preference"}, "topic": "travel", "is_answerable": true, "evidence_position": 9, "filler_seed": 2015038825, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "P017", "cand_id": "PC103", "memory_type": "preference", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "On my commute to work I usually listen to true crime and self-improvement podcasts. Lately I want to use that time a bit differently. Any other ideas for the ride?", "assistant_reply": "You could learn a language with an audio course, listen to an audiobook series, or try short guided breathing sessions to arrive calmer. Calling a friend once a week is nice too.", "followup_turn": "Which language app works well without looking at the screen?", "followup_reply": "Pimsleur is designed to be audio-only, so it suits commuting well. Language Transfer is a free option that also works just by listening.", "q1": "Which podcast genres do I listen to on my commute?", "q2": "What kinds of podcasts do I usually play on the way to work?"}, "hi": {"memory_turn": "ऑफ़िस आते-जाते मैं आमतौर पर ट्रू क्राइम और सेल्फ़-इम्प्रूवमेंट वाले पॉडकास्ट सुनती हूँ। आजकल मैं उस समय का कुछ अलग इस्तेमाल करना चाहती हूँ। सफ़र के लिए कोई और आइडिया?", "assistant_reply": "आप ऑडियो कोर्स से कोई भाषा सीख सकती हैं, ऑडियोबुक की कोई सीरीज़ सुन सकती हैं, या शांत होकर पहुँचने के लिए छोटे गाइडेड ब्रीदिंग सेशन आज़मा सकती हैं। हफ़्ते में एक बार किसी दोस्त को फ़ोन करना भी अच्छा है।", "followup_turn": "कौन-सा भाषा ऐप स्क्रीन देखे बिना अच्छा चलता है?", "followup_reply": "Pimsleur सिर्फ़ ऑडियो के लिए बना है, इसलिए सफ़र में अच्छा रहता है। Language Transfer एक मुफ़्त विकल्प है जो सिर्फ़ सुनकर भी काम करता है।", "q1": "सफ़र में मैं किस तरह के पॉडकास्ट सुनती हूँ?", "q2": "ऑफ़िस जाते समय मैं आमतौर पर कौन-से पॉडकास्ट चलाती हूँ?"}, "hg": {"memory_turn": "Office aate-jaate main usually true crime aur self-improvement wale podcasts sunti hoon. Aajkal main us time ka kuch alag use karna chahti hoon. Commute ke liye koi aur ideas?", "assistant_reply": "Aap audio course se koi language seekh sakti hain, audiobook ki koi series sun sakti hain, ya calm hokar pahunchne ke liye chhote guided breathing sessions try kar sakti hain. Hafte mein ek baar kisi dost ko call karna bhi accha hai.", "followup_turn": "Kaunsa language app screen dekhe bina accha chalta hai?", "followup_reply": "Pimsleur sirf audio ke liye bana hai, isliye commute mein accha rehta hai. Language Transfer ek free option hai jo sirf sun ke bhi kaam karta hai.", "q1": "Commute mein main kis type ke podcasts sunti hoon?", "q2": "Office jaate time main usually kaunse podcasts chalati hoon?"}}, "gold": {"answer_en": "true crime and self-improvement", "slots": [{"name": "genre1", "aliases": ["true crime", "ट्रू क्राइम"]}, {"name": "genre2", "aliases": ["self-improvement", "self improvement", "self-help", "सेल्फ़-इम्प्रूवमेंट", "सेल्फ-इम्प्रूवमेंट", "आत्म-सुधार"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "1c0ddc50", "lme_type": "single-session-preference"}, "topic": "work/study", "is_answerable": true, "evidence_position": 2, "filler_seed": 3054638298, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": ["FL03", "FL25"]}
{"item_id": "P018", "cand_id": "PX0013", "memory_type": "preference", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "For gardening resources, I'm a big fan of the Gardening Know How website. I'm planning a small vegetable patch this spring. Which vegetables are easiest for a beginner?", "assistant_reply": "Lettuce, radishes, bush beans, zucchini and cherry tomatoes are all very forgiving. They grow quickly, need little special care, and give you results early in the season.", "followup_turn": "How much sun do they need?", "followup_reply": "Most of them need at least six hours of direct sun a day. Lettuce and radishes can manage with a little less, especially in warm weather.", "q1": "Which gardening website am I a big fan of?", "q2": "What is my favourite website for gardening advice?", "distractor_sentence": "My neighbour swears by Garden Gate magazine instead."}, "hi": {"memory_turn": "बाग़बानी की जानकारी के लिए मैं Gardening Know How वेबसाइट का बड़ा फ़ैन हूँ। इस वसंत में मैं एक छोटी सब्ज़ी की क्यारी लगाने का सोच रहा हूँ। नौसिखिए के लिए कौन-सी सब्ज़ियाँ सबसे आसान हैं?", "assistant_reply": "लेट्यूस, मूली, बीन्स, ज़ुकिनी और चेरी टमाटर बहुत आसान हैं। ये जल्दी उगते हैं, ख़ास देखभाल नहीं माँगते और सीज़न की शुरुआत में ही नतीजे देते हैं।", "followup_turn": "इन्हें कितनी धूप चाहिए?", "followup_reply": "ज़्यादातर को दिन में कम से कम छह घंटे सीधी धूप चाहिए। लेट्यूस और मूली थोड़ी कम धूप में भी चल जाती हैं, ख़ासकर गर्म मौसम में।", "q1": "मैं बाग़बानी की किस वेबसाइट का बड़ा फ़ैन हूँ?", "q2": "बाग़बानी की सलाह के लिए मेरी पसंदीदा वेबसाइट कौन-सी है?", "distractor_sentence": "मेरे पड़ोसी तो Garden Gate मैगज़ीन की ही क़सम खाते हैं।"}, "hg": {"memory_turn": "Gardening resources ke liye main Gardening Know How website ka bada fan hoon. Is spring mein main ek chhota vegetable patch lagane ka soch raha hoon. Beginner ke liye kaunsi sabziyan sabse easy hain?", "assistant_reply": "Lettuce, mooli, beans, zucchini aur cherry tomatoes bahut easy hain. Yeh jaldi ugte hain, khaas care nahi maangte aur season ki shuruaat mein hi results dete hain.", "followup_turn": "Inhe kitni dhoop chahiye?", "followup_reply": "Zyaadatar ko din mein kam se kam chhe ghante direct dhoop chahiye. Lettuce aur mooli thodi kam dhoop mein bhi chal jaati hain, especially garam mausam mein.", "q1": "Main gardening ki kis website ka bada fan hoon?", "q2": "Gardening advice ke liye meri favourite website kaunsi hai?", "distractor_sentence": "Mere padosi toh Garden Gate magazine ki hi kasam khaate hain."}}, "gold": {"answer_en": "Gardening Know How", "slots": [{"name": "website", "aliases": ["Gardening Know How", "गार्डनिंग नो हाउ"]}]}, "distractor": {"value_aliases": ["Garden Gate", "गार्डन गेट"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 5, "filler_seed": 410071893, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P019", "cand_id": "PX0070", "memory_type": "preference", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I'm really enjoying my Hatha yoga classes, and my favourite pose is probably downward-facing dog. How can I make my practice more consistent at home?", "assistant_reply": "Pick a fixed time, keep your mat rolled out somewhere visible, and start with just fifteen minutes. A simple sequence you repeat each day is easier to stick with than a new one every time.", "followup_turn": "Is it okay to practise right after dinner?", "followup_reply": "It's better to wait two to three hours after a full meal. Twists and inversions on a full stomach can feel uncomfortable.", "q1": "Which yoga pose is my favourite?", "q2": "What's my favourite pose in yoga?", "q3": "My instructor wants to end Friday's class with each student's best-loved asana. Which one should I name?"}, "hi": {"memory_turn": "मुझे अपनी हठ योग की क्लास बहुत अच्छी लग रही हैं, और मेरा सबसे पसंदीदा आसन शायद अधोमुख श्वानासन यानी डाउनवर्ड डॉग है। घर पर अपनी प्रैक्टिस नियमित कैसे बनाऊँ?", "assistant_reply": "एक तय समय चुनें, अपनी मैट ऐसी जगह बिछाकर रखें जहाँ दिखती रहे, और सिर्फ़ पंद्रह मिनट से शुरुआत करें। हर दिन दोहराया जाने वाला एक सरल क्रम, रोज़ नए क्रम से आसानी से निभता है।", "followup_turn": "क्या रात के खाने के तुरंत बाद योग करना ठीक है?", "followup_reply": "भरपेट खाने के बाद दो-तीन घंटे रुकना बेहतर है। भरे पेट पर ट्विस्ट और उलटे आसन असहज लग सकते हैं।", "q1": "मेरा पसंदीदा योग आसन कौन-सा है?", "q2": "योग में मुझे सबसे ज़्यादा कौन-सा आसन पसंद है?", "q3": "मेरी टीचर शुक्रवार की क्लास हर स्टूडेंट के सबसे प्यारे आसन से ख़त्म करना चाहती हैं। मुझे कौन-सा आसन बताना चाहिए?"}, "hg": {"memory_turn": "Mujhe apni Hatha yoga classes bahut acchi lag rahi hain, aur mera sabse favourite pose shayad downward-facing dog hai. Ghar pe apni practice regular kaise banaoon?", "assistant_reply": "Ek fixed time chunein, apni mat aisi jagah bichha ke rakhein jahan dikhti rahe, aur sirf pandrah minute se shuru karein. Roz repeat hone wala ek simple sequence, roz naye sequence se aasani se nibhta hai.", "followup_turn": "Kya dinner ke turant baad yoga karna theek hai?", "followup_reply": "Bharpet khaane ke baad do-teen ghante rukna better hai. Bhare pet pe twists aur inversions uncomfortable lag sakte hain.", "q1": "Mera favourite yoga pose kaunsa hai?", "q2": "Yoga mein mujhe sabse zyaada kaunsa pose pasand hai?", "q3": "Meri teacher Friday ki class har student ke sabse pyaare asana se khatam karna chahti hain. Mujhe kaunsa asana batana chahiye?"}}, "gold": {"answer_en": "downward-facing dog", "slots": [{"name": "pose", "aliases": ["downward-facing dog", "downward facing dog", "downward dog", "down dog", "adho mukha svanasana", "अधोमुख श्वानासन", "डाउनवर्ड डॉग", "डाउनवर्ड-फ़ेसिंग डॉग"]}]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 9, "filler_seed": 626500150, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": true, "abl_q3": true, "abl_digits": false, "banned_fillers": ["FL06", "FL31"]}
{"item_id": "P020", "cand_id": "PX0121", "memory_type": "preference", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I'm a big fan of scrambled eggs with spinach and feta for breakfast. I want to add more protein to my mornings. What else could I eat?", "assistant_reply": "Greek yogurt with nuts and seeds, cottage cheese on toast, or a smoothie with a scoop of protein powder all add a lot of protein. Leftover chicken or lentils in a wrap works too.", "followup_turn": "Is a protein smoothie filling enough on its own?", "followup_reply": "It can be if you add fibre and fat, like nuts, banana and peanut butter. Otherwise you may feel hungry again within a couple of hours.", "q1": "Which breakfast dish am I a big fan of?", "q2": "What is my favourite thing to eat for breakfast?", "distractor_sentence": "My roommate eats masala oats every single morning."}, "hi": {"memory_turn": "नाश्ते में मुझे पालक और फ़ेटा चीज़ वाले स्क्रैम्बल्ड एग्स बहुत पसंद हैं। मैं अपनी सुबह में और प्रोटीन जोड़ना चाहता हूँ। और क्या खा सकता हूँ?", "assistant_reply": "मेवे और बीज के साथ ग्रीक योगर्ट, टोस्ट पर पनीर, या प्रोटीन पाउडर वाली स्मूदी, सबमें काफ़ी प्रोटीन होता है। बचा हुआ चिकन या दाल रैप में डालकर भी खा सकते हैं।", "followup_turn": "क्या सिर्फ़ प्रोटीन स्मूदी से पेट भर जाएगा?", "followup_reply": "अगर उसमें फ़ाइबर और फ़ैट डालें, जैसे मेवे, केला और पीनट बटर, तो हाँ। वरना दो घंटे में फिर भूख लग सकती है।", "q1": "नाश्ते की किस डिश का मैं बड़ा फ़ैन हूँ?", "q2": "नाश्ते में मुझे सबसे ज़्यादा क्या खाना पसंद है?", "distractor_sentence": "मेरा रूममेट हर सुबह मसाला ओट्स ही खाता है।"}, "hg": {"memory_turn": "Breakfast mein mujhe spinach aur feta wale scrambled eggs bahut pasand hain. Main apni subah mein aur protein add karna chahta hoon. Aur kya kha sakta hoon?", "assistant_reply": "Nuts aur seeds ke saath Greek yogurt, toast pe paneer, ya protein powder wali smoothie, sabmein kaafi protein hota hai. Bacha hua chicken ya dal wrap mein daal ke bhi kha sakte hain.", "followup_turn": "Kya sirf protein smoothie se pet bhar jayega?", "followup_reply": "Agar usmein fibre aur fat daalein, jaise nuts, kela aur peanut butter, toh haan. Warna do ghante mein phir bhookh lag sakti hai.", "q1": "Breakfast ki kis dish ka main bada fan hoon?", "q2": "Breakfast mein mujhe sabse zyaada kya khaana pasand hai?", "distractor_sentence": "Mera roommate har subah masala oats hi khaata hai."}}, "gold": {"answer_en": "scrambled eggs with spinach and feta", "slots": [{"name": "dish", "aliases": ["scrambled eggs", "scrambled egg", "स्क्रैम्बल्ड एग्स", "स्क्रैम्बल्ड एग", "अंडा भुर्जी", "anda bhurji"]}]}, "distractor": {"value_aliases": ["masala oats", "मसाला ओट्स"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 2, "filler_seed": 3949397245, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "E011", "cand_id": "EC002", "memory_type": "event", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I'm researching the new hybrid version of my Toyota Corolla. By the way, I washed my current Corolla on February 27th after more than a month, since it was really dirty after the snowstorm. How fuel-efficient is the hybrid?", "assistant_reply": "The Corolla Hybrid is one of the most efficient compact cars, often around 20 to 25 kilometres per litre in mixed driving. It shines most in city traffic, where the electric motor does more of the work.", "followup_turn": "Does the hybrid battery need replacing often?", "followup_reply": "Not usually. Toyota hybrid batteries commonly last well over ten years, and they come with a long warranty in most markets.", "q1": "When did I last wash my Corolla?", "q2": "On what date did I wash my current Corolla?", "q3": "I want to keep a regular cleaning routine for my car. When was it last cleaned?", "distractor_sentence": "My brother washed his car on March 12th."}, "hi": {"memory_turn": "मैं अपनी Toyota Corolla के नए हाइब्रिड मॉडल के बारे में पढ़ रही हूँ। वैसे, मैंने अपनी मौजूदा Corolla को 27 फ़रवरी को धोया था, एक महीने से ज़्यादा के बाद, क्योंकि बर्फ़ीले तूफ़ान के बाद वह बहुत गंदी हो गई थी। हाइब्रिड का माइलेज कैसा है?", "assistant_reply": "Corolla Hybrid सबसे किफ़ायती कॉम्पैक्ट कारों में से है, मिली-जुली ड्राइविंग में अक्सर 20 से 25 किलोमीटर प्रति लीटर। शहर के ट्रैफ़िक में यह सबसे अच्छा चलता है, जहाँ इलेक्ट्रिक मोटर ज़्यादा काम करती है।", "followup_turn": "क्या हाइब्रिड बैटरी बार-बार बदलनी पड़ती है?", "followup_reply": "आमतौर पर नहीं। Toyota की हाइब्रिड बैटरियाँ अक्सर दस साल से ज़्यादा चलती हैं, और ज़्यादातर बाज़ारों में लंबी वारंटी के साथ आती हैं।", "q1": "मैंने आख़िरी बार अपनी Corolla कब धोई थी?", "q2": "मैंने अपनी मौजूदा Corolla किस तारीख़ को धोई?", "q3": "मैं अपनी गाड़ी की सफ़ाई का एक नियमित रूटीन रखना चाहती हूँ। पिछली बार वह कब साफ़ हुई थी?", "distractor_sentence": "मेरे भाई ने अपनी कार 12 मार्च को धोई थी।"}, "hg": {"memory_turn": "Main apni Toyota Corolla ke naye hybrid model ke baare mein padh rahi hoon. Waise, maine apni current Corolla ko 27 February ko dhoya tha, ek mahine se zyaada ke baad, kyunki snowstorm ke baad woh bahut gandi ho gayi thi. Hybrid ka mileage kaisa hai?", "assistant_reply": "Corolla Hybrid sabse efficient compact cars mein se hai, mixed driving mein aksar 20 se 25 km per litre. City traffic mein yeh sabse accha chalta hai, jahan electric motor zyaada kaam karti hai.", "followup_turn": "Kya hybrid battery baar-baar badalni padti hai?", "followup_reply": "Usually nahi. Toyota ki hybrid batteries aksar das saal se zyaada chalti hain, aur zyaadatar markets mein lambi warranty ke saath aati hain.", "q1": "Maine last time apni Corolla kab dhoyi thi?", "q2": "Maine apni current Corolla kis date ko dhoyi?", "q3": "Main apni gaadi ki safai ka ek regular routine rakhna chahti hoon. Pichhli baar woh kab saaf hui thi?", "distractor_sentence": "Mere bhai ne apni car 12 March ko dhoyi thi."}}, "gold": {"answer_en": "February 27", "date": {"day": 27, "month": 2}, "slots": [{"name": "date", "aliases": ["February 27", "February 27th", "27 February", "27th February", "27th of February", "Feb 27", "27 Feb", "27 फ़रवरी", "27 फरवरी"]}]}, "distractor": {"date": {"day": 12, "month": 3}, "value_aliases": ["March 12", "March 12th", "12 March", "12th March", "12th of March", "Mar 12", "12 Mar", "12 मार्च"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "gpt4_76048e76", "lme_type": "temporal-reasoning"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 3561391180, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": true, "abl_q3": true, "abl_digits": true, "banned_fillers": []}
{"item_id": "E012", "cand_id": "EC004", "memory_type": "event", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I'm packing for a trip to Hawaii and need adapters for my new Dell XPS 13. By the way, I pre-ordered the laptop on January 28th, and it arrived after a delay. Will I need a voltage converter?", "assistant_reply": "Probably not. Hawaii uses the same plugs and voltage as the rest of the United States, and laptop chargers usually accept a wide voltage range. Check the label on your charger to be sure.", "followup_turn": "Should I bring a power bank too?", "followup_reply": "It's a good idea for long beach days or flights. Pick one that supports USB-C power delivery so it can top up both your phone and the laptop.", "q1": "When did I pre-order my Dell XPS 13?", "q2": "On what date did I place the pre-order for my new laptop?", "distractor_sentence": "One of my colleagues ordered the same laptop on February 9th."}, "hi": {"memory_turn": "मैं हवाई की ट्रिप के लिए पैकिंग कर रहा हूँ और मुझे अपने नए Dell XPS 13 के लिए अडैप्टर चाहिए। वैसे, मैंने यह लैपटॉप 28 जनवरी को प्री-ऑर्डर किया था, और यह देर से पहुँचा। क्या मुझे वोल्टेज कन्वर्टर की ज़रूरत पड़ेगी?", "assistant_reply": "शायद नहीं। हवाई में बाक़ी अमेरिका जैसे ही प्लग और वोल्टेज हैं, और लैपटॉप चार्जर आमतौर पर कई वोल्टेज पर चल जाते हैं। पक्का करने के लिए चार्जर का लेबल देख लें।", "followup_turn": "क्या पावर बैंक भी साथ ले जाऊँ?", "followup_reply": "लंबे बीच वाले दिनों या फ़्लाइट के लिए अच्छा रहेगा। ऐसा चुनें जो USB-C पावर डिलीवरी सपोर्ट करे, ताकि फ़ोन और लैपटॉप दोनों चार्ज हो सकें।", "q1": "मैंने अपना Dell XPS 13 कब प्री-ऑर्डर किया था?", "q2": "मैंने अपने नए लैपटॉप का प्री-ऑर्डर किस तारीख़ को दिया था?", "distractor_sentence": "मेरे एक साथी ने वही लैपटॉप 9 फ़रवरी को ऑर्डर किया था।"}, "hg": {"memory_turn": "Main Hawaii trip ke liye packing kar raha hoon aur mujhe apne naye Dell XPS 13 ke liye adapters chahiye. Waise, maine yeh laptop 28 January ko pre-order kiya tha, aur yeh late pahuncha. Kya mujhe voltage converter ki zaroorat padegi?", "assistant_reply": "Shayad nahi. Hawaii mein baaki America jaise hi plugs aur voltage hain, aur laptop chargers usually kai voltages pe chal jaate hain. Pakka karne ke liye charger ka label check kar lein.", "followup_turn": "Kya power bank bhi saath le jaaun?", "followup_reply": "Lambe beach days ya flights ke liye accha rahega. Aisa chunein jo USB-C power delivery support kare, taaki phone aur laptop dono charge ho sakein.", "q1": "Maine apna Dell XPS 13 kab pre-order kiya tha?", "q2": "Maine apne naye laptop ka pre-order kis date ko diya tha?", "distractor_sentence": "Mere ek colleague ne wahi laptop 9 February ko order kiya tha."}}, "gold": {"answer_en": "January 28", "date": {"day": 28, "month": 1}, "slots": [{"name": "date", "aliases": ["January 28", "January 28th", "28 January", "28th January", "28th of January", "Jan 28", "28 Jan", "28 जनवरी"]}]}, "distractor": {"date": {"day": 9, "month": 2}, "value_aliases": ["February 9", "February 9th", "9 February", "9th February", "9th of February", "Feb 9", "9 Feb", "9 फ़रवरी", "9 फरवरी"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "gpt4_2312f94c", "lme_type": "temporal-reasoning"}, "topic": "shopping/home", "is_answerable": true, "evidence_position": 5, "filler_seed": 3357666641, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": true, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E013", "cand_id": "EC009", "memory_type": "event", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I just started some marigold seeds that arrived on March 3rd, and they're already germinating. I'm so excited to see them grow! When should I move the seedlings into bigger pots?", "assistant_reply": "Wait until each seedling has two sets of true leaves, usually two to three weeks after sprouting. Handle them by the leaves, not the stem, and water them well after moving.", "followup_turn": "Do marigolds need a lot of fertiliser?", "followup_reply": "No, too much fertiliser gives lots of leaves but fewer flowers. A light feed once a month is plenty.", "q1": "When did my marigold seeds arrive?", "q2": "On what date were the marigold seeds I'm growing delivered?"}, "hi": {"memory_turn": "मैंने गेंदे के कुछ बीज बोए हैं जो 3 मार्च को आए थे, और वे अभी से अंकुरित हो रहे हैं। उन्हें बढ़ते देखने के लिए मैं बहुत उत्साहित हूँ! पौधों को बड़े गमलों में कब लगाना चाहिए?", "assistant_reply": "जब हर पौधे पर असली पत्तियों के दो जोड़े आ जाएँ, आमतौर पर अंकुर निकलने के दो-तीन हफ़्ते बाद। उन्हें तने से नहीं, पत्तियों से पकड़ें, और लगाने के बाद अच्छे से पानी दें।", "followup_turn": "क्या गेंदे को बहुत खाद चाहिए?", "followup_reply": "नहीं, ज़्यादा खाद से पत्तियाँ तो ख़ूब आती हैं पर फूल कम। महीने में एक बार हल्की खाद काफ़ी है।", "q1": "मेरे गेंदे के बीज कब आए थे?", "q2": "जो गेंदे के बीज मैं उगा रही हूँ, वे किस तारीख़ को डिलीवर हुए थे?"}, "hg": {"memory_turn": "Maine marigold ke kuch seeds boye hain jo 3 March ko aaye the, aur woh abhi se germinate ho rahe hain. Unhe badhte dekhne ke liye main bahut excited hoon! Seedlings ko bade gamlon mein kab lagana chahiye?", "assistant_reply": "Jab har seedling pe asli pattiyon ke do jode aa jaayein, usually ankur nikalne ke do-teen hafte baad. Unhe stem se nahi, pattiyon se pakdein, aur lagane ke baad acche se paani dein.", "followup_turn": "Kya marigold ko bahut khaad chahiye?", "followup_reply": "Nahi, zyaada khaad se pattiyan toh khoob aati hain par phool kam. Mahine mein ek baar halki khaad kaafi hai.", "q1": "Mere marigold ke seeds kab aaye the?", "q2": "Jo marigold seeds main uga rahi hoon, woh kis date ko deliver hue the?"}}, "gold": {"answer_en": "March 3", "date": {"day": 3, "month": 3}, "slots": [{"name": "date", "aliases": ["March 3", "March 3rd", "3 March", "3rd March", "3rd of March", "Mar 3", "3 Mar", "3 मार्च"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "gpt4_385a5000", "lme_type": "temporal-reasoning"}, "topic": "other", "is_answerable": true, "evidence_position": 9, "filler_seed": 3413521209, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E014", "cand_id": "EC010", "memory_type": "event", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I've been starting seeds indoors under grow lights in my basement since February 20th. The tomatoes, peppers and cucumbers are all doing well. When can I move them outside?", "assistant_reply": "After your last frost, once night temperatures stay above about 10°C. Harden them off first by putting them outside for a few hours a day for a week, increasing the time gradually.", "followup_turn": "How long should the grow lights stay on each day?", "followup_reply": "Around 14 to 16 hours a day works well for most vegetable seedlings. Keep the lights a few centimetres above the plants and raise them as they grow.", "q1": "Since when have I been starting seeds indoors under grow lights?", "q2": "On what date did I start growing seeds under the grow lights in my basement?", "q3": "My neighbour asked how long my basement seedlings have been under the lamps. From which date should I count?", "distractor_sentence": "My uncle sowed his seeds outdoors on April 5th."}, "hi": {"memory_turn": "मैं 20 फ़रवरी से अपने बेसमेंट में ग्रो लाइट के नीचे घर के अंदर बीज उगा रहा हूँ। टमाटर, शिमला मिर्च और खीरे सब अच्छे से बढ़ रहे हैं। इन्हें बाहर कब ले जा सकता हूँ?", "assistant_reply": "आख़िरी पाले के बाद, जब रात का तापमान लगातार लगभग 10°C से ऊपर रहने लगे। पहले एक हफ़्ते तक इन्हें रोज़ कुछ घंटे बाहर रखकर आदत डालें, और धीरे-धीरे समय बढ़ाएँ।", "followup_turn": "ग्रो लाइट रोज़ कितनी देर जलानी चाहिए?", "followup_reply": "ज़्यादातर सब्ज़ियों के पौधों के लिए दिन में 14 से 16 घंटे ठीक हैं। लाइट को पौधों से कुछ सेंटीमीटर ऊपर रखें और बढ़ने पर ऊपर करते जाएँ।", "q1": "मैं कब से घर के अंदर ग्रो लाइट के नीचे बीज उगा रहा हूँ?", "q2": "मैंने अपने बेसमेंट में ग्रो लाइट के नीचे बीज किस तारीख़ से उगाने शुरू किए?", "q3": "मेरे पड़ोसी ने पूछा कि बेसमेंट वाले पौधे कितने समय से लैंप के नीचे हैं। मुझे किस तारीख़ से गिनना चाहिए?", "distractor_sentence": "मेरे चाचा ने अपने बीज 5 अप्रैल को बाहर बोए थे।"}, "hg": {"memory_turn": "Main 20 February se apne basement mein grow lights ke neeche ghar ke andar seeds uga raha hoon. Tomatoes, shimla mirch aur kheere sab acche se badh rahe hain. Inhe bahar kab le ja sakta hoon?", "assistant_reply": "Last frost ke baad, jab raat ka temperature lagataar lagbhag 10°C se upar rehne lage. Pehle ek hafte tak inhe roz kuch ghante bahar rakh ke aadat daalein, aur dheere-dheere time badhayein.", "followup_turn": "Grow lights roz kitni der jalani chahiye?", "followup_reply": "Zyaadatar vegetable seedlings ke liye din mein 14 se 16 ghante theek hain. Lights ko plants se kuch centimetre upar rakhein aur badhne pe upar karte jaayein.", "q1": "Main kab se ghar ke andar grow lights ke neeche seeds uga raha hoon?", "q2": "Maine apne basement mein grow lights ke neeche seeds kis date se ugane shuru kiye?", "q3": "Mere padosi ne poocha ki basement wale plants kitne time se lamps ke neeche hain. Mujhe kis date se ginna chahiye?", "distractor_sentence": "Mere chacha ne apne seeds 5 April ko bahar boye the."}}, "gold": {"answer_en": "February 20", "date": {"day": 20, "month": 2}, "slots": [{"name": "date", "aliases": ["February 20", "February 20th", "20 February", "20th February", "20th of February", "Feb 20", "20 Feb", "20 फ़रवरी", "20 फरवरी"]}]}, "distractor": {"date": {"day": 5, "month": 4}, "value_aliases": ["April 5", "April 5th", "5 April", "5th April", "5th of April", "Apr 5", "5 Apr", "5 अप्रैल"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "gpt4_385a5000", "lme_type": "temporal-reasoning"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 3498298790, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": true, "abl_q3": true, "abl_digits": true, "banned_fillers": []}
{"item_id": "E015", "cand_id": "EC013", "memory_type": "event", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I took part in the Turbocharged Tuesdays racing event at the local track on June 14th and set a personal best of 12.8 seconds in the quarter-mile drag race! How can I improve my launch off the line?", "assistant_reply": "Practise consistent reaction times with a tree simulator app, check your tyre pressure for better grip, and avoid too much wheelspin by easing into the throttle rather than flooring it instantly.", "followup_turn": "Do I need special tyres for drag racing?", "followup_reply": "Not to start with. Many people race on regular performance tyres. Drag radials help a lot later, but they wear quickly on normal roads.", "q1": "When did I take part in the Turbocharged Tuesdays racing event?", "q2": "On what date was the Turbocharged Tuesdays event where I raced?"}, "hi": {"memory_turn": "मैंने 14 जून को लोकल ट्रैक पर Turbocharged Tuesdays रेसिंग इवेंट में हिस्सा लिया और क्वार्टर-माइल ड्रैग रेस में 12.8 सेकंड का अपना सबसे अच्छा टाइम निकाला! लाइन से तेज़ शुरुआत कैसे सुधारूँ?", "assistant_reply": "ट्री सिम्युलेटर ऐप से रिएक्शन टाइम की प्रैक्टिस करें, बेहतर पकड़ के लिए टायर प्रेशर जाँचें, और एकदम पूरा एक्सेलरेटर दबाने की बजाय धीरे से दबाकर व्हीलस्पिन कम करें।", "followup_turn": "क्या ड्रैग रेसिंग के लिए ख़ास टायर चाहिए?", "followup_reply": "शुरुआत में नहीं। बहुत लोग सामान्य परफ़ॉर्मेंस टायर पर ही रेस करते हैं। आगे चलकर ड्रैग रेडियल बहुत मदद करते हैं, पर सामान्य सड़कों पर जल्दी घिसते हैं।", "q1": "मैंने Turbocharged Tuesdays रेसिंग इवेंट में कब हिस्सा लिया था?", "q2": "जिस Turbocharged Tuesdays इवेंट में मैंने रेस की, वह किस तारीख़ को था?"}, "hg": {"memory_turn": "Maine 14 June ko local track pe Turbocharged Tuesdays racing event mein hissa liya aur quarter-mile drag race mein 12.8 second ka apna best time nikala! Line se fast start kaise improve karoon?", "assistant_reply": "Tree simulator app se reaction time ki practice karein, better grip ke liye tyre pressure check karein, aur ekdum full throttle ki bajaye dheere se daba ke wheelspin kam karein.", "followup_turn": "Kya drag racing ke liye special tyres chahiye?", "followup_reply": "Shuru mein nahi. Bahut log normal performance tyres pe hi race karte hain. Aage chal ke drag radials bahut help karte hain, par normal roads pe jaldi ghiste hain.", "q1": "Maine Turbocharged Tuesdays racing event mein kab hissa liya tha?", "q2": "Jis Turbocharged Tuesdays event mein maine race ki, woh kis date ko tha?"}}, "gold": {"answer_en": "June 14", "date": {"day": 14, "month": 6}, "slots": [{"name": "date", "aliases": ["June 14", "June 14th", "14 June", "14th June", "14th of June", "Jun 14", "14 Jun", "14 जून"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "bbf86515", "lme_type": "temporal-reasoning"}, "topic": "other", "is_answerable": true, "evidence_position": 5, "filler_seed": 435984637, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E016", "cand_id": "EC014", "memory_type": "event", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I just got back from the Rack Fest car show in a nearby city on June 18th, and it was amazing! I saw so many modified cars. How do I start modifying my own car on a small budget?", "assistant_reply": "Start with things that improve the driving feel: good tyres, better brake pads and a proper alignment. After that, a simple wheel upgrade changes the look a lot without a big cost.", "followup_turn": "Will modifications affect my car insurance?", "followup_reply": "They can. Tell your insurer about any changes, because undeclared modifications may let them refuse a claim. Some changes raise the premium only slightly.", "q1": "When did I go to the Rack Fest car show?", "q2": "On what date did I attend Rack Fest?"}, "hi": {"memory_turn": "मैं अभी पास के शहर में 18 जून को हुए Rack Fest कार शो से लौटा हूँ, और वह शानदार था! मैंने बहुत सारी मॉडिफ़ाइड कारें देखीं। कम बजट में अपनी कार मॉडिफ़ाई करना कैसे शुरू करूँ?", "assistant_reply": "उन चीज़ों से शुरू करें जो ड्राइविंग का अनुभव बेहतर बनाती हैं: अच्छे टायर, बेहतर ब्रेक पैड और सही अलाइनमेंट। उसके बाद व्हील अपग्रेड बिना ज़्यादा ख़र्च के लुक बहुत बदल देता है।", "followup_turn": "क्या मॉडिफ़िकेशन से मेरी कार का बीमा प्रभावित होगा?", "followup_reply": "हो सकता है। बीमा कंपनी को हर बदलाव के बारे में बताएँ, क्योंकि बिना बताए बदलाव होने पर वे क्लेम ठुकरा सकते हैं। कुछ बदलावों से प्रीमियम बस थोड़ा बढ़ता है।", "q1": "मैं Rack Fest कार शो में कब गया था?", "q2": "मैं किस तारीख़ को Rack Fest में शामिल हुआ था?"}, "hg": {"memory_turn": "Main abhi paas ke sheher mein 18 June ko hue Rack Fest car show se lauta hoon, aur woh amazing tha! Maine bahut saari modified cars dekhin. Kam budget mein apni car modify karna kaise shuru karoon?", "assistant_reply": "Un cheezon se shuru karein jo driving feel better banati hain: acche tyres, better brake pads aur sahi alignment. Uske baad wheel upgrade bina zyaada kharche ke look bahut badal deta hai.", "followup_turn": "Kya modifications se meri car ka insurance affect hoga?", "followup_reply": "Ho sakta hai. Insurance company ko har change ke baare mein batayein, kyunki bina bataye changes hone pe woh claim reject kar sakte hain. Kuch changes se premium bas thoda badhta hai.", "q1": "Main Rack Fest car show mein kab gaya tha?", "q2": "Main kis date ko Rack Fest mein gaya tha?"}}, "gold": {"answer_en": "June 18", "date": {"day": 18, "month": 6}, "slots": [{"name": "date", "aliases": ["June 18", "June 18th", "18 June", "18th June", "18th of June", "Jun 18", "18 Jun", "18 जून"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "bbf86515", "lme_type": "temporal-reasoning"}, "topic": "other", "is_answerable": true, "evidence_position": 9, "filler_seed": 1918348445, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E017", "cand_id": "EC015", "memory_type": "event", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I recently volunteered at the Coastal Cleanup event on March 7th, and it was amazing to see the impact we made on our beaches. Can you suggest other environmental volunteering I could try?", "assistant_reply": "Tree-planting drives, river and lake cleanups, and helping at a local recycling or composting centre are all good options. Many city councils and nature groups post upcoming events on their websites.", "followup_turn": "Do I need any training for tree planting?", "followup_reply": "Usually not. Organisers show you how to dig the hole, place the sapling and firm the soil. Just bring gloves, water and comfortable shoes.", "q1": "When did I volunteer at the Coastal Cleanup event?", "q2": "On what date was the Coastal Cleanup I volunteered at?", "q3": "I'm filling in my volunteer hours log. What date should I enter for the beach work I did?", "distractor_sentence": "My friend joined a river cleanup on April 19th."}, "hi": {"memory_turn": "हाल ही में मैंने 7 मार्च को Coastal Cleanup इवेंट में वॉलंटियर किया, और हमारे बीच पर जो असर हुआ वह देखकर बहुत अच्छा लगा। क्या आप पर्यावरण से जुड़ा कोई और वॉलंटियर काम सुझा सकते हैं?", "assistant_reply": "पेड़ लगाने के अभियान, नदी और झील की सफ़ाई, और लोकल रीसाइक्लिंग या कम्पोस्टिंग सेंटर में मदद, सब अच्छे विकल्प हैं। कई नगर निगम और प्रकृति से जुड़े समूह अपनी वेबसाइट पर आने वाले इवेंट डालते हैं।", "followup_turn": "क्या पेड़ लगाने के लिए कोई ट्रेनिंग चाहिए?", "followup_reply": "आमतौर पर नहीं। आयोजक बताते हैं कि गड्ढा कैसे खोदें, पौधा कैसे रखें और मिट्टी कैसे दबाएँ। बस दस्ताने, पानी और आरामदायक जूते ले जाएँ।", "q1": "मैंने Coastal Cleanup इवेंट में कब वॉलंटियर किया था?", "q2": "जिस Coastal Cleanup में मैंने वॉलंटियर किया, वह किस तारीख़ को था?", "q3": "मैं अपने वॉलंटियर घंटों का रिकॉर्ड भर रही हूँ। बीच पर किए गए काम के लिए कौन-सी तारीख़ लिखूँ?", "distractor_sentence": "मेरी दोस्त 19 अप्रैल को नदी की सफ़ाई में शामिल हुई थी।"}, "hg": {"memory_turn": "Haal hi mein maine 7 March ko Coastal Cleanup event mein volunteer kiya, aur hamare beaches pe jo impact hua woh dekh ke bahut accha laga. Kya aap environment se juda koi aur volunteering kaam suggest kar sakte hain?", "assistant_reply": "Tree-planting drives, nadi aur jheel ki safai, aur local recycling ya composting centre mein help, sab acche options hain. Kai city councils aur nature groups apni website pe upcoming events daalte hain.", "followup_turn": "Kya tree planting ke liye koi training chahiye?", "followup_reply": "Usually nahi. Organisers batate hain ki gaddha kaise khodein, paudha kaise rakhein aur mitti kaise dabayein. Bas gloves, paani aur comfortable shoes le jaayein.", "q1": "Maine Coastal Cleanup event mein kab volunteer kiya tha?", "q2": "Jis Coastal Cleanup mein maine volunteer kiya, woh kis date ko tha?", "q3": "Main apne volunteer hours ka record bhar rahi hoon. Beach pe kiye gaye kaam ke liye kaunsi date likhun?", "distractor_sentence": "Meri friend 19 April ko river cleanup mein shaamil hui thi."}}, "gold": {"answer_en": "March 7", "date": {"day": 7, "month": 3}, "slots": [{"name": "date", "aliases": ["March 7", "March 7th", "7 March", "7th March", "7th of March", "Mar 7", "7 Mar", "7 मार्च"]}]}, "distractor": {"date": {"day": 19, "month": 4}, "value_aliases": ["April 19", "April 19th", "19 April", "19th April", "19th of April", "Apr 19", "19 Apr", "19 अप्रैल"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "f0853d11", "lme_type": "temporal-reasoning"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 2094459473, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": true, "abl_digits": true, "banned_fillers": []}
{"item_id": "E018", "cand_id": "EC019", "memory_type": "event", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I attended a charity golf tournament on July 17th, and while I had a blast playing with colleagues, I realised I need to work on my swing. What drills can help a beginner?", "assistant_reply": "Try the towel drill, holding a towel under both armpits to keep your arms connected, and slow half-swings focused on a smooth tempo. Hitting balls with your feet together also improves balance.", "followup_turn": "How often should I practise at the range?", "followup_reply": "Two short sessions a week are better than one long one. Focus on a single drill each time instead of hitting a whole bucket without a plan.", "q1": "When did I attend the charity golf tournament?", "q2": "On what date did I play in the charity golf tournament?", "q3": "My manager wants to thank everyone who represented the company at the golf fundraiser. What date did I play there?", "distractor_sentence": "My boss played in a different golf tournament on August 2nd."}, "hi": {"memory_turn": "मैं 17 जुलाई को एक चैरिटी गोल्फ़ टूर्नामेंट में गया था, और साथियों के साथ खेलकर मज़ा तो बहुत आया, पर समझ आया कि मुझे अपने स्विंग पर काम करना होगा। नौसिखिए के लिए कौन-सी ड्रिल मदद करेंगी?", "assistant_reply": "तौलिया ड्रिल आज़माइए, जिसमें बाँहें जुड़ी रखने के लिए दोनों बगलों में तौलिया दबाते हैं, और आराम की लय पर ध्यान देते हुए धीमे हाफ़-स्विंग करें। पैर साथ रखकर गेंद मारने से संतुलन भी सुधरता है।", "followup_turn": "मुझे रेंज पर कितनी बार प्रैक्टिस करनी चाहिए?", "followup_reply": "एक लंबे सेशन से हफ़्ते में दो छोटे सेशन बेहतर हैं। हर बार बिना योजना के पूरी बाल्टी मारने की बजाय एक ड्रिल पर ध्यान दें।", "q1": "मैं चैरिटी गोल्फ़ टूर्नामेंट में कब गया था?", "q2": "मैंने चैरिटी गोल्फ़ टूर्नामेंट किस तारीख़ को खेला था?", "q3": "मेरे मैनेजर गोल्फ़ वाले फ़ंडरेज़र में कंपनी की तरफ़ से खेलने वाले सबको धन्यवाद देना चाहते हैं। मैं वहाँ किस तारीख़ को खेला था?", "distractor_sentence": "मेरे बॉस 2 अगस्त को एक दूसरे गोल्फ़ टूर्नामेंट में खेले थे।"}, "hg": {"memory_turn": "Main 17 July ko ek charity golf tournament mein gaya tha, aur colleagues ke saath khel ke maza toh bahut aaya, par samajh aaya ki mujhe apne swing pe kaam karna hoga. Beginner ke liye kaunse drills help karenge?", "assistant_reply": "Towel drill try kijiye, jismein arms connected rakhne ke liye dono baghlon mein towel dabate hain, aur smooth tempo pe dhyaan dete hue slow half-swings karein. Pair saath rakh ke ball maarne se balance bhi sudharta hai.", "followup_turn": "Mujhe range pe kitni baar practice karni chahiye?", "followup_reply": "Ek lambe session se hafte mein do chhote sessions better hain. Har baar bina plan ke poori bucket maarne ki bajaye ek drill pe focus karein.", "q1": "Main charity golf tournament mein kab gaya tha?", "q2": "Maine charity golf tournament kis date ko khela tha?", "q3": "Mere manager golf fundraiser mein company ki taraf se khelne wale sabko thank you bolna chahte hain. Main wahan kis date ko khela tha?", "distractor_sentence": "Mere boss 2 August ko ek doosre golf tournament mein khele the."}}, "gold": {"answer_en": "July 17", "date": {"day": 17, "month": 7}, "slots": [{"name": "date", "aliases": ["July 17", "July 17th", "17 July", "17th July", "17th of July", "Jul 17", "17 Jul", "17 जुलाई"]}]}, "distractor": {"date": {"day": 2, "month": 8}, "value_aliases": ["August 2", "August 2nd", "2 August", "2nd August", "2nd of August", "Aug 2", "2 Aug", "2 अगस्त"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "a3838d2b", "lme_type": "temporal-reasoning"}, "topic": "work/study", "is_answerable": true, "evidence_position": 5, "filler_seed": 2626614892, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": true, "abl_q3": true, "abl_digits": true, "banned_fillers": []}
{"item_id": "E019", "cand_id": "EC020", "memory_type": "event", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I volunteered at the Food for Thought charity gala on September 25th, which was a great experience. Now I'd like to organise a small fundraiser myself. Where should I start?", "assistant_reply": "Choose one clear cause and a realistic goal, then pick a simple format like a bake sale or a quiz night. Ask a local café or community hall to host it for free in exchange for publicity.", "followup_turn": "How do I get people to actually show up?", "followup_reply": "Personal invitations work best. Ask each helper to bring five friends, post reminders a week and a day before, and make it easy to donate online for anyone who can't come.", "q1": "When did I volunteer at the Food for Thought charity gala?", "q2": "On what date was the Food for Thought gala where I volunteered?"}, "hi": {"memory_turn": "मैंने 25 सितंबर को Food for Thought चैरिटी गाला में वॉलंटियर किया था, जो बहुत अच्छा अनुभव रहा। अब मैं ख़ुद एक छोटा फ़ंडरेज़र आयोजित करना चाहती हूँ। कहाँ से शुरू करूँ?", "assistant_reply": "एक साफ़ मक़सद और एक व्यावहारिक लक्ष्य चुनें, फिर बेक सेल या क्विज़ नाइट जैसा कोई आसान तरीक़ा तय करें। किसी लोकल कैफ़े या सामुदायिक हॉल से प्रचार के बदले मुफ़्त में जगह देने को कहें।", "followup_turn": "लोगों को सच में आने के लिए कैसे मनाऊँ?", "followup_reply": "निजी न्योते सबसे अच्छे काम करते हैं। हर मददगार से पाँच दोस्त लाने को कहें, एक हफ़्ता और एक दिन पहले याद दिलाएँ, और जो न आ सकें उनके लिए ऑनलाइन दान आसान बनाएँ।", "q1": "मैंने Food for Thought चैरिटी गाला में कब वॉलंटियर किया था?", "q2": "जिस Food for Thought गाला में मैंने वॉलंटियर किया, वह किस तारीख़ को था?"}, "hg": {"memory_turn": "Maine 25 September ko Food for Thought charity gala mein volunteer kiya tha, jo bahut accha experience raha. Ab main khud ek chhota fundraiser organise karna chahti hoon. Kahan se shuru karoon?", "assistant_reply": "Ek clear cause aur ek realistic goal chunein, phir bake sale ya quiz night jaisa koi simple format decide karein. Kisi local cafe ya community hall se publicity ke badle free mein jagah dene ko kahein.", "followup_turn": "Logon ko sach mein aane ke liye kaise manaun?", "followup_reply": "Personal invites sabse accha kaam karte hain. Har helper se paanch dost laane ko kahein, ek hafta aur ek din pehle reminder dein, aur jo na aa sakein unke liye online donation easy banayein.", "q1": "Maine Food for Thought charity gala mein kab volunteer kiya tha?", "q2": "Jis Food for Thought gala mein maine volunteer kiya, woh kis date ko tha?"}}, "gold": {"answer_en": "September 25", "date": {"day": 25, "month": 9}, "slots": [{"name": "date", "aliases": ["September 25", "September 25th", "25 September", "25th September", "25th of September", "Sep 25", "25 Sep", "25 सितंबर", "25 सितम्बर"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "a3838d2b", "lme_type": "temporal-reasoning"}, "topic": "food/cooking", "is_answerable": true, "evidence_position": 9, "filler_seed": 319567452, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E020", "cand_id": "EC022", "memory_type": "event", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I went to a backyard BBQ party at my colleague's house on June 3rd, and they had an amazing selection of BBQ sauces. I want to make my own sauce now. Any simple recipe ideas?", "assistant_reply": "Start with ketchup, apple cider vinegar, brown sugar, smoked paprika, garlic powder and a little mustard. Simmer for fifteen minutes and adjust the sweetness and tang to your taste.", "followup_turn": "How long does homemade sauce keep?", "followup_reply": "In a clean, sealed jar in the fridge it usually lasts about two weeks. Use a clean spoon each time so it doesn't spoil early.", "q1": "When did I go to the backyard BBQ at my colleague's house?", "q2": "On what date was the BBQ party I attended at my colleague's place?", "distractor_sentence": "My neighbours are hosting their own BBQ on June 24th."}, "hi": {"memory_turn": "मैं 3 जून को अपने एक साथी के घर के आँगन में हुई BBQ पार्टी में गया था, और वहाँ BBQ सॉस की कमाल की वैरायटी थी। अब मैं अपना सॉस बनाना चाहता हूँ। कोई आसान रेसिपी?", "assistant_reply": "केचप, ऐप्पल साइडर विनेगर, ब्राउन शुगर, स्मोक्ड पपरिका, लहसुन पाउडर और थोड़ी सरसों से शुरू करें। पंद्रह मिनट धीमी आँच पर पकाएँ और मिठास व खटास अपने स्वाद के हिसाब से ठीक करें।", "followup_turn": "घर का बना सॉस कितने दिन चलता है?", "followup_reply": "साफ़, बंद जार में फ़्रिज में रखें तो आमतौर पर लगभग दो हफ़्ते चलता है। हर बार साफ़ चम्मच इस्तेमाल करें ताकि यह जल्दी ख़राब न हो।", "q1": "मैं अपने साथी के घर के आँगन वाली BBQ पार्टी में कब गया था?", "q2": "अपने साथी के घर जिस BBQ पार्टी में मैं गया, वह किस तारीख़ को थी?", "distractor_sentence": "मेरे पड़ोसी 24 जून को अपनी BBQ पार्टी रख रहे हैं।"}, "hg": {"memory_turn": "Main 3 June ko apne ek colleague ke ghar ke backyard mein hui BBQ party mein gaya tha, aur wahan BBQ sauces ki kamaal ki variety thi. Ab main apna sauce banana chahta hoon. Koi easy recipe?", "assistant_reply": "Ketchup, apple cider vinegar, brown sugar, smoked paprika, garlic powder aur thodi mustard se shuru karein. Pandrah minute dheemi aanch pe pakayein aur meethapan aur khattapan apne taste ke hisaab se adjust karein.", "followup_turn": "Ghar ka bana sauce kitne din chalta hai?", "followup_reply": "Saaf, band jar mein fridge mein rakhein toh usually lagbhag do hafte chalta hai. Har baar saaf chammach use karein taaki yeh jaldi kharab na ho.", "q1": "Main apne colleague ke ghar ke backyard wali BBQ party mein kab gaya tha?", "q2": "Apne colleague ke ghar jis BBQ party mein main gaya, woh kis date ko thi?", "distractor_sentence": "Mere padosi 24 June ko apni BBQ party rakh rahe hain."}}, "gold": {"answer_en": "June 3", "date": {"day": 3, "month": 6}, "slots": [{"name": "date", "aliases": ["June 3", "June 3rd", "3 June", "3rd June", "3rd of June", "Jun 3", "3 Jun", "3 जून"]}]}, "distractor": {"date": {"day": 24, "month": 6}, "value_aliases": ["June 24", "June 24th", "24 June", "24th June", "24th of June", "Jun 24", "24 Jun", "24 जून"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "gpt4_4edbafa2", "lme_type": "temporal-reasoning"}, "topic": "family/social", "is_answerable": true, "evidence_position": 2, "filler_seed": 2177627882, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "F021", "cand_id": "FC123", "memory_type": "fact", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I'm trying to get better at packing because I always overpack. On my 5-day trip to Costa Rica I brought 7 shirts and ended up wearing barely half of them. How can I pack lighter next time?", "assistant_reply": "Plan outfits around one or two colours so everything mixes, pack for about half the days and do a quick laundry midway, and wear your bulkiest shoes on the plane.", "followup_turn": "Are packing cubes actually worth it?", "followup_reply": "For many people, yes. They keep items grouped, make it easy to see what you have, and stop the bag turning into a mess after the first day.", "q1": "How many shirts did I pack for my 5-day trip to Costa Rica?", "q2": "For the Costa Rica trip, how many shirts did I take with me?", "q3": "I'm telling a friend how badly I overpacked tops on my Central America holiday. What number should I mention?", "distractor_sentence": "My sister packed just 4 shirts for her week in Goa and still had spare ones."}, "hi": {"memory_turn": "मैं पैकिंग में बेहतर होने की कोशिश कर रही हूँ क्योंकि मैं हमेशा ज़रूरत से ज़्यादा सामान ले जाती हूँ। कोस्टा रिका की 5 दिन की ट्रिप पर मैं 7 शर्ट ले गई थी और उनमें से मुश्किल से आधी पहनीं। अगली बार हल्का सामान कैसे पैक करूँ?", "assistant_reply": "एक या दो रंगों के आसपास कपड़े चुनें ताकि सब आपस में मैच हो जाएँ, लगभग आधे दिनों के कपड़े रखें और बीच में एक बार कपड़े धो लें, और सबसे भारी जूते सफ़र में पहनकर जाएँ।", "followup_turn": "क्या पैकिंग क्यूब सच में काम के हैं?", "followup_reply": "बहुत लोगों के लिए हाँ। इनसे सामान अलग-अलग रहता है, दिखता है कि क्या रखा है, और पहले ही दिन के बाद बैग बिखरता नहीं।", "q1": "कोस्टा रिका की 5 दिन की ट्रिप के लिए मैंने कितनी शर्ट पैक की थीं?", "q2": "कोस्टा रिका ट्रिप पर मैं अपने साथ कितनी शर्ट ले गई थी?", "q3": "मैं एक दोस्त को बता रही हूँ कि सेंट्रल अमेरिका की छुट्टी पर मैंने कितने ज़्यादा टॉप पैक कर लिए थे। मुझे कौन-सी संख्या बतानी चाहिए?", "distractor_sentence": "मेरी बहन गोवा में एक हफ़्ते के लिए बस 4 शर्ट ले गई थी और फिर भी कुछ बच गईं।"}, "hg": {"memory_turn": "Main packing mein better hone ki koshish kar rahi hoon kyunki main hamesha zaroorat se zyaada saamaan le jaati hoon. Costa Rica ki 5 din ki trip pe main 7 shirts le gayi thi aur unmein se mushkil se aadhi pehni. Agli baar halka saamaan kaise pack karoon?", "assistant_reply": "Ek ya do colours ke around kapde chunein taaki sab aapas mein match ho jaayein, lagbhag aadhe dinon ke kapde rakhein aur beech mein ek baar laundry kar lein, aur sabse bhaari shoes travel mein pehen ke jaayein.", "followup_turn": "Kya packing cubes sach mein kaam ke hain?", "followup_reply": "Bahut logon ke liye haan. Inse saamaan alag-alag rehta hai, dikhta hai ki kya rakha hai, aur pehle hi din ke baad bag bikharta nahi.", "q1": "Costa Rica ki 5 din ki trip ke liye maine kitni shirts pack ki thi?", "q2": "Costa Rica trip pe main apne saath kitni shirts le gayi thi?", "q3": "Main ek friend ko bata rahi hoon ki Central America ki chhutti pe maine kitne zyaada tops pack kar liye the. Mujhe kaunsa number batana chahiye?", "distractor_sentence": "Meri behen Goa mein ek hafte ke liye bas 4 shirts le gayi thi aur phir bhi kuch bach gayi."}}, "gold": {"answer_en": "7 shirts", "slots": [{"name": "count", "aliases": ["7", "seven", "सात", "saat"]}]}, "distractor": {"value_aliases": ["4 shirts", "4 शर्ट", "4"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "af8d2e46", "lme_type": "single-session-user"}, "topic": "travel", "is_answerable": true, "evidence_position": 2, "filler_seed": 2087399000, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": true, "abl_q3": true, "abl_digits": false, "banned_fillers": ["FL04", "FL24"]}
{"item_id": "F022", "cand_id": "FC126", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I completed a certification in Data Science last month, and I need to add it to my LinkedIn profile. Do you have any tips on how to showcase certifications well?", "assistant_reply": "Add it under Licenses and Certifications with the issuer, date and credential link. Then mention one project you built during the course in your About section, so recruiters see the skill in action.", "followup_turn": "Should I also post about it?", "followup_reply": "Yes, a short post works well. Share what you learned, one result from your project and thank the instructors. Keep it to a few lines with a screenshot or link.", "q1": "What certification did I complete last month?", "q2": "In which field did I finish a certification last month?"}, "hi": {"memory_turn": "मैंने पिछले महीने डेटा साइंस में एक सर्टिफ़िकेशन पूरा किया, और मुझे इसे अपनी LinkedIn प्रोफ़ाइल में जोड़ना है। क्या आप सर्टिफ़िकेशन को अच्छे से दिखाने के लिए कुछ टिप्स दे सकते हैं?", "assistant_reply": "इसे Licenses and Certifications में जारी करने वाली संस्था, तारीख़ और क्रेडेंशियल लिंक के साथ जोड़ें। फिर About सेक्शन में कोर्स के दौरान बनाया एक प्रोजेक्ट लिखें, ताकि रिक्रूटर उस हुनर को काम करते देख सकें।", "followup_turn": "क्या मुझे इसके बारे में पोस्ट भी करनी चाहिए?", "followup_reply": "हाँ, एक छोटी पोस्ट अच्छी रहती है। बताएँ आपने क्या सीखा, प्रोजेक्ट का एक नतीजा, और इंस्ट्रक्टर को धन्यवाद दें। स्क्रीनशॉट या लिंक के साथ कुछ ही लाइनें रखें।", "q1": "मैंने पिछले महीने कौन-सा सर्टिफ़िकेशन पूरा किया?", "q2": "पिछले महीने मैंने किस विषय में सर्टिफ़िकेशन ख़त्म किया?"}, "hg": {"memory_turn": "Maine pichhle mahine Data Science mein ek certification complete kiya, aur mujhe ise apni LinkedIn profile mein add karna hai. Kya aap certifications ko acche se showcase karne ke liye kuch tips de sakte hain?", "assistant_reply": "Ise Licenses and Certifications mein issuer, date aur credential link ke saath add karein. Phir About section mein course ke dauraan banaya ek project likhein, taaki recruiters us skill ko kaam karte dekh sakein.", "followup_turn": "Kya mujhe iske baare mein post bhi karni chahiye?", "followup_reply": "Haan, ek chhoti post acchi rehti hai. Batayein aapne kya seekha, project ka ek result, aur instructors ko thank you bolein. Screenshot ya link ke saath kuch hi lines rakhein.", "q1": "Maine pichhle mahine kaunsa certification complete kiya?", "q2": "Pichhle mahine maine kis field mein certification khatam kiya?"}}, "gold": {"answer_en": "Data Science", "slots": [{"name": "field", "aliases": ["Data Science", "डेटा साइंस", "डाटा साइंस"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "8ebdbe50", "lme_type": "single-session-user"}, "topic": "other", "is_answerable": true, "evidence_position": 5, "filler_seed": 1584952539, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F023", "cand_id": "FC127", "memory_type": "fact", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I'm planning to buy carbon fibre wheels in a few months. Speaking of my bikes, I've got three of them: a road bike, a mountain bike and a commuter bike. Any tips on keeping them organised in a small garage?", "assistant_reply": "Wall-mounted hooks or a vertical rack save the most floor space. Hang the bikes you use least highest up, and keep a small shelf nearby for helmets, pumps and tools.", "followup_turn": "Is it bad for the wheels to hang a bike by one wheel?", "followup_reply": "No, it's fine for most bikes. Just make sure the hook is padded so it doesn't scratch the rim, and check carbon wheels with the maker's guidance.", "q1": "How many bikes do I own?", "q2": "What is the total number of bicycles I have?", "q3": "I'm buying a wall rack for my garage. How many hooks do I need so every one of my bikes has a spot?"}, "hi": {"memory_turn": "मैं कुछ महीनों में कार्बन फ़ाइबर व्हील ख़रीदने का सोच रही हूँ। मेरी साइकिलों की बात करें तो, मेरे पास तीन साइकिलें हैं: एक रोड बाइक, एक माउंटेन बाइक और एक रोज़ आने-जाने वाली साइकिल। छोटे गैराज में इन्हें ठीक से रखने के लिए कोई टिप्स?", "assistant_reply": "दीवार पर लगे हुक या खड़ा रैक सबसे ज़्यादा जगह बचाते हैं। जो साइकिल सबसे कम चलाती हैं उसे सबसे ऊपर टाँगें, और पास में हेलमेट, पंप और औज़ारों के लिए एक छोटी शेल्फ़ रखें।", "followup_turn": "क्या साइकिल को एक पहिए से टाँगना पहियों के लिए बुरा है?", "followup_reply": "नहीं, ज़्यादातर साइकिलों के लिए यह ठीक है। बस हुक पर गद्दी हो ताकि रिम पर खरोंच न आए, और कार्बन व्हील के लिए कंपनी की सलाह देख लें।", "q1": "मेरे पास कितनी साइकिलें हैं?", "q2": "मेरी कुल कितनी साइकिलें हैं?", "q3": "मैं अपने गैराज के लिए दीवार वाला रैक ख़रीद रही हूँ। मुझे कितने हुक चाहिए ताकि मेरी हर साइकिल को जगह मिल जाए?"}, "hg": {"memory_turn": "Main kuch mahinon mein carbon fibre wheels khareedne ka soch rahi hoon. Meri bikes ki baat karein toh, mere paas teen bikes hain: ek road bike, ek mountain bike aur ek daily commute wali bike. Chhote garage mein inhe theek se rakhne ke liye koi tips?", "assistant_reply": "Deewar pe lage hooks ya vertical rack sabse zyaada jagah bachate hain. Jo bike sabse kam chalati hain use sabse upar taangein, aur paas mein helmets, pump aur tools ke liye ek chhoti shelf rakhein.", "followup_turn": "Kya bike ko ek wheel se taangna wheels ke liye bura hai?", "followup_reply": "Nahi, zyaadatar bikes ke liye yeh theek hai. Bas hook pe padding ho taaki rim pe scratch na aaye, aur carbon wheels ke liye company ki guidance dekh lein.", "q1": "Mere paas kitni bikes hain?", "q2": "Meri total kitni cycles hain?", "q3": "Main apne garage ke liye wall rack khareed rahi hoon. Mujhe kitne hooks chahiye taaki meri har bike ko jagah mil jaaye?"}}, "gold": {"answer_en": "three", "slots": [{"name": "count", "aliases": ["three", "3", "तीन", "teen"]}]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "6b168ec8", "lme_type": "single-session-user"}, "topic": "vehicles", "is_answerable": true, "evidence_position": 9, "filler_seed": 2758821564, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": ["FL04", "FL06", "FL07", "FL11", "FL12", "FL14", "FL16", "FL20", "FL21", "FL22", "FL29", "FL30", "FL32", "FL33", "FL35", "FL36"]}
{"item_id": "F024", "cand_id": "FC129", "memory_type": "fact", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I've fished in Lake Michigan before, and spinner lures worked better for trout than live bait. On my last trip there I caught 12 largemouth bass. What's the best spinner lure to use for trout?", "assistant_reply": "Small inline spinners like a Panther Martin or a Mepps Aglia in sizes 1 to 3 work well. Pick silver or gold blades on bright days and darker colours when the water is murky.", "followup_turn": "What time of day is best for trout?", "followup_reply": "Early morning and late evening are usually best, when the light is low and the water is cooler. Overcast days can be good all day.", "q1": "How many largemouth bass did I catch on my last trip to Lake Michigan?", "q2": "On my last Lake Michigan fishing trip, how many largemouth bass did I land?", "distractor_sentence": "My fishing buddy caught only 5 bass when he went there last summer."}, "hi": {"memory_turn": "मैं पहले भी मिशिगन झील में मछली पकड़ चुका हूँ, और ट्राउट के लिए ज़िंदा चारे से बेहतर स्पिनर ल्योर चले। पिछली ट्रिप पर मैंने वहाँ 12 लार्जमाउथ बास पकड़ीं। ट्राउट के लिए सबसे अच्छा स्पिनर ल्योर कौन-सा है?", "assistant_reply": "छोटे इनलाइन स्पिनर जैसे Panther Martin या Mepps Aglia, साइज़ 1 से 3, अच्छे रहते हैं। धूप वाले दिन चाँदी या सुनहरे ब्लेड चुनें और पानी गंदला हो तो गहरे रंग।", "followup_turn": "ट्राउट के लिए दिन का कौन-सा समय सबसे अच्छा है?", "followup_reply": "आमतौर पर सुबह-सुबह और देर शाम, जब रोशनी कम हो और पानी ठंडा हो। बादल वाले दिन पूरे दिन अच्छे रह सकते हैं।", "q1": "मिशिगन झील की पिछली ट्रिप पर मैंने कितनी लार्जमाउथ बास पकड़ी थीं?", "q2": "पिछली बार मिशिगन झील में मछली पकड़ते हुए मेरे हाथ कितनी लार्जमाउथ बास लगीं?", "distractor_sentence": "मेरे मछली पकड़ने वाले दोस्त ने पिछली गर्मियों में वहाँ बस 5 बास पकड़ी थीं।"}, "hg": {"memory_turn": "Main pehle bhi Lake Michigan mein fishing kar chuka hoon, aur trout ke liye live bait se better spinner lures chale. Pichhli trip pe maine wahan 12 largemouth bass pakdi. Trout ke liye sabse accha spinner lure kaunsa hai?", "assistant_reply": "Chhote inline spinners jaise Panther Martin ya Mepps Aglia, size 1 se 3, acche rehte hain. Dhoop wale din silver ya golden blades chunein aur paani gandla ho toh dark colours.", "followup_turn": "Trout ke liye din ka kaunsa time sabse accha hai?", "followup_reply": "Usually subah-subah aur der shaam, jab roshni kam ho aur paani thanda ho. Badal wale din poore din acche reh sakte hain.", "q1": "Lake Michigan ki pichhli trip pe maine kitni largemouth bass pakdi thi?", "q2": "Pichhli baar Lake Michigan mein fishing karte hue mere haath kitni largemouth bass lagi?", "distractor_sentence": "Mere fishing wale dost ne pichhli garmiyon mein wahan bas 5 bass pakdi thi."}}, "gold": {"answer_en": "12", "slots": [{"name": "count", "aliases": ["12", "twelve", "बारह", "barah"]}]}, "distractor": {"value_aliases": ["5 bass", "5 बास", "5"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "21436231", "lme_type": "single-session-user"}, "topic": "travel", "is_answerable": true, "evidence_position": 2, "filler_seed": 323160299, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": ["FL36"]}
{"item_id": "F025", "cand_id": "FC130", "memory_type": "fact", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I've been taking a stand-up comedy class. I recently went to an open mic night and saw 10 amateur comedians perform, which was really inspiring. Do you have advice on how to write a good joke?", "assistant_reply": "Start from something true that annoys or surprises you, find the unexpected angle, and put the funniest word at the very end. Then cut every word that the joke doesn't need.", "followup_turn": "How long should my first set be?", "followup_reply": "Most open mics give new comics about five minutes. Prepare three or four short bits you know well rather than one long story.", "q1": "How many amateur comedians did I watch perform at the open mic night?", "q2": "At the open mic I went to, how many amateur comics performed?", "distractor_sentence": "My classmate says she saw 15 comedians at a much bigger show downtown."}, "hi": {"memory_turn": "मैं स्टैंड-अप कॉमेडी की क्लास ले रही हूँ। हाल ही में मैं एक ओपन माइक नाइट में गई थी और वहाँ 10 नए कॉमेडियन को परफ़ॉर्म करते देखा, जो बहुत प्रेरणा देने वाला था। अच्छा जोक लिखने के लिए कोई सलाह?", "assistant_reply": "किसी सच्ची बात से शुरू करें जो आपको चिढ़ाती या चौंकाती हो, उसका अनोखा पहलू खोजें, और सबसे मज़ेदार शब्द बिल्कुल आख़िर में रखें। फिर हर वह शब्द हटा दें जिसकी जोक को ज़रूरत नहीं।", "followup_turn": "मेरा पहला सेट कितना लंबा होना चाहिए?", "followup_reply": "ज़्यादातर ओपन माइक में नए कॉमेडियन को करीब पाँच मिनट मिलते हैं। एक लंबी कहानी की बजाय तीन-चार छोटे हिस्से तैयार करें जो आपको अच्छे से याद हों।", "q1": "ओपन माइक नाइट में मैंने कितने नए कॉमेडियन को परफ़ॉर्म करते देखा?", "q2": "जिस ओपन माइक में मैं गई थी, वहाँ कितने नौसिखिए कॉमेडियन ने परफ़ॉर्म किया?", "distractor_sentence": "मेरी क्लास की एक लड़की कहती है कि उसने शहर के एक बड़े शो में 15 कॉमेडियन देखे।"}, "hg": {"memory_turn": "Main stand-up comedy ki class le rahi hoon. Haal hi mein main ek open mic night mein gayi thi aur wahan 10 amateur comedians ko perform karte dekha, jo bahut inspiring tha. Accha joke likhne ke liye koi advice?", "assistant_reply": "Kisi sacchi baat se shuru karein jo aapko irritate ya surprise karti ho, uska unexpected angle dhoondhein, aur sabse funny word bilkul end mein rakhein. Phir har woh word hata dein jiski joke ko zaroorat nahi.", "followup_turn": "Mera pehla set kitna lamba hona chahiye?", "followup_reply": "Zyaadatar open mics mein naye comics ko kareeb paanch minute milte hain. Ek lambi kahani ki bajaye teen-chaar chhote bits taiyaar karein jo aapko acche se yaad hon.", "q1": "Open mic night mein maine kitne amateur comedians ko perform karte dekha?", "q2": "Jis open mic mein main gayi thi, wahan kitne amateur comics ne perform kiya?", "distractor_sentence": "Meri class ki ek ladki kehti hai ki usne sheher ke ek bade show mein 15 comedians dekhe."}}, "gold": {"answer_en": "10", "slots": [{"name": "count", "aliases": ["10", "ten", "दस"]}]}, "distractor": {"value_aliases": ["15"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "95bcc1c8", "lme_type": "single-session-user"}, "topic": "other", "is_answerable": true, "evidence_position": 5, "filler_seed": 2981395773, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": ["FL04", "FL06", "FL12", "FL21", "FL23", "FL25", "FL30", "FL32", "FL35"]}
{"item_id": "F026", "cand_id": "FC132", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I like the idea of a 'Story' column in my jewellery log. I'll start with the most sentimental piece: the silver chain my grandma gave me on my 18th birthday. Sadly I lost it a couple of months ago. What other columns should I add?", "assistant_reply": "Useful columns are a description, the material, where it is kept, its approximate value, a photo link and a condition note. A 'last seen' column helps a lot if something goes missing.", "followup_turn": "Should I get anything appraised for insurance?", "followup_reply": "For pieces that are valuable or irreplaceable, yes. A written appraisal with photos makes any insurance claim much easier.", "q1": "How old was I when my grandma gave me the silver chain?", "q2": "At what age did I receive the silver chain from my grandmother?", "q3": "I'm writing the story behind my most sentimental keepsake for my log. At what age did it come into my life?"}, "hi": {"memory_turn": "मुझे अपने गहनों के रिकॉर्ड में 'कहानी' वाला कॉलम रखने का आइडिया पसंद है। मैं सबसे भावनात्मक चीज़ से शुरू करूँगा: वह चाँदी की चेन जो दादी ने मुझे मेरे 18वें जन्मदिन पर दी थी। अफ़सोस, कुछ महीने पहले वह मुझसे खो गई। और कौन-से कॉलम जोड़ूँ?", "assistant_reply": "काम के कॉलम हैं: विवरण, धातु, कहाँ रखी है, अनुमानित क़ीमत, फ़ोटो लिंक और हालत का नोट। 'आख़िरी बार कब देखी' वाला कॉलम किसी चीज़ के खोने पर बहुत मदद करता है।", "followup_turn": "क्या बीमा के लिए किसी चीज़ का मूल्यांकन करवाना चाहिए?", "followup_reply": "क़ीमती या कभी न मिल सकने वाली चीज़ों के लिए हाँ। फ़ोटो के साथ लिखित मूल्यांकन होने से बीमा क्लेम बहुत आसान हो जाता है।", "q1": "जब दादी ने मुझे चाँदी की चेन दी, तब मेरी उम्र क्या थी?", "q2": "दादी से चाँदी की चेन मुझे किस उम्र में मिली थी?", "q3": "मैं अपने रिकॉर्ड के लिए अपनी सबसे भावनात्मक यादगार चीज़ की कहानी लिख रहा हूँ। वह मेरी ज़िंदगी में किस उम्र में आई?"}, "hg": {"memory_turn": "Mujhe apne jewellery log mein 'Story' column rakhne ka idea pasand hai. Main sabse sentimental cheez se shuru karunga: woh silver chain jo dadi ne mujhe mere 18th birthday pe di thi. Afsos, kuch mahine pehle woh mujhse kho gayi. Aur kaunse columns add karoon?", "assistant_reply": "Kaam ke columns hain: description, metal, kahan rakhi hai, approximate value, photo link aur condition note. 'Last seen' wala column kisi cheez ke khone pe bahut help karta hai.", "followup_turn": "Kya insurance ke liye kisi cheez ka appraisal karwana chahiye?", "followup_reply": "Keemti ya kabhi na mil sakne wali cheezon ke liye haan. Photos ke saath written appraisal hone se insurance claim bahut easy ho jaata hai.", "q1": "Jab dadi ne mujhe silver chain di, tab meri umar kya thi?", "q2": "Dadi se silver chain mujhe kis umar mein mili thi?", "q3": "Main apne log ke liye apni sabse sentimental keepsake ki story likh raha hoon. Woh meri life mein kis umar mein aayi?"}}, "gold": {"answer_en": "18", "slots": [{"name": "age", "aliases": ["18", "18th", "eighteen", "अठारह", "athaarah"]}]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "853b0a1d", "lme_type": "single-session-user"}, "topic": "other", "is_answerable": true, "evidence_position": 9, "filler_seed": 775595626, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": true, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "F027", "cand_id": "FC133", "memory_type": "fact", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I've been experimenting with gin-to-vermouth ratios in my classic martini, and I've settled on 3:1 with a dash of citrus bitters. It really brings out the gin. Can you recommend a good vermouth brand to pair with it?", "assistant_reply": "Dolin Dry is a popular choice for a clean, light martini, and Noilly Prat Extra Dry gives a slightly richer, more herbal taste. Keep opened vermouth in the fridge, since it fades within a few weeks.", "followup_turn": "Should I stir or shake a martini?", "followup_reply": "Stir it. Stirring chills and dilutes it without clouding the drink or adding air bubbles, so it stays silky and clear.", "q1": "What is my preferred gin-to-vermouth ratio for a classic martini?", "q2": "In what proportion of gin to vermouth do I like my classic martini?", "distractor_sentence": "My brother likes his martinis much drier, at 5:1."}, "hi": {"memory_turn": "मैं अपनी क्लासिक मार्टिनी में जिन और वरमाउथ के अनुपात के साथ प्रयोग कर रही थी, और अब मैंने 3:1 तय किया है, साथ में सिट्रस बिटर्स की एक बूँद। इससे जिन का स्वाद अच्छे से उभरता है। इसके साथ कौन-सा वरमाउथ ब्रांड अच्छा रहेगा?", "assistant_reply": "हल्की, साफ़ मार्टिनी के लिए Dolin Dry लोकप्रिय है, और Noilly Prat Extra Dry थोड़ा भरपूर, जड़ी-बूटी वाला स्वाद देता है। खुला हुआ वरमाउथ फ़्रिज में रखें, क्योंकि कुछ हफ़्तों में उसका स्वाद उड़ जाता है।", "followup_turn": "मार्टिनी को हिलाना चाहिए या शेक करना चाहिए?", "followup_reply": "चम्मच से हिलाएँ। इससे ड्रिंक ठंडी और थोड़ी पतली हो जाती है, पर धुंधली नहीं होती और हवा के बुलबुले नहीं आते, इसलिए वह मुलायम और साफ़ रहती है।", "q1": "क्लासिक मार्टिनी के लिए जिन और वरमाउथ का मेरा पसंदीदा अनुपात क्या है?", "q2": "मुझे अपनी क्लासिक मार्टिनी में जिन और वरमाउथ किस अनुपात में पसंद है?", "distractor_sentence": "मेरे भाई को अपनी मार्टिनी काफ़ी ड्राई पसंद है, 5:1 वाली।"}, "hg": {"memory_turn": "Main apni classic martini mein gin aur vermouth ke ratio ke saath experiment kar rahi thi, aur ab maine 3:1 final kiya hai, saath mein citrus bitters ki ek boond. Isse gin ka flavour acche se ubharta hai. Iske saath kaunsa vermouth brand accha rahega?", "assistant_reply": "Halki, clean martini ke liye Dolin Dry popular hai, aur Noilly Prat Extra Dry thoda rich, herbal taste deta hai. Khula hua vermouth fridge mein rakhein, kyunki kuch hafton mein uska flavour ud jaata hai.", "followup_turn": "Martini ko stir karna chahiye ya shake?", "followup_reply": "Stir karein. Isse drink thandi aur thodi dilute ho jaati hai, par cloudy nahi hoti aur air bubbles nahi aate, isliye woh smooth aur clear rehti hai.", "q1": "Classic martini ke liye gin aur vermouth ka mera favourite ratio kya hai?", "q2": "Mujhe apni classic martini mein gin aur vermouth kis ratio mein pasand hai?", "distractor_sentence": "Mere bhai ko apni martini kaafi dry pasand hai, 5:1 wali."}}, "gold": {"answer_en": "3:1", "slots": [{"name": "ratio", "aliases": ["3:1", "3 : 1", "3 to 1", "three to one", "3 parts gin"]}]}, "distractor": {"value_aliases": ["5:1"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "a06e4cfe", "lme_type": "single-session-user"}, "topic": "food/cooking", "is_answerable": true, "evidence_position": 2, "filler_seed": 2306761778, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F028", "cand_id": "FC134", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "My work laptop is a Dell Inspiron from 2018. Before I upgraded its RAM to 16GB, the battery lasted 6 to 7 hours, but now it lasts only 4 to 5. I haven't changed anything else. Could the upgrade be the cause?", "assistant_reply": "More memory uses a little more power, but rarely that much. It's more likely that the battery is ageing or that you now run heavier apps. Check the battery wear level and which apps use the most energy.", "followup_turn": "How do I check the battery wear on Windows?", "followup_reply": "Open Command Prompt and run 'powercfg /batteryreport'. Compare the full charge capacity with the design capacity in the report it creates.", "q1": "How much RAM did I upgrade my laptop to?", "q2": "What is the RAM size in my laptop after the upgrade?", "q3": "A new game lists its memory requirements. How much memory does my work machine have now?"}, "hi": {"memory_turn": "मेरा काम वाला लैपटॉप 2018 का Dell Inspiron है। इसकी RAM को 16GB करने से पहले बैटरी 6 से 7 घंटे चलती थी, पर अब सिर्फ़ 4 से 5 घंटे चलती है। मैंने और कुछ नहीं बदला। क्या अपग्रेड इसकी वजह हो सकता है?", "assistant_reply": "ज़्यादा मेमोरी थोड़ी ज़्यादा बिजली लेती है, पर इतनी नहीं। ज़्यादा संभावना है कि बैटरी पुरानी हो रही है या आप अब भारी ऐप चलाते हैं। बैटरी की घिसावट और सबसे ज़्यादा ऊर्जा लेने वाले ऐप देखें।", "followup_turn": "Windows पर बैटरी की घिसावट कैसे देखूँ?", "followup_reply": "Command Prompt खोलें और 'powercfg /batteryreport' चलाएँ। उससे बनी रिपोर्ट में फ़ुल चार्ज कैपेसिटी की तुलना डिज़ाइन कैपेसिटी से करें।", "q1": "मैंने अपने लैपटॉप की RAM कितनी करवाई?", "q2": "अपग्रेड के बाद मेरे लैपटॉप में कितनी RAM है?", "q3": "एक नए गेम ने अपनी मेमोरी की ज़रूरत बताई है। मेरी काम वाली मशीन में अब कितनी मेमोरी है?"}, "hg": {"memory_turn": "Mera kaam wala laptop 2018 ka Dell Inspiron hai. Iski RAM ko 16GB karne se pehle battery 6 se 7 ghante chalti thi, par ab sirf 4 se 5 ghante chalti hai. Maine aur kuch nahi badla. Kya upgrade iski wajah ho sakta hai?", "assistant_reply": "Zyaada memory thodi zyaada power leti hai, par itni nahi. Zyaada chance hai ki battery purani ho rahi hai ya aap ab heavy apps chalate hain. Battery wear level aur sabse zyaada energy lene wale apps check karein.", "followup_turn": "Windows pe battery wear kaise check karoon?", "followup_reply": "Command Prompt kholein aur 'powercfg /batteryreport' run karein. Usse bani report mein full charge capacity ko design capacity se compare karein.", "q1": "Maine apne laptop ki RAM kitni karwayi?", "q2": "Upgrade ke baad mere laptop mein kitni RAM hai?", "q3": "Ek naye game ne apni memory requirement batayi hai. Meri kaam wali machine mein ab kitni memory hai?"}}, "gold": {"answer_en": "16GB", "slots": [{"name": "ram", "aliases": ["16GB", "16 GB", "16 gigabytes", "16 जीबी", "16"]}]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "37d43f65", "lme_type": "single-session-user"}, "topic": "work/study", "is_answerable": true, "evidence_position": 5, "filler_seed": 2574525652, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": ["FL28"]}
{"item_id": "F029", "cand_id": "FC136", "memory_type": "fact", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I was just at my cousin's wedding at the Grand Ballroom last weekend, and my mom looked absolutely stunning. I want to get her a skincare gift to help her keep that glow. Any ideas?", "assistant_reply": "A gentle vitamin C serum, a rich night cream and a good sunscreen make a thoughtful set. Add a soft silk pillowcase or a facial roller if you'd like it to feel more like a treat.", "followup_turn": "Is vitamin C okay for sensitive skin?", "followup_reply": "Usually, if you choose a lower strength, around 10 percent, or a gentler form like sodium ascorbyl phosphate. Suggest she patch-tests it first.", "q1": "Where did I attend my cousin's wedding?", "q2": "At which venue was my cousin's wedding that I went to?", "distractor_sentence": "My other cousin is getting married at Lakeview Gardens next year."}, "hi": {"memory_turn": "मैं पिछले वीकेंड अपनी कज़िन की शादी में ग्रैंड बॉलरूम गई थी, और मेरी मम्मी बेहद ख़ूबसूरत लग रही थीं। मैं उन्हें स्किनकेयर का कोई तोहफ़ा देना चाहती हूँ ताकि उनका वह निखार बना रहे। कोई आइडिया?", "assistant_reply": "हल्का विटामिन C सीरम, एक भरपूर नाइट क्रीम और अच्छा सनस्क्रीन एक सोच-समझकर बना सेट है। इसे और ख़ास बनाना हो तो रेशमी तकिये का कवर या फ़ेस रोलर जोड़ दें।", "followup_turn": "क्या संवेदनशील त्वचा के लिए विटामिन C ठीक है?", "followup_reply": "आमतौर पर हाँ, अगर हल्की मात्रा वाला, करीब 10 प्रतिशत, या सोडियम एस्कॉर्बिल फ़ॉस्फ़ेट जैसा नरम रूप चुनें। उन्हें पहले थोड़ी जगह पर लगाकर देखने को कहें।", "q1": "मैं अपनी कज़िन की शादी में कहाँ गई थी?", "q2": "जिस कज़िन की शादी में मैं गई, वह किस जगह पर हुई थी?", "distractor_sentence": "मेरी दूसरी कज़िन की शादी अगले साल लेकव्यू गार्डन्स में होने वाली है।"}, "hg": {"memory_turn": "Main pichhle weekend apni cousin ki shaadi mein Grand Ballroom gayi thi, aur meri mummy bahut khoobsurat lag rahi thi. Main unhe skincare ka koi gift dena chahti hoon taaki unka woh glow bana rahe. Koi idea?", "assistant_reply": "Halka vitamin C serum, ek rich night cream aur accha sunscreen ek thoughtful set hai. Ise aur special banana ho toh silk pillowcase ya face roller add kar dein.", "followup_turn": "Kya sensitive skin ke liye vitamin C theek hai?", "followup_reply": "Usually haan, agar halki strength wala, kareeb 10 percent, ya sodium ascorbyl phosphate jaisa gentle form chunein. Unhe pehle thodi jagah pe laga ke dekhne ko kahein.", "q1": "Main apni cousin ki shaadi mein kahan gayi thi?", "q2": "Jis cousin ki shaadi mein main gayi, woh kis venue pe hui thi?", "distractor_sentence": "Meri doosri cousin ki shaadi next year Lakeview Gardens mein hone wali hai."}}, "gold": {"answer_en": "The Grand Ballroom", "slots": [{"name": "venue", "aliases": ["Grand Ballroom", "ग्रैंड बॉलरूम", "ग्रांड बॉलरूम"]}]}, "distractor": {"value_aliases": ["Lakeview Gardens", "Lakeview", "लेकव्यू गार्डन्स", "लेकव्यू"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "d52b4f67", "lme_type": "single-session-user"}, "topic": "family/social", "is_answerable": true, "evidence_position": 9, "filler_seed": 530246910, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F030", "cand_id": "FC137", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I completed my Bachelor's in Computer Science at UCLA, and now I'm leaning towards Stanford for a Master's, but I'm worried about the cost. Is it worth taking out loans for it?", "assistant_reply": "It can be if the degree clearly raises your salary or opens roles you can't reach now. Compare the total cost with the expected pay increase, and ask about assistantships, which can cover tuition.", "followup_turn": "Do employers really care about a Master's in tech?", "followup_reply": "For many software roles, experience matters more. It helps most for research, machine learning and specialised positions, or when changing fields.", "q1": "Where did I complete my Bachelor's degree in Computer Science?", "q2": "Which university did I get my undergraduate CS degree from?", "q3": "A recruiter asked which university is printed on my undergraduate diploma. What should I tell her?"}, "hi": {"memory_turn": "मैंने कंप्यूटर साइंस में बैचलर डिग्री यूसीएलए से पूरी की, और अब मास्टर्स के लिए मेरा झुकाव स्टैनफ़ोर्ड की तरफ़ है, पर ख़र्च की चिंता है। क्या इसके लिए लोन लेना ठीक रहेगा?", "assistant_reply": "हो सकता है, अगर डिग्री से आपकी सैलरी साफ़ बढ़े या ऐसे रोल खुलें जो अभी नहीं मिल सकते। कुल ख़र्च की तुलना अनुमानित सैलरी बढ़ोतरी से करें, और असिस्टेंटशिप के बारे में पूछें, जिससे फ़ीस निकल सकती है।", "followup_turn": "क्या टेक में कंपनियों को सच में मास्टर्स से फ़र्क पड़ता है?", "followup_reply": "कई सॉफ़्टवेयर रोल में अनुभव ज़्यादा मायने रखता है। रिसर्च, मशीन लर्निंग और ख़ास पदों के लिए, या फ़ील्ड बदलते समय, यह सबसे ज़्यादा मदद करता है।", "q1": "मैंने कंप्यूटर साइंस में बैचलर डिग्री कहाँ से पूरी की?", "q2": "मुझे कंप्यूटर साइंस की ग्रेजुएशन डिग्री किस यूनिवर्सिटी से मिली?", "q3": "एक रिक्रूटर ने पूछा कि मेरी ग्रेजुएशन की डिग्री पर किस यूनिवर्सिटी का नाम छपा है। उसे क्या बताऊँ?"}, "hg": {"memory_turn": "Maine Computer Science mein Bachelor's degree UCLA se complete ki, aur ab Master's ke liye mera jhukaav Stanford ki taraf hai, par kharche ki chinta hai. Kya iske liye loan lena theek rahega?", "assistant_reply": "Ho sakta hai, agar degree se aapki salary clearly badhe ya aise roles khulein jo abhi nahi mil sakte. Total cost ko expected salary increase se compare karein, aur assistantships ke baare mein poochhein, jisse fees nikal sakti hai.", "followup_turn": "Kya tech mein companies ko sach mein Master's se fark padta hai?", "followup_reply": "Kai software roles mein experience zyaada matter karta hai. Research, machine learning aur specialised positions ke liye, ya field badalte time, yeh sabse zyaada help karta hai.", "q1": "Maine Computer Science mein Bachelor's degree kahan se complete ki?", "q2": "Mujhe CS ki undergraduate degree kis university se mili?", "q3": "Ek recruiter ne poocha ki meri undergraduate degree pe kis university ka naam chhapa hai. Use kya bataun?"}}, "gold": {"answer_en": "UCLA (University of California, Los Angeles)", "slots": [{"name": "university", "aliases": ["UCLA", "University of California, Los Angeles", "University of California Los Angeles", "यूसीएलए"]}]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "25e5aa4f", "lme_type": "single-session-user"}, "topic": "work/study", "is_answerable": true, "evidence_position": 2, "filler_seed": 2103985978, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "P021", "cand_id": "PX0127", "memory_type": "preference", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I've always loved the Elder Scrolls series, and Skyrim is one of my favourite games of all time. I finally have a free weekend. Can you suggest a cosy indie game to try as a change?", "assistant_reply": "Stardew Valley is a classic for relaxed farming and friendships, and A Short Hike is a calm little exploration game you can finish in an afternoon. Unpacking is also lovely and very low stress.", "followup_turn": "Which of those works best on a laptop without a good graphics card?", "followup_reply": "All three run fine on most laptops. A Short Hike and Unpacking are especially light, so even older machines should handle them.", "q1": "Which video game series have I always loved?", "q2": "What is my all-time favourite video game series?", "q3": "I'm picking a game to replay from the franchise I've loved the longest. Which franchise should I go back to?", "distractor_sentence": "My brother is obsessed with the Witcher games and keeps telling me to play them."}, "hi": {"memory_turn": "मुझे हमेशा से Elder Scrolls सीरीज़ बहुत पसंद रही है, और Skyrim मेरे अब तक के सबसे पसंदीदा गेम में से एक है। आख़िरकार मेरे पास एक ख़ाली वीकेंड है। बदलाव के लिए कोई आरामदायक इंडी गेम बताइए?", "assistant_reply": "आराम से खेती और दोस्ती के लिए Stardew Valley क्लासिक है, और A Short Hike एक शांत, छोटा खोजबीन वाला गेम है जो एक दोपहर में ख़त्म हो जाता है। Unpacking भी प्यारा और बिल्कुल तनाव-मुक्त है।", "followup_turn": "इनमें से कौन-सा बिना अच्छे ग्राफ़िक्स कार्ड वाले लैपटॉप पर सबसे अच्छा चलेगा?", "followup_reply": "तीनों ज़्यादातर लैपटॉप पर ठीक चलते हैं। A Short Hike और Unpacking ख़ास तौर पर हल्के हैं, इसलिए पुरानी मशीनें भी इन्हें चला लेंगी।", "q1": "मुझे हमेशा से कौन-सी वीडियो गेम सीरीज़ पसंद रही है?", "q2": "मेरी अब तक की सबसे पसंदीदा वीडियो गेम सीरीज़ कौन-सी है?", "q3": "मैं उस फ़्रैंचाइज़ी का कोई गेम दोबारा खेलना चाहती हूँ जो मुझे सबसे लंबे समय से पसंद है। मुझे किस फ़्रैंचाइज़ी पर लौटना चाहिए?", "distractor_sentence": "मेरा भाई Witcher गेम्स का दीवाना है और मुझसे भी खेलने को कहता रहता है।"}, "hg": {"memory_turn": "Mujhe hamesha se Elder Scrolls series bahut pasand rahi hai, aur Skyrim mere ab tak ke sabse favourite games mein se ek hai. Aakhirkaar mere paas ek free weekend hai. Change ke liye koi cosy indie game batayiye?", "assistant_reply": "Relaxed farming aur dosti ke liye Stardew Valley classic hai, aur A Short Hike ek shaant, chhota exploration game hai jo ek dopahar mein khatam ho jaata hai. Unpacking bhi pyaara aur bilkul stress-free hai.", "followup_turn": "Inmein se kaunsa bina acche graphics card wale laptop pe sabse accha chalega?", "followup_reply": "Teeno zyaadatar laptops pe theek chalte hain. A Short Hike aur Unpacking khaas taur pe halke hain, isliye purani machines bhi inhe chala lengi.", "q1": "Mujhe hamesha se kaunsi video game series pasand rahi hai?", "q2": "Meri ab tak ki sabse favourite video game series kaunsi hai?", "q3": "Main us franchise ka koi game dobara khelna chahti hoon jo mujhe sabse lambe time se pasand hai. Mujhe kis franchise pe lautna chahiye?", "distractor_sentence": "Mera bhai Witcher games ka deewana hai aur mujhse bhi khelne ko kehta rehta hai."}}, "gold": {"answer_en": "The Elder Scrolls (e.g. Skyrim)", "slots": [{"name": "series", "aliases": ["Elder Scrolls", "Skyrim", "एल्डर स्क्रॉल्स", "स्काईरिम"]}]}, "distractor": {"value_aliases": ["Witcher", "विचर"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 2, "filler_seed": 3121022853, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": true, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "P022", "cand_id": "PX0137", "memory_type": "preference", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "My favourite restaurant so far is a tiny ramen shop in Shibuya called Ichiran. I'm back in Tokyo next spring for a week. What neighbourhoods should I explore this time?", "assistant_reply": "Try Shimokitazawa for vintage shops and small cafés, Yanaka for an old-Tokyo feel, and Kichijoji for the park and relaxed streets. All three are easy to reach by train.", "followup_turn": "Is spring a busy time to visit?", "followup_reply": "Yes, late March to early April is cherry blossom season and very popular. Book your hotel early and visit famous parks on weekday mornings.", "q1": "Which is my favourite restaurant so far?", "q2": "What restaurant do I like the most up to now?", "q3": "My friend is visiting Tokyo and wants the one place I'd most recommend for dinner. Where should I send him?"}, "hi": {"memory_turn": "अब तक मेरा सबसे पसंदीदा रेस्टोरेंट शिबुया में इचिरान नाम की एक छोटी-सी रामेन की दुकान है। मैं अगले वसंत में एक हफ़्ते के लिए फिर टोक्यो जा रहा हूँ। इस बार कौन-से इलाक़े घूमूँ?", "assistant_reply": "पुरानी चीज़ों की दुकानों और छोटे कैफ़े के लिए शिमोकिताज़ावा, पुराने टोक्यो के अहसास के लिए यानाका, और पार्क व आरामदायक गलियों के लिए किचिजोजी जाइए। तीनों ट्रेन से आसानी से पहुँच सकते हैं।", "followup_turn": "क्या वसंत में वहाँ बहुत भीड़ होती है?", "followup_reply": "हाँ, मार्च के आख़िर से अप्रैल की शुरुआत तक चेरी ब्लॉसम का मौसम होता है और यह बहुत लोकप्रिय है। होटल जल्दी बुक करें और मशहूर पार्क हफ़्ते के दिनों में सुबह जाएँ।", "q1": "अब तक मेरा सबसे पसंदीदा रेस्टोरेंट कौन-सा है?", "q2": "अभी तक मुझे कौन-सा रेस्टोरेंट सबसे ज़्यादा पसंद है?", "q3": "मेरा दोस्त टोक्यो जा रहा है और रात के खाने के लिए मेरी सबसे बड़ी सिफ़ारिश चाहता है। उसे कहाँ भेजूँ?"}, "hg": {"memory_turn": "Ab tak mera sabse favourite restaurant Shibuya mein Ichiran naam ki ek chhoti-si ramen shop hai. Main agle spring mein ek hafte ke liye phir Tokyo ja raha hoon. Is baar kaunse areas ghoomun?", "assistant_reply": "Vintage shops aur chhote cafes ke liye Shimokitazawa, purane Tokyo ki feel ke liye Yanaka, aur park aur relaxed galiyon ke liye Kichijoji jaaiye. Teeno train se aasani se pahunch sakte hain.", "followup_turn": "Kya spring mein wahan bahut bheed hoti hai?", "followup_reply": "Haan, March ke end se April ki shuruaat tak cherry blossom season hota hai aur yeh bahut popular hai. Hotel jaldi book karein aur famous parks weekdays mein subah jaayein.", "q1": "Ab tak mera sabse favourite restaurant kaunsa hai?", "q2": "Abhi tak mujhe kaunsa restaurant sabse zyaada pasand hai?", "q3": "Mera dost Tokyo ja raha hai aur dinner ke liye meri sabse badi recommendation chahta hai. Use kahan bhejun?"}}, "gold": {"answer_en": "Ichiran (ramen shop in Shibuya)", "slots": [{"name": "restaurant", "aliases": ["Ichiran", "इचिरान"]}]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 5, "filler_seed": 2338811528, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "P023", "cand_id": "PX0180", "memory_type": "preference", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I'm a big fan of sci-fi and fantasy shows, and I just finished everything on my watchlist. I want something with a strong female lead next. Any suggestions?", "assistant_reply": "The Expanse has several excellent female leads, Orphan Black is a gripping thriller built around one brilliant performance, and His Dark Materials follows a brave young girl across worlds.", "followup_turn": "Which one has the shortest total runtime?", "followup_reply": "His Dark Materials is the shortest, with three seasons and fewer episodes. The Expanse is the longest at six seasons.", "q1": "Which genres of TV shows do I like?", "q2": "What kinds of TV shows am I a big fan of?"}, "hi": {"memory_turn": "मैं साइंस फ़िक्शन और फ़ैंटेसी शो की बड़ी फ़ैन हूँ, और मैंने अपनी वॉचलिस्ट की हर चीज़ देख ली है। अब मुझे कुछ ऐसा चाहिए जिसमें मुख्य किरदार कोई दमदार महिला हो। कोई सुझाव?", "assistant_reply": "The Expanse में कई बेहतरीन महिला किरदार हैं, Orphan Black एक शानदार अभिनय पर टिका ज़बरदस्त थ्रिलर है, और His Dark Materials एक बहादुर लड़की की अलग-अलग दुनियाओं की यात्रा है।", "followup_turn": "इनमें से किसका कुल रनटाइम सबसे कम है?", "followup_reply": "His Dark Materials सबसे छोटा है, तीन सीज़न और कम एपिसोड के साथ। The Expanse छह सीज़न के साथ सबसे लंबा है।", "q1": "मुझे किन जॉनर के टीवी शो पसंद हैं?", "q2": "मैं किस तरह के टीवी शो की बड़ी फ़ैन हूँ?"}, "hg": {"memory_turn": "Main sci-fi aur fantasy shows ki badi fan hoon, aur maine apni watchlist ki har cheez dekh li hai. Ab mujhe kuch aisa chahiye jismein lead koi strong female character ho. Koi suggestions?", "assistant_reply": "The Expanse mein kai behtareen female characters hain, Orphan Black ek shaandaar performance pe tika zabardast thriller hai, aur His Dark Materials ek bahadur ladki ki alag-alag duniyaon ki journey hai.", "followup_turn": "Inmein se kiska total runtime sabse kam hai?", "followup_reply": "His Dark Materials sabse chhota hai, teen seasons aur kam episodes ke saath. The Expanse chhe seasons ke saath sabse lamba hai.", "q1": "Mujhe kin genres ke TV shows pasand hain?", "q2": "Main kis type ke TV shows ki badi fan hoon?"}}, "gold": {"answer_en": "sci-fi and fantasy", "slots": [{"name": "genre1", "aliases": ["sci-fi", "sci fi", "science fiction", "साइंस फ़िक्शन", "साइंस फिक्शन", "साई-फ़ाई"]}, {"name": "genre2", "aliases": ["fantasy", "फ़ैंटेसी", "फैंटेसी"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 9, "filler_seed": 2788748486, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P024", "cand_id": "PX0193", "memory_type": "preference", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I've started smoking food on weekends, and for wood chips I'm a big fan of hickory. This Sunday I want to try smoking a whole chicken. How long will it take?", "assistant_reply": "At about 120°C, a 1.5 kg chicken usually takes three to four hours. Go by temperature, not time: it's done when the thickest part of the thigh reaches 74°C.", "followup_turn": "Should I brine it first?", "followup_reply": "A simple brine of salt and water for a few hours keeps the meat juicy. Pat the skin dry afterwards so it crisps up in the smoker.", "q1": "Which wood chips do I prefer for smoking food?", "q2": "What kind of wood chips am I a big fan of when I smoke food?", "distractor_sentence": "My neighbour only ever uses mesquite chips in his smoker."}, "hi": {"memory_turn": "मैंने वीकेंड पर खाना स्मोक करना शुरू किया है, और लकड़ी के चिप्स में मैं हिकोरी का बड़ा फ़ैन हूँ। इस रविवार मैं पूरा चिकन स्मोक करके देखना चाहता हूँ। इसमें कितना समय लगेगा?", "assistant_reply": "लगभग 120°C पर डेढ़ किलो का चिकन आमतौर पर तीन से चार घंटे लेता है। समय से नहीं, तापमान से तय करें: जाँघ का सबसे मोटा हिस्सा 74°C पर पहुँचे तो चिकन तैयार है।", "followup_turn": "क्या पहले इसे नमक के पानी में भिगोना चाहिए?", "followup_reply": "कुछ घंटे नमक और पानी में रखने से मांस रसदार रहता है। बाद में खाल को पोंछकर सुखा लें ताकि स्मोकर में वह कुरकुरी बने।", "q1": "खाना स्मोक करने के लिए मुझे कौन-से लकड़ी के चिप्स पसंद हैं?", "q2": "खाना स्मोक करते समय मैं किस तरह के लकड़ी के चिप्स का बड़ा फ़ैन हूँ?", "distractor_sentence": "मेरे पड़ोसी अपने स्मोकर में हमेशा सिर्फ़ मेस्क्वीट चिप्स ही डालते हैं।"}, "hg": {"memory_turn": "Maine weekends pe khaana smoke karna shuru kiya hai, aur wood chips mein main hickory ka bada fan hoon. Is Sunday main poora chicken smoke karke dekhna chahta hoon. Ismein kitna time lagega?", "assistant_reply": "Lagbhag 120°C pe dedh kilo ka chicken usually teen se chaar ghante leta hai. Time se nahi, temperature se decide karein: thigh ka sabse mota hissa 74°C pe pahunche toh chicken ready hai.", "followup_turn": "Kya pehle ise brine karna chahiye?", "followup_reply": "Kuch ghante namak aur paani mein rakhne se meat juicy rehta hai. Baad mein skin ko pochh ke sukha lein taaki smoker mein woh crispy bane.", "q1": "Khaana smoke karne ke liye mujhe kaunse wood chips pasand hain?", "q2": "Khaana smoke karte time main kis type ke wood chips ka bada fan hoon?", "distractor_sentence": "Mere padosi apne smoker mein hamesha sirf mesquite chips hi daalte hain."}}, "gold": {"answer_en": "hickory", "slots": [{"name": "wood", "aliases": ["hickory", "हिकोरी", "हिकरी"]}]}, "distractor": {"value_aliases": ["mesquite", "मेस्क्वीट"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 2, "filler_seed": 3778626705, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P025", "cand_id": "PX0219", "memory_type": "preference", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "When I go on road trips, I prefer scenic routes, especially ones that pass through national parks. I'm driving from Seattle to San Francisco next month. How many days should I plan?", "assistant_reply": "If you take the coastal road through Oregon and the redwoods, plan four to five days. That leaves time for short walks, viewpoints and a relaxed pace instead of long days at the wheel.", "followup_turn": "Where should I stop for the first night?", "followup_reply": "Cannon Beach or Astoria in Oregon make a good first stop. Both are about four hours from Seattle and have plenty of places to eat and stay.", "q1": "What kind of driving routes do I prefer?", "q2": "Which type of roads do I like to take on road trips?", "distractor_sentence": "My husband always takes the fastest highway, no matter how boring it is."}, "hi": {"memory_turn": "रोड ट्रिप पर मुझे ख़ूबसूरत नज़ारों वाले रास्ते पसंद हैं, ख़ासकर वे जो नेशनल पार्क से होकर जाते हैं। अगले महीने मैं सिएटल से सैन फ़्रांसिस्को गाड़ी से जा रही हूँ। मुझे कितने दिन का प्लान बनाना चाहिए?", "assistant_reply": "अगर आप ओरेगन और रेडवुड जंगलों से होकर समंदर किनारे वाली सड़क लेती हैं, तो चार-पाँच दिन रखें। इससे छोटी सैर, व्यूपॉइंट और आराम से चलने का समय मिल जाता है।", "followup_turn": "पहली रात कहाँ रुकना चाहिए?", "followup_reply": "ओरेगन में कैनन बीच या एस्टोरिया पहले पड़ाव के लिए अच्छे हैं। दोनों सिएटल से करीब चार घंटे दूर हैं और वहाँ खाने-रुकने की कई जगहें हैं।", "q1": "मुझे किस तरह के ड्राइविंग रास्ते पसंद हैं?", "q2": "रोड ट्रिप पर मुझे किस तरह की सड़कें लेना अच्छा लगता है?", "distractor_sentence": "मेरे पति हमेशा सबसे तेज़ हाईवे लेते हैं, चाहे वह कितना भी उबाऊ हो।"}, "hg": {"memory_turn": "Road trips pe mujhe scenic routes pasand hain, especially woh jo national parks se hokar jaate hain. Next month main Seattle se San Francisco gaadi se ja rahi hoon. Mujhe kitne din ka plan banana chahiye?", "assistant_reply": "Agar aap Oregon aur redwood forests se hokar coastal road leti hain, toh chaar-paanch din rakhein. Isse chhoti walks, viewpoints aur aaraam se chalne ka time mil jaata hai.", "followup_turn": "Pehli raat kahan rukna chahiye?", "followup_reply": "Oregon mein Cannon Beach ya Astoria pehle stop ke liye acche hain. Dono Seattle se kareeb chaar ghante door hain aur wahan khaane-rukne ki kai jagah hain.", "q1": "Mujhe kis type ke driving routes pasand hain?", "q2": "Road trips pe mujhe kis type ki roads lena accha lagta hai?", "distractor_sentence": "Mere husband hamesha sabse fast highway lete hain, chahe woh kitna bhi boring ho."}}, "gold": {"answer_en": "scenic routes, especially through national parks", "slots": [{"name": "route", "aliases": ["scenic", "national park", "नज़ारों वाले", "नेशनल पार्क"]}]}, "distractor": {"value_aliases": ["fastest highway", "highway", "हाईवे"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 5, "filler_seed": 2190814720, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P026", "cand_id": "PX0221", "memory_type": "preference", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "My favourite football team is Manchester United, and just following them now isn't enough for me; I want to learn the club's history too. Can you recommend a good book about it?", "assistant_reply": "Try 'The Official Illustrated History', which is full of photos and key moments, or Sir Alex Ferguson's autobiography for an inside view of the most successful era.", "followup_turn": "Is there a good documentary as well?", "followup_reply": "'The Class of 92' is a popular documentary about a famous group of young players who came through the academy together and won many trophies.", "q1": "Which football team is my favourite?", "q2": "What is my favourite football club?", "q3": "I'm buying myself a new football jersey. Which club's shirt should I get?"}, "hi": {"memory_turn": "मेरी पसंदीदा फ़ुटबॉल टीम मैनचेस्टर यूनाइटेड है, और अब सिर्फ़ उसे फ़ॉलो करना मेरे लिए काफ़ी नहीं है; मैं क्लब का इतिहास भी जानना चाहता हूँ। क्या आप इस पर कोई अच्छी किताब बता सकते हैं?", "assistant_reply": "'The Official Illustrated History' पढ़िए, जिसमें बहुत सारी तस्वीरें और ख़ास पल हैं, या सबसे कामयाब दौर को अंदर से जानने के लिए सर एलेक्स फ़र्ग्युसन की आत्मकथा।", "followup_turn": "क्या कोई अच्छी डॉक्यूमेंट्री भी है?", "followup_reply": "'The Class of 92' एक लोकप्रिय डॉक्यूमेंट्री है, जो उन मशहूर युवा खिलाड़ियों के बारे में है जो साथ में अकादमी से आगे आए और कई ट्रॉफ़ी जीतीं।", "q1": "मेरी पसंदीदा फ़ुटबॉल टीम कौन-सी है?", "q2": "मेरा पसंदीदा फ़ुटबॉल क्लब कौन-सा है?", "q3": "मैं अपने लिए एक नई फ़ुटबॉल जर्सी ख़रीद रहा हूँ। मुझे किस क्लब की जर्सी लेनी चाहिए?"}, "hg": {"memory_turn": "Meri favourite football team Manchester United hai, aur ab sirf use follow karna mere liye kaafi nahi hai; main club ki history bhi jaanna chahta hoon. Kya aap is pe koi acchi book bata sakte hain?", "assistant_reply": "'The Official Illustrated History' padhiye, jismein bahut saari photos aur khaas moments hain, ya sabse successful daur ko andar se jaanne ke liye Sir Alex Ferguson ki autobiography.", "followup_turn": "Kya koi acchi documentary bhi hai?", "followup_reply": "'The Class of 92' ek popular documentary hai, jo un famous young players ke baare mein hai jo saath mein academy se aage aaye aur kai trophies jeeti.", "q1": "Meri favourite football team kaunsi hai?", "q2": "Mera favourite football club kaunsa hai?", "q3": "Main apne liye ek nayi football jersey khareed raha hoon. Mujhe kis club ki jersey leni chahiye?"}}, "gold": {"answer_en": "Manchester United", "slots": [{"name": "team", "aliases": ["Manchester United", "Man United", "Man Utd", "मैनचेस्टर यूनाइटेड", "मैन यूनाइटेड"]}]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 9, "filler_seed": 1441530076, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "P027", "cand_id": "PX0370", "memory_type": "preference", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "My favourite camera for street photography is my Canon QL17. I want to start developing black-and-white film at home. What basic equipment do I need?", "assistant_reply": "You'll need a changing bag, a developing tank with reels, a thermometer, measuring jugs, and three chemicals: developer, stop bath and fixer. Clips for hanging the film to dry help too.", "followup_turn": "Which developer is easiest for a beginner?", "followup_reply": "Kodak HC-110 and Ilford ID-11 are both forgiving and widely used. Many beginners also like Rodinal because one bottle lasts a very long time.", "q1": "Which camera do I like best for street photography?", "q2": "What is my favourite camera when I shoot on the street?", "q3": "I'm heading out this evening to capture life in the city's markets and lanes. Which camera should I take from my shelf?"}, "hi": {"memory_turn": "स्ट्रीट फ़ोटोग्राफ़ी के लिए मेरा पसंदीदा कैमरा मेरा Canon QL17 है। मैं घर पर ब्लैक-एंड-व्हाइट फ़िल्म डेवलप करना शुरू करना चाहती हूँ। मुझे कौन-सा बुनियादी सामान चाहिए?", "assistant_reply": "आपको एक चेंजिंग बैग, रील वाला डेवलपिंग टैंक, थर्मामीटर, नापने के जग और तीन केमिकल चाहिए: डेवलपर, स्टॉप बाथ और फ़िक्सर। फ़िल्म सुखाने के लिए टाँगने वाली क्लिप भी काम आती हैं।", "followup_turn": "नौसिखिए के लिए कौन-सा डेवलपर सबसे आसान है?", "followup_reply": "Kodak HC-110 और Ilford ID-11 दोनों ग़लतियाँ झेल लेते हैं और ख़ूब इस्तेमाल होते हैं। कई नए लोग Rodinal भी पसंद करते हैं क्योंकि एक बोतल बहुत लंबे समय तक चलती है।", "q1": "स्ट्रीट फ़ोटोग्राफ़ी के लिए मुझे कौन-सा कैमरा सबसे ज़्यादा पसंद है?", "q2": "सड़क पर फ़ोटो खींचते समय मेरा पसंदीदा कैमरा कौन-सा है?", "q3": "मैं आज शाम शहर के बाज़ारों और गलियों की ज़िंदगी क़ैद करने जा रही हूँ। अपनी शेल्फ़ से कौन-सा कैमरा उठाऊँ?"}, "hg": {"memory_turn": "Street photography ke liye mera favourite camera mera Canon QL17 hai. Main ghar pe black-and-white film develop karna shuru karna chahti hoon. Mujhe kaunsa basic saamaan chahiye?", "assistant_reply": "Aapko ek changing bag, reels wala developing tank, thermometer, measuring jugs aur teen chemicals chahiye: developer, stop bath aur fixer. Film sukhane ke liye taangne wale clips bhi kaam aate hain.", "followup_turn": "Beginner ke liye kaunsa developer sabse easy hai?", "followup_reply": "Kodak HC-110 aur Ilford ID-11 dono galtiyan jhel lete hain aur khoob use hote hain. Kai naye log Rodinal bhi pasand karte hain kyunki ek bottle bahut lambe time tak chalti hai.", "q1": "Street photography ke liye mujhe kaunsa camera sabse zyaada pasand hai?", "q2": "Street pe photos kheenchte time mera favourite camera kaunsa hai?", "q3": "Main aaj shaam sheher ke bazaaron aur galiyon ki zindagi capture karne ja rahi hoon. Apni shelf se kaunsa camera uthaun?"}}, "gold": {"answer_en": "Canon QL17", "slots": [{"name": "camera", "aliases": ["Canon QL17", "QL17", "QL 17", "Canonet QL17", "कैनन QL17"]}]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 2, "filler_seed": 2115010801, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "P028", "cand_id": "PX0375", "memory_type": "preference", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I just finished Attack on Titan, and my favourite character is Levi Ackerman; I love his cool and collected personality. Can you suggest another anime with similarly intense action?", "assistant_reply": "Jujutsu Kaisen has fast, stylish fights, Vinland Saga mixes brutal battles with a strong story about revenge, and Demon Slayer is known for stunning animation in its action scenes.", "followup_turn": "Which of those has the most serious story?", "followup_reply": "Vinland Saga is the most serious. Its second season slows down and becomes a thoughtful story about violence, guilt and finding peace.", "q1": "Who is my favourite character from Attack on Titan?", "q2": "Which Attack on Titan character do I like the most?", "distractor_sentence": "My cousin's favourite is Eren, but I find him too angry."}, "hi": {"memory_turn": "मैंने अभी-अभी Attack on Titan ख़त्म किया, और मेरा पसंदीदा किरदार लेवी एकरमैन है; मुझे उसका ठंडा और संयमित स्वभाव बहुत पसंद है। क्या आप ऐसी ही ज़बरदस्त एक्शन वाली कोई और एनीमे बता सकते हैं?", "assistant_reply": "Jujutsu Kaisen में तेज़ और स्टाइलिश लड़ाइयाँ हैं, Vinland Saga में ख़ूँख़ार जंग के साथ बदले की दमदार कहानी है, और Demon Slayer अपने एक्शन सीन की शानदार एनिमेशन के लिए मशहूर है।", "followup_turn": "इनमें से किसकी कहानी सबसे गंभीर है?", "followup_reply": "Vinland Saga सबसे गंभीर है। इसका दूसरा सीज़न धीमा होकर हिंसा, पछतावे और सुकून पाने की एक गहरी कहानी बन जाता है।", "q1": "Attack on Titan में मेरा पसंदीदा किरदार कौन है?", "q2": "Attack on Titan का कौन-सा किरदार मुझे सबसे ज़्यादा पसंद है?", "distractor_sentence": "मेरे कज़िन का पसंदीदा एरेन है, पर मुझे वह बहुत ग़ुस्सैल लगता है।"}, "hg": {"memory_turn": "Maine abhi-abhi Attack on Titan khatam kiya, aur mera favourite character Levi Ackerman hai; mujhe uski cool aur calm personality bahut pasand hai. Kya aap aisi hi intense action wali koi aur anime bata sakte hain?", "assistant_reply": "Jujutsu Kaisen mein fast aur stylish fights hain, Vinland Saga mein khatarnak battles ke saath revenge ki strong story hai, aur Demon Slayer apne action scenes ki shaandaar animation ke liye famous hai.", "followup_turn": "Inmein se kiski story sabse serious hai?", "followup_reply": "Vinland Saga sabse serious hai. Iska doosra season slow hokar violence, guilt aur sukoon paane ki ek gehri kahani ban jaata hai.", "q1": "Attack on Titan mein mera favourite character kaun hai?", "q2": "Attack on Titan ka kaunsa character mujhe sabse zyaada pasand hai?", "distractor_sentence": "Mere cousin ka favourite Eren hai, par mujhe woh bahut gussail lagta hai."}}, "gold": {"answer_en": "Levi Ackerman", "slots": [{"name": "character", "aliases": ["Levi", "लेवी"]}]}, "distractor": {"value_aliases": ["Eren", "एरेन"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 5, "filler_seed": 3964428379, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P029", "cand_id": "PX0377", "memory_type": "preference", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I prefer to work out in the morning, around 6:30 am, before work. Lately I feel low on energy during those sessions. Should I eat something before training?", "assistant_reply": "A small, easy snack 30 to 60 minutes before helps many people: a banana, a slice of toast with honey, or a few dates. Drink a glass of water too, since you wake up slightly dehydrated.", "followup_turn": "What should I eat after the workout?", "followup_reply": "Within an hour or two, have a meal with protein and carbohydrates, like eggs on toast, yogurt with fruit, or poha with peanuts.", "q1": "At what time do I prefer to work out?", "q2": "What time of day do I like to exercise?", "q3": "I'm booking a fixed slot at the new gym. Which time should I choose so it matches my usual training habit?"}, "hi": {"memory_turn": "मुझे सुबह काम से पहले, लगभग 6:30 बजे वर्कआउट करना पसंद है। आजकल उन सेशन में मुझे ऊर्जा कम लगती है। क्या ट्रेनिंग से पहले कुछ खाना चाहिए?", "assistant_reply": "बहुत लोगों को 30 से 60 मिनट पहले कोई हल्का नाश्ता मदद करता है: एक केला, शहद के साथ टोस्ट का एक स्लाइस, या कुछ खजूर। एक गिलास पानी भी पिएँ, क्योंकि सुबह शरीर में थोड़ी पानी की कमी होती है।", "followup_turn": "वर्कआउट के बाद क्या खाना चाहिए?", "followup_reply": "एक-दो घंटे के अंदर प्रोटीन और कार्बोहाइड्रेट वाला खाना लें, जैसे टोस्ट के साथ अंडे, फल के साथ दही, या मूँगफली वाला पोहा।", "q1": "मुझे किस समय वर्कआउट करना पसंद है?", "q2": "मुझे दिन में कब व्यायाम करना अच्छा लगता है?", "q3": "मैं नए जिम में एक तय स्लॉट बुक कर रही हूँ। अपनी ट्रेनिंग की आदत के हिसाब से कौन-सा समय चुनूँ?"}, "hg": {"memory_turn": "Mujhe subah kaam se pehle, lagbhag 6:30 baje workout karna pasand hai. Aajkal un sessions mein mujhe energy kam lagti hai. Kya training se pehle kuch khaana chahiye?", "assistant_reply": "Bahut logon ko 30 se 60 minute pehle koi halka snack help karta hai: ek kela, shahad ke saath toast ka ek slice, ya kuch khajoor. Ek glass paani bhi piyein, kyunki subah body mein thodi paani ki kami hoti hai.", "followup_turn": "Workout ke baad kya khaana chahiye?", "followup_reply": "Ek-do ghante ke andar protein aur carbs wala khaana lein, jaise toast ke saath ande, fruit ke saath dahi, ya moongfali wala poha.", "q1": "Mujhe kis time workout karna pasand hai?", "q2": "Mujhe din mein kab exercise karna accha lagta hai?", "q3": "Main naye gym mein ek fixed slot book kar rahi hoon. Apni training ki aadat ke hisaab se kaunsa time choose karoon?"}}, "gold": {"answer_en": "around 6:30 am", "slots": [{"name": "time", "aliases": ["6:30", "6.30", "साढ़े छह", "sadhe chhe", "half past six"]}]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 9, "filler_seed": 363900253, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "P030", "cand_id": "PX0399", "memory_type": "preference", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I've always loved the combination of espresso and mascarpone. I'm hosting a dinner next week and want a dessert I can make the day before. Any ideas?", "assistant_reply": "Tiramisu is the obvious choice and actually tastes better after a night in the fridge. A coffee panna cotta or a no-bake cheesecake in small glasses would also work well.", "followup_turn": "Can I make tiramisu without raw eggs?", "followup_reply": "Yes. Many recipes skip the eggs and use whipped cream folded into the cheese instead, which gives a light, stable texture.", "q1": "Which flavour combination have I always loved?", "q2": "What pairing of flavours has always been my favourite?", "distractor_sentence": "My wife prefers chocolate and orange together in her desserts."}, "hi": {"memory_turn": "मुझे हमेशा से एस्प्रेसो और मस्कार्पोने का मेल बहुत पसंद रहा है। अगले हफ़्ते मैं डिनर रख रहा हूँ और ऐसी मिठाई चाहता हूँ जो एक दिन पहले बन जाए। कोई आइडिया?", "assistant_reply": "तिरामिसु सबसे सही रहेगी, और फ़्रिज में एक रात रखने के बाद तो और स्वादिष्ट लगती है। कॉफ़ी पन्ना कोटा या छोटे गिलासों में बिना बेक वाला चीज़केक भी अच्छा रहेगा।", "followup_turn": "क्या कच्चे अंडों के बिना तिरामिसु बन सकती है?", "followup_reply": "हाँ। कई रेसिपी में अंडों की जगह फेंटी हुई क्रीम चीज़ में मिलाई जाती है, जिससे हल्की और टिकाऊ बनावट मिलती है।", "q1": "मुझे हमेशा से किन स्वादों का मेल पसंद रहा है?", "q2": "स्वादों की कौन-सी जोड़ी हमेशा से मेरी पसंदीदा रही है?", "distractor_sentence": "मेरी पत्नी को मिठाई में चॉकलेट और संतरा साथ में पसंद है।"}, "hg": {"memory_turn": "Mujhe hamesha se espresso aur mascarpone ka combination bahut pasand raha hai. Next week main dinner rakh raha hoon aur aisi dessert chahta hoon jo ek din pehle ban jaaye. Koi idea?", "assistant_reply": "Tiramisu sabse sahi rahegi, aur fridge mein ek raat rakhne ke baad toh aur tasty lagti hai. Coffee panna cotta ya chhote glasses mein no-bake cheesecake bhi accha rahega.", "followup_turn": "Kya kacche ande ke bina tiramisu ban sakti hai?", "followup_reply": "Haan. Kai recipes mein ande ki jagah whipped cream cheese mein milayi jaati hai, jisse halka aur stable texture milta hai.", "q1": "Mujhe hamesha se kin flavours ka combination pasand raha hai?", "q2": "Flavours ki kaunsi jodi hamesha se meri favourite rahi hai?", "distractor_sentence": "Meri wife ko dessert mein chocolate aur orange saath mein pasand hai."}}, "gold": {"answer_en": "espresso and mascarpone", "slots": [{"name": "flavour1", "aliases": ["espresso", "एस्प्रेसो"]}, {"name": "flavour2", "aliases": ["mascarpone", "मस्कार्पोने", "मास्कार्पोने"]}]}, "distractor": {"value_aliases": ["chocolate and orange", "chocolate aur orange", "चॉकलेट और संतरा"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 2, "filler_seed": 1302923492, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "E021", "cand_id": "EC023", "memory_type": "event", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I recently got a new pair of Adidas running shoes on January 10th, and I want to make sure I take good care of them. How should I clean them without damaging the foam?", "assistant_reply": "Remove the insoles and laces, brush off dry dirt, then scrub gently with lukewarm water and mild soap. Never put them in the dryer; stuff them with newspaper and let them air-dry.", "followup_turn": "How many kilometres do running shoes usually last?", "followup_reply": "Most last between 500 and 800 kilometres. Lighter racing shoes wear out sooner, while heavier cushioned trainers can last longer.", "q1": "When did I get my new Adidas running shoes?", "q2": "On what date did I buy my new pair of Adidas running shoes?", "q3": "I like to replace my running shoes after six months. From which date should I start counting for my current pair?", "distractor_sentence": "My brother bought the same model on January 24th."}, "hi": {"memory_turn": "मैंने हाल ही में 10 जनवरी को Adidas के नए रनिंग शूज़ लिए, और मैं उनका अच्छे से ध्यान रखना चाहती हूँ। फ़ोम को नुक़सान पहुँचाए बिना उन्हें कैसे साफ़ करूँ?", "assistant_reply": "इनसोल और फीते निकाल लें, सूखी मिट्टी ब्रश से झाड़ें, फिर गुनगुने पानी और हल्के साबुन से धीरे-धीरे रगड़ें। इन्हें कभी ड्रायर में न डालें; अंदर अख़बार भरकर हवा में सूखने दें।", "followup_turn": "रनिंग शूज़ आमतौर पर कितने किलोमीटर चलते हैं?", "followup_reply": "ज़्यादातर 500 से 800 किलोमीटर तक चलते हैं। हल्के रेसिंग शूज़ जल्दी घिसते हैं, जबकि भारी कुशन वाले ज़्यादा चल सकते हैं।", "q1": "मैंने अपने नए Adidas रनिंग शूज़ कब लिए?", "q2": "मैंने Adidas के नए रनिंग शूज़ किस तारीख़ को ख़रीदे?", "q3": "मैं अपने रनिंग शूज़ छह महीने बाद बदलती हूँ। अभी वाले जोड़े के लिए मुझे किस तारीख़ से गिनती शुरू करनी चाहिए?", "distractor_sentence": "मेरे भाई ने वही मॉडल 24 जनवरी को ख़रीदा था।"}, "hg": {"memory_turn": "Maine haal hi mein 10 January ko Adidas ke naye running shoes liye, aur main unka acche se dhyaan rakhna chahti hoon. Foam ko nuksaan pahunchaye bina unhe kaise saaf karoon?", "assistant_reply": "Insoles aur laces nikaal lein, sookhi mitti brush se jhaadein, phir gunguna paani aur halke sabun se dheere-dheere ragdein. Inhe kabhi dryer mein na daalein; andar newspaper bhar ke hawa mein sookhne dein.", "followup_turn": "Running shoes usually kitne kilometre chalte hain?", "followup_reply": "Zyaadatar 500 se 800 kilometre tak chalte hain. Halke racing shoes jaldi ghiste hain, jabki bhaari cushioned trainers zyaada chal sakte hain.", "q1": "Maine apne naye Adidas running shoes kab liye?", "q2": "Maine Adidas ke naye running shoes kis date ko khareede?", "q3": "Main apne running shoes chhe mahine baad badalti hoon. Abhi wale pair ke liye mujhe kis date se ginti shuru karni chahiye?", "distractor_sentence": "Mere bhai ne wahi model 24 January ko khareeda tha."}}, "gold": {"answer_en": "January 10", "date": {"day": 10, "month": 1}, "slots": [{"name": "date", "aliases": ["January 10", "January 10th", "10 January", "10th January", "10th of January", "Jan 10", "10 Jan", "10 जनवरी"]}]}, "distractor": {"date": {"day": 24, "month": 1}, "value_aliases": ["January 24", "January 24th", "24 January", "24th January", "24th of January", "Jan 24", "24 Jan", "24 जनवरी"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "dcfa8644", "lme_type": "temporal-reasoning"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 4004256290, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": true, "abl_digits": true, "banned_fillers": []}
{"item_id": "E022", "cand_id": "EC024", "memory_type": "event", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I wore my white sneakers to play basketball on February 1st, and they got pretty dirty. What's the best way to get them clean again?", "assistant_reply": "Mix a little baking soda with dish soap and warm water, scrub with an old toothbrush, and wipe clean. For the soles, a melamine sponge removes scuffs very well.", "followup_turn": "Can I put them in the washing machine?", "followup_reply": "Many canvas and mesh sneakers survive a gentle cold cycle inside a pillowcase. Avoid it for leather pairs, and always let them air-dry.", "q1": "When did I wear my sneakers to play basketball?", "q2": "On what date did I play basketball in my white sneakers?", "q3": "My white sneakers still have stains from that one game on the court. On which date did that game happen?"}, "hi": {"memory_turn": "मैंने 1 फ़रवरी को बास्केटबॉल खेलते समय अपने सफ़ेद स्नीकर्स पहने थे, और वे काफ़ी गंदे हो गए। उन्हें फिर से साफ़ करने का सबसे अच्छा तरीक़ा क्या है?", "assistant_reply": "थोड़ा बेकिंग सोडा, बर्तन धोने का साबुन और गुनगुना पानी मिलाएँ, पुराने टूथब्रश से रगड़ें और पोंछ लें। तलवों के लिए मेलामाइन स्पंज निशान बहुत अच्छे से हटाता है।", "followup_turn": "क्या मैं इन्हें वॉशिंग मशीन में डाल सकता हूँ?", "followup_reply": "कई कैनवस और जाली वाले स्नीकर्स तकिये के कवर में रखकर ठंडे, हल्के साइकल में ठीक रहते हैं। चमड़े वालों के लिए ऐसा न करें, और हमेशा हवा में सुखाएँ।", "q1": "मैंने अपने स्नीकर्स पहनकर बास्केटबॉल कब खेला था?", "q2": "मैंने अपने सफ़ेद स्नीकर्स में किस तारीख़ को बास्केटबॉल खेला?", "q3": "मेरे सफ़ेद स्नीकर्स पर अभी भी कोर्ट वाले उस एक मैच के दाग़ हैं। वह मैच किस तारीख़ को हुआ था?"}, "hg": {"memory_turn": "Maine 1 February ko basketball khelte time apne white sneakers pehne the, aur woh kaafi gande ho gaye. Unhe phir se saaf karne ka sabse accha tareeka kya hai?", "assistant_reply": "Thoda baking soda, bartan dhone ka sabun aur garam paani milayein, purane toothbrush se ragdein aur pochh lein. Soles ke liye melamine sponge nishaan bahut acche se hatata hai.", "followup_turn": "Kya main inhe washing machine mein daal sakta hoon?", "followup_reply": "Kai canvas aur mesh sneakers pillowcase mein rakh ke thande, gentle cycle mein theek rehte hain. Leather walon ke liye aisa na karein, aur hamesha hawa mein sukhayein.", "q1": "Maine apne sneakers pehen ke basketball kab khela tha?", "q2": "Maine apne white sneakers mein kis date ko basketball khela?", "q3": "Mere white sneakers pe abhi bhi court wale us ek match ke daag hain. Woh match kis date ko hua tha?"}}, "gold": {"answer_en": "February 1", "date": {"day": 1, "month": 2}, "slots": [{"name": "date", "aliases": ["February 1", "February 1st", "1 February", "1st February", "1st of February", "Feb 1", "1 Feb", "1 फ़रवरी", "1 फरवरी"]}]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "dcfa8644", "lme_type": "temporal-reasoning"}, "topic": "other", "is_answerable": true, "evidence_position": 5, "filler_seed": 1352470443, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": true, "abl_q3": true, "abl_digits": true, "banned_fillers": []}
{"item_id": "E023", "cand_id": "EC026", "memory_type": "event", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I got a new router on January 15th, and it has improved my Wi-Fi signal a lot, but I still need to set up the guest network. How do I do that?", "assistant_reply": "Open the router's app or its admin page in a browser, find the Guest Network setting, turn it on, and give it its own name and password. Keep the option that blocks guests from seeing your devices switched on.", "followup_turn": "Should the guest password be different from my main one?", "followup_reply": "Yes, always. That way you can share or change it freely without ever giving anyone access to your main network.", "q1": "When did I get my new router?", "q2": "On what date did my new router arrive?"}, "hi": {"memory_turn": "मैंने 15 जनवरी को नया राउटर लिया, और इससे मेरा Wi-Fi सिग्नल बहुत बेहतर हो गया है, पर मुझे अभी गेस्ट नेटवर्क सेट करना है। यह कैसे करूँ?", "assistant_reply": "राउटर का ऐप या ब्राउज़र में उसका एडमिन पेज खोलें, Guest Network सेटिंग ढूँढें, उसे चालू करें और उसे अलग नाम और पासवर्ड दें। मेहमानों को आपके डिवाइस देखने से रोकने वाला विकल्प चालू रखें।", "followup_turn": "क्या गेस्ट पासवर्ड मेरे मुख्य पासवर्ड से अलग होना चाहिए?", "followup_reply": "हाँ, हमेशा। इससे आप उसे आराम से शेयर या बदल सकती हैं, और किसी को आपके मुख्य नेटवर्क तक पहुँच नहीं मिलती।", "q1": "मैंने अपना नया राउटर कब लिया?", "q2": "मेरा नया राउटर किस तारीख़ को आया?"}, "hg": {"memory_turn": "Maine 15 January ko naya router liya, aur isse mera Wi-Fi signal bahut better ho gaya hai, par mujhe abhi guest network set karna hai. Yeh kaise karoon?", "assistant_reply": "Router ka app ya browser mein uska admin page kholein, Guest Network setting dhoondhein, use on karein aur use alag naam aur password dein. Guests ko aapke devices dekhne se rokne wala option on rakhein.", "followup_turn": "Kya guest password mere main password se alag hona chahiye?", "followup_reply": "Haan, hamesha. Isse aap use aaraam se share ya change kar sakti hain, aur kisi ko aapke main network tak access nahi milta.", "q1": "Maine apna naya router kab liya?", "q2": "Mera naya router kis date ko aaya?"}}, "gold": {"answer_en": "January 15", "date": {"day": 15, "month": 1}, "slots": [{"name": "date", "aliases": ["January 15", "January 15th", "15 January", "15th January", "15th of January", "Jan 15", "15 Jan", "15 जनवरी"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "gpt4_d9af6064", "lme_type": "temporal-reasoning"}, "topic": "other", "is_answerable": true, "evidence_position": 9, "filler_seed": 1366866113, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": true, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E024", "cand_id": "EC029", "memory_type": "event", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I got a great deal on a personalised photo album from Shutterfly. I ordered it on April 15th for my best friend's birthday, and it turned out amazing! What other personalised gifts are good for friends?", "assistant_reply": "A custom star map of a special night, a recipe book of dishes you've shared, or a framed print of a favourite trip photo all feel personal. Engraved items like a wallet or keychain work well too.", "followup_turn": "How early should I order personalised gifts?", "followup_reply": "Give yourself two to three weeks. Production and shipping can be slow, especially around holidays, and you may want time to fix any mistakes.", "q1": "When did I order the photo album from Shutterfly?", "q2": "On what date did I place my order for the personalised photo album?", "distractor_sentence": "My sister ordered a calendar from the same site on May 2nd."}, "hi": {"memory_turn": "मुझे Shutterfly से एक पर्सनलाइज़्ड फ़ोटो एल्बम पर बढ़िया डील मिली। मैंने इसे अपने सबसे अच्छे दोस्त के जन्मदिन के लिए 15 अप्रैल को ऑर्डर किया था, और यह कमाल का बना! दोस्तों के लिए और कौन-से पर्सनलाइज़्ड तोहफ़े अच्छे हैं?", "assistant_reply": "किसी ख़ास रात का कस्टम स्टार मैप, साथ खाए पकवानों की रेसिपी बुक, या किसी पसंदीदा ट्रिप की फ़्रेम की हुई फ़ोटो, सब निजी लगते हैं। नाम खुदे बटुए या चाबी के छल्ले भी अच्छे रहते हैं।", "followup_turn": "पर्सनलाइज़्ड तोहफ़े कितने पहले ऑर्डर करने चाहिए?", "followup_reply": "दो-तीन हफ़्ते का समय रखें। बनने और पहुँचने में देर हो सकती है, ख़ासकर त्योहारों के आसपास, और ग़लती सुधारने का समय भी चाहिए।", "q1": "मैंने Shutterfly से फ़ोटो एल्बम कब ऑर्डर किया?", "q2": "पर्सनलाइज़्ड फ़ोटो एल्बम का ऑर्डर मैंने किस तारीख़ को दिया?", "distractor_sentence": "मेरी बहन ने उसी साइट से 2 मई को एक कैलेंडर ऑर्डर किया था।"}, "hg": {"memory_turn": "Mujhe Shutterfly se ek personalised photo album pe badhiya deal mili. Maine ise apne best friend ke birthday ke liye 15 April ko order kiya tha, aur yeh kamaal ka bana! Doston ke liye aur kaunse personalised gifts acche hain?", "assistant_reply": "Kisi khaas raat ka custom star map, saath khaaye dishes ki recipe book, ya kisi favourite trip ki framed photo, sab personal lagte hain. Naam engrave kiye wallet ya keychain bhi acche rehte hain.", "followup_turn": "Personalised gifts kitne pehle order karne chahiye?", "followup_reply": "Do-teen hafte ka time rakhein. Banne aur pahunchne mein der ho sakti hai, especially tyohaaron ke aaspaas, aur galti sudharne ka time bhi chahiye.", "q1": "Maine Shutterfly se photo album kab order kiya?", "q2": "Personalised photo album ka order maine kis date ko diya?", "distractor_sentence": "Meri behen ne usi site se 2 May ko ek calendar order kiya tha."}}, "gold": {"answer_en": "April 15", "date": {"day": 15, "month": 4}, "slots": [{"name": "date", "aliases": ["April 15", "April 15th", "15 April", "15th April", "15th of April", "Apr 15", "15 Apr", "15 अप्रैल"]}]}, "distractor": {"date": {"day": 2, "month": 5}, "value_aliases": ["May 2", "May 2nd", "2 May", "2nd May", "2nd of May", "May 2", "2 May", "2 मई"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "a3045048", "lme_type": "temporal-reasoning"}, "topic": "media/books/games", "is_answerable": true, "evidence_position": 2, "filler_seed": 2413420439, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E025", "cand_id": "EC031", "memory_type": "event", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I submitted my application to the exchange program on February 10th, and I just got accepted, so I'm really excited to finally be going! What should I sort out first before I leave?", "assistant_reply": "Start with the visa, since it takes the longest. Then arrange housing, check how your credits will transfer, and get travel insurance. A list of key documents with copies saved online is very useful.", "followup_turn": "How much money should I take with me at the start?", "followup_reply": "Plan for at least the first month's rent and deposit plus a few weeks of living costs, since grants and new bank accounts can take time to start working.", "q1": "When did I submit my application to the exchange program?", "q2": "On what date did I send in my exchange program application?", "q3": "My university office needs the date I sent off my study-abroad form for their records. What date should I give them?", "distractor_sentence": "My roommate applied to the same program on March 1st."}, "hi": {"memory_turn": "मैंने एक्सचेंज प्रोग्राम के लिए 10 फ़रवरी को आवेदन भेजा था, और अभी मेरा चयन हो गया है, आख़िरकार जाने को लेकर मैं बहुत उत्साहित हूँ! जाने से पहले सबसे पहले क्या तैयार करूँ?", "assistant_reply": "वीज़ा से शुरू करें, क्योंकि उसमें सबसे ज़्यादा समय लगता है। फिर रहने की जगह तय करें, देखें कि आपके क्रेडिट कैसे ट्रांसफ़र होंगे, और ट्रैवल इंश्योरेंस लें। ज़रूरी काग़ज़ों की लिस्ट और उनकी ऑनलाइन कॉपी बहुत काम आती है।", "followup_turn": "शुरुआत में अपने साथ कितने पैसे ले जाऊँ?", "followup_reply": "कम से कम पहले महीने का किराया और डिपॉज़िट, और कुछ हफ़्तों का ख़र्च रखें, क्योंकि स्कॉलरशिप और नए बैंक खाते चालू होने में समय लग सकता है।", "q1": "मैंने एक्सचेंज प्रोग्राम के लिए आवेदन कब भेजा था?", "q2": "एक्सचेंज प्रोग्राम का आवेदन मैंने किस तारीख़ को जमा किया?", "q3": "मेरी यूनिवर्सिटी के दफ़्तर को रिकॉर्ड के लिए वह तारीख़ चाहिए जब मैंने विदेश पढ़ाई वाला फ़ॉर्म भेजा था। उन्हें कौन-सी तारीख़ बताऊँ?", "distractor_sentence": "मेरी रूममेट ने उसी प्रोग्राम के लिए 1 मार्च को आवेदन किया था।"}, "hg": {"memory_turn": "Maine exchange program ke liye 10 February ko application bheji thi, aur abhi mera selection ho gaya hai, aakhirkaar jaane ko lekar main bahut excited hoon! Jaane se pehle sabse pehle kya ready karoon?", "assistant_reply": "Visa se shuru karein, kyunki usmein sabse zyaada time lagta hai. Phir rehne ki jagah decide karein, dekhein ki aapke credits kaise transfer honge, aur travel insurance lein. Zaroori documents ki list aur unki online copy bahut kaam aati hai.", "followup_turn": "Shuruaat mein apne saath kitne paise le jaaun?", "followup_reply": "Kam se kam pehle mahine ka rent aur deposit, aur kuch hafton ka kharcha rakhein, kyunki scholarship aur naye bank accounts chalu hone mein time lag sakta hai.", "q1": "Maine exchange program ke liye application kab bheji thi?", "q2": "Exchange program ki application maine kis date ko submit ki?", "q3": "Meri university ke office ko record ke liye woh date chahiye jab maine study abroad wala form bheja tha. Unhe kaunsi date bataun?", "distractor_sentence": "Meri roommate ne usi program ke liye 1 March ko apply kiya tha."}}, "gold": {"answer_en": "February 10", "date": {"day": 10, "month": 2}, "slots": [{"name": "date", "aliases": ["February 10", "February 10th", "10 February", "10th February", "10th of February", "Feb 10", "10 Feb", "10 फ़रवरी", "10 फरवरी"]}]}, "distractor": {"date": {"day": 1, "month": 3}, "value_aliases": ["March 1", "March 1st", "1 March", "1st March", "1st of March", "Mar 1", "1 Mar", "1 मार्च"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "gpt4_4cd9eba1", "lme_type": "temporal-reasoning"}, "topic": "other", "is_answerable": true, "evidence_position": 5, "filler_seed": 3113028245, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": true, "abl_digits": true, "banned_fillers": []}
{"item_id": "E026", "cand_id": "EC032", "memory_type": "event", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "Speaking of engagements, my close friend Rachel got engaged on May 15th, and we're already planning a surprise party for her. Can you suggest some fun party themes?", "assistant_reply": "A 'love around the world' theme with food from places the couple wants to visit is fun, as is a retro disco night or a garden picnic with fairy lights and lawn games.", "followup_turn": "How do we keep it a surprise?", "followup_reply": "Use a separate group chat without her, give her partner the job of getting her there on time, and tell guests to arrive at least thirty minutes early.", "q1": "When did my friend Rachel get engaged?", "q2": "On what date did Rachel's engagement happen?"}, "hi": {"memory_turn": "सगाई की बात चली तो, मेरी क़रीबी दोस्त रेचल की 15 मई को सगाई हुई, और हम अभी से उसके लिए एक सरप्राइज़ पार्टी की योजना बना रहे हैं। क्या आप पार्टी की कुछ मज़ेदार थीम बता सकते हैं?", "assistant_reply": "'दुनिया भर का प्यार' थीम, जिसमें उन जगहों का खाना हो जहाँ जोड़ा जाना चाहता है, मज़ेदार रहेगी। रेट्रो डिस्को नाइट या जगमगाती लाइटों और खेलों वाली बग़ीचा पिकनिक भी अच्छी है।", "followup_turn": "इसे सरप्राइज़ कैसे रखें?", "followup_reply": "उसके बिना एक अलग ग्रुप चैट बनाएँ, उसके पार्टनर को उसे समय पर लाने का ज़िम्मा दें, और मेहमानों को कम से कम आधा घंटा पहले आने को कहें।", "q1": "मेरी दोस्त रेचल की सगाई कब हुई?", "q2": "रेचल की सगाई किस तारीख़ को हुई थी?"}, "hg": {"memory_turn": "Engagement ki baat chali toh, meri close friend Rachel ki 15 May ko engagement hui, aur hum abhi se uske liye ek surprise party plan kar rahe hain. Kya aap party ke kuch mazedaar themes bata sakte hain?", "assistant_reply": "'Love around the world' theme, jismein un jagahon ka khaana ho jahan couple jaana chahta hai, mazedaar rahegi. Retro disco night ya fairy lights aur games wali garden picnic bhi acchi hai.", "followup_turn": "Ise surprise kaise rakhein?", "followup_reply": "Uske bina ek alag group chat banayein, uske partner ko use time pe laane ki zimmedari dein, aur guests ko kam se kam aadha ghanta pehle aane ko kahein.", "q1": "Meri friend Rachel ki engagement kab hui?", "q2": "Rachel ki engagement kis date ko hui thi?"}}, "gold": {"answer_en": "May 15", "date": {"day": 15, "month": 5}, "slots": [{"name": "date", "aliases": ["May 15", "May 15th", "15 May", "15th May", "15th of May", "May 15", "15 May", "15 मई"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "6613b389", "lme_type": "temporal-reasoning"}, "topic": "family/social", "is_answerable": true, "evidence_position": 9, "filler_seed": 2930300006, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": true, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E027", "cand_id": "EC034", "memory_type": "event", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I attended a pride parade in my city on May 1st, and it got me thinking about diversity and inclusion at work. How can I support inclusion in my team as a regular employee?", "assistant_reply": "Make sure quieter colleagues get space in meetings, use people's correct names and pronouns, speak up when you hear exclusionary jokes, and suggest team events that everyone can join.", "followup_turn": "Is it worth starting an employee resource group?", "followup_reply": "It can be. Start by finding a few interested colleagues and an executive sponsor, then agree on simple goals like monthly talks or a mentoring programme.", "q1": "When did I attend the pride parade?", "q2": "On what date did I go to the pride parade in my city?"}, "hi": {"memory_turn": "मैं 1 मई को अपने शहर की प्राइड परेड में गई थी, और तब से मैं काम की जगह पर विविधता और समावेश के बारे में सोच रही हूँ। एक आम कर्मचारी होकर मैं अपनी टीम में समावेश को कैसे बढ़ावा दूँ?", "assistant_reply": "ध्यान रखें कि मीटिंग में चुप रहने वाले साथियों को भी बोलने का मौक़ा मिले, लोगों के सही नाम और सर्वनाम इस्तेमाल करें, किसी को अलग करने वाले मज़ाक़ पर आवाज़ उठाएँ, और ऐसे टीम इवेंट सुझाएँ जिनमें सब शामिल हो सकें।", "followup_turn": "क्या एम्प्लॉई रिसोर्स ग्रुप शुरू करना ठीक रहेगा?", "followup_reply": "हो सकता है। पहले कुछ इच्छुक साथी और एक वरिष्ठ प्रायोजक ढूँढें, फिर मासिक बातचीत या मेंटरिंग जैसे आसान लक्ष्य तय करें।", "q1": "मैं प्राइड परेड में कब गई थी?", "q2": "मैं अपने शहर की प्राइड परेड में किस तारीख़ को गई?"}, "hg": {"memory_turn": "Main 1 May ko apne sheher ki pride parade mein gayi thi, aur tab se main kaam ki jagah pe diversity aur inclusion ke baare mein soch rahi hoon. Ek normal employee hokar main apni team mein inclusion ko kaise support karoon?", "assistant_reply": "Dhyaan rakhein ki meetings mein chup rehne wale colleagues ko bhi bolne ka mauka mile, logon ke sahi naam aur pronouns use karein, kisi ko alag karne wale jokes pe awaaz uthayein, aur aise team events suggest karein jinmein sab shaamil ho sakein.", "followup_turn": "Kya employee resource group shuru karna theek rahega?", "followup_reply": "Ho sakta hai. Pehle kuch interested colleagues aur ek senior sponsor dhoondhein, phir monthly talks ya mentoring jaise simple goals decide karein.", "q1": "Main pride parade mein kab gayi thi?", "q2": "Main apne sheher ki pride parade mein kis date ko gayi?"}}, "gold": {"answer_en": "May 1", "date": {"day": 1, "month": 5}, "slots": [{"name": "date", "aliases": ["May 1", "May 1st", "1 May", "1st May", "1st of May", "May 1", "1 May", "1 मई"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "gpt4_1a1dc16d", "lme_type": "temporal-reasoning"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 1899684798, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": false, "abl_digits": true, "banned_fillers": ["FL33"]}
{"item_id": "E028", "cand_id": "EC035", "memory_type": "event", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I'm thinking about switching from engineering to product management. I discussed it with my mentor, Rachel, when we met on April 10th. What skills should I build first?", "assistant_reply": "Focus on understanding users, writing clear problem statements, prioritising features, and working with data to measure results. Your engineering background is a real strength for technical discussions.", "followup_turn": "Should I take a product management course?", "followup_reply": "A short course can help with vocabulary and frameworks, but the best proof is doing product work: volunteer to own a small feature or write a spec at your current job.", "q1": "When did I meet my mentor Rachel?", "q2": "On what date did I have my meeting with my mentor, Rachel?", "q3": "My mentor asked me to recap what we discussed in our last session. When did that session take place?", "distractor_sentence": "My next session with my career coach is on April 28th."}, "hi": {"memory_turn": "मैं इंजीनियरिंग छोड़कर प्रोडक्ट मैनेजमेंट में जाने का सोच रहा हूँ। मैंने इस बारे में अपनी मेंटर रेचल से बात की थी, जब हम 10 अप्रैल को मिले थे। मुझे पहले कौन-से हुनर सीखने चाहिए?", "assistant_reply": "यूज़र्स को समझने, समस्या को साफ़ शब्दों में लिखने, फ़ीचर की प्राथमिकता तय करने और नतीजे नापने के लिए डेटा के साथ काम करने पर ध्यान दें। तकनीकी चर्चाओं में आपकी इंजीनियरिंग की पृष्ठभूमि बड़ी ताक़त है।", "followup_turn": "क्या मुझे प्रोडक्ट मैनेजमेंट का कोई कोर्स करना चाहिए?", "followup_reply": "छोटा कोर्स शब्दावली और फ़्रेमवर्क में मदद करता है, पर सबसे अच्छा सबूत ख़ुद प्रोडक्ट का काम करना है: अपनी मौजूदा नौकरी में कोई छोटा फ़ीचर सँभालें या एक स्पेक लिखें।", "q1": "मैं अपनी मेंटर रेचल से कब मिला था?", "q2": "अपनी मेंटर रेचल के साथ मेरी मीटिंग किस तारीख़ को हुई थी?", "q3": "मेरी मेंटर ने कहा कि पिछली बातचीत में जो चर्चा हुई उसका सार बताऊँ। वह बातचीत कब हुई थी?", "distractor_sentence": "मेरे करियर कोच के साथ मेरा अगला सेशन 28 अप्रैल को है।"}, "hg": {"memory_turn": "Main engineering chhod ke product management mein jaane ka soch raha hoon. Maine is baare mein apni mentor Rachel se baat ki thi, jab hum 10 April ko mile the. Mujhe pehle kaunse skills seekhne chahiye?", "assistant_reply": "Users ko samajhne, problem ko clear words mein likhne, features ki priority decide karne aur results naapne ke liye data ke saath kaam karne pe focus karein. Technical discussions mein aapka engineering background badi strength hai.", "followup_turn": "Kya mujhe product management ka koi course karna chahiye?", "followup_reply": "Chhota course vocabulary aur frameworks mein help karta hai, par sabse accha proof khud product ka kaam karna hai: apni current job mein koi chhota feature sambhalein ya ek spec likhein.", "q1": "Main apni mentor Rachel se kab mila tha?", "q2": "Apni mentor Rachel ke saath meri meeting kis date ko hui thi?", "q3": "Meri mentor ne kaha ki pichhli baatcheet mein jo discuss hua uska summary bataun. Woh baatcheet kab hui thi?", "distractor_sentence": "Mere career coach ke saath mera next session 28 April ko hai."}}, "gold": {"answer_en": "April 10", "date": {"day": 10, "month": 4}, "slots": [{"name": "date", "aliases": ["April 10", "April 10th", "10 April", "10th April", "10th of April", "Apr 10", "10 Apr", "10 अप्रैल"]}]}, "distractor": {"date": {"day": 28, "month": 4}, "value_aliases": ["April 28", "April 28th", "28 April", "28th April", "28th of April", "Apr 28", "28 Apr", "28 अप्रैल"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "gpt4_1a1dc16d", "lme_type": "temporal-reasoning"}, "topic": "other", "is_answerable": true, "evidence_position": 5, "filler_seed": 1614385674, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": false, "abl_q3": true, "abl_digits": true, "banned_fillers": []}
{"item_id": "E029", "cand_id": "EC036", "memory_type": "event", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I bought a silver necklace with a tiny pendant for my best friend's 30th birthday on March 15th, and she loved it. How do I keep silver jewellery from tarnishing?", "assistant_reply": "Store silver in a zip bag or an anti-tarnish pouch, away from humidity. Take it off before swimming or using perfume, and polish it gently with a soft silver cloth now and then.", "followup_turn": "Can I clean tarnished silver at home?", "followup_reply": "Yes. Line a bowl with foil, add hot water, baking soda and a pinch of salt, and soak the piece for a few minutes. Rinse and dry it well afterwards.", "q1": "When did I buy the silver necklace for my best friend?", "q2": "On what date did I get the silver necklace for my best friend's birthday?", "distractor_sentence": "My sister's birthday party is on March 29th, and I still need a gift for her."}, "hi": {"memory_turn": "मैंने 15 मार्च को अपनी सबसे अच्छी दोस्त के 30वें जन्मदिन के लिए छोटे-से पेंडेंट वाला चाँदी का हार ख़रीदा, और उसे बहुत पसंद आया। चाँदी के गहनों को काला पड़ने से कैसे बचाऊँ?", "assistant_reply": "चाँदी को ज़िप बैग या एंटी-टार्निश पाउच में नमी से दूर रखें। तैरने या परफ़्यूम लगाने से पहले उतार दें, और कभी-कभी मुलायम सिल्वर क्लॉथ से हल्के हाथ से चमकाएँ।", "followup_turn": "क्या काली पड़ी चाँदी घर पर साफ़ कर सकती हूँ?", "followup_reply": "हाँ। एक कटोरे में फ़ॉइल बिछाएँ, गर्म पानी, बेकिंग सोडा और चुटकी भर नमक डालें, और गहने को कुछ मिनट भिगोएँ। बाद में धोकर अच्छे से सुखा लें।", "q1": "मैंने अपनी सबसे अच्छी दोस्त के लिए चाँदी का हार कब ख़रीदा?", "q2": "सबसे अच्छी दोस्त के जन्मदिन के लिए चाँदी का हार मैंने किस तारीख़ को लिया?", "distractor_sentence": "मेरी बहन की जन्मदिन पार्टी 29 मार्च को है, और मुझे अभी उसके लिए तोहफ़ा लेना है।"}, "hg": {"memory_turn": "Maine 15 March ko apni best friend ke 30th birthday ke liye chhote-se pendant wala silver necklace khareeda, aur use bahut pasand aaya. Silver jewellery ko kaala padne se kaise bachaun?", "assistant_reply": "Silver ko zip bag ya anti-tarnish pouch mein nami se door rakhein. Swimming ya perfume lagane se pehle utaar dein, aur kabhi-kabhi soft silver cloth se halke haath se polish karein.", "followup_turn": "Kya kaali padi silver ghar pe saaf kar sakti hoon?", "followup_reply": "Haan. Ek bowl mein foil bichhayein, garam paani, baking soda aur chutki bhar namak daalein, aur piece ko kuch minute bhigoyein. Baad mein dho ke acche se sukha lein.", "q1": "Maine apni best friend ke liye silver necklace kab khareeda?", "q2": "Best friend ke birthday ke liye silver necklace maine kis date ko liya?", "distractor_sentence": "Meri behen ki birthday party 29 March ko hai, aur mujhe abhi uske liye gift lena hai."}}, "gold": {"answer_en": "March 15", "date": {"day": 15, "month": 3}, "slots": [{"name": "date", "aliases": ["March 15", "March 15th", "15 March", "15th March", "15th of March", "Mar 15", "15 Mar", "15 मार्च"]}]}, "distractor": {"date": {"day": 29, "month": 3}, "value_aliases": ["March 29", "March 29th", "29 March", "29th March", "29th of March", "Mar 29", "29 Mar", "29 मार्च"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "8c18457d", "lme_type": "temporal-reasoning"}, "topic": "family/social", "is_answerable": true, "evidence_position": 9, "filler_seed": 2816770142, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": true, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E030", "cand_id": "EC040", "memory_type": "event", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I viewed a 1-bedroom condo on February 10th, but the noise from the highway was a deal-breaker. What questions should I ask about noise when I view the next place?", "assistant_reply": "Ask which direction the bedroom faces, whether the windows are double-glazed, and how noisy it is at rush hour and at night. If you can, visit again at a different time of day.", "followup_turn": "Do double-glazed windows really make a big difference?", "followup_reply": "Yes, they can cut outside noise noticeably, especially traffic hum. Laminated glass or secondary glazing helps even more for very busy roads.", "q1": "When did I view the 1-bedroom condo near the highway?", "q2": "On what date did I go to see the 1-bedroom condo with the highway noise?"}, "hi": {"memory_turn": "मैंने 10 फ़रवरी को एक 1-बेडरूम कॉन्डो देखा था, पर हाईवे का शोर मेरे लिए बड़ी रुकावट था। अगली जगह देखते समय शोर के बारे में क्या-क्या पूछूँ?", "assistant_reply": "पूछें कि बेडरूम किस दिशा में है, खिड़कियों में डबल ग्लास है या नहीं, और भीड़ के समय और रात में कितना शोर होता है। हो सके तो दिन के किसी और समय दोबारा जाकर देखें।", "followup_turn": "क्या डबल ग्लास वाली खिड़कियों से सच में बड़ा फ़र्क पड़ता है?", "followup_reply": "हाँ, इनसे बाहर का शोर काफ़ी कम हो सकता है, ख़ासकर ट्रैफ़िक की भनभनाहट। बहुत व्यस्त सड़कों के लिए लैमिनेटेड ग्लास या दूसरी परत और ज़्यादा मदद करती है।", "q1": "मैंने हाईवे के पास वाला 1-बेडरूम कॉन्डो कब देखा था?", "q2": "हाईवे के शोर वाला 1-बेडरूम कॉन्डो देखने मैं किस तारीख़ को गया था?"}, "hg": {"memory_turn": "Maine 10 February ko ek 1-bedroom condo dekha tha, par highway ka shor mere liye deal-breaker tha. Agli jagah dekhte time shor ke baare mein kya-kya poochhun?", "assistant_reply": "Poochhein ki bedroom kis direction mein hai, windows mein double glass hai ya nahi, aur rush hour aur raat mein kitna shor hota hai. Ho sake toh din ke kisi aur time dobara jaake dekhein.", "followup_turn": "Kya double-glazed windows se sach mein bada fark padta hai?", "followup_reply": "Haan, inse bahar ka shor kaafi kam ho sakta hai, especially traffic ki bhanbhanahat. Bahut busy roads ke liye laminated glass ya secondary glazing aur zyaada help karti hai.", "q1": "Maine highway ke paas wala 1-bedroom condo kab dekha tha?", "q2": "Highway ke shor wala 1-bedroom condo dekhne main kis date ko gaya tha?"}}, "gold": {"answer_en": "February 10", "date": {"day": 10, "month": 2}, "slots": [{"name": "date", "aliases": ["February 10", "February 10th", "10 February", "10th February", "10th of February", "Feb 10", "10 Feb", "10 फ़रवरी", "10 फरवरी"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "gpt4_7fce9456", "lme_type": "multi-session"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 2338953681, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "F031", "cand_id": "FC138", "memory_type": "fact", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "It took me and my friends around 5 hours to move everything into the new apartment, but it was worth it to be closer to work. Now I want to find a good café in the new area. How should I go about it?", "assistant_reply": "Walk the nearby streets on a weekend morning, check recent map reviews with photos, and ask the baristas at the first café you like where else they go. Locals usually know the hidden spots.", "followup_turn": "What should I look for if I want to work from a café?", "followup_reply": "Reliable Wi-Fi, a few power sockets, comfortable tables and a calm atmosphere matter most. Visit at the time you plan to work to see how crowded it gets.", "q1": "How long did it take to move into the new apartment?", "q2": "How many hours did my move to the new apartment take?", "distractor_sentence": "My cousin's move took a whole 9 hours because the lift was broken."}, "hi": {"memory_turn": "मुझे और मेरे दोस्तों को नए अपार्टमेंट में सारा सामान शिफ़्ट करने में लगभग 5 घंटे लगे, पर काम के पास रहने के लिए यह मेहनत ठीक थी। अब मैं नए इलाक़े में कोई अच्छा कैफ़े ढूँढना चाहती हूँ। कैसे ढूँढूँ?", "assistant_reply": "वीकेंड की सुबह आसपास की गलियों में घूमें, मैप पर फ़ोटो वाले ताज़ा रिव्यू देखें, और जो पहला कैफ़े पसंद आए वहाँ के बरिस्ता से पूछें कि वे और कहाँ जाते हैं। स्थानीय लोग अक्सर छुपी जगहें जानते हैं।", "followup_turn": "अगर कैफ़े से काम करना हो तो किन चीज़ों का ध्यान रखूँ?", "followup_reply": "भरोसेमंद Wi-Fi, कुछ पावर सॉकेट, आरामदायक टेबल और शांत माहौल सबसे ज़रूरी हैं। जिस समय काम करना हो उसी समय जाकर देखें कि कितनी भीड़ होती है।", "q1": "नए अपार्टमेंट में शिफ़्ट होने में कितना समय लगा?", "q2": "नए अपार्टमेंट में मेरी शिफ़्टिंग में कितने घंटे लगे?", "distractor_sentence": "मेरे कज़िन की शिफ़्टिंग में पूरे 9 घंटे लगे क्योंकि लिफ़्ट ख़राब थी।"}, "hg": {"memory_turn": "Mujhe aur mere doston ko naye apartment mein saara saamaan shift karne mein lagbhag 5 ghante lage, par kaam ke paas rehne ke liye yeh mehnat theek thi. Ab main naye area mein koi accha cafe dhoondhna chahti hoon. Kaise dhoondhun?", "assistant_reply": "Weekend ki subah aaspaas ki galiyon mein ghoomein, map pe photos wale recent reviews dekhein, aur jo pehla cafe pasand aaye wahan ke baristas se poochhein ki woh aur kahan jaate hain. Local log aksar hidden spots jaante hain.", "followup_turn": "Agar cafe se kaam karna ho toh kin cheezon ka dhyaan rakhun?", "followup_reply": "Reliable Wi-Fi, kuch power sockets, comfortable tables aur shaant mahaul sabse zaroori hain. Jis time kaam karna ho usi time jaake dekhein ki kitni bheed hoti hai.", "q1": "Naye apartment mein shift hone mein kitna time laga?", "q2": "Naye apartment mein meri shifting mein kitne ghante lage?", "distractor_sentence": "Mere cousin ki shifting mein poore 9 ghante lage kyunki lift kharab thi."}}, "gold": {"answer_en": "about 5 hours", "slots": [{"name": "duration", "aliases": ["5 hours", "five hours", "5 hrs", "5 घंटे", "पाँच घंटे", "5 ghante", "paanch ghante"]}]}, "distractor": {"value_aliases": ["9 hours", "9 घंटे", "9 ghante"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "caf9ead2", "lme_type": "single-session-user"}, "topic": "work/study", "is_answerable": true, "evidence_position": 2, "filler_seed": 1232754730, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F032", "cand_id": "FC143", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I've been building a better evening routine and now stop checking work emails and messages by 7 pm to separate work from my personal life. It really helps me relax. Can you suggest a calming book to read at night?", "assistant_reply": "'The Midnight Library' by Matt Haig is gentle and thoughtful, and 'A Gentleman in Moscow' is warm and slow-paced. For non-fiction, 'Wintering' by Katherine May is very soothing.", "followup_turn": "How long should I read before bed?", "followup_reply": "Twenty to thirty minutes is enough for most people to wind down. Use a warm, dim lamp rather than a bright screen so it's easier to fall asleep afterwards.", "q1": "What time do I stop checking work emails and messages?", "q2": "By what time in the evening do I stop looking at work messages?", "q3": "My colleague wants to message me tonight about a project. After what time should she assume I won't see it until tomorrow?"}, "hi": {"memory_turn": "मैं शाम का एक बेहतर रूटीन बना रहा हूँ और अब काम और निजी ज़िंदगी को अलग रखने के लिए शाम 7 बजे तक काम के ईमेल और मैसेज देखना बंद कर देता हूँ। इससे मुझे सच में सुकून मिलता है। क्या रात को पढ़ने के लिए कोई शांत किताब बता सकते हैं?", "assistant_reply": "मैट हेग की 'The Midnight Library' कोमल और विचारशील है, और 'A Gentleman in Moscow' गर्मजोशी भरी और धीमी गति की है। नॉन-फ़िक्शन में कैथरीन मे की 'Wintering' बहुत सुकून देती है।", "followup_turn": "सोने से पहले कितनी देर पढ़ना चाहिए?", "followup_reply": "ज़्यादातर लोगों के लिए बीस-तीस मिनट काफ़ी हैं। तेज़ स्क्रीन की जगह हल्की, गर्म रोशनी वाला लैंप इस्तेमाल करें ताकि बाद में नींद आसानी से आए।", "q1": "मैं काम के ईमेल और मैसेज देखना किस समय बंद करता हूँ?", "q2": "शाम को किस समय तक मैं काम के मैसेज देखना बंद कर देता हूँ?", "q3": "मेरी एक साथी आज रात किसी प्रोजेक्ट के बारे में मुझे मैसेज करना चाहती है। किस समय के बाद उसे मान लेना चाहिए कि मैं वह कल ही देखूँगा?"}, "hg": {"memory_turn": "Main shaam ka ek better routine bana raha hoon aur ab kaam aur personal life ko alag rakhne ke liye shaam 7 baje tak work emails aur messages dekhna band kar deta hoon. Isse mujhe sach mein sukoon milta hai. Kya raat ko padhne ke liye koi calming book bata sakte hain?", "assistant_reply": "Matt Haig ki 'The Midnight Library' gentle aur thoughtful hai, aur 'A Gentleman in Moscow' warm aur slow-paced hai. Non-fiction mein Katherine May ki 'Wintering' bahut soothing hai.", "followup_turn": "Sone se pehle kitni der padhna chahiye?", "followup_reply": "Zyaadatar logon ke liye bees-tees minute kaafi hain. Bright screen ki jagah halki, warm roshni wala lamp use karein taaki baad mein neend aasani se aaye.", "q1": "Main work emails aur messages dekhna kis time band karta hoon?", "q2": "Shaam ko kis time tak main kaam ke messages dekhna band kar deta hoon?", "q3": "Meri ek colleague aaj raat kisi project ke baare mein mujhe message karna chahti hai. Kis time ke baad use maan lena chahiye ki main woh kal hi dekhunga?"}}, "gold": {"answer_en": "7 pm", "slots": [{"name": "time", "aliases": ["7 pm", "7pm", "7 p.m.", "7:00", "7 बजे", "7 baje", "saat baje", "seven"]}]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "577d4d32", "lme_type": "single-session-user"}, "topic": "work/study", "is_answerable": true, "evidence_position": 5, "filler_seed": 4114262840, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": true, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "F033", "cand_id": "FC145", "memory_type": "fact", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I've been collecting vintage cameras for three months now, and I've already built a pretty impressive collection. Any advice on keeping them organised and easy to reach for cleaning?", "assistant_reply": "Use a glass-front cabinet away from direct sunlight, keep silica gel packs inside to control humidity, and label each shelf by era or brand. A simple spreadsheet helps you track condition and cleaning dates.", "followup_turn": "How do I stop fungus from growing on the lenses?", "followup_reply": "Keep humidity between about 40 and 50 percent, store lenses with caps off now and then so air moves, and use them regularly, since light and airflow slow fungus growth.", "q1": "How long have I been collecting vintage cameras?", "q2": "For how much time have I been building my vintage camera collection?", "distractor_sentence": "My uncle has been collecting old radios for 20 years."}, "hi": {"memory_turn": "मैं तीन महीने से पुराने विंटेज कैमरे इकट्ठा कर रही हूँ, और अभी से मेरे पास अच्छा-ख़ासा कलेक्शन बन गया है। इन्हें व्यवस्थित रखने और सफ़ाई के लिए आसानी से निकालने की कोई सलाह?", "assistant_reply": "धूप से दूर शीशे वाली अलमारी इस्तेमाल करें, नमी कम रखने के लिए अंदर सिलिका जेल के पैकेट रखें, और हर शेल्फ़ पर दौर या ब्रांड का लेबल लगाएँ। एक आसान स्प्रेडशीट में हालत और सफ़ाई की तारीख़ें लिख सकती हैं।", "followup_turn": "लेंस पर फफूँद लगने से कैसे रोकूँ?", "followup_reply": "नमी लगभग 40 से 50 प्रतिशत के बीच रखें, कभी-कभी लेंस के ढक्कन हटाकर हवा लगने दें, और उन्हें नियमित इस्तेमाल करें, क्योंकि रोशनी और हवा से फफूँद धीमी बढ़ती है।", "q1": "मैं कब से विंटेज कैमरे इकट्ठा कर रही हूँ?", "q2": "मैं कितने समय से अपना विंटेज कैमरा कलेक्शन बना रही हूँ?", "distractor_sentence": "मेरे मामा 20 साल से पुराने रेडियो इकट्ठा कर रहे हैं।"}, "hg": {"memory_turn": "Main teen mahine se purane vintage cameras collect kar rahi hoon, aur abhi se mere paas accha-khaasa collection ban gaya hai. Inhe organised rakhne aur cleaning ke liye aasani se nikaalne ki koi advice?", "assistant_reply": "Dhoop se door glass-front cabinet use karein, nami kam rakhne ke liye andar silica gel packets rakhein, aur har shelf pe era ya brand ka label lagayein. Ek simple spreadsheet mein condition aur cleaning ki dates likh sakti hain.", "followup_turn": "Lens pe fungus lagne se kaise rokun?", "followup_reply": "Nami lagbhag 40 se 50 percent ke beech rakhein, kabhi-kabhi lens caps hata ke hawa lagne dein, aur unhe regularly use karein, kyunki roshni aur hawa se fungus dheere badhta hai.", "q1": "Main kab se vintage cameras collect kar rahi hoon?", "q2": "Main kitne time se apna vintage camera collection bana rahi hoon?", "distractor_sentence": "Mere mama 20 saal se purane radios collect kar rahe hain."}}, "gold": {"answer_en": "three months", "slots": [{"name": "duration", "aliases": ["three months", "3 months", "तीन महीने", "3 महीने", "teen mahine", "3 mahine"]}]}, "distractor": {"value_aliases": ["20 years", "20 साल", "20 saal"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "15745da0", "lme_type": "single-session-user"}, "topic": "other", "is_answerable": true, "evidence_position": 9, "filler_seed": 2357595376, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": ["FL36"]}
{"item_id": "F034", "cand_id": "FC146", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "Last month I went to Hawaii with my family for a week, and I loved it so much that I'm already thinking of going back. What could I do on the Big Island next time?", "assistant_reply": "Visit Volcanoes National Park to see the craters and lava fields, snorkel at Kahaluu Bay, and go stargazing near Mauna Kea. The black sand beach at Punaluu is worth a stop too.", "followup_turn": "Is it better to rent a car there?", "followup_reply": "Yes, the island is large and public transport is limited. A regular car is enough for most sights, though some summit roads need a four-wheel drive.", "q1": "Where did I go on a week-long trip with my family?", "q2": "Which place did I visit with my family for a week last month?", "q3": "My cousin wants a family holiday spot I can personally vouch for from my own recent trip. Where should I suggest?"}, "hi": {"memory_turn": "पिछले महीने मैं अपने परिवार के साथ एक हफ़्ते के लिए हवाई गया था, और मुझे वह इतना पसंद आया कि मैं अभी से दोबारा जाने का सोच रहा हूँ। अगली बार बिग आइलैंड पर क्या कर सकता हूँ?", "assistant_reply": "ज्वालामुखी के गड्ढे और लावा के मैदान देखने के लिए वॉल्केनोज़ नेशनल पार्क जाएँ, काहालू खाड़ी में स्नॉर्कलिंग करें, और मौना केआ के पास तारे देखें। पुनालू का काली रेत वाला बीच भी देखने लायक है।", "followup_turn": "क्या वहाँ कार किराये पर लेना बेहतर है?", "followup_reply": "हाँ, द्वीप बड़ा है और सार्वजनिक परिवहन कम है। ज़्यादातर जगहों के लिए सामान्य कार काफ़ी है, हालाँकि चोटी तक की कुछ सड़कों के लिए फ़ोर-व्हील ड्राइव चाहिए।", "q1": "मैं अपने परिवार के साथ एक हफ़्ते की ट्रिप पर कहाँ गया था?", "q2": "पिछले महीने मैं परिवार के साथ एक हफ़्ते के लिए किस जगह घूमने गया?", "q3": "मेरा कज़िन ऐसी फ़ैमिली छुट्टी की जगह चाहता है जिसकी मैं अपनी हाल की ट्रिप से गारंटी दे सकूँ। उसे कौन-सी जगह सुझाऊँ?"}, "hg": {"memory_turn": "Pichhle mahine main apni family ke saath ek hafte ke liye Hawaii gaya tha, aur mujhe woh itna pasand aaya ki main abhi se dobara jaane ka soch raha hoon. Agli baar Big Island pe kya kar sakta hoon?", "assistant_reply": "Craters aur lava fields dekhne ke liye Volcanoes National Park jaayein, Kahaluu Bay mein snorkelling karein, aur Mauna Kea ke paas stargazing karein. Punaluu ka black sand beach bhi dekhne laayak hai.", "followup_turn": "Kya wahan car rent pe lena better hai?", "followup_reply": "Haan, island bada hai aur public transport kam hai. Zyaadatar jagahon ke liye normal car kaafi hai, halanki summit ki kuch roads ke liye four-wheel drive chahiye.", "q1": "Main apni family ke saath ek hafte ki trip pe kahan gaya tha?", "q2": "Pichhle mahine main family ke saath ek hafte ke liye kis jagah ghoomne gaya?", "q3": "Mera cousin aisi family holiday ki jagah chahta hai jiski main apni recent trip se guarantee de sakoon. Use kaunsi jagah suggest karoon?"}}, "gold": {"answer_en": "Hawaii", "slots": [{"name": "place", "aliases": ["Hawaii", "हवाई"]}]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "e01b8e2f", "lme_type": "single-session-user"}, "topic": "travel", "is_answerable": true, "evidence_position": 2, "filler_seed": 768099288, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": true, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "F035", "cand_id": "FC147", "memory_type": "fact", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I recently made a lemon blueberry cake for my niece's birthday party, and it was a huge hit! I was nervous about it, but it worked. Now I want to try a fancier dessert. Any ideas?", "assistant_reply": "A raspberry pavlova looks impressive but is mostly egg whites and sugar, a French apple tart teaches you neat pastry work, and a chocolate mousse cake is rich but forgiving.", "followup_turn": "Which one is safest for a beginner?", "followup_reply": "The apple tart is the safest, especially with ready-made pastry. Pavlova can crack if the oven runs hot, and mousse needs time to set.", "q1": "What did I bake for my niece's birthday party?", "q2": "Which cake did I make for my niece's birthday?", "distractor_sentence": "My sister made a red velvet cake for her son's party last week."}, "hi": {"memory_turn": "हाल ही में मैंने अपनी भतीजी की जन्मदिन पार्टी के लिए लेमन ब्लूबेरी केक बनाया, और वह सबको बहुत पसंद आया! मैं घबराई हुई थी, पर बन गया। अब मैं कोई और शानदार मिठाई बनाना चाहती हूँ। कोई आइडिया?", "assistant_reply": "रास्पबेरी पावलोवा दिखने में शानदार है पर ज़्यादातर अंडे की सफ़ेदी और चीनी से बनता है, फ़्रेंच एप्पल टार्ट से पेस्ट्री का साफ़ काम सीखती हैं, और चॉकलेट मूस केक भरपूर है पर ग़लतियाँ झेल लेता है।", "followup_turn": "नौसिखिए के लिए इनमें सबसे सुरक्षित कौन-सा है?", "followup_reply": "एप्पल टार्ट सबसे सुरक्षित है, ख़ासकर बनी-बनाई पेस्ट्री के साथ। ओवन ज़्यादा गर्म हो तो पावलोवा फट सकता है, और मूस को जमने में समय लगता है।", "q1": "मैंने अपनी भतीजी की जन्मदिन पार्टी के लिए क्या बेक किया?", "q2": "भतीजी के जन्मदिन के लिए मैंने कौन-सा केक बनाया?", "distractor_sentence": "मेरी बहन ने पिछले हफ़्ते अपने बेटे की पार्टी के लिए रेड वेलवेट केक बनाया।"}, "hg": {"memory_turn": "Haal hi mein maine apni bhatiji ki birthday party ke liye lemon blueberry cake banaya, aur woh sabko bahut pasand aaya! Main nervous thi, par ban gaya. Ab main koi aur fancy dessert banana chahti hoon. Koi idea?", "assistant_reply": "Raspberry pavlova dikhne mein shaandaar hai par zyaadatar egg whites aur sugar se banta hai, French apple tart se pastry ka neat kaam seekhti hain, aur chocolate mousse cake rich hai par galtiyan jhel leta hai.", "followup_turn": "Beginner ke liye inmein sabse safe kaunsa hai?", "followup_reply": "Apple tart sabse safe hai, especially ready-made pastry ke saath. Oven zyaada garam ho toh pavlova crack ho sakta hai, aur mousse ko set hone mein time lagta hai.", "q1": "Maine apni bhatiji ki birthday party ke liye kya bake kiya?", "q2": "Bhatiji ke birthday ke liye maine kaunsa cake banaya?", "distractor_sentence": "Meri behen ne pichhle hafte apne bete ki party ke liye red velvet cake banaya."}}, "gold": {"answer_en": "a lemon blueberry cake", "slots": [{"name": "cake", "aliases": ["lemon blueberry", "लेमन ब्लूबेरी", "नींबू ब्लूबेरी"]}]}, "distractor": {"value_aliases": ["red velvet", "रेड वेलवेट"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "bc8a6e93", "lme_type": "single-session-user"}, "topic": "food/cooking", "is_answerable": true, "evidence_position": 5, "filler_seed": 1942370237, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F036", "cand_id": "FC148", "memory_type": "fact", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I'm really into indie and alternative rock right now, and I've been listening to Arctic Monkeys and The Neighbourhood a lot on Spotify lately. Can you suggest a few similar bands?", "assistant_reply": "Try Cage the Elephant, The Strokes, Wallows and Cigarettes After Sex. If you like the moodier side of The Neighbourhood, Glass Animals and The 1975 are worth a listen too.", "followup_turn": "Which of those would be good to see live?", "followup_reply": "Cage the Elephant and The Strokes are known for energetic live shows. Glass Animals also put on a very visual, fun concert.", "q1": "Which music streaming service have I been using lately?", "q2": "What app have I been listening to music on recently?", "distractor_sentence": "My brother still swears by Apple Music for everything."}, "hi": {"memory_turn": "आजकल मुझे इंडी और ऑल्टरनेटिव रॉक बहुत पसंद है, और पिछले कुछ समय से मैं Spotify पर Arctic Monkeys और The Neighbourhood ख़ूब सुन रहा हूँ। क्या आप ऐसे ही कुछ बैंड बता सकते हैं?", "assistant_reply": "Cage the Elephant, The Strokes, Wallows और Cigarettes After Sex सुनिए। अगर आपको The Neighbourhood का उदास-सा अंदाज़ पसंद है, तो Glass Animals और The 1975 भी सुनने लायक हैं।", "followup_turn": "इनमें से किसे लाइव देखना अच्छा रहेगा?", "followup_reply": "Cage the Elephant और The Strokes अपने जोशीले लाइव शो के लिए जाने जाते हैं। Glass Animals का कॉन्सर्ट भी बहुत रंगीन और मज़ेदार होता है।", "q1": "आजकल मैं कौन-सी म्यूज़िक स्ट्रीमिंग सर्विस इस्तेमाल कर रहा हूँ?", "q2": "हाल में मैं किस ऐप पर गाने सुन रहा हूँ?", "distractor_sentence": "मेरा भाई आज भी हर चीज़ के लिए Apple Music की ही क़सम खाता है।"}, "hg": {"memory_turn": "Aajkal mujhe indie aur alternative rock bahut pasand hai, aur pichhle kuch time se main Spotify pe Arctic Monkeys aur The Neighbourhood khoob sun raha hoon. Kya aap aise hi kuch bands bata sakte hain?", "assistant_reply": "Cage the Elephant, The Strokes, Wallows aur Cigarettes After Sex suniye. Agar aapko The Neighbourhood ka moody andaaz pasand hai, toh Glass Animals aur The 1975 bhi sunne laayak hain.", "followup_turn": "Inmein se kise live dekhna accha rahega?", "followup_reply": "Cage the Elephant aur The Strokes apne energetic live shows ke liye jaane jaate hain. Glass Animals ka concert bhi bahut colourful aur fun hota hai.", "q1": "Aajkal main kaunsi music streaming service use kar raha hoon?", "q2": "Haal mein main kis app pe gaane sun raha hoon?", "distractor_sentence": "Mera bhai aaj bhi har cheez ke liye Apple Music ki hi kasam khaata hai."}}, "gold": {"answer_en": "Spotify", "slots": [{"name": "service", "aliases": ["Spotify", "स्पॉटिफ़ाई", "स्पॉटिफाई"]}]}, "distractor": {"value_aliases": ["Apple Music", "ऐप्पल म्यूज़िक"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "ccb36322", "lme_type": "single-session-user"}, "topic": "media/books/games", "is_answerable": true, "evidence_position": 9, "filler_seed": 946100210, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F037", "cand_id": "FC150", "memory_type": "fact", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I've been on a roll with my collectibles lately: I just got a rare blue Snaggletooth action figure from a thrift store a few weeks ago. Now I need a display case for my vintage watches. Any recommendations?", "assistant_reply": "A glass-top watch box with soft cushions lets you see every piece at once. For a wall display, a shallow shadow box with a velvet back looks elegant and keeps dust off.", "followup_turn": "Should I keep the watches wound while they're displayed?", "followup_reply": "It isn't necessary. Most mechanical watches are fine resting, but winding them every few weeks keeps the oils moving and the movement healthy.", "q1": "What type of action figure did I buy from a thrift store?", "q2": "Which action figure did I find at the thrift store a few weeks ago?"}, "hi": {"memory_turn": "आजकल मेरी कलेक्शन ख़ूब बढ़ रही है: कुछ हफ़्ते पहले मुझे एक थ्रिफ़्ट स्टोर से दुर्लभ नीला Snaggletooth एक्शन फ़िगर मिला। अब मुझे अपनी विंटेज घड़ियों के लिए डिस्प्ले केस चाहिए। कोई सुझाव?", "assistant_reply": "मुलायम गद्दियों वाला शीशे के ढक्कन का वॉच बॉक्स हर घड़ी को एक साथ दिखाता है। दीवार के लिए, मख़मली पीछे वाला पतला शैडो बॉक्स सुंदर लगता है और धूल से बचाता है।", "followup_turn": "क्या डिस्प्ले में रखी घड़ियों में चाबी भरते रहना चाहिए?", "followup_reply": "ज़रूरी नहीं है। ज़्यादातर मैकेनिकल घड़ियाँ रुकी रहें तो भी ठीक हैं, पर हर कुछ हफ़्तों में चाबी भरने से तेल चलता रहता है और मशीन ठीक रहती है।", "q1": "मैंने थ्रिफ़्ट स्टोर से किस तरह का एक्शन फ़िगर ख़रीदा?", "q2": "कुछ हफ़्ते पहले थ्रिफ़्ट स्टोर में मुझे कौन-सा एक्शन फ़िगर मिला?"}, "hg": {"memory_turn": "Aajkal meri collection khoob badh rahi hai: kuch hafte pehle mujhe ek thrift store se rare blue Snaggletooth action figure mila. Ab mujhe apni vintage watches ke liye display case chahiye. Koi suggestion?", "assistant_reply": "Soft cushions wala glass-top watch box har ghadi ko ek saath dikhata hai. Deewar ke liye, velvet back wala patla shadow box sundar lagta hai aur dhool se bachata hai.", "followup_turn": "Kya display mein rakhi watches mein chaabi bharte rehna chahiye?", "followup_reply": "Zaroori nahi hai. Zyaadatar mechanical watches ruki rahein toh bhi theek hain, par har kuch hafton mein chaabi bharne se oil chalta rehta hai aur movement theek rehta hai.", "q1": "Maine thrift store se kis type ka action figure khareeda?", "q2": "Kuch hafte pehle thrift store mein mujhe kaunsa action figure mila?"}}, "gold": {"answer_en": "a rare blue Snaggletooth", "slots": [{"name": "figure", "aliases": ["Snaggletooth", "स्नैगलटूथ"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "b320f3f8", "lme_type": "single-session-user"}, "topic": "shopping/home", "is_answerable": true, "evidence_position": 2, "filler_seed": 4187810158, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F038", "cand_id": "FC152", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I've got tickets to see the Jonas Brothers at TD Garden next month. Oh, and I just remembered the amazing Imagine Dragons concert I went to at the Xfinity Center. How early should I arrive at a big arena show?", "assistant_reply": "Aim to arrive 45 minutes to an hour before the doors open. That gives time for security lines, finding your seats and grabbing food before the opening act starts.", "followup_turn": "Can I bring a bag inside?", "followup_reply": "Most arenas only allow small clear bags or tiny clutches. Check the venue's website the day before, since rules change from show to show.", "q1": "Where did I attend the Imagine Dragons concert?", "q2": "At which venue did I see Imagine Dragons perform?", "q3": "My friend wants to know where I saw the band behind 'Radioactive' play live. Which venue was it?"}, "hi": {"memory_turn": "मेरे पास अगले महीने TD Garden में Jonas Brothers के शो के टिकट हैं। अरे, मुझे Xfinity Center में देखा Imagine Dragons का शानदार कॉन्सर्ट भी याद आ गया। बड़े एरीना शो में कितना पहले पहुँचना चाहिए?", "assistant_reply": "दरवाज़े खुलने से 45 मिनट से एक घंटा पहले पहुँचने की कोशिश करें। इससे सिक्योरिटी की लाइन, अपनी सीट ढूँढने और शुरुआती कलाकार से पहले कुछ खाने का समय मिल जाता है।", "followup_turn": "क्या मैं अंदर बैग ले जा सकता हूँ?", "followup_reply": "ज़्यादातर एरीना में सिर्फ़ छोटे पारदर्शी बैग या बहुत छोटे क्लच की इजाज़त होती है। एक दिन पहले जगह की वेबसाइट देख लें, क्योंकि नियम हर शो में बदलते हैं।", "q1": "मैंने Imagine Dragons का कॉन्सर्ट कहाँ देखा था?", "q2": "मैंने Imagine Dragons को किस जगह परफ़ॉर्म करते देखा?", "q3": "मेरा दोस्त जानना चाहता है कि मैंने 'Radioactive' वाले बैंड को लाइव कहाँ देखा था। वह कौन-सी जगह थी?"}, "hg": {"memory_turn": "Mere paas next month TD Garden mein Jonas Brothers ke show ke tickets hain. Arre, mujhe Xfinity Center mein dekha Imagine Dragons ka shaandaar concert bhi yaad aa gaya. Bade arena show mein kitna pehle pahunchna chahiye?", "assistant_reply": "Doors khulne se 45 minute se ek ghanta pehle pahunchne ki koshish karein. Isse security ki line, apni seat dhoondhne aur opening act se pehle kuch khaane ka time mil jaata hai.", "followup_turn": "Kya main andar bag le ja sakta hoon?", "followup_reply": "Zyaadatar arenas mein sirf chhote clear bags ya bahut chhote clutches allowed hote hain. Ek din pehle venue ki website check kar lein, kyunki rules har show mein badalte hain.", "q1": "Maine Imagine Dragons ka concert kahan dekha tha?", "q2": "Maine Imagine Dragons ko kis venue pe perform karte dekha?", "q3": "Mera dost jaanna chahta hai ki maine 'Radioactive' wale band ko live kahan dekha tha. Woh kaunsa venue tha?"}}, "gold": {"answer_en": "Xfinity Center", "slots": [{"name": "venue", "aliases": ["Xfinity Center", "Xfinity Centre", "Xfinity", "एक्सफ़िनिटी"]}]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "4fd1909e", "lme_type": "single-session-user"}, "topic": "media/books/games", "is_answerable": true, "evidence_position": 5, "filler_seed": 1600365713, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "F039", "cand_id": "FC153", "memory_type": "fact", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I've been averaging around 2 hours of screen time on Instagram per day for the past two weeks, which is way too much. Would a 30-minute daily limit be a good starting point?", "assistant_reply": "That's a big jump, so it may feel hard at first. Many people do better cutting back in steps, for example to an hour a day this week, then 45 minutes, then 30.", "followup_turn": "What can I do instead when I feel the urge to scroll?", "followup_reply": "Keep a book or puzzle within reach, move the app off your home screen, and replace the habit with something short, like a five-minute walk or messaging a friend directly.", "q1": "How much screen time have I been averaging on Instagram per day?", "q2": "On average, how long do I spend on Instagram each day?", "q3": "A digital-wellbeing app asks how long I currently spend each day on that photo-sharing app. What should I enter?", "distractor_sentence": "My friend says she spends only 45 minutes a day on it."}, "hi": {"memory_turn": "पिछले दो हफ़्तों से मेरा Instagram पर औसतन रोज़ लगभग 2 घंटे का स्क्रीन टाइम है, जो बहुत ज़्यादा है। क्या रोज़ 30 मिनट की सीमा से शुरुआत करना ठीक रहेगा?", "assistant_reply": "यह बहुत बड़ी छलाँग है, इसलिए शुरू में मुश्किल लग सकती है। कई लोगों के लिए धीरे-धीरे घटाना बेहतर रहता है, जैसे इस हफ़्ते रोज़ एक घंटा, फिर 45 मिनट, फिर 30।", "followup_turn": "जब स्क्रॉल करने का मन करे तो उसकी जगह क्या करूँ?", "followup_reply": "कोई किताब या पहेली पास रखें, ऐप को होम स्क्रीन से हटा दें, और आदत की जगह कोई छोटी चीज़ करें, जैसे पाँच मिनट टहलना या किसी दोस्त को सीधे मैसेज करना।", "q1": "Instagram पर मेरा रोज़ का औसत स्क्रीन टाइम कितना है?", "q2": "औसतन मैं हर दिन Instagram पर कितना समय बिताती हूँ?", "q3": "एक डिजिटल-वेलबीइंग ऐप पूछ रहा है कि मैं अभी उस फ़ोटो शेयर करने वाले ऐप पर रोज़ कितना समय बिताती हूँ। मुझे क्या भरना चाहिए?", "distractor_sentence": "मेरी दोस्त कहती है कि वह उस पर रोज़ बस 45 मिनट बिताती है।"}, "hg": {"memory_turn": "Pichhle do hafton se mera Instagram pe average roz lagbhag 2 ghante ka screen time hai, jo bahut zyaada hai. Kya roz 30 minute ki limit se shuru karna theek rahega?", "assistant_reply": "Yeh bahut bada jump hai, isliye shuru mein mushkil lag sakta hai. Kai logon ke liye dheere-dheere kam karna better rehta hai, jaise is hafte roz ek ghanta, phir 45 minute, phir 30.", "followup_turn": "Jab scroll karne ka mann kare toh uski jagah kya karoon?", "followup_reply": "Koi book ya puzzle paas rakhein, app ko home screen se hata dein, aur aadat ki jagah koi chhoti cheez karein, jaise paanch minute walk ya kisi dost ko seedha message karna.", "q1": "Instagram pe mera roz ka average screen time kitna hai?", "q2": "Average mein main har din Instagram pe kitna time bitati hoon?", "q3": "Ek digital-wellbeing app pooch raha hai ki main abhi us photo sharing app pe roz kitna time bitati hoon. Mujhe kya bharna chahiye?", "distractor_sentence": "Meri friend kehti hai ki woh us pe roz bas 45 minute bitati hai."}}, "gold": {"answer_en": "about 2 hours a day", "slots": [{"name": "duration", "aliases": ["2 hours", "two hours", "2 hrs", "2 घंटे", "दो घंटे", "2 ghante"]}]}, "distractor": {"value_aliases": ["45 minutes", "45 मिनट", "45 minute"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "545bd2b5", "lme_type": "single-session-user"}, "topic": "other", "is_answerable": true, "evidence_position": 9, "filler_seed": 2664273575, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "F040", "cand_id": "FC154", "memory_type": "fact", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I like the idea of layered lighting, so I'll put a floor lamp behind the sofa. Can you suggest warm-toned floor lamps that would match the Philips LED bulb I have in my bedside lamp?", "assistant_reply": "Look for lamps that take a standard screw-in bulb, then use a warm white bulb around 2700K to match. An arc lamp or a simple tripod lamp with a fabric shade both give a soft, cosy glow.", "followup_turn": "How bright should a reading lamp be?", "followup_reply": "Around 450 to 800 lumens is comfortable for reading. A dimmable bulb lets you switch between bright reading light and a softer evening glow.", "q1": "What brand of bulb do I have in my bedside lamp?", "q2": "Which company's LED bulb is in my bedside lamp?", "distractor_sentence": "My roommate uses a Syska bulb in his desk lamp."}, "hi": {"memory_turn": "मुझे लेयर्ड लाइटिंग का आइडिया पसंद है, इसलिए मैं सोफ़े के पीछे एक फ़्लोर लैंप रखूँगा। क्या आप गर्म रोशनी वाले फ़्लोर लैंप बता सकते हैं जो मेरे बेडसाइड लैंप के Philips LED बल्ब से मेल खाएँ?", "assistant_reply": "ऐसे लैंप देखें जिनमें सामान्य पेंच वाला बल्ब लगता हो, फिर मेल के लिए लगभग 2700K का वॉर्म व्हाइट बल्ब लगाएँ। आर्क लैंप या कपड़े के शेड वाला सादा तिपाई लैंप, दोनों हल्की और आरामदायक रोशनी देते हैं।", "followup_turn": "पढ़ने के लैंप की रोशनी कितनी तेज़ होनी चाहिए?", "followup_reply": "पढ़ने के लिए लगभग 450 से 800 ल्यूमेन आरामदायक है। डिम होने वाले बल्ब से आप तेज़ पढ़ने की रोशनी और शाम की हल्की रोशनी के बीच बदल सकते हैं।", "q1": "मेरे बेडसाइड लैंप में किस ब्रांड का बल्ब है?", "q2": "मेरे बेडसाइड लैंप में किस कंपनी का LED बल्ब लगा है?", "distractor_sentence": "मेरा रूममेट अपने डेस्क लैंप में Syska का बल्ब इस्तेमाल करता है।"}, "hg": {"memory_turn": "Mujhe layered lighting ka idea pasand hai, isliye main sofa ke peeche ek floor lamp rakhunga. Kya aap warm roshni wale floor lamps bata sakte hain jo mere bedside lamp ke Philips LED bulb se match karein?", "assistant_reply": "Aise lamps dekhein jinmein normal screw wala bulb lagta ho, phir match ke liye lagbhag 2700K ka warm white bulb lagayein. Arc lamp ya fabric shade wala simple tripod lamp, dono halki aur cosy roshni dete hain.", "followup_turn": "Padhne ke lamp ki roshni kitni tez honi chahiye?", "followup_reply": "Padhne ke liye lagbhag 450 se 800 lumens comfortable hai. Dimmable bulb se aap tez padhne wali roshni aur shaam ki halki roshni ke beech switch kar sakte hain.", "q1": "Mere bedside lamp mein kis brand ka bulb hai?", "q2": "Mere bedside lamp mein kis company ka LED bulb laga hai?", "distractor_sentence": "Mera roommate apne desk lamp mein Syska ka bulb use karta hai."}}, "gold": {"answer_en": "a Philips LED bulb", "slots": [{"name": "brand", "aliases": ["Philips", "फ़िलिप्स", "फिलिप्स"]}]}, "distractor": {"value_aliases": ["Syska", "सिस्का"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "8a137a7f", "lme_type": "single-session-user"}, "topic": "work/study", "is_answerable": true, "evidence_position": 2, "filler_seed": 2509852883, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P031", "cand_id": "PX0421", "memory_type": "preference", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I'm a big fan of Jules Verne's work, especially his adventure stories. I'd like to start a small book club with friends. How do I keep it going past the first few meetings?", "assistant_reply": "Keep the group small, around five to eight people, meet on a fixed day each month, and rotate who picks the book. Short, discussable books work better than long ones at the start.", "followup_turn": "Should we meet in person or online?", "followup_reply": "In person usually feels livelier, but a mix works well if people live far apart. Whatever you choose, keep the same format so it becomes a habit.", "q1": "Which classic author am I a big fan of?", "q2": "Whose classic novels do I really love?", "distractor_sentence": "My dad prefers Charles Dickens and keeps rereading his novels."}, "hi": {"memory_turn": "मैं जूल्स वर्न की रचनाओं की बड़ी फ़ैन हूँ, ख़ासकर उनकी रोमांचक कहानियों की। मैं दोस्तों के साथ एक छोटा बुक क्लब शुरू करना चाहती हूँ। पहली कुछ मीटिंग के बाद भी इसे कैसे चलाए रखूँ?", "assistant_reply": "समूह छोटा रखें, लगभग पाँच से आठ लोग, हर महीने एक तय दिन मिलें, और किताब चुनने की बारी बदलते रहें। शुरुआत में लंबी किताबों की जगह छोटी और चर्चा लायक़ किताबें बेहतर रहती हैं।", "followup_turn": "हमें आमने-सामने मिलना चाहिए या ऑनलाइन?", "followup_reply": "आमने-सामने आमतौर पर ज़्यादा जीवंत लगता है, पर लोग दूर रहते हों तो दोनों का मेल अच्छा है। जो भी चुनें, तरीक़ा एक जैसा रखें ताकि यह आदत बन जाए।", "q1": "मैं किस क्लासिक लेखक की बड़ी फ़ैन हूँ?", "q2": "मुझे किसके क्लासिक उपन्यास बहुत पसंद हैं?", "distractor_sentence": "मेरे पापा को चार्ल्स डिकेंस पसंद हैं और वे उनके उपन्यास बार-बार पढ़ते हैं।"}, "hg": {"memory_turn": "Main Jules Verne ke kaam ki badi fan hoon, especially unki adventure stories ki. Main doston ke saath ek chhota book club shuru karna chahti hoon. Pehli kuch meetings ke baad bhi ise kaise chalaye rakhun?", "assistant_reply": "Group chhota rakhein, lagbhag paanch se aath log, har mahine ek fixed din milein, aur book chunne ki baari badalte rahein. Shuru mein lambi books ki jagah chhoti aur discussion laayak books better rehti hain.", "followup_turn": "Humein in person milna chahiye ya online?", "followup_reply": "In person usually zyaada lively lagta hai, par log door rehte hon toh dono ka mix accha hai. Jo bhi chunein, format ek jaisa rakhein taaki yeh aadat ban jaaye.", "q1": "Main kis classic author ki badi fan hoon?", "q2": "Mujhe kiske classic novels bahut pasand hain?", "distractor_sentence": "Mere papa ko Charles Dickens pasand hain aur woh unke novels baar-baar padhte hain."}}, "gold": {"answer_en": "Jules Verne", "slots": [{"name": "author", "aliases": ["Jules Verne", "Verne", "जूल्स वर्न", "वर्न"]}]}, "distractor": {"value_aliases": ["Dickens", "डिकेंस"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 2, "filler_seed": 1631341496, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P032", "cand_id": "PX0459", "memory_type": "preference", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I mostly play piano and some synth sounds, and I prefer weighted keys. I'm looking for a digital keyboard for my small flat. What should I consider?", "assistant_reply": "Check that it has 88 keys with a hammer action, a decent built-in speaker or good headphone output, and a slim body if space is tight. A sustain pedal input is a must for piano playing.", "followup_turn": "Do I need a separate stand?", "followup_reply": "Many slim models come with an optional matching stand, which is more stable than an X-stand. A sturdy stand at the right height also helps your posture.", "q1": "What kind of keys do I prefer on a keyboard instrument?", "q2": "Which type of keys do I like when playing piano?"}, "hi": {"memory_turn": "मैं ज़्यादातर पियानो और कुछ सिंथ साउंड बजाता हूँ, और मुझे वेटेड कीज़ पसंद हैं। मैं अपने छोटे फ़्लैट के लिए एक डिजिटल कीबोर्ड ढूँढ रहा हूँ। किन बातों का ध्यान रखूँ?", "assistant_reply": "देखें कि उसमें हैमर एक्शन वाली 88 कीज़ हों, ठीक-ठाक बिल्ट-इन स्पीकर या अच्छा हेडफ़ोन आउटपुट हो, और जगह कम हो तो पतली बॉडी हो। पियानो बजाने के लिए सस्टेन पैडल का इनपुट ज़रूरी है।", "followup_turn": "क्या मुझे अलग स्टैंड चाहिए?", "followup_reply": "कई पतले मॉडल के साथ मेल खाता स्टैंड अलग से मिलता है, जो X-स्टैंड से ज़्यादा मज़बूत होता है। सही ऊँचाई का मज़बूत स्टैंड बैठने का तरीक़ा भी ठीक रखता है।", "q1": "कीबोर्ड वाद्य पर मुझे किस तरह की कीज़ पसंद हैं?", "q2": "पियानो बजाते समय मुझे कौन-सी तरह की कीज़ अच्छी लगती हैं?"}, "hg": {"memory_turn": "Main zyaadatar piano aur kuch synth sounds bajata hoon, aur mujhe weighted keys pasand hain. Main apne chhote flat ke liye ek digital keyboard dhoondh raha hoon. Kin baaton ka dhyaan rakhun?", "assistant_reply": "Dekhein ki usmein hammer action wali 88 keys hon, theek-thaak built-in speaker ya accha headphone output ho, aur jagah kam ho toh slim body ho. Piano bajane ke liye sustain pedal ka input zaroori hai.", "followup_turn": "Kya mujhe alag stand chahiye?", "followup_reply": "Kai slim models ke saath match karta stand alag se milta hai, jo X-stand se zyaada stable hota hai. Sahi height ka mazboot stand baithne ka posture bhi theek rakhta hai.", "q1": "Keyboard instrument pe mujhe kis type ki keys pasand hain?", "q2": "Piano bajate time mujhe kaunsi type ki keys acchi lagti hain?"}}, "gold": {"answer_en": "weighted keys", "slots": [{"name": "keys", "aliases": ["weighted", "वेटेड"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 5, "filler_seed": 3721383471, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P033", "cand_id": "PX0465", "memory_type": "preference", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I've always loved the way sunflowers look, and I've heard they're great for attracting helpful insects. I want to plant some this year. When should I sow the seeds?", "assistant_reply": "Sow them directly outdoors after the last frost, once the soil has warmed up. Pick the sunniest spot you have, and plant a few seeds every two weeks for flowers over a longer season.", "followup_turn": "Do they need a lot of water?", "followup_reply": "Water deeply once or twice a week while they're young. Once established, they cope well with dry spells, but tall varieties may need a stake on windy days.", "q1": "Which flowers have I always loved?", "q2": "What are my favourite flowers?", "q3": "I'm ordering a bouquet for myself to brighten up my desk. Which flower should it be made of?", "distractor_sentence": "My mother has always preferred roses in her garden."}, "hi": {"memory_turn": "मुझे हमेशा से सूरजमुखी के फूल देखने में बहुत अच्छे लगते हैं, और सुना है कि वे काम के कीड़ों को भी आकर्षित करते हैं। मैं इस साल कुछ लगाना चाहती हूँ। बीज कब बोने चाहिए?", "assistant_reply": "आख़िरी पाले के बाद, जब मिट्टी गर्म हो जाए, तो सीधे बाहर बोएँ। सबसे धूप वाली जगह चुनें, और हर दो हफ़्ते में कुछ बीज बोती रहें ताकि लंबे समय तक फूल आते रहें।", "followup_turn": "क्या इन्हें बहुत पानी चाहिए?", "followup_reply": "छोटे पौधों को हफ़्ते में एक-दो बार गहराई तक पानी दें। जम जाने के बाद ये सूखे दिन झेल लेते हैं, पर लंबी क़िस्मों को तेज़ हवा में सहारे की ज़रूरत पड़ सकती है।", "q1": "मुझे हमेशा से कौन-से फूल पसंद रहे हैं?", "q2": "मेरे पसंदीदा फूल कौन-से हैं?", "q3": "मैं अपनी डेस्क सजाने के लिए अपने लिए एक गुलदस्ता मँगवा रही हूँ। वह किस फूल का होना चाहिए?", "distractor_sentence": "मेरी माँ को अपने बग़ीचे में हमेशा गुलाब ही पसंद रहे हैं।"}, "hg": {"memory_turn": "Mujhe hamesha se sunflowers dekhne mein bahut acche lagte hain, aur suna hai ki woh helpful insects ko bhi attract karte hain. Main is saal kuch lagana chahti hoon. Seeds kab bone chahiye?", "assistant_reply": "Last frost ke baad, jab mitti garam ho jaaye, toh seedha bahar boyein. Sabse dhoop wali jagah chunein, aur har do hafte mein kuch seeds boti rahein taaki lambe time tak phool aate rahein.", "followup_turn": "Kya inhe bahut paani chahiye?", "followup_reply": "Chhote paudhon ko hafte mein ek-do baar gehraai tak paani dein. Jam jaane ke baad yeh sookhe din jhel lete hain, par lambi varieties ko tez hawa mein sahaare ki zaroorat pad sakti hai.", "q1": "Mujhe hamesha se kaunse phool pasand rahe hain?", "q2": "Mere favourite phool kaunse hain?", "q3": "Main apni desk sajane ke liye apne liye ek bouquet mangwa rahi hoon. Woh kis phool ka hona chahiye?", "distractor_sentence": "Meri maa ko apne garden mein hamesha roses hi pasand rahe hain."}}, "gold": {"answer_en": "sunflowers", "slots": [{"name": "flower", "aliases": ["sunflower", "sunflowers", "सूरजमुखी", "surajmukhi"]}]}, "distractor": {"value_aliases": ["roses", "rose", "गुलाब", "gulab"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 9, "filler_seed": 3915060738, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": true, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "P034", "cand_id": "PX0474", "memory_type": "preference", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "When I travel, I'm a big fan of street food, so on my trip to Mumbai I'll definitely be trying the local chaat and dosa. How do I avoid getting an upset stomach?", "assistant_reply": "Choose busy stalls where food is cooked fresh in front of you, drink sealed bottled water, and skip raw salads and ice at first. Carrying oral rehydration salts is a good backup.", "followup_turn": "Is it safe to try pani puri?", "followup_reply": "It's usually fine at popular, clean stalls. Some places offer pani made with bottled water, which is a safer choice for visitors.", "q1": "What kind of food am I a big fan of when I travel?", "q2": "Which type of food do I love trying on my trips?", "distractor_sentence": "My wife sticks to fine dining restaurants whenever we go on holiday."}, "hi": {"memory_turn": "घूमते समय मैं स्ट्रीट फ़ूड का बड़ा फ़ैन हूँ, इसलिए अपनी मुंबई ट्रिप पर मैं वहाँ की चाट और डोसा ज़रूर खाऊँगा। पेट ख़राब होने से कैसे बचूँ?", "assistant_reply": "भीड़ वाले ठेले चुनें जहाँ खाना आपके सामने ताज़ा बनता हो, सील बंद बोतल का पानी पिएँ, और शुरू में कच्चे सलाद और बर्फ़ से बचें। साथ में ORS रखना अच्छा बैकअप है।", "followup_turn": "क्या पानी पूरी खाना सुरक्षित है?", "followup_reply": "मशहूर और साफ़ ठेलों पर आमतौर पर ठीक रहता है। कुछ जगहें बोतल के पानी से बना पानी देती हैं, जो बाहर से आने वालों के लिए ज़्यादा सुरक्षित है।", "q1": "घूमते समय मैं किस तरह के खाने का बड़ा फ़ैन हूँ?", "q2": "अपनी ट्रिप पर मुझे किस तरह का खाना आज़माना बहुत पसंद है?", "distractor_sentence": "मेरी पत्नी छुट्टियों में हमेशा फ़ाइन डाइनिंग रेस्टोरेंट में ही खाती है।"}, "hg": {"memory_turn": "Travel karte time main street food ka bada fan hoon, isliye apni Mumbai trip pe main wahan ki chaat aur dosa zaroor khaunga. Pet kharab hone se kaise bachun?", "assistant_reply": "Bheed wale stalls chunein jahan khaana aapke saamne fresh banta ho, sealed bottle ka paani piyein, aur shuru mein kacche salad aur barf se bachein. Saath mein ORS rakhna accha backup hai.", "followup_turn": "Kya pani puri khaana safe hai?", "followup_reply": "Famous aur saaf stalls pe usually theek rehta hai. Kuch jagah bottle ke paani se bana pani deti hain, jo bahar se aane walon ke liye zyaada safe hai.", "q1": "Travel karte time main kis type ke khaane ka bada fan hoon?", "q2": "Apni trips pe mujhe kis type ka khaana try karna bahut pasand hai?", "distractor_sentence": "Meri wife chhuttiyon mein hamesha fine dining restaurants mein hi khaati hai."}}, "gold": {"answer_en": "street food (e.g. chaat, dosa)", "slots": [{"name": "food", "aliases": ["street food", "स्ट्रीट फ़ूड", "स्ट्रीट फूड"]}]}, "distractor": {"value_aliases": ["fine dining", "फ़ाइन डाइनिंग"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 2, "filler_seed": 2008505817, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P035", "cand_id": "PX0495", "memory_type": "preference", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I'm a big fan of Miles Morales, and I've got a few variant covers featuring him in my comic collection. How should I store my comics so they stay in good condition?", "assistant_reply": "Put each issue in an acid-free bag with a backing board, store them upright in a comic box, and keep the box somewhere cool, dry and dark. Avoid attics and basements with changing temperatures.", "followup_turn": "Is it worth getting valuable issues graded?", "followup_reply": "For rare or high-value issues, grading can raise their resale value and protect them in a sealed case. For comics you want to reread, it's usually not worth the cost.", "q1": "Which Spider-Man character am I a big fan of?", "q2": "Who is my favourite version of Spider-Man?", "q3": "I'm choosing a poster of my favourite web-slinging hero for my room. Which version of the hero should it show?"}, "hi": {"memory_turn": "मैं माइल्स मोरालेस की बड़ी फ़ैन हूँ, और मेरे कॉमिक कलेक्शन में उनके कुछ वेरिएंट कवर भी हैं। कॉमिक्स को अच्छी हालत में रखने के लिए कैसे सँभालूँ?", "assistant_reply": "हर अंक को एसिड-फ़्री बैग में बैकिंग बोर्ड के साथ रखें, कॉमिक बॉक्स में सीधा खड़ा करें, और बॉक्स को ठंडी, सूखी और अँधेरी जगह पर रखें। बदलते तापमान वाली अटारी और तहख़ाने से बचें।", "followup_turn": "क्या क़ीमती अंकों की ग्रेडिंग करवाना ठीक है?", "followup_reply": "दुर्लभ या महँगे अंकों के लिए ग्रेडिंग से दोबारा बेचने की क़ीमत बढ़ सकती है और वे बंद केस में सुरक्षित रहते हैं। जिन कॉमिक्स को दोबारा पढ़ना हो, उनके लिए यह ख़र्च आमतौर पर बेकार है।", "q1": "स्पाइडर-मैन का कौन-सा किरदार मुझे बहुत पसंद है?", "q2": "स्पाइडर-मैन का मेरा पसंदीदा रूप कौन-सा है?", "q3": "मैं अपने कमरे के लिए अपने पसंदीदा जाल फेंकने वाले हीरो का पोस्टर चुन रही हूँ। उसमें हीरो का कौन-सा रूप होना चाहिए?"}, "hg": {"memory_turn": "Main Miles Morales ki badi fan hoon, aur mere comic collection mein unke kuch variant covers bhi hain. Comics ko acchi condition mein rakhne ke liye kaise sambhaalun?", "assistant_reply": "Har issue ko acid-free bag mein backing board ke saath rakhein, comic box mein seedha khada karein, aur box ko thandi, sookhi aur andheri jagah pe rakhein. Badalte temperature wali attic aur basement se bachein.", "followup_turn": "Kya keemti issues ki grading karwana theek hai?", "followup_reply": "Rare ya mehenge issues ke liye grading se resale value badh sakti hai aur woh sealed case mein safe rehte hain. Jin comics ko dobara padhna ho, unke liye yeh kharcha usually bekaar hai.", "q1": "Spider-Man ka kaunsa character mujhe bahut pasand hai?", "q2": "Spider-Man ka mera favourite version kaunsa hai?", "q3": "Main apne room ke liye apne favourite web-slinging hero ka poster choose kar rahi hoon. Usmein hero ka kaunsa version hona chahiye?"}}, "gold": {"answer_en": "Miles Morales", "slots": [{"name": "character", "aliases": ["Miles Morales", "Miles", "माइल्स मोरालेस", "माइल्स"]}]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 5, "filler_seed": 4278800194, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": true, "abl_q3": true, "abl_digits": false, "banned_fillers": ["FL14"]}
{"item_id": "P036", "cand_id": "PX0503", "memory_type": "preference", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I've started doing Pilates, and my favourite exercise is the hundred; it really helps me engage my core and improve my posture. How many times a week should I practise?", "assistant_reply": "Two to three sessions a week is a good start. That gives your muscles time to recover while still building strength, and you can add a short session at home on other days.", "followup_turn": "Can Pilates help with lower back pain?", "followup_reply": "It often helps by strengthening the deep core and improving posture. Start with gentle classes and tell your instructor about the pain so they can adapt the exercises.", "q1": "Which is my favourite Pilates exercise?", "q2": "What Pilates move do I like the most?", "q3": "My Pilates teacher lets each of us choose one move to open the class. Which one should I pick for myself?", "distractor_sentence": "My trainer's favourite move is the roll-up, but I find it tough."}, "hi": {"memory_turn": "मैंने पिलाटेज़ शुरू किया है, और मेरी पसंदीदा एक्सरसाइज़ 'द हंड्रेड' है; इससे मेरा कोर अच्छे से काम करता है और पोस्चर सुधरता है। मुझे हफ़्ते में कितनी बार प्रैक्टिस करनी चाहिए?", "assistant_reply": "हफ़्ते में दो-तीन सेशन अच्छी शुरुआत है। इससे मांसपेशियों को आराम का समय मिलता है और ताक़त भी बढ़ती है, और बाक़ी दिनों में घर पर छोटा सेशन जोड़ सकते हैं।", "followup_turn": "क्या पिलाटेज़ से कमर के निचले दर्द में मदद मिलती है?", "followup_reply": "अक्सर हाँ, क्योंकि यह अंदर की कोर मांसपेशियों को मज़बूत करता है और पोस्चर सुधारता है। हल्की क्लास से शुरू करें और इंस्ट्रक्टर को दर्द के बारे में बताएँ ताकि वे एक्सरसाइज़ बदल सकें।", "q1": "मेरी पसंदीदा पिलाटेज़ एक्सरसाइज़ कौन-सी है?", "q2": "पिलाटेज़ की कौन-सी मूव मुझे सबसे ज़्यादा पसंद है?", "q3": "मेरी पिलाटेज़ टीचर हम सबको क्लास शुरू करने के लिए एक मूव चुनने देती हैं। मुझे अपने लिए कौन-सी चुननी चाहिए?", "distractor_sentence": "मेरे ट्रेनर की पसंदीदा मूव रोल-अप है, पर मुझे वह मुश्किल लगती है।"}, "hg": {"memory_turn": "Maine Pilates shuru kiya hai, aur meri favourite exercise the hundred hai; isse mera core acche se kaam karta hai aur posture sudharta hai. Mujhe hafte mein kitni baar practice karni chahiye?", "assistant_reply": "Hafte mein do-teen sessions accha start hai. Isse muscles ko recover hone ka time milta hai aur strength bhi badhti hai, aur baaki dinon mein ghar pe chhota session add kar sakte hain.", "followup_turn": "Kya Pilates se lower back pain mein help milti hai?", "followup_reply": "Aksar haan, kyunki yeh andar ki core muscles ko mazboot karta hai aur posture sudharta hai. Halki class se shuru karein aur instructor ko dard ke baare mein batayein taaki woh exercises badal sakein.", "q1": "Meri favourite Pilates exercise kaunsi hai?", "q2": "Pilates ki kaunsi move mujhe sabse zyaada pasand hai?", "q3": "Meri Pilates teacher hum sabko class shuru karne ke liye ek move chunne deti hain. Mujhe apne liye kaunsi chunni chahiye?", "distractor_sentence": "Mere trainer ki favourite move roll-up hai, par mujhe woh mushkil lagti hai."}}, "gold": {"answer_en": "the hundred", "slots": [{"name": "exercise", "aliases": ["hundred", "हंड्रेड"]}]}, "distractor": {"value_aliases": ["roll-up", "roll up", "रोल-अप"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 9, "filler_seed": 2146840307, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": true, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "P037", "cand_id": "PX0546", "memory_type": "preference", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I've been following along with yoga videos on YouTube, and my favourite instructor is Adriene; her calm voice really helps me relax. Should I also try an in-person class?", "assistant_reply": "It's worth trying. A teacher in the room can correct your alignment, which videos can't do. Many studios offer a cheap first class, so you can test it without committing.", "followup_turn": "What should I bring to my first class?", "followup_reply": "Bring water, wear comfortable clothes that let you stretch, and arrive ten minutes early to tell the teacher you're new. Most studios lend mats if you don't have one.", "q1": "Who is my favourite yoga instructor on YouTube?", "q2": "Which YouTube yoga teacher do I like the most?"}, "hi": {"memory_turn": "मैं YouTube पर योगा वीडियो देखकर साथ-साथ कर रही हूँ, और मेरी पसंदीदा इंस्ट्रक्टर एड्रीन हैं; उनकी शांत आवाज़ से मुझे सच में सुकून मिलता है। क्या मुझे कोई आमने-सामने वाली क्लास भी आज़मानी चाहिए?", "assistant_reply": "आज़माने लायक़ है। कमरे में मौजूद टीचर आपकी मुद्रा ठीक कर सकते हैं, जो वीडियो नहीं कर सकते। कई स्टूडियो पहली क्लास सस्ते में देते हैं, तो आप बिना बँधे आज़मा सकती हैं।", "followup_turn": "पहली क्लास में क्या लेकर जाऊँ?", "followup_reply": "पानी लेकर जाएँ, ऐसे आरामदायक कपड़े पहनें जिनमें खिंचाव हो सके, और दस मिनट पहले पहुँचकर टीचर को बताएँ कि आप नई हैं। मैट न हो तो ज़्यादातर स्टूडियो दे देते हैं।", "q1": "YouTube पर मेरी पसंदीदा योगा इंस्ट्रक्टर कौन है?", "q2": "YouTube की कौन-सी योगा टीचर मुझे सबसे ज़्यादा पसंद है?"}, "hg": {"memory_turn": "Main YouTube pe yoga videos dekh ke saath-saath kar rahi hoon, aur meri favourite instructor Adriene hain; unki calm awaaz se mujhe sach mein sukoon milta hai. Kya mujhe koi in-person class bhi try karni chahiye?", "assistant_reply": "Try karne laayak hai. Room mein maujood teacher aapka alignment theek kar sakte hain, jo videos nahi kar sakte. Kai studios pehli class saste mein dete hain, toh aap bina commit kiye try kar sakti hain.", "followup_turn": "Pehli class mein kya lekar jaaun?", "followup_reply": "Paani lekar jaayein, aise comfortable kapde pehnein jinmein stretch ho sake, aur das minute pehle pahunch ke teacher ko batayein ki aap nayi hain. Mat na ho toh zyaadatar studios de dete hain.", "q1": "YouTube pe meri favourite yoga instructor kaun hai?", "q2": "YouTube ki kaunsi yoga teacher mujhe sabse zyaada pasand hai?"}}, "gold": {"answer_en": "Adriene (Yoga With Adriene)", "slots": [{"name": "instructor", "aliases": ["Adriene", "एड्रीन", "एड्रियन"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 2, "filler_seed": 3111482174, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P038", "cand_id": "PX0589", "memory_type": "preference", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I've been reading more poems lately, and I think I prefer modern poetry, though I'm open to other styles. Can you suggest a few collections to start with?", "assistant_reply": "Try 'Ariel' by Sylvia Plath, 'Milk and Honey' by Rupi Kaur, and 'The Carrying' by Ada Limón. They are quite different from each other, so you'll quickly see which voice you like.", "followup_turn": "How should I read poetry to get more out of it?", "followup_reply": "Read each poem twice, once quickly and once slowly out loud. Notice the line breaks and images, and don't worry about understanding everything the first time.", "q1": "What kind of poetry do I prefer?", "q2": "Which style of poetry do I like most?"}, "hi": {"memory_turn": "आजकल मैं ज़्यादा कविताएँ पढ़ रहा हूँ, और मुझे लगता है मुझे आधुनिक कविता पसंद है, हालाँकि मैं दूसरी शैलियों के लिए भी तैयार हूँ। शुरुआत के लिए कुछ संग्रह बता सकते हैं?", "assistant_reply": "सिल्विया प्लाथ की 'Ariel', रूपी कौर की 'Milk and Honey' और एडा लिमोन की 'The Carrying' पढ़िए। ये एक-दूसरे से काफ़ी अलग हैं, तो जल्दी पता चल जाएगा कि आपको कौन-सी आवाज़ पसंद है।", "followup_turn": "कविता से ज़्यादा पाने के लिए उसे कैसे पढ़ूँ?", "followup_reply": "हर कविता दो बार पढ़ें, एक बार जल्दी और एक बार धीरे-धीरे बोलकर। पंक्तियों के टूटने और बिंबों पर ध्यान दें, और पहली बार में सब समझने की चिंता न करें।", "q1": "मुझे किस तरह की कविता पसंद है?", "q2": "कविता की कौन-सी शैली मुझे सबसे ज़्यादा पसंद है?"}, "hg": {"memory_turn": "Aajkal main zyaada poems padh raha hoon, aur mujhe lagta hai mujhe modern poetry pasand hai, halanki main doosre styles ke liye bhi open hoon. Shuruaat ke liye kuch collections bata sakte hain?", "assistant_reply": "Sylvia Plath ki 'Ariel', Rupi Kaur ki 'Milk and Honey' aur Ada Limón ki 'The Carrying' padhiye. Yeh ek-doosre se kaafi alag hain, toh jaldi pata chal jayega ki aapko kaunsi awaaz pasand hai.", "followup_turn": "Poetry se zyaada paane ke liye use kaise padhun?", "followup_reply": "Har poem do baar padhein, ek baar jaldi aur ek baar dheere-dheere bol ke. Line breaks aur images pe dhyaan dein, aur pehli baar mein sab samajhne ki chinta na karein.", "q1": "Mujhe kis type ki poetry pasand hai?", "q2": "Poetry ka kaunsa style mujhe sabse zyaada pasand hai?"}}, "gold": {"answer_en": "modern poetry", "slots": [{"name": "style", "aliases": ["modern", "आधुनिक", "मॉडर्न", "contemporary"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 5, "filler_seed": 329123671, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": ["FL22", "FL37"]}
{"item_id": "P039", "cand_id": "PX0616", "memory_type": "preference", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I've been thinking about it, and my favourite synth-pop album is probably 'Violator' by Depeche Mode. I want to start collecting vinyl. Is it worth buying a turntable with a built-in speaker?", "assistant_reply": "They're convenient but usually sound thin and can wear records faster. A basic belt-drive turntable with a pair of powered bookshelf speakers costs a little more and sounds much better.", "followup_turn": "How should I store my records?", "followup_reply": "Keep them upright, never stacked flat, in inner and outer sleeves, and away from heat and sunlight. A sturdy shelf made for records is ideal.", "q1": "What is my favourite synth-pop album?", "q2": "Which synth-pop album do I like most?", "distractor_sentence": "My sister thinks Black Celebration is their best album, but I disagree."}, "hi": {"memory_turn": "मैं इस बारे में सोच रही थी, और शायद मेरा पसंदीदा सिंथ-पॉप एल्बम डेपेश मोड का 'Violator' है। मैं विनाइल रिकॉर्ड इकट्ठा करना शुरू करना चाहती हूँ। क्या बिल्ट-इन स्पीकर वाला टर्नटेबल लेना ठीक है?", "assistant_reply": "वे सुविधाजनक हैं पर आमतौर पर आवाज़ पतली होती है और रिकॉर्ड जल्दी घिस सकते हैं। एक सादा बेल्ट-ड्राइव टर्नटेबल और पावर वाले छोटे बुकशेल्फ़ स्पीकर थोड़े महँगे पड़ते हैं, पर आवाज़ कहीं बेहतर होती है।", "followup_turn": "रिकॉर्ड को कैसे रखूँ?", "followup_reply": "उन्हें सीधा खड़ा रखें, कभी एक के ऊपर एक न रखें, अंदर और बाहर के कवर में रखें, और गर्मी व धूप से दूर रखें। रिकॉर्ड के लिए बनी मज़बूत शेल्फ़ सबसे अच्छी है।", "q1": "मेरा पसंदीदा सिंथ-पॉप एल्बम कौन-सा है?", "q2": "मुझे सबसे ज़्यादा कौन-सा सिंथ-पॉप एल्बम पसंद है?", "distractor_sentence": "मेरी बहन को लगता है कि Black Celebration उनका सबसे अच्छा एल्बम है, पर मैं सहमत नहीं हूँ।"}, "hg": {"memory_turn": "Main is baare mein soch rahi thi, aur shayad mera favourite synth-pop album Depeche Mode ka 'Violator' hai. Main vinyl records collect karna shuru karna chahti hoon. Kya built-in speaker wala turntable lena theek hai?", "assistant_reply": "Woh convenient hain par usually awaaz patli hoti hai aur records jaldi ghis sakte hain. Ek simple belt-drive turntable aur powered bookshelf speakers thode mehenge padte hain, par awaaz kahin better hoti hai.", "followup_turn": "Records ko kaise rakhun?", "followup_reply": "Unhe seedha khada rakhein, kabhi ek ke upar ek na rakhein, inner aur outer sleeves mein rakhein, aur garmi aur dhoop se door rakhein. Records ke liye bani mazboot shelf sabse acchi hai.", "q1": "Mera favourite synth-pop album kaunsa hai?", "q2": "Mujhe sabse zyaada kaunsa synth-pop album pasand hai?", "distractor_sentence": "Meri behen ko lagta hai ki Black Celebration unka sabse accha album hai, par main agree nahi karti."}}, "gold": {"answer_en": "Violator (Depeche Mode)", "slots": [{"name": "album", "aliases": ["Violator", "वायलेटर"]}]}, "distractor": {"value_aliases": ["Black Celebration"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 9, "filler_seed": 3027308341, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P040", "cand_id": "PX0687", "memory_type": "preference", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I've been experimenting with different guitar tunings, and I think I prefer the sound of D tuning over standard tuning. Will tuning down often damage my guitar?", "assistant_reply": "No, tuning down slightly is safe. The strings just feel a bit looser. If you stay in the lower tuning for a long time, a slightly heavier string set and a quick setup will keep it playing well.", "followup_turn": "Do I need to adjust the neck?", "followup_reply": "Only if you notice buzzing or the action changes a lot. A small truss rod adjustment fixes most problems, but a music shop can do it if you're unsure.", "q1": "Which guitar tuning do I prefer?", "q2": "What tuning do I like the sound of most on my guitar?", "q3": "Before tonight's jam session I'm retuning my guitar to the sound I like best. Which tuning should I set?", "distractor_sentence": "My bandmate prefers open G for his slide playing."}, "hi": {"memory_turn": "मैं गिटार की अलग-अलग ट्यूनिंग आज़मा रहा हूँ, और मुझे लगता है कि मुझे स्टैंडर्ड ट्यूनिंग से ज़्यादा D ट्यूनिंग की आवाज़ पसंद है। क्या बार-बार नीचे ट्यून करने से गिटार ख़राब होगा?", "assistant_reply": "नहीं, थोड़ा नीचे ट्यून करना सुरक्षित है। तार बस थोड़े ढीले लगते हैं। अगर लंबे समय तक नीचे वाली ट्यूनिंग में रहें, तो थोड़े मोटे तार और एक छोटा सेटअप इसे अच्छा बजता रखेंगे।", "followup_turn": "क्या मुझे नेक ठीक करनी पड़ेगी?", "followup_reply": "सिर्फ़ तब, जब भिनभिनाहट आए या तारों की ऊँचाई बहुत बदल जाए। ट्रस रॉड का छोटा-सा बदलाव ज़्यादातर दिक़्क़तें ठीक कर देता है, और यक़ीन न हो तो म्यूज़िक शॉप कर देगी।", "q1": "मुझे गिटार की कौन-सी ट्यूनिंग पसंद है?", "q2": "अपने गिटार पर मुझे किस ट्यूनिंग की आवाज़ सबसे ज़्यादा पसंद है?", "q3": "आज रात के जैम सेशन से पहले मैं अपना गिटार अपनी सबसे पसंदीदा आवाज़ पर ट्यून कर रहा हूँ। कौन-सी ट्यूनिंग लगाऊँ?", "distractor_sentence": "मेरे बैंड के साथी को स्लाइड बजाने के लिए open G पसंद है।"}, "hg": {"memory_turn": "Main guitar ki alag-alag tunings try kar raha hoon, aur mujhe lagta hai ki mujhe standard tuning se zyaada D tuning ki awaaz pasand hai. Kya baar-baar neeche tune karne se guitar kharab hoga?", "assistant_reply": "Nahi, thoda neeche tune karna safe hai. Strings bas thodi dheeli lagti hain. Agar lambe time tak neeche wali tuning mein rahein, toh thodi moti strings aur ek chhota setup ise accha bajta rakhenge.", "followup_turn": "Kya mujhe neck adjust karni padegi?", "followup_reply": "Sirf tab, jab buzzing aaye ya action bahut badal jaaye. Truss rod ka chhota-sa adjustment zyaadatar problems theek kar deta hai, aur yakeen na ho toh music shop kar degi.", "q1": "Mujhe guitar ki kaunsi tuning pasand hai?", "q2": "Apne guitar pe mujhe kis tuning ki awaaz sabse zyaada pasand hai?", "q3": "Aaj raat ke jam session se pehle main apna guitar apni sabse favourite awaaz pe tune kar raha hoon. Kaunsi tuning lagaun?", "distractor_sentence": "Mere band ke saathi ko slide bajane ke liye open G pasand hai."}}, "gold": {"answer_en": "D tuning", "slots": [{"name": "tuning", "aliases": ["D tuning", "D-tuning", "D ट्यूनिंग", "डी ट्यूनिंग"]}]}, "distractor": {"value_aliases": ["open G"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 2, "filler_seed": 1942903413, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "E031", "cand_id": "EC041", "memory_type": "event", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I saw a beautiful 3-bedroom bungalow in the Oakwood neighbourhood on January 22nd that I really liked, but the kitchen needed serious renovation. How much should I budget for a kitchen remodel?", "assistant_reply": "A basic refresh with new cabinet doors, counters and paint is far cheaper than a full remodel with new layout and plumbing. Get at least three quotes and keep 10 to 15 percent extra for surprises.", "followup_turn": "Should I get an inspection before making an offer?", "followup_reply": "Yes. A home inspection can reveal hidden problems with wiring, plumbing or the roof, and gives you a stronger position to negotiate the price.", "q1": "When did I see the 3-bedroom bungalow in Oakwood?", "q2": "On what date did I view the Oakwood bungalow?", "q3": "The estate agent wants to know when I last came by to look at that house with the old kitchen. What date should I tell her?", "distractor_sentence": "My parents viewed a house in the next street on February 5th."}, "hi": {"memory_turn": "मैंने 22 जनवरी को ओकवुड इलाक़े में एक ख़ूबसूरत 3-बेडरूम बंगला देखा जो मुझे बहुत पसंद आया, पर उसकी रसोई में बड़ी मरम्मत की ज़रूरत थी। रसोई नए सिरे से बनाने के लिए कितना बजट रखूँ?", "assistant_reply": "नए कैबिनेट दरवाज़े, काउंटर और पेंट वाला साधारण बदलाव, नए लेआउट और प्लंबिंग वाले पूरे काम से कहीं सस्ता है। कम से कम तीन कोटेशन लें और अचानक ख़र्चों के लिए 10 से 15 प्रतिशत ज़्यादा रखें।", "followup_turn": "क्या ऑफ़र देने से पहले घर की जाँच करवानी चाहिए?", "followup_reply": "हाँ। घर की जाँच से वायरिंग, प्लंबिंग या छत की छुपी दिक़्क़तें पता चल सकती हैं, और क़ीमत पर मोलभाव करने में आपकी स्थिति मज़बूत होती है।", "q1": "मैंने ओकवुड में 3-बेडरूम बंगला कब देखा था?", "q2": "मैंने ओकवुड वाला बंगला किस तारीख़ को देखा?", "q3": "प्रॉपर्टी एजेंट जानना चाहती है कि पुरानी रसोई वाला वह घर देखने मैं आख़िरी बार कब आई थी। उसे कौन-सी तारीख़ बताऊँ?", "distractor_sentence": "मेरे मम्मी-पापा ने 5 फ़रवरी को अगली गली में एक घर देखा था।"}, "hg": {"memory_turn": "Maine 22 January ko Oakwood area mein ek khoobsurat 3-bedroom bungalow dekha jo mujhe bahut pasand aaya, par uski kitchen mein badi renovation ki zaroorat thi. Kitchen remodel ke liye kitna budget rakhun?", "assistant_reply": "Naye cabinet doors, counters aur paint wala basic refresh, naye layout aur plumbing wale full remodel se kahin sasta hai. Kam se kam teen quotes lein aur surprises ke liye 10 se 15 percent zyaada rakhein.", "followup_turn": "Kya offer dene se pehle home inspection karwana chahiye?", "followup_reply": "Haan. Home inspection se wiring, plumbing ya chhat ki chhupi problems pata chal sakti hain, aur price negotiate karne mein aapki position strong hoti hai.", "q1": "Maine Oakwood mein 3-bedroom bungalow kab dekha tha?", "q2": "Maine Oakwood wala bungalow kis date ko dekha?", "q3": "Property agent jaanna chahti hai ki purani kitchen wala woh ghar dekhne main last time kab aayi thi. Use kaunsi date bataun?", "distractor_sentence": "Mere mummy-papa ne 5 February ko agli gali mein ek ghar dekha tha."}}, "gold": {"answer_en": "January 22", "date": {"day": 22, "month": 1}, "slots": [{"name": "date", "aliases": ["January 22", "January 22nd", "22 January", "22nd January", "22nd of January", "Jan 22", "22 Jan", "22 जनवरी"]}]}, "distractor": {"date": {"day": 5, "month": 2}, "value_aliases": ["February 5", "February 5th", "5 February", "5th February", "5th of February", "Feb 5", "5 Feb", "5 फ़रवरी", "5 फरवरी"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "gpt4_7fce9456", "lme_type": "multi-session"}, "topic": "work/study", "is_answerable": true, "evidence_position": 2, "filler_seed": 3491454187, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": true, "abl_q3": true, "abl_digits": true, "banned_fillers": []}
{"item_id": "E032", "cand_id": "EC042", "memory_type": "event", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I put in an offer on a 3-bedroom townhouse in the Brookside neighbourhood on February 25th, and after some negotiation we agreed on a price. What are the next steps before closing?", "assistant_reply": "Usually you'll finalise the mortgage, arrange a home inspection and appraisal, review the title search, and buy home insurance. Your agent or lawyer will give you a timeline with deadlines.", "followup_turn": "How long does closing usually take?", "followup_reply": "Often around 30 to 45 days after the offer is accepted, but it depends on the lender and any repairs that need to be agreed.", "q1": "When did I put in an offer on the Brookside townhouse?", "q2": "On what date did I make my offer on the 3-bedroom townhouse in Brookside?", "distractor_sentence": "My friend made an offer on a flat across town on March 8th."}, "hi": {"memory_turn": "मैंने 25 फ़रवरी को ब्रुकसाइड इलाक़े में एक 3-बेडरूम टाउनहाउस पर ऑफ़र दिया, और कुछ मोलभाव के बाद क़ीमत पर बात बन गई। सौदा पूरा होने से पहले अगले क़दम क्या हैं?", "assistant_reply": "आमतौर पर आप होम लोन पक्का करेंगे, घर की जाँच और मूल्यांकन करवाएँगे, मालिकाना हक़ के काग़ज़ देखेंगे, और घर का बीमा लेंगे। आपका एजेंट या वकील आपको तारीख़ों के साथ पूरा क्रम देगा।", "followup_turn": "सौदा पूरा होने में आमतौर पर कितना समय लगता है?", "followup_reply": "अक्सर ऑफ़र मंज़ूर होने के 30 से 45 दिन बाद, पर यह बैंक और तय होने वाली मरम्मत पर निर्भर करता है।", "q1": "मैंने ब्रुकसाइड वाले टाउनहाउस पर ऑफ़र कब दिया था?", "q2": "ब्रुकसाइड के 3-बेडरूम टाउनहाउस पर मैंने किस तारीख़ को ऑफ़र दिया?", "distractor_sentence": "मेरे दोस्त ने 8 मार्च को शहर के दूसरे छोर पर एक फ़्लैट के लिए ऑफ़र दिया था।"}, "hg": {"memory_turn": "Maine 25 February ko Brookside area mein ek 3-bedroom townhouse pe offer diya, aur kuch negotiation ke baad price pe baat ban gayi. Closing se pehle next steps kya hain?", "assistant_reply": "Usually aap mortgage final karenge, home inspection aur appraisal karwayenge, title search dekhenge, aur home insurance lenge. Aapka agent ya lawyer aapko deadlines ke saath poori timeline dega.", "followup_turn": "Closing mein usually kitna time lagta hai?", "followup_reply": "Aksar offer accept hone ke 30 se 45 din baad, par yeh lender aur agree hone wali repairs pe depend karta hai.", "q1": "Maine Brookside wale townhouse pe offer kab diya tha?", "q2": "Brookside ke 3-bedroom townhouse pe maine kis date ko offer diya?", "distractor_sentence": "Mere dost ne 8 March ko sheher ke doosre kone pe ek flat ke liye offer diya tha."}}, "gold": {"answer_en": "February 25", "date": {"day": 25, "month": 2}, "slots": [{"name": "date", "aliases": ["February 25", "February 25th", "25 February", "25th February", "25th of February", "Feb 25", "25 Feb", "25 फ़रवरी", "25 फरवरी"]}]}, "distractor": {"date": {"day": 8, "month": 3}, "value_aliases": ["March 8", "March 8th", "8 March", "8th March", "8th of March", "Mar 8", "8 Mar", "8 मार्च"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "gpt4_7fce9456", "lme_type": "multi-session"}, "topic": "other", "is_answerable": true, "evidence_position": 5, "filler_seed": 1711409012, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": false, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E033", "cand_id": "EC043", "memory_type": "event", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I fell in love with a 2-bedroom condo on February 15th; it had modern appliances and a community pool, but my offer got rejected a few days later because of a higher bid. How can I make my next offer stronger?", "assistant_reply": "Get a mortgage pre-approval letter, keep conditions to a minimum, offer a flexible closing date, and ask your agent what matters most to the seller. A short personal letter sometimes helps too.", "followup_turn": "Should I offer above the asking price?", "followup_reply": "In a competitive market it may be necessary, but set your maximum in advance based on recent sales nearby so you don't overpay in the heat of the moment.", "q1": "When did I see the 2-bedroom condo with the community pool?", "q2": "On what date did I fall in love with the 2-bedroom condo?", "q3": "I'm writing in my house-hunting diary about the flat with the shared pool that I lost to a higher bid. Which date did I first see it?"}, "hi": {"memory_turn": "15 फ़रवरी को मुझे एक 2-बेडरूम कॉन्डो से प्यार हो गया; उसमें आधुनिक उपकरण और सोसाइटी का पूल था, पर कुछ दिन बाद ज़्यादा ऊँची बोली की वजह से मेरा ऑफ़र ठुकरा दिया गया। अगला ऑफ़र मज़बूत कैसे बनाऊँ?", "assistant_reply": "होम लोन का प्री-अप्रूवल लेटर लें, शर्तें कम से कम रखें, सौदा पूरा करने की तारीख़ में लचीलापन दें, और अपने एजेंट से पूछें कि बेचने वाले के लिए सबसे ज़रूरी क्या है। कभी-कभी एक छोटा निजी पत्र भी मदद करता है।", "followup_turn": "क्या मुझे माँगी गई क़ीमत से ज़्यादा ऑफ़र करना चाहिए?", "followup_reply": "होड़ वाले बाज़ार में ज़रूरी हो सकता है, पर आसपास की हाल की बिक्री देखकर अपनी अधिकतम सीमा पहले तय कर लें, ताकि जोश में ज़्यादा न दे दें।", "q1": "मैंने सोसाइटी पूल वाला 2-बेडरूम कॉन्डो कब देखा था?", "q2": "मुझे किस तारीख़ को 2-बेडरूम कॉन्डो से प्यार हुआ था?", "q3": "मैं घर ढूँढने की अपनी डायरी में साझा पूल वाले उस फ़्लैट के बारे में लिख रही हूँ जो ऊँची बोली में हाथ से निकल गया। मैंने उसे पहली बार किस तारीख़ को देखा था?"}, "hg": {"memory_turn": "15 February ko mujhe ek 2-bedroom condo se pyaar ho gaya; usmein modern appliances aur community pool tha, par kuch din baad higher bid ki wajah se mera offer reject ho gaya. Agla offer strong kaise banaun?", "assistant_reply": "Mortgage pre-approval letter lein, conditions kam se kam rakhein, closing date mein flexibility dein, aur apne agent se poochhein ki seller ke liye sabse zaroori kya hai. Kabhi-kabhi ek chhota personal letter bhi help karta hai.", "followup_turn": "Kya mujhe asking price se zyaada offer karna chahiye?", "followup_reply": "Competitive market mein zaroori ho sakta hai, par aaspaas ki recent sales dekh ke apni maximum limit pehle decide kar lein, taaki josh mein zyaada na de dein.", "q1": "Maine community pool wala 2-bedroom condo kab dekha tha?", "q2": "Mujhe kis date ko 2-bedroom condo se pyaar hua tha?", "q3": "Main ghar dhoondhne ki apni diary mein shared pool wale us flat ke baare mein likh rahi hoon jo higher bid mein haath se nikal gaya. Maine use pehli baar kis date ko dekha tha?"}}, "gold": {"answer_en": "February 15", "date": {"day": 15, "month": 2}, "slots": [{"name": "date", "aliases": ["February 15", "February 15th", "15 February", "15th February", "15th of February", "Feb 15", "15 Feb", "15 फ़रवरी", "15 फरवरी"]}]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "gpt4_7fce9456", "lme_type": "multi-session"}, "topic": "other", "is_answerable": true, "evidence_position": 9, "filler_seed": 166847694, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": true, "abl_q3": true, "abl_digits": true, "banned_fillers": []}
{"item_id": "E034", "cand_id": "EC044", "memory_type": "event", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I've been searching for a home for a while, and some places just didn't fit my budget, like the one in Cedar Creek I saw on February 1st; it was way out of my league. How do I work out a realistic budget?", "assistant_reply": "A common guide is to keep total housing costs below about 30 percent of your take-home pay. Include the deposit, closing costs, taxes, insurance and maintenance, not just the monthly loan payment.", "followup_turn": "How big should my emergency fund be after buying?", "followup_reply": "Aim for three to six months of essential expenses. Homes come with surprise repairs, so it's wise not to spend every last rupee or dollar on the deposit.", "q1": "When did I see the property in Cedar Creek?", "q2": "On what date did I look at the house in Cedar Creek that was beyond my budget?", "distractor_sentence": "My sister looked at a place in Maple Heights on February 19th."}, "hi": {"memory_turn": "मैं काफ़ी समय से घर ढूँढ रहा हूँ, और कुछ घर मेरे बजट में बिल्कुल नहीं आए, जैसे सीडर क्रीक वाला जो मैंने 1 फ़रवरी को देखा था; वह मेरी पहुँच से बहुत बाहर था। सही बजट कैसे तय करूँ?", "assistant_reply": "एक आम नियम है कि घर का कुल ख़र्च आपकी हाथ में आने वाली तनख़्वाह के लगभग 30 प्रतिशत से कम रहे। सिर्फ़ मासिक किस्त नहीं, बल्कि डाउन पेमेंट, सौदे का ख़र्च, टैक्स, बीमा और रखरखाव भी जोड़ें।", "followup_turn": "घर ख़रीदने के बाद इमरजेंसी फ़ंड कितना होना चाहिए?", "followup_reply": "ज़रूरी ख़र्चों के तीन से छह महीने के बराबर रखें। घरों में अचानक मरम्मत आती रहती है, इसलिए डाउन पेमेंट में आख़िरी पैसा तक लगा देना समझदारी नहीं।", "q1": "मैंने सीडर क्रीक वाली प्रॉपर्टी कब देखी थी?", "q2": "सीडर क्रीक का वह घर जो मेरे बजट से बाहर था, मैंने किस तारीख़ को देखा?", "distractor_sentence": "मेरी बहन ने 19 फ़रवरी को मेपल हाइट्स में एक घर देखा था।"}, "hg": {"memory_turn": "Main kaafi time se ghar dhoondh raha hoon, aur kuch ghar mere budget mein bilkul nahi aaye, jaise Cedar Creek wala jo maine 1 February ko dekha tha; woh meri pahunch se bahut bahar tha. Sahi budget kaise decide karoon?", "assistant_reply": "Ek common rule hai ki ghar ka total kharcha aapki take-home salary ke lagbhag 30 percent se kam rahe. Sirf monthly EMI nahi, balki down payment, closing costs, tax, insurance aur maintenance bhi jodein.", "followup_turn": "Ghar khareedne ke baad emergency fund kitna hona chahiye?", "followup_reply": "Zaroori kharchon ke teen se chhe mahine ke barabar rakhein. Gharon mein achaanak repairs aati rehti hain, isliye down payment mein aakhri paisa tak laga dena samajhdaari nahi.", "q1": "Maine Cedar Creek wali property kab dekhi thi?", "q2": "Cedar Creek ka woh ghar jo mere budget se bahar tha, maine kis date ko dekha?", "distractor_sentence": "Meri behen ne 19 February ko Maple Heights mein ek ghar dekha tha."}}, "gold": {"answer_en": "February 1", "date": {"day": 1, "month": 2}, "slots": [{"name": "date", "aliases": ["February 1", "February 1st", "1 February", "1st February", "1st of February", "Feb 1", "1 Feb", "1 फ़रवरी", "1 फरवरी"]}]}, "distractor": {"date": {"day": 19, "month": 2}, "value_aliases": ["February 19", "February 19th", "19 February", "19th February", "19th of February", "Feb 19", "19 Feb", "19 फ़रवरी", "19 फरवरी"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "gpt4_7fce9456", "lme_type": "multi-session"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 3478217432, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": false, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E035", "cand_id": "EC046", "memory_type": "event", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I attended a lecture at the Art Gallery on 'The Evolution of Street Art' on March 3rd, and it got me thinking about the role of street art in cities. Which street artists should I read about?", "assistant_reply": "Start with Banksy for political stencils, Shepard Fairey for bold poster art, and Os Gêmeos for colourful murals. In India, the St+art India Foundation has documented many large public murals.", "followup_turn": "Are there street art tours I could join?", "followup_reply": "Many cities have guided walking tours, often run by local artists. Delhi's Lodhi Art District and Mumbai's Bandra area are well known for their murals.", "q1": "When did I attend the lecture on 'The Evolution of Street Art'?", "q2": "On what date was the street art lecture I went to at the Art Gallery?", "q3": "I'm writing a blog post about the gallery talk on graffiti and murals that I went to. What date should I mention?"}, "hi": {"memory_turn": "मैं 3 मार्च को आर्ट गैलरी में 'The Evolution of Street Art' नाम के एक लेक्चर में गई थी, और उसने मुझे शहरों में स्ट्रीट आर्ट की भूमिका के बारे में सोचने पर लगा दिया। मुझे किन स्ट्रीट आर्टिस्ट के बारे में पढ़ना चाहिए?", "assistant_reply": "राजनीतिक स्टेंसिल के लिए बैंक्सी, बोल्ड पोस्टर आर्ट के लिए शेपर्ड फ़ेयरी, और रंगीन म्यूरल के लिए ओस जेमियोस से शुरू करें। भारत में St+art India Foundation ने कई बड़े सार्वजनिक म्यूरल दर्ज किए हैं।", "followup_turn": "क्या कोई स्ट्रीट आर्ट टूर है जिसमें मैं शामिल हो सकूँ?", "followup_reply": "कई शहरों में गाइड के साथ पैदल टूर होते हैं, अक्सर स्थानीय कलाकार चलाते हैं। दिल्ली का लोधी आर्ट डिस्ट्रिक्ट और मुंबई का बांद्रा इलाक़ा अपने म्यूरल के लिए मशहूर हैं।", "q1": "मैं 'The Evolution of Street Art' वाले लेक्चर में कब गई थी?", "q2": "आर्ट गैलरी में जिस स्ट्रीट आर्ट लेक्चर में मैं गई, वह किस तारीख़ को था?", "q3": "मैं ग्रैफ़िटी और म्यूरल पर गैलरी में हुई उस बातचीत के बारे में ब्लॉग लिख रही हूँ जिसमें मैं गई थी। मुझे कौन-सी तारीख़ लिखनी चाहिए?"}, "hg": {"memory_turn": "Main 3 March ko Art Gallery mein 'The Evolution of Street Art' naam ke ek lecture mein gayi thi, aur usne mujhe sheheron mein street art ke role ke baare mein sochne pe laga diya. Mujhe kin street artists ke baare mein padhna chahiye?", "assistant_reply": "Political stencils ke liye Banksy, bold poster art ke liye Shepard Fairey, aur colourful murals ke liye Os Gêmeos se shuru karein. India mein St+art India Foundation ne kai bade public murals document kiye hain.", "followup_turn": "Kya koi street art tour hai jismein main join kar sakoon?", "followup_reply": "Kai sheheron mein guided walking tours hote hain, aksar local artists chalate hain. Delhi ka Lodhi Art District aur Mumbai ka Bandra area apne murals ke liye famous hain.", "q1": "Main 'The Evolution of Street Art' wale lecture mein kab gayi thi?", "q2": "Art Gallery mein jis street art lecture mein main gayi, woh kis date ko tha?", "q3": "Main graffiti aur murals pe gallery mein hui us talk ke baare mein blog likh rahi hoon jismein main gayi thi. Mujhe kaunsi date likhni chahiye?"}}, "gold": {"answer_en": "March 3", "date": {"day": 3, "month": 3}, "slots": [{"name": "date", "aliases": ["March 3", "March 3rd", "3 March", "3rd March", "3rd of March", "Mar 3", "3 Mar", "3 मार्च"]}]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "2ce6a0f2", "lme_type": "multi-session"}, "topic": "other", "is_answerable": true, "evidence_position": 5, "filler_seed": 1006987074, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": false, "abl_q3": true, "abl_digits": true, "banned_fillers": []}
{"item_id": "E036", "cand_id": "EC047", "memory_type": "event", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I attended the 'Women in Art' exhibition on February 10th, and I was especially drawn to the works of local artist Rachel Lee. How can I support local artists like her?", "assistant_reply": "Buy prints or small works when you can, follow and share their pages, attend their openings, and recommend them to friends. Commissioning a small piece is a lovely way to help too.", "followup_turn": "Is it okay to ask an artist about prices directly?", "followup_reply": "Yes, most artists appreciate a polite direct question. Galleries usually have a price list, and many artists also offer payment plans for larger works.", "q1": "When did I attend the 'Women in Art' exhibition?", "q2": "On what date did I go to the 'Women in Art' exhibition?", "distractor_sentence": "My cousin is visiting a photography exhibition on March 14th."}, "hi": {"memory_turn": "मैं 10 फ़रवरी को 'Women in Art' प्रदर्शनी में गया था, और मुझे ख़ासकर स्थानीय कलाकार रेचल ली का काम बहुत पसंद आया। मैं उनके जैसे स्थानीय कलाकारों की मदद कैसे कर सकता हूँ?", "assistant_reply": "जब हो सके उनके प्रिंट या छोटी कलाकृतियाँ ख़रीदें, उनके पेज फ़ॉलो और शेयर करें, उनकी प्रदर्शनियों में जाएँ, और दोस्तों को उनके बारे में बताएँ। कोई छोटा काम बनवाना भी अच्छा तरीक़ा है।", "followup_turn": "क्या कलाकार से सीधे क़ीमत पूछना ठीक है?", "followup_reply": "हाँ, ज़्यादातर कलाकार विनम्र सीधे सवाल की क़द्र करते हैं। गैलरी में आमतौर पर क़ीमतों की सूची होती है, और कई कलाकार बड़े कामों के लिए किस्तों का विकल्प भी देते हैं।", "q1": "मैं 'Women in Art' प्रदर्शनी में कब गया था?", "q2": "मैं किस तारीख़ को 'Women in Art' प्रदर्शनी देखने गया?", "distractor_sentence": "मेरा कज़िन 14 मार्च को एक फ़ोटोग्राफ़ी प्रदर्शनी देखने जा रहा है।"}, "hg": {"memory_turn": "Main 10 February ko 'Women in Art' exhibition mein gaya tha, aur mujhe khaaskar local artist Rachel Lee ka kaam bahut pasand aaya. Main unke jaise local artists ki madad kaise kar sakta hoon?", "assistant_reply": "Jab ho sake unke prints ya chhote artworks khareedein, unke pages follow aur share karein, unke openings mein jaayein, aur doston ko unke baare mein batayein. Koi chhota piece commission karna bhi accha tareeka hai.", "followup_turn": "Kya artist se seedha price poochhna theek hai?", "followup_reply": "Haan, zyaadatar artists polite seedhe sawaal ki kadar karte hain. Galleries mein usually price list hoti hai, aur kai artists bade kaamon ke liye installments ka option bhi dete hain.", "q1": "Main 'Women in Art' exhibition mein kab gaya tha?", "q2": "Main kis date ko 'Women in Art' exhibition dekhne gaya?", "distractor_sentence": "Mera cousin 14 March ko ek photography exhibition dekhne ja raha hai."}}, "gold": {"answer_en": "February 10", "date": {"day": 10, "month": 2}, "slots": [{"name": "date", "aliases": ["February 10", "February 10th", "10 February", "10th February", "10th of February", "Feb 10", "10 Feb", "10 फ़रवरी", "10 फरवरी"]}]}, "distractor": {"date": {"day": 14, "month": 3}, "value_aliases": ["March 14", "March 14th", "14 March", "14th March", "14th of March", "Mar 14", "14 Mar", "14 मार्च"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "2ce6a0f2", "lme_type": "multi-session"}, "topic": "other", "is_answerable": true, "evidence_position": 9, "filler_seed": 3324571055, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E037", "cand_id": "EC048", "memory_type": "event", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I volunteered at the Children's Museum for their 'Art Afternoon' event on February 17th, and it was amazing to see the kids create artwork inspired by famous paintings. Can you suggest easy art activities for kids aged six to ten?", "assistant_reply": "Try leaf printing with paint, collage portraits from old magazines, and 'starry night' pictures with oil pastels. Salt painting with glue and watercolours is also simple and very satisfying.", "followup_turn": "How do I keep the mess under control?", "followup_reply": "Cover tables with old sheets or newspaper, give each child an apron or an old T-shirt, and use washable paints. Setting up a clean-up station with wipes saves a lot of trouble.", "q1": "When did I volunteer at the Children's Museum 'Art Afternoon'?", "q2": "On what date did I help out at the Children's Museum art event?"}, "hi": {"memory_turn": "मैंने 17 फ़रवरी को चिल्ड्रन्स म्यूज़ियम के 'Art Afternoon' इवेंट में वॉलंटियर किया, और बच्चों को मशहूर पेंटिंग से प्रेरित होकर अपनी कला बनाते देखना कमाल था। छह से दस साल के बच्चों के लिए आसान आर्ट एक्टिविटी बता सकते हैं?", "assistant_reply": "पत्तियों पर रंग लगाकर छाप बनाना, पुरानी पत्रिकाओं से कोलाज वाले चेहरे, और ऑयल पेस्टल से 'तारों भरी रात' वाली तस्वीरें आज़माइए। गोंद और वॉटरकलर से नमक वाली पेंटिंग भी आसान और मज़ेदार है।", "followup_turn": "गंदगी को कैसे क़ाबू में रखूँ?", "followup_reply": "टेबल को पुरानी चादर या अख़बार से ढकें, हर बच्चे को एप्रन या पुरानी टी-शर्ट दें, और धुल जाने वाले रंग इस्तेमाल करें। वाइप्स के साथ सफ़ाई का एक कोना बनाने से बहुत झंझट बचती है।", "q1": "मैंने चिल्ड्रन्स म्यूज़ियम के 'Art Afternoon' में कब वॉलंटियर किया?", "q2": "चिल्ड्रन्स म्यूज़ियम के आर्ट इवेंट में मैंने किस तारीख़ को मदद की?"}, "hg": {"memory_turn": "Maine 17 February ko Children's Museum ke 'Art Afternoon' event mein volunteer kiya, aur bacchon ko famous paintings se inspire hokar apni art banate dekhna kamaal tha. Chhe se das saal ke bacchon ke liye easy art activities bata sakte hain?", "assistant_reply": "Pattiyon pe paint laga ke prints banana, purani magazines se collage wale chehre, aur oil pastels se 'starry night' wali pictures try kijiye. Glue aur watercolours se salt painting bhi easy aur mazedaar hai.", "followup_turn": "Gandagi ko kaise control mein rakhun?", "followup_reply": "Tables ko purani chaadar ya newspaper se dhakein, har bacche ko apron ya purani T-shirt dein, aur washable paints use karein. Wipes ke saath ek cleaning corner banane se bahut jhanjhat bachti hai.", "q1": "Maine Children's Museum ke 'Art Afternoon' mein kab volunteer kiya?", "q2": "Children's Museum ke art event mein maine kis date ko help ki?"}}, "gold": {"answer_en": "February 17", "date": {"day": 17, "month": 2}, "slots": [{"name": "date", "aliases": ["February 17", "February 17th", "17 February", "17th February", "17th of February", "Feb 17", "17 Feb", "17 फ़रवरी", "17 फरवरी"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "2ce6a0f2", "lme_type": "multi-session"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 4187298928, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E038", "cand_id": "EC052", "memory_type": "event", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I've been really busy with photography lately. My new remote shutter release arrived on February 10th, and it's been a game-changer for my long exposures. What should I try shooting with it?", "assistant_reply": "Try light trails from traffic at dusk, silky waterfalls with a slow shutter, star trails on a clear night, or self-portraits where you can't reach the camera.", "followup_turn": "What settings work for light trails?", "followup_reply": "Use a tripod, ISO 100, an aperture around f/8 to f/11 and a shutter speed of 10 to 30 seconds. Adjust the time until the trails look as long as you like.", "q1": "When did my remote shutter release arrive?", "q2": "On what date was my new remote shutter release delivered?", "q3": "I want to check if my camera remote is still under its 30-day return window. On what date did it reach me?", "distractor_sentence": "My friend's new tripod arrived on February 21st."}, "hi": {"memory_turn": "आजकल मैं फ़ोटोग्राफ़ी में बहुत व्यस्त हूँ। मेरा नया रिमोट शटर रिलीज़ 10 फ़रवरी को आया, और लंबे एक्सपोज़र के लिए इसने कमाल कर दिया है। इससे क्या-क्या शूट करके देखूँ?", "assistant_reply": "शाम के समय ट्रैफ़िक की रोशनी की लकीरें, धीमे शटर से रेशमी झरने, साफ़ रात में तारों की लकीरें, या ऐसी सेल्फ़-पोर्ट्रेट आज़माइए जिनमें आप कैमरे तक नहीं पहुँच सकते।", "followup_turn": "रोशनी की लकीरों के लिए कौन-सी सेटिंग ठीक रहती है?", "followup_reply": "तिपाई इस्तेमाल करें, ISO 100, अपर्चर f/8 से f/11 के आसपास और शटर स्पीड 10 से 30 सेकंड रखें। समय तब तक बदलें जब तक लकीरें मनचाही लंबी न दिखें।", "q1": "मेरा रिमोट शटर रिलीज़ कब आया था?", "q2": "मेरा नया रिमोट शटर रिलीज़ किस तारीख़ को डिलीवर हुआ?", "q3": "मैं देखना चाहता हूँ कि मेरा कैमरा रिमोट अभी 30 दिन की वापसी की सीमा में है या नहीं। वह मेरे पास किस तारीख़ को पहुँचा था?", "distractor_sentence": "मेरे दोस्त का नया ट्राइपॉड 21 फ़रवरी को आया था।"}, "hg": {"memory_turn": "Aajkal main photography mein bahut busy hoon. Mera naya remote shutter release 10 February ko aaya, aur long exposures ke liye isne kamaal kar diya hai. Isse kya-kya shoot karke dekhun?", "assistant_reply": "Shaam ke time traffic ki light trails, slow shutter se silky waterfalls, saaf raat mein star trails, ya aise self-portraits try kijiye jinmein aap camera tak nahi pahunch sakte.", "followup_turn": "Light trails ke liye kaunsi settings theek rehti hain?", "followup_reply": "Tripod use karein, ISO 100, aperture f/8 se f/11 ke aaspaas aur shutter speed 10 se 30 second rakhein. Time tab tak badlein jab tak trails mann-chahi lambi na dikhein.", "q1": "Mera remote shutter release kab aaya tha?", "q2": "Mera naya remote shutter release kis date ko deliver hua?", "q3": "Main dekhna chahta hoon ki mera camera remote abhi 30 din ki return window mein hai ya nahi. Woh mere paas kis date ko pahuncha tha?", "distractor_sentence": "Mere dost ka naya tripod 21 February ko aaya tha."}}, "gold": {"answer_en": "February 10", "date": {"day": 10, "month": 2}, "slots": [{"name": "date", "aliases": ["February 10", "February 10th", "10 February", "10th February", "10th of February", "Feb 10", "10 Feb", "10 फ़रवरी", "10 फरवरी"]}]}, "distractor": {"date": {"day": 21, "month": 2}, "value_aliases": ["February 21", "February 21st", "21 February", "21st February", "21st of February", "Feb 21", "21 Feb", "21 फ़रवरी", "21 फरवरी"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "b3c15d39", "lme_type": "multi-session"}, "topic": "media/books/games", "is_answerable": true, "evidence_position": 5, "filler_seed": 2285137155, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": true, "abl_q3": true, "abl_digits": true, "banned_fillers": []}
{"item_id": "E039", "cand_id": "EC054", "memory_type": "event", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "My road bike has been running great since I cleaned and lubricated the chain on March 2nd; it's been shifting smoothly and quietly! How often should I do chain maintenance?", "assistant_reply": "Wipe the chain after every few rides, and do a proper clean and lube roughly every 300 kilometres, or sooner after riding in rain or dust. Check chain wear with a simple gauge every couple of months.", "followup_turn": "Wet lube or dry lube?", "followup_reply": "Dry lube suits dry, dusty conditions and stays cleaner. Wet lube lasts longer in rain but attracts more dirt, so it needs more frequent wiping.", "q1": "When did I clean and lubricate my road bike chain?", "q2": "On what date did I last clean and oil the chain on my road bike?"}, "hi": {"memory_turn": "2 मार्च को चेन साफ़ करके तेल लगाने के बाद से मेरी रोड बाइक बहुत अच्छी चल रही है; गियर आराम से और बिना आवाज़ बदल रहे हैं! चेन की देखभाल कितनी बार करनी चाहिए?", "assistant_reply": "हर कुछ राइड के बाद चेन पोंछ लें, और लगभग हर 300 किलोमीटर पर अच्छे से साफ़ करके तेल लगाएँ, बारिश या धूल में चलाने के बाद और जल्दी। हर दो-एक महीने में एक सादे गेज से चेन की घिसावट जाँचें।", "followup_turn": "गीला ल्यूब या सूखा ल्यूब?", "followup_reply": "सूखा ल्यूब सूखे, धूल भरे मौसम के लिए ठीक है और साफ़ रहता है। गीला ल्यूब बारिश में ज़्यादा टिकता है पर गंदगी ज़्यादा पकड़ता है, इसलिए उसे बार-बार पोंछना पड़ता है।", "q1": "मैंने अपनी रोड बाइक की चेन कब साफ़ करके तेल लगाया?", "q2": "मैंने आख़िरी बार अपनी रोड बाइक की चेन किस तारीख़ को साफ़ की और तेल लगाया?"}, "hg": {"memory_turn": "2 March ko chain saaf karke oil lagane ke baad se meri road bike bahut acchi chal rahi hai; gears aaraam se aur bina awaaz badal rahe hain! Chain ki maintenance kitni baar karni chahiye?", "assistant_reply": "Har kuch rides ke baad chain pochh lein, aur lagbhag har 300 kilometre pe acche se saaf karke lube lagayein, baarish ya dhool mein chalane ke baad aur jaldi. Har do-ek mahine mein ek simple gauge se chain ki ghisaavat check karein.", "followup_turn": "Wet lube ya dry lube?", "followup_reply": "Dry lube sookhe, dhool bhare mausam ke liye theek hai aur saaf rehta hai. Wet lube baarish mein zyaada tikta hai par gandagi zyaada pakadta hai, isliye use baar-baar pochhna padta hai.", "q1": "Maine apni road bike ki chain kab saaf karke oil lagaya?", "q2": "Maine last time apni road bike ki chain kis date ko saaf ki aur oil lagaya?"}}, "gold": {"answer_en": "March 2", "date": {"day": 2, "month": 3}, "slots": [{"name": "date", "aliases": ["March 2", "March 2nd", "2 March", "2nd March", "2nd of March", "Mar 2", "2 Mar", "2 मार्च"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "a9f6b44c", "lme_type": "multi-session"}, "topic": "vehicles", "is_answerable": true, "evidence_position": 9, "filler_seed": 3102401676, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": true, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E040", "cand_id": "EC055", "memory_type": "event", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I'm really looking forward to Sunday's group ride after getting my road bike serviced at Pedal Power on March 10th; it's been running so smoothly since then! What should I bring on a 60-kilometre ride?", "assistant_reply": "Bring two water bottles, a couple of energy bars or bananas, a spare tube, tyre levers, a mini pump and a light rain jacket. A phone and a small amount of cash are useful in case of problems.", "followup_turn": "How do I keep up with a faster group?", "followup_reply": "Ride in the middle of the group to benefit from drafting, keep a steady effort instead of sprinting, and eat a little every 30 to 45 minutes before you feel hungry.", "q1": "When did I get my road bike serviced at Pedal Power?", "q2": "On what date was my road bike serviced at Pedal Power?", "q3": "The bike shop offers a free check-up 30 days after a service. From which date should I count for my road bike?", "distractor_sentence": "My friend got his bike serviced at the same shop on March 24th."}, "hi": {"memory_turn": "10 मार्च को Pedal Power में अपनी रोड बाइक की सर्विस करवाने के बाद मैं रविवार की ग्रुप राइड का बेसब्री से इंतज़ार कर रहा हूँ; तब से वह बहुत स्मूद चल रही है! 60 किलोमीटर की राइड पर क्या लेकर जाऊँ?", "assistant_reply": "पानी की दो बोतलें, दो-एक एनर्जी बार या केले, एक अतिरिक्त ट्यूब, टायर लीवर, छोटा पंप और हल्की बरसाती जैकेट ले जाएँ। किसी दिक़्क़त के लिए फ़ोन और थोड़े नक़द पैसे भी काम आते हैं।", "followup_turn": "तेज़ ग्रुप के साथ कैसे टिका रहूँ?", "followup_reply": "हवा से बचने के लिए ग्रुप के बीच में चलाएँ, अचानक ज़ोर लगाने की बजाय एक जैसी मेहनत रखें, और भूख लगने से पहले हर 30 से 45 मिनट में थोड़ा खाते रहें।", "q1": "मैंने Pedal Power में अपनी रोड बाइक की सर्विस कब करवाई?", "q2": "Pedal Power में मेरी रोड बाइक की सर्विस किस तारीख़ को हुई?", "q3": "बाइक की दुकान सर्विस के 30 दिन बाद मुफ़्त जाँच देती है। मुझे अपनी रोड बाइक के लिए किस तारीख़ से गिनना चाहिए?", "distractor_sentence": "मेरे दोस्त ने 24 मार्च को उसी दुकान से अपनी बाइक की सर्विस करवाई।"}, "hg": {"memory_turn": "10 March ko Pedal Power mein apni road bike ki service karwane ke baad main Sunday ki group ride ka besabri se intezaar kar raha hoon; tab se woh bahut smooth chal rahi hai! 60 kilometre ki ride pe kya lekar jaaun?", "assistant_reply": "Paani ki do bottles, do-ek energy bars ya kele, ek extra tube, tyre levers, chhota pump aur halki rain jacket le jaayein. Kisi problem ke liye phone aur thoda cash bhi kaam aata hai.", "followup_turn": "Fast group ke saath kaise tika rahun?", "followup_reply": "Hawa se bachne ke liye group ke beech mein chalayein, achaanak zor lagane ki bajaye steady effort rakhein, aur bhookh lagne se pehle har 30 se 45 minute mein thoda khaate rahein.", "q1": "Maine Pedal Power mein apni road bike ki service kab karwayi?", "q2": "Pedal Power mein meri road bike ki service kis date ko hui?", "q3": "Bike shop service ke 30 din baad free check-up deti hai. Mujhe apni road bike ke liye kis date se ginna chahiye?", "distractor_sentence": "Mere dost ne 24 March ko usi shop se apni bike ki service karwayi."}}, "gold": {"answer_en": "March 10", "date": {"day": 10, "month": 3}, "slots": [{"name": "date", "aliases": ["March 10", "March 10th", "10 March", "10th March", "10th of March", "Mar 10", "10 Mar", "10 मार्च"]}]}, "distractor": {"date": {"day": 24, "month": 3}, "value_aliases": ["March 24", "March 24th", "24 March", "24th March", "24th of March", "Mar 24", "24 Mar", "24 मार्च"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "a9f6b44c", "lme_type": "multi-session"}, "topic": "vehicles", "is_answerable": true, "evidence_position": 2, "filler_seed": 3958080884, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": false, "abl_q3": true, "abl_digits": true, "banned_fillers": []}
{"item_id": "F041", "cand_id": "FC156", "memory_type": "fact", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I'm currently devouring 'The Seven Husbands of Evelyn Hugo' and it's hard to put down! But I struggle to fit reading into my busy days. Any tips on making time for it?", "assistant_reply": "Keep your book with you for short waits, read ten pages before checking your phone in the morning, and swap some evening scrolling for reading in bed. Audiobooks help on commutes and chores.", "followup_turn": "Is it okay to read more than one book at a time?", "followup_reply": "Yes, many people do. A common approach is one novel for evenings and one lighter or non-fiction book for short breaks, so each suits a different mood.", "q1": "What book am I currently reading?", "q2": "Which novel am I in the middle of right now?", "q3": "A friend wants to borrow the novel that's on my bedside table at the moment. Which one is it?", "distractor_sentence": "My book club is reading Normal People this month, but I haven't started it."}, "hi": {"memory_turn": "मैं अभी 'The Seven Husbands of Evelyn Hugo' बड़े चाव से पढ़ रही हूँ और इसे छोड़ना मुश्किल है! पर व्यस्त दिनों में पढ़ने का समय निकालना मेरे लिए मुश्किल होता है। इसके लिए कोई टिप्स?", "assistant_reply": "छोटे इंतज़ार के लिए किताब साथ रखें, सुबह फ़ोन देखने से पहले दस पन्ने पढ़ें, और शाम की कुछ स्क्रॉलिंग की जगह बिस्तर में पढ़ें। सफ़र और घर के कामों में ऑडियोबुक मदद करती हैं।", "followup_turn": "क्या एक साथ एक से ज़्यादा किताबें पढ़ना ठीक है?", "followup_reply": "हाँ, बहुत लोग ऐसा करते हैं। एक आम तरीक़ा है शाम के लिए एक उपन्यास और छोटे ब्रेक के लिए कोई हल्की या नॉन-फ़िक्शन किताब, ताकि हर एक अलग मूड में फ़िट हो।", "q1": "मैं अभी कौन-सी किताब पढ़ रही हूँ?", "q2": "इस समय मैं कौन-सा उपन्यास पढ़ने के बीच में हूँ?", "q3": "एक दोस्त वह उपन्यास उधार लेना चाहती है जो अभी मेरे बिस्तर के पास रखा है। वह कौन-सा है?", "distractor_sentence": "मेरे बुक क्लब में इस महीने Normal People पढ़ी जा रही है, पर मैंने अभी शुरू नहीं की।"}, "hg": {"memory_turn": "Main abhi 'The Seven Husbands of Evelyn Hugo' bade chaav se padh rahi hoon aur ise chhodna mushkil hai! Par busy dinon mein padhne ka time nikalna mere liye mushkil hota hai. Iske liye koi tips?", "assistant_reply": "Chhote intezaar ke liye book saath rakhein, subah phone dekhne se pehle das pages padhein, aur shaam ki kuch scrolling ki jagah bed mein padhein. Commute aur ghar ke kaamon mein audiobooks help karti hain.", "followup_turn": "Kya ek saath ek se zyaada books padhna theek hai?", "followup_reply": "Haan, bahut log aisa karte hain. Ek common tareeka hai shaam ke liye ek novel aur chhote breaks ke liye koi halki ya non-fiction book, taaki har ek alag mood mein fit ho.", "q1": "Main abhi kaunsi book padh rahi hoon?", "q2": "Is time main kaunsa novel padhne ke beech mein hoon?", "q3": "Ek friend woh novel udhaar lena chahti hai jo abhi mere bed ke paas rakha hai. Woh kaunsa hai?", "distractor_sentence": "Mere book club mein is mahine Normal People padhi ja rahi hai, par maine abhi shuru nahi ki."}}, "gold": {"answer_en": "The Seven Husbands of Evelyn Hugo", "slots": [{"name": "book", "aliases": ["Seven Husbands of Evelyn Hugo", "Evelyn Hugo", "एवलिन ह्यूगो"]}]}, "distractor": {"value_aliases": ["Normal People"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "86f00804", "lme_type": "single-session-user"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 667282773, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "F042", "cand_id": "FC157", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I've started making amigurumi toys, and I recently found a stash of 17 skeins of worsted weight yarn. Can I use that yarn for amigurumi, or do I need something thinner?", "assistant_reply": "Worsted weight works well. Just use a smaller hook than the label suggests, often 3.5 to 4 mm, so the stitches stay tight and the stuffing doesn't show through.", "followup_turn": "What stuffing should I use?", "followup_reply": "Polyester fibrefill is the most common: it's light, washable and doesn't clump much. Stuff firmly, adding small pieces at a time, so the toy keeps its shape.", "q1": "How many skeins of worsted weight yarn did I find in my stash?", "q2": "What number of worsted weight skeins did I recently discover?"}, "hi": {"memory_turn": "मैंने अमिगुरुमी खिलौने बनाना शुरू किया है, और हाल ही में मुझे वर्स्टेड वेट ऊन की 17 लच्छियों का ज़ख़ीरा मिला। क्या मैं उस ऊन से अमिगुरुमी बना सकता हूँ, या पतली ऊन चाहिए?", "assistant_reply": "वर्स्टेड वेट अच्छा चलता है। बस लेबल पर लिखे से छोटा हुक इस्तेमाल करें, अक्सर 3.5 से 4 मिमी, ताकि टाँके कसे रहें और अंदर की रुई बाहर न दिखे।", "followup_turn": "भरने के लिए क्या इस्तेमाल करूँ?", "followup_reply": "पॉलिएस्टर फ़ाइबरफ़िल सबसे आम है: हल्का होता है, धुल जाता है और ज़्यादा गुच्छे नहीं बनाता। छोटे-छोटे टुकड़ों में कसकर भरें ताकि खिलौने का आकार बना रहे।", "q1": "मुझे अपने ज़ख़ीरे में वर्स्टेड वेट ऊन की कितनी लच्छियाँ मिलीं?", "q2": "हाल ही में मुझे वर्स्टेड वेट ऊन की कितनी लच्छियाँ मिली थीं?"}, "hg": {"memory_turn": "Maine amigurumi toys banana shuru kiya hai, aur haal hi mein mujhe worsted weight yarn ki 17 skeins ka stash mila. Kya main us yarn se amigurumi bana sakta hoon, ya patli yarn chahiye?", "assistant_reply": "Worsted weight accha chalta hai. Bas label pe likhe se chhota hook use karein, aksar 3.5 se 4 mm, taaki stitches tight rahein aur andar ki stuffing bahar na dikhe.", "followup_turn": "Bharne ke liye kya use karoon?", "followup_reply": "Polyester fibrefill sabse common hai: halka hota hai, dhul jaata hai aur zyaada gucche nahi banata. Chhote-chhote tukdon mein kas ke bharein taaki toy ka shape bana rahe.", "q1": "Mujhe apne stash mein worsted weight yarn ki kitni skeins mili?", "q2": "Haal hi mein mujhe worsted weight yarn ki kitni skeins mili thi?"}}, "gold": {"answer_en": "17", "slots": [{"name": "count", "aliases": ["17", "seventeen", "सत्रह", "satrah"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "8e9d538c", "lme_type": "single-session-user"}, "topic": "other", "is_answerable": true, "evidence_position": 5, "filler_seed": 3147554416, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F043", "cand_id": "FC158", "memory_type": "fact", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I've been watching a lot of educational content: I think I spent 10 hours last month watching documentaries on Netflix. Now I want to apply machine learning at work. Where should a beginner start?", "assistant_reply": "Start with Python basics, then a practical course such as Andrew Ng's Machine Learning Specialization. Pick one small problem from your own work, like predicting delays, to practise on.", "followup_turn": "Do I need a lot of maths?", "followup_reply": "Basic statistics and some linear algebra help, but you can begin with libraries like scikit-learn and learn the maths gradually as concepts come up.", "q1": "How many hours did I spend watching documentaries on Netflix last month?", "q2": "Last month, how much time did I spend on Netflix documentaries?", "q3": "For my monthly screen journal, how much time should I log for the factual films I streamed last month?", "distractor_sentence": "My brother watched about 25 hours of anime last month."}, "hi": {"memory_turn": "मैं आजकल बहुत ज्ञान वाली चीज़ें देख रही हूँ: मुझे लगता है पिछले महीने मैंने Netflix पर डॉक्यूमेंट्री देखने में 10 घंटे बिताए। अब मैं काम में मशीन लर्निंग लगाना चाहती हूँ। नौसिखिए को कहाँ से शुरू करना चाहिए?", "assistant_reply": "पहले Python की बुनियादी बातें सीखें, फिर एंड्रयू एनजी की Machine Learning Specialization जैसा प्रैक्टिकल कोर्स करें। अभ्यास के लिए अपने काम से कोई छोटी समस्या चुनें, जैसे देरी का अनुमान लगाना।", "followup_turn": "क्या बहुत ज़्यादा गणित चाहिए?", "followup_reply": "थोड़ी सांख्यिकी और कुछ लीनियर अलजेब्रा मदद करते हैं, पर आप scikit-learn जैसी लाइब्रेरी से शुरू कर सकती हैं और ज़रूरत पड़ने पर गणित धीरे-धीरे सीख सकती हैं।", "q1": "पिछले महीने मैंने Netflix पर डॉक्यूमेंट्री देखने में कितने घंटे बिताए?", "q2": "पिछले महीने Netflix की डॉक्यूमेंट्री पर मेरा कितना समय गया?", "q3": "अपनी मासिक स्क्रीन डायरी के लिए, पिछले महीने स्ट्रीम की गई तथ्य वाली फ़िल्मों का कितना समय लिखूँ?", "distractor_sentence": "मेरे भाई ने पिछले महीने लगभग 25 घंटे एनीमे देखा।"}, "hg": {"memory_turn": "Main aajkal bahut educational content dekh rahi hoon: mujhe lagta hai pichhle mahine maine Netflix pe documentaries dekhne mein 10 ghante bitaye. Ab main kaam mein machine learning lagana chahti hoon. Beginner ko kahan se shuru karna chahiye?", "assistant_reply": "Pehle Python ke basics seekhein, phir Andrew Ng ki Machine Learning Specialization jaisa practical course karein. Practice ke liye apne kaam se koi chhoti problem chunein, jaise delays predict karna.", "followup_turn": "Kya bahut zyaada maths chahiye?", "followup_reply": "Thodi statistics aur kuch linear algebra help karte hain, par aap scikit-learn jaisi libraries se shuru kar sakti hain aur zaroorat padne pe maths dheere-dheere seekh sakti hain.", "q1": "Pichhle mahine maine Netflix pe documentaries dekhne mein kitne ghante bitaye?", "q2": "Pichhle mahine Netflix ki documentaries pe mera kitna time gaya?", "q3": "Apni monthly screen diary ke liye, pichhle mahine stream ki gayi factual films ka kitna time likhun?", "distractor_sentence": "Mere bhai ne pichhle mahine lagbhag 25 ghante anime dekha."}}, "gold": {"answer_en": "about 10 hours", "slots": [{"name": "duration", "aliases": ["10 hours", "ten hours", "10 hrs", "10 घंटे", "दस घंटे", "10 ghante"]}]}, "distractor": {"value_aliases": ["25 hours", "25 घंटे", "25 ghante"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "311778f1", "lme_type": "single-session-user"}, "topic": "work/study", "is_answerable": true, "evidence_position": 9, "filler_seed": 3918439262, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "F044", "cand_id": "FC159", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I usually get home from work around 6:30 pm on weekdays, then I make dinner and do some yoga before an hour of language learning on Tuesdays and Thursdays. How can I make faster progress in Spanish?", "assistant_reply": "Add short daily listening, even ten minutes of a podcast, so you hear the language every day. Try to speak with a tutor or exchange partner once a week, and review vocabulary with spaced repetition.", "followup_turn": "Is it better to study grammar or just practise speaking?", "followup_reply": "Both matter, but speaking practice with a little focused grammar works best for most people. Use grammar to fix the mistakes you notice while speaking.", "q1": "What time do I usually get home from work on weekdays?", "q2": "At what time do I normally reach home after work on weeknights?"}, "hi": {"memory_turn": "हफ़्ते के दिनों में मैं आमतौर पर शाम लगभग 6:30 बजे काम से घर पहुँचता हूँ, फिर खाना बनाता हूँ और थोड़ा योग करता हूँ, और मंगलवार व गुरुवार को एक घंटा भाषा सीखता हूँ। स्पैनिश में जल्दी सुधार कैसे करूँ?", "assistant_reply": "रोज़ थोड़ा सुनना जोड़ें, जैसे दस मिनट का पॉडकास्ट, ताकि भाषा हर दिन कान में पड़े। हफ़्ते में एक बार किसी ट्यूटर या भाषा-साथी से बात करें, और शब्दावली को अंतराल पर दोहराने वाले तरीक़े से याद करें।", "followup_turn": "व्याकरण पढ़ना बेहतर है या सिर्फ़ बोलने का अभ्यास?", "followup_reply": "दोनों ज़रूरी हैं, पर ज़्यादातर लोगों के लिए बोलने का अभ्यास और थोड़ा केंद्रित व्याकरण सबसे अच्छा रहता है। बोलते समय जो ग़लतियाँ दिखें, उन्हें ठीक करने के लिए व्याकरण इस्तेमाल करें।", "q1": "हफ़्ते के दिनों में मैं आमतौर पर काम से कितने बजे घर आता हूँ?", "q2": "काम के दिनों में शाम को मैं सामान्यतः किस समय घर पहुँचता हूँ?"}, "hg": {"memory_turn": "Weekdays mein main usually shaam lagbhag 6:30 baje kaam se ghar pahunchta hoon, phir khaana banata hoon aur thoda yoga karta hoon, aur Tuesday aur Thursday ko ek ghanta language seekhta hoon. Spanish mein jaldi improve kaise karoon?", "assistant_reply": "Roz thoda sunna add karein, jaise das minute ka podcast, taaki language har din kaan mein pade. Hafte mein ek baar kisi tutor ya exchange partner se baat karein, aur vocabulary ko spaced repetition se revise karein.", "followup_turn": "Grammar padhna better hai ya sirf speaking practice?", "followup_reply": "Dono zaroori hain, par zyaadatar logon ke liye speaking practice aur thoda focused grammar sabse accha rehta hai. Bolte time jo galtiyan dikhein, unhe theek karne ke liye grammar use karein.", "q1": "Weekdays mein main usually kaam se kitne baje ghar aata hoon?", "q2": "Kaam ke dinon mein shaam ko main normally kis time ghar pahunchta hoon?"}}, "gold": {"answer_en": "around 6:30 pm", "slots": [{"name": "time", "aliases": ["6:30", "6.30", "साढ़े छह", "sadhe chhe", "half past six"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "c19f7a0b", "lme_type": "single-session-user"}, "topic": "food/cooking", "is_answerable": true, "evidence_position": 2, "filler_seed": 790709788, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F045", "cand_id": "FC162", "memory_type": "fact", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I'm thinking of visiting my sister Emily in Denver soon, and her kids will be with us. Do you know any kid-friendly attractions there?", "assistant_reply": "The Children's Museum and the Downtown Aquarium are big favourites, the zoo in City Park is great for a full day, and the Museum of Nature and Science has a planetarium kids love.", "followup_turn": "What's a good day trip from there with children?", "followup_reply": "Rocky Mountain National Park is about two hours away and has easy, short trails. Georgetown's old steam train ride is another fun option for families.", "q1": "Where does my sister Emily live?", "q2": "Which city is my sister Emily based in?", "distractor_sentence": "My cousin lives in Austin and keeps inviting me to visit."}, "hi": {"memory_turn": "मैं जल्द ही डेनवर में अपनी बहन एमिली से मिलने जाने का सोच रही हूँ, और उसके बच्चे भी हमारे साथ होंगे। क्या आप वहाँ बच्चों के लिए अच्छी जगहें जानते हैं?", "assistant_reply": "चिल्ड्रन्स म्यूज़ियम और डाउनटाउन एक्वेरियम बहुत पसंद किए जाते हैं, सिटी पार्क का चिड़ियाघर पूरे दिन के लिए बढ़िया है, और म्यूज़ियम ऑफ़ नेचर एंड साइंस का प्लैनेटेरियम बच्चों को बहुत भाता है।", "followup_turn": "वहाँ से बच्चों के साथ एक दिन की अच्छी सैर कौन-सी है?", "followup_reply": "रॉकी माउंटेन नेशनल पार्क करीब दो घंटे दूर है और वहाँ आसान, छोटे रास्ते हैं। जॉर्जटाउन की पुरानी भाप वाली ट्रेन की सवारी भी परिवारों के लिए मज़ेदार है।", "q1": "मेरी बहन एमिली कहाँ रहती है?", "q2": "मेरी बहन एमिली किस शहर में रहती है?", "distractor_sentence": "मेरी कज़िन ऑस्टिन में रहती है और बार-बार मुझे आने को कहती है।"}, "hg": {"memory_turn": "Main jaldi hi Denver mein apni behen Emily se milne jaane ka soch rahi hoon, aur uske bacche bhi hamare saath honge. Kya aap wahan bacchon ke liye acchi jagah jaante hain?", "assistant_reply": "Children's Museum aur Downtown Aquarium bahut pasand kiye jaate hain, City Park ka zoo poore din ke liye badhiya hai, aur Museum of Nature and Science ka planetarium bacchon ko bahut pasand aata hai.", "followup_turn": "Wahan se bacchon ke saath ek din ki acchi trip kaunsi hai?", "followup_reply": "Rocky Mountain National Park kareeb do ghante door hai aur wahan easy, chhote trails hain. Georgetown ki purani steam train ki ride bhi families ke liye mazedaar hai.", "q1": "Meri behen Emily kahan rehti hai?", "q2": "Meri behen Emily kis city mein rehti hai?", "distractor_sentence": "Meri cousin Austin mein rehti hai aur baar-baar mujhe aane ko kehti hai."}}, "gold": {"answer_en": "Denver", "slots": [{"name": "city", "aliases": ["Denver", "डेनवर"]}]}, "distractor": {"value_aliases": ["Austin", "ऑस्टिन"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "1faac195", "lme_type": "single-session-user"}, "topic": "family/social", "is_answerable": true, "evidence_position": 5, "filler_seed": 3483501393, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F046", "cand_id": "FC164", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "My new stand mixer was a birthday gift from my sister last month, and it's been a game-changer for making caramel. My caramel sauce sometimes turns grainy, though. Why does that happen?", "assistant_reply": "Graininess usually means the sugar crystallised. Stirring too early, sugar crystals on the pan's sides, or impurities can trigger it. A spoonful of corn syrup or a squeeze of lemon juice helps prevent it.", "followup_turn": "Can I fix caramel once it has gone grainy?", "followup_reply": "Often yes. Add a little water, heat it gently until the crystals dissolve, then let it cook back to the right colour without stirring.", "q1": "Who gave me my new stand mixer as a birthday gift?", "q2": "From whom did I get the stand mixer for my birthday?", "q3": "I want to send a thank-you note to whoever gave me the kitchen gadget I got for my birthday. Who should I address it to?"}, "hi": {"memory_turn": "मेरा नया स्टैंड मिक्सर पिछले महीने मेरी बहन ने जन्मदिन पर तोहफ़े में दिया था, और कैरामेल बनाने में इसने कमाल कर दिया है। पर कभी-कभी मेरा कैरामेल सॉस दानेदार हो जाता है। ऐसा क्यों होता है?", "assistant_reply": "दानेदार होने का मतलब आमतौर पर चीनी के क्रिस्टल बन जाना है। बहुत जल्दी चलाना, बर्तन के किनारों पर चीनी के दाने, या कोई गंदगी इसकी वजह बन सकती है। एक चम्मच कॉर्न सिरप या थोड़ा नींबू का रस इसे रोकने में मदद करता है।", "followup_turn": "क्या दानेदार हो चुका कैरामेल ठीक हो सकता है?", "followup_reply": "अक्सर हाँ। थोड़ा पानी डालें, धीमी आँच पर क्रिस्टल घुलने तक गर्म करें, फिर बिना चलाए दोबारा सही रंग आने तक पकाएँ।", "q1": "मुझे जन्मदिन पर नया स्टैंड मिक्सर किसने तोहफ़े में दिया?", "q2": "जन्मदिन का स्टैंड मिक्सर मुझे किससे मिला था?", "q3": "मैं उस व्यक्ति को धन्यवाद का नोट भेजना चाहता हूँ जिसने मुझे जन्मदिन पर रसोई का वह उपकरण दिया। नोट किसके नाम लिखूँ?"}, "hg": {"memory_turn": "Mera naya stand mixer pichhle mahine meri behen ne birthday pe gift kiya tha, aur caramel banane mein isne kamaal kar diya hai. Par kabhi-kabhi mera caramel sauce grainy ho jaata hai. Aisa kyun hota hai?", "assistant_reply": "Grainy hone ka matlab usually sugar ke crystals ban jaana hai. Bahut jaldi chalana, pan ke kinaron pe sugar ke daane, ya koi gandagi iski wajah ban sakti hai. Ek chammach corn syrup ya thoda lemon juice ise rokne mein help karta hai.", "followup_turn": "Kya grainy ho chuka caramel theek ho sakta hai?", "followup_reply": "Aksar haan. Thoda paani daalein, dheemi aanch pe crystals ghulne tak garam karein, phir bina chalaye dobara sahi colour aane tak pakayein.", "q1": "Mujhe birthday pe naya stand mixer kisne gift kiya?", "q2": "Birthday ka stand mixer mujhe kisse mila tha?", "q3": "Main us person ko thank you note bhejna chahta hoon jisne mujhe birthday pe kitchen ka woh gadget diya. Note kiske naam likhun?"}}, "gold": {"answer_en": "my sister", "slots": [{"name": "person", "aliases": ["sister", "बहन", "behen", "behan"]}]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "f4f1d8a4", "lme_type": "single-session-user"}, "topic": "food/cooking", "is_answerable": true, "evidence_position": 9, "filler_seed": 1856036271, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": ["FL11", "FL17", "FL29"]}
{"item_id": "F047", "cand_id": "FC165", "memory_type": "fact", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I've been using a lavender-scented shampoo that I picked up on a whim at Trader Joe's, and it's been doing wonders for my hair. Do you have tips for keeping hair healthy and shiny?", "assistant_reply": "Wash with lukewarm water, use conditioner on the ends, and rinse with cool water at the end. Limit heat styling, sleep on a silk pillowcase, and get a small trim every couple of months.", "followup_turn": "How often should I wash my hair?", "followup_reply": "It depends on your hair type. Many people do well washing every two or three days; oily hair may need it more often, while curly or dry hair often needs it less.", "q1": "Where did I buy the shampoo I currently use?", "q2": "From which store did I pick up my lavender shampoo?"}, "hi": {"memory_turn": "मैं एक लैवेंडर ख़ुशबू वाला शैम्पू इस्तेमाल कर रही हूँ जो मैंने यूँ ही Trader Joe's से उठा लिया था, और इसने मेरे बालों पर कमाल कर दिया है। बालों को स्वस्थ और चमकदार रखने के लिए कोई टिप्स?", "assistant_reply": "गुनगुने पानी से धोएँ, कंडीशनर सिरों पर लगाएँ, और आख़िर में ठंडे पानी से धो लें। गर्म स्टाइलिंग कम करें, रेशमी तकिये पर सोएँ, और हर दो-एक महीने में थोड़े बाल कटवाएँ।", "followup_turn": "मुझे बाल कितनी बार धोने चाहिए?", "followup_reply": "यह बालों के प्रकार पर निर्भर करता है। कई लोगों के लिए हर दो-तीन दिन में धोना ठीक रहता है; तैलीय बालों को ज़्यादा बार और घुँघराले या रूखे बालों को अक्सर कम बार धोना पड़ता है।", "q1": "जो शैम्पू मैं अभी इस्तेमाल कर रही हूँ, वह मैंने कहाँ से लिया?", "q2": "मैंने अपना लैवेंडर शैम्पू किस दुकान से ख़रीदा था?"}, "hg": {"memory_turn": "Main ek lavender khushbu wala shampoo use kar rahi hoon jo maine yunhi Trader Joe's se utha liya tha, aur isne mere baalon pe kamaal kar diya hai. Baalon ko healthy aur shiny rakhne ke liye koi tips?", "assistant_reply": "Gunguna paani se dhoyein, conditioner ends pe lagayein, aur end mein thande paani se dho lein. Heat styling kam karein, silk pillowcase pe soyein, aur har do-ek mahine mein thode baal trim karwayein.", "followup_turn": "Mujhe baal kitni baar dhone chahiye?", "followup_reply": "Yeh baalon ke type pe depend karta hai. Kai logon ke liye har do-teen din mein dhona theek rehta hai; oily baalon ko zyaada baar aur curly ya dry baalon ko aksar kam baar dhona padta hai.", "q1": "Jo shampoo main abhi use kar rahi hoon, woh maine kahan se liya?", "q2": "Maine apna lavender shampoo kis store se khareeda tha?"}}, "gold": {"answer_en": "Trader Joe's", "slots": [{"name": "store", "aliases": ["Trader Joe's", "Trader Joe", "Trader Joes", "ट्रेडर जोज़"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "c14c00dd", "lme_type": "single-session-user"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 2400481851, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "F048", "cand_id": "FC166", "memory_type": "fact", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I recently had a bad case of bronchitis that I first thought was just a cold, but it turned out to be much more serious. Are there foods or spices that can help support my immune system?", "assistant_reply": "A varied diet matters most: plenty of fruit and vegetables, protein, and enough sleep. Ginger, garlic and turmeric are pleasant additions, and warm soups help when you're recovering.", "followup_turn": "Should I take vitamin supplements as well?", "followup_reply": "Only if a doctor finds a deficiency, for example in vitamin D. For most people, food covers their needs, and high doses can sometimes cause harm.", "q1": "What health issue did I first think was just a cold?", "q2": "Which illness did I recently have that I mistook for a common cold?", "q3": "My new doctor's form asks about any chest infection I had recently. What should I write?", "distractor_sentence": "My colleague had pneumonia last winter and was off work for weeks."}, "hi": {"memory_turn": "हाल ही में मुझे ब्रोंकाइटिस हुआ था जिसे मैंने पहले साधारण ज़ुकाम समझा, पर वह कहीं ज़्यादा गंभीर निकला। क्या कुछ खाने की चीज़ें या मसाले हैं जो रोग प्रतिरोधक क्षमता में मदद करें?", "assistant_reply": "सबसे ज़रूरी है अलग-अलग तरह का खाना: ख़ूब फल और सब्ज़ियाँ, प्रोटीन, और पूरी नींद। अदरक, लहसुन और हल्दी अच्छे साथी हैं, और ठीक होते समय गर्म सूप मदद करता है।", "followup_turn": "क्या विटामिन की गोलियाँ भी लेनी चाहिए?", "followup_reply": "सिर्फ़ तब, जब डॉक्टर को कोई कमी मिले, जैसे विटामिन D की। ज़्यादातर लोगों की ज़रूरत खाने से पूरी हो जाती है, और ज़्यादा ख़ुराक कभी-कभी नुक़सान कर सकती है।", "q1": "मैंने पहले किस बीमारी को साधारण ज़ुकाम समझा था?", "q2": "हाल में मुझे कौन-सी बीमारी हुई जिसे मैंने आम सर्दी-ज़ुकाम समझ लिया?", "q3": "मेरे नए डॉक्टर के फ़ॉर्म में हाल के किसी छाती के संक्रमण के बारे में पूछा गया है। मुझे क्या लिखना चाहिए?", "distractor_sentence": "मेरे एक साथी को पिछली सर्दियों में निमोनिया हुआ था और वह हफ़्तों छुट्टी पर रहा।"}, "hg": {"memory_turn": "Haal hi mein mujhe bronchitis hua tha jise maine pehle normal cold samjha, par woh kahin zyaada serious nikla. Kya kuch khaane ki cheezein ya masale hain jo immunity mein help karein?", "assistant_reply": "Sabse zaroori hai alag-alag tarah ka khaana: khoob fruits aur sabziyan, protein, aur poori neend. Adrak, lehsun aur haldi acche saathi hain, aur theek hote time garam soup help karta hai.", "followup_turn": "Kya vitamin ki goliyan bhi leni chahiye?", "followup_reply": "Sirf tab, jab doctor ko koi kami mile, jaise vitamin D ki. Zyaadatar logon ki zaroorat khaane se poori ho jaati hai, aur zyaada dose kabhi-kabhi nuksaan kar sakti hai.", "q1": "Maine pehle kis bimaari ko normal cold samjha tha?", "q2": "Haal mein mujhe kaunsi bimaari hui jise maine aam sardi-zukaam samajh liya?", "q3": "Mere naye doctor ke form mein haal ke kisi chest infection ke baare mein poocha gaya hai. Mujhe kya likhna chahiye?", "distractor_sentence": "Mere ek colleague ko pichhli sardiyon mein pneumonia hua tha aur woh hafton chhutti pe raha."}}, "gold": {"answer_en": "bronchitis", "slots": [{"name": "illness", "aliases": ["bronchitis", "ब्रोंकाइटिस", "ब्रॉन्काइटिस"]}]}, "distractor": {"value_aliases": ["pneumonia", "निमोनिया"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "36580ce8", "lme_type": "single-session-user"}, "topic": "fitness/health", "is_answerable": true, "evidence_position": 5, "filler_seed": 590166416, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "F049", "cand_id": "FC167", "memory_type": "fact", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I'm filling out the 'How We Met' part of my friendship journal. For Sophia, it was at a coffee shop in the city. What other sections would make the journal more meaningful?", "assistant_reply": "Add a favourite memory together, an inside joke, something you've learned from each person, and a short note about what you'd like to do together next. Photos or ticket stubs bring it to life.", "followup_turn": "Should I show the journal to my friends?", "followup_reply": "It's up to you. Some people share a page as a birthday gift, which can be very touching. Keeping it private lets you write more honestly.", "q1": "Where did I meet Sophia?", "q2": "At what place did I first meet my friend Sophia?"}, "hi": {"memory_turn": "मैं अपनी दोस्ती वाली डायरी में 'हम कैसे मिले' वाला हिस्सा भर रही हूँ। सोफ़िया से मैं शहर के एक कॉफ़ी शॉप में मिली थी। डायरी को और ख़ास बनाने के लिए कौन-से हिस्से जोड़ूँ?", "assistant_reply": "साथ की कोई पसंदीदा याद, कोई अंदर का मज़ाक़, हर व्यक्ति से सीखी कोई बात, और आगे साथ में क्या करना चाहती हैं इस पर छोटा नोट जोड़ें। फ़ोटो या टिकट के टुकड़े इसे जीवंत बना देते हैं।", "followup_turn": "क्या मुझे यह डायरी अपने दोस्तों को दिखानी चाहिए?", "followup_reply": "यह आप पर है। कुछ लोग जन्मदिन के तोहफ़े के तौर पर एक पन्ना शेयर करते हैं, जो बहुत दिल छूने वाला हो सकता है। निजी रखने से आप ज़्यादा ईमानदारी से लिख पाती हैं।", "q1": "मैं सोफ़िया से कहाँ मिली थी?", "q2": "अपनी दोस्त सोफ़िया से मैं पहली बार किस जगह मिली?"}, "hg": {"memory_turn": "Main apni friendship journal mein 'How We Met' wala hissa bhar rahi hoon. Sophia se main sheher ke ek coffee shop mein mili thi. Journal ko aur special banane ke liye kaunse sections add karoon?", "assistant_reply": "Saath ki koi favourite memory, koi inside joke, har person se seekhi koi baat, aur aage saath mein kya karna chahti hain is pe chhota note add karein. Photos ya ticket stubs ise lively bana dete hain.", "followup_turn": "Kya mujhe yeh journal apne doston ko dikhani chahiye?", "followup_reply": "Yeh aap pe hai. Kuch log birthday gift ke taur pe ek page share karte hain, jo bahut touching ho sakta hai. Private rakhne se aap zyaada honestly likh paati hain.", "q1": "Main Sophia se kahan mili thi?", "q2": "Apni friend Sophia se main pehli baar kis jagah mili?"}}, "gold": {"answer_en": "a coffee shop in the city", "slots": [{"name": "place", "aliases": ["coffee shop", "café", "cafe", "कॉफ़ी शॉप", "कॉफी शॉप", "कैफ़े", "कैफे"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "3d86fd0a", "lme_type": "single-session-user"}, "topic": "food/cooking", "is_answerable": true, "evidence_position": 9, "filler_seed": 340506979, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": ["FL37"]}
{"item_id": "F050", "cand_id": "FC168", "memory_type": "fact", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "Speaking of gaming, I finally beat the last boss in the Dark Souls 3 DLC last weekend after weeks of trying! I'm planning a new PC build in the higher price range. Which graphics card should I aim for?", "assistant_reply": "In the higher range, an Nvidia RTX 4070 Super or an AMD RX 7900 GRE gives smooth 1440p gaming with room to spare. Pair it with a quality 750-watt power supply.", "followup_turn": "How much RAM do I need for gaming?", "followup_reply": "32 GB is a comfortable amount today and leaves headroom for streaming or browser tabs in the background. Fast dual-channel memory matters more than going beyond that.", "q1": "What game did I finally beat last weekend?", "q2": "Which game's final boss did I defeat last weekend?", "distractor_sentence": "My roommate just finished Elden Ring and won't stop talking about it."}, "hi": {"memory_turn": "गेमिंग की बात चली तो, हफ़्तों की कोशिश के बाद पिछले वीकेंड आख़िरकार मैंने Dark Souls 3 DLC का आख़िरी बॉस हरा दिया! मैं ऊँचे बजट में एक नया PC बनाने की सोच रहा हूँ। मुझे कौन-सा ग्राफ़िक्स कार्ड लेना चाहिए?", "assistant_reply": "ऊँचे बजट में Nvidia RTX 4070 Super या AMD RX 7900 GRE आराम से 1440p पर स्मूद गेमिंग देते हैं। इसके साथ अच्छी क्वालिटी की 750-वॉट पावर सप्लाई लगाएँ।", "followup_turn": "गेमिंग के लिए कितनी RAM चाहिए?", "followup_reply": "आजकल 32 GB आरामदायक है और बैकग्राउंड में स्ट्रीमिंग या ब्राउज़र टैब के लिए भी जगह रहती है। इससे ज़्यादा लेने से ज़्यादा ज़रूरी तेज़ ड्यूल-चैनल मेमोरी है।", "q1": "पिछले वीकेंड मैंने आख़िरकार कौन-सा गेम पूरा किया?", "q2": "पिछले वीकेंड मैंने किस गेम का आख़िरी बॉस हराया?", "distractor_sentence": "मेरे रूममेट ने अभी Elden Ring ख़त्म किया है और उसके बारे में बात करना बंद ही नहीं करता।"}, "hg": {"memory_turn": "Gaming ki baat chali toh, hafton ki koshish ke baad pichhle weekend aakhirkaar maine Dark Souls 3 DLC ka last boss hara diya! Main high budget mein ek naya PC build karne ka soch raha hoon. Mujhe kaunsa graphics card lena chahiye?", "assistant_reply": "High budget mein Nvidia RTX 4070 Super ya AMD RX 7900 GRE aaraam se 1440p pe smooth gaming dete hain. Iske saath acchi quality ki 750-watt power supply lagayein.", "followup_turn": "Gaming ke liye kitni RAM chahiye?", "followup_reply": "Aajkal 32 GB comfortable hai aur background mein streaming ya browser tabs ke liye bhi jagah rehti hai. Isse zyaada lene se zyaada zaroori fast dual-channel memory hai.", "q1": "Pichhle weekend maine aakhirkaar kaunsa game complete kiya?", "q2": "Pichhle weekend maine kis game ka last boss haraya?", "distractor_sentence": "Mere roommate ne abhi Elden Ring khatam kiya hai aur uske baare mein baat karna band hi nahi karta."}}, "gold": {"answer_en": "the Dark Souls 3 DLC", "slots": [{"name": "game", "aliases": ["Dark Souls 3", "Dark Souls III", "Dark Souls", "डार्क सोल्स"]}]}, "distractor": {"value_aliases": ["Elden Ring"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "a82c026e", "lme_type": "single-session-user"}, "topic": "work/study", "is_answerable": true, "evidence_position": 2, "filler_seed": 1786985805, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P041", "cand_id": "PX0702", "memory_type": "preference", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I'm a huge fan of LeBron James, though I've also been impressed with Dennis Schroder lately. I want to start playing basketball myself. What should a beginner work on first?", "assistant_reply": "Start with ball handling using both hands, a consistent shooting form close to the basket, and basic footwork like pivots and layups. Short daily sessions build skill faster than rare long ones.", "followup_turn": "Do I need special shoes?", "followup_reply": "Basketball shoes help, because they support your ankles and grip the court well. Running shoes are not built for quick side-to-side moves.", "q1": "Which basketball player am I a huge fan of?", "q2": "Who is my favourite basketball player?", "q3": "I'm ordering a jersey with my favourite player's name printed on the back. Whose name should it have?", "distractor_sentence": "My brother thinks Stephen Curry is the greatest of all time."}, "hi": {"memory_turn": "मैं लेब्रॉन जेम्स की बहुत बड़ी फ़ैन हूँ, हालाँकि हाल में डेनिस श्रोडर ने भी मुझे प्रभावित किया है। मैं ख़ुद बास्केटबॉल खेलना शुरू करना चाहती हूँ। नौसिखिए को पहले किस पर काम करना चाहिए?", "assistant_reply": "दोनों हाथों से गेंद सँभालने, बास्केट के पास एक जैसा शूटिंग तरीक़ा, और पिवट व लेअप जैसे बुनियादी फ़ुटवर्क से शुरू करें। कभी-कभार लंबे सेशन से रोज़ के छोटे सेशन जल्दी हुनर बनाते हैं।", "followup_turn": "क्या मुझे ख़ास जूते चाहिए?", "followup_reply": "बास्केटबॉल के जूते मदद करते हैं, क्योंकि वे टखनों को सहारा देते हैं और कोर्ट पर अच्छी पकड़ बनाते हैं। रनिंग शूज़ तेज़ी से दाएँ-बाएँ हिलने के लिए नहीं बने।", "q1": "मैं किस बास्केटबॉल खिलाड़ी की बहुत बड़ी फ़ैन हूँ?", "q2": "मेरा पसंदीदा बास्केटबॉल खिलाड़ी कौन है?", "q3": "मैं एक जर्सी मँगवा रही हूँ जिसकी पीठ पर मेरे पसंदीदा खिलाड़ी का नाम छपा हो। उस पर किसका नाम होना चाहिए?", "distractor_sentence": "मेरे भाई को लगता है कि स्टीफ़न करी अब तक के सबसे महान खिलाड़ी हैं।"}, "hg": {"memory_turn": "Main LeBron James ki bahut badi fan hoon, halanki haal mein Dennis Schroder ne bhi mujhe impress kiya hai. Main khud basketball khelna shuru karna chahti hoon. Beginner ko pehle kis pe kaam karna chahiye?", "assistant_reply": "Dono haathon se ball handling, basket ke paas ek jaisa shooting form, aur pivots aur layups jaisa basic footwork se shuru karein. Kabhi-kabhar lambe sessions se roz ke chhote sessions jaldi skill banate hain.", "followup_turn": "Kya mujhe special shoes chahiye?", "followup_reply": "Basketball shoes help karte hain, kyunki woh ankles ko support dete hain aur court pe acchi grip banate hain. Running shoes tezi se side-to-side move karne ke liye nahi bane.", "q1": "Main kis basketball player ki bahut badi fan hoon?", "q2": "Mera favourite basketball player kaun hai?", "q3": "Main ek jersey mangwa rahi hoon jiski peeth pe mere favourite player ka naam chhapa ho. Us pe kiska naam hona chahiye?", "distractor_sentence": "Mere bhai ko lagta hai ki Stephen Curry ab tak ke sabse great player hain."}}, "gold": {"answer_en": "LeBron James", "slots": [{"name": "player", "aliases": ["LeBron James", "LeBron", "लेब्रॉन जेम्स", "लेब्रॉन", "लेब्रोन"]}]}, "distractor": {"value_aliases": ["Stephen Curry", "Curry", "स्टीफ़न करी", "करी"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 2, "filler_seed": 1804102774, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "P042", "cand_id": "PX0753", "memory_type": "preference", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I prefer milder weather and fewer crowds, so spring is the perfect time for me to visit Greece. Which islands would you suggest for a first trip?", "assistant_reply": "Naxos is great for beaches, villages and food, Crete has history and gorges to hike, and Milos is known for unusual rock formations. Santorini is beautiful but busy, so a short stop there is enough.", "followup_turn": "How do I get between the islands?", "followup_reply": "Ferries connect most islands, and high-speed boats save time on longer routes. Book popular crossings a few days in advance, and allow a buffer day in case of strong winds.", "q1": "In which season do I prefer to visit Greece?", "q2": "What time of year do I like to travel to Greece?"}, "hi": {"memory_turn": "मुझे हल्का मौसम और कम भीड़ पसंद है, इसलिए ग्रीस घूमने के लिए वसंत मेरे लिए सबसे सही समय है। पहली ट्रिप के लिए आप कौन-से द्वीप सुझाएँगे?", "assistant_reply": "समुद्र तटों, गाँवों और खाने के लिए नक्सोस बढ़िया है, क्रीट में इतिहास और पैदल घूमने के लिए घाटियाँ हैं, और मिलोस अनोखी चट्टानों के लिए मशहूर है। सैंटोरिनी ख़ूबसूरत है पर भीड़ भरा, इसलिए वहाँ थोड़ा रुकना काफ़ी है।", "followup_turn": "द्वीपों के बीच कैसे आना-जाना करूँ?", "followup_reply": "ज़्यादातर द्वीप फ़ेरी से जुड़े हैं, और लंबे रास्तों पर तेज़ नावें समय बचाती हैं। मशहूर रास्तों की बुकिंग कुछ दिन पहले कर लें, और तेज़ हवा के लिए एक दिन अतिरिक्त रखें।", "q1": "मुझे किस मौसम में ग्रीस जाना पसंद है?", "q2": "साल के किस समय मुझे ग्रीस घूमना अच्छा लगता है?"}, "hg": {"memory_turn": "Mujhe halka mausam aur kam bheed pasand hai, isliye Greece ghoomne ke liye spring mere liye sabse sahi time hai. Pehli trip ke liye aap kaunse islands suggest karenge?", "assistant_reply": "Beaches, gaon aur khaane ke liye Naxos badhiya hai, Crete mein history aur hiking ke liye gorges hain, aur Milos anokhi rock formations ke liye famous hai. Santorini khoobsurat hai par bheed bhara, isliye wahan thoda rukna kaafi hai.", "followup_turn": "Islands ke beech kaise aana-jaana karoon?", "followup_reply": "Zyaadatar islands ferries se jude hain, aur lambe routes pe fast boats time bachati hain. Popular routes ki booking kuch din pehle kar lein, aur tez hawa ke liye ek din extra rakhein.", "q1": "Mujhe kis season mein Greece jaana pasand hai?", "q2": "Saal ke kis time mujhe Greece ghoomna accha lagta hai?"}}, "gold": {"answer_en": "spring (milder weather, fewer crowds)", "slots": [{"name": "season", "aliases": ["spring", "वसंत", "बसंत", "basant"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 5, "filler_seed": 3314719855, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P043", "cand_id": "PX0780", "memory_type": "preference", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I never worry about lunch on Fridays, because I meet my friends at a nearby café where I always order their vegetarian wrap with sweet potato fries. For the other weekdays, what easy lunches can I pack?", "assistant_reply": "Try a grain bowl with chickpeas and roasted vegetables, pasta salad with pesto, or a big batch of soup portioned into jars. Make two recipes on Sunday and alternate them through the week.", "followup_turn": "How do I keep packed salads from getting soggy?", "followup_reply": "Put the dressing at the bottom of the jar, then sturdy vegetables, grains, and leafy greens on top. Shake it into a bowl just before eating.", "q1": "What do I always order at the café on Fridays?", "q2": "What's my usual Friday lunch order when I meet my friends?"}, "hi": {"memory_turn": "शुक्रवार को मुझे दोपहर के खाने की चिंता नहीं होती, क्योंकि मैं पास के एक कैफ़े में दोस्तों से मिलती हूँ और हमेशा उनका वेजिटेरियन रैप शकरकंद फ़्राइज़ के साथ मँगवाती हूँ। बाक़ी दिनों के लिए कौन-से आसान लंच पैक कर सकती हूँ?", "assistant_reply": "छोले और भुनी सब्ज़ियों वाला ग्रेन बाउल, पेस्टो वाला पास्ता सलाद, या सूप की बड़ी मात्रा बनाकर जार में बाँट लें। रविवार को दो रेसिपी बनाएँ और हफ़्ते भर बारी-बारी से खाएँ।", "followup_turn": "पैक किया सलाद गीला होने से कैसे बचाऊँ?", "followup_reply": "ड्रेसिंग जार में सबसे नीचे डालें, फिर कड़ी सब्ज़ियाँ, अनाज, और सबसे ऊपर हरी पत्तियाँ। खाने से ठीक पहले हिलाकर कटोरे में निकालें।", "q1": "शुक्रवार को कैफ़े में मैं हमेशा क्या मँगवाती हूँ?", "q2": "दोस्तों से मिलते समय शुक्रवार के लंच में मेरा रोज़ का ऑर्डर क्या होता है?"}, "hg": {"memory_turn": "Friday ko mujhe lunch ki chinta nahi hoti, kyunki main paas ke ek cafe mein doston se milti hoon aur hamesha unka vegetarian wrap sweet potato fries ke saath order karti hoon. Baaki dinon ke liye kaunse easy lunch pack kar sakti hoon?", "assistant_reply": "Chhole aur roasted veggies wala grain bowl, pesto wala pasta salad, ya soup ka bada batch bana ke jars mein baant lein. Sunday ko do recipes banayein aur hafte bhar baari-baari se khaayein.", "followup_turn": "Pack kiya salad geela hone se kaise bachaun?", "followup_reply": "Dressing jar mein sabse neeche daalein, phir sakht sabziyan, grains, aur sabse upar hari pattiyan. Khaane se theek pehle hila ke bowl mein nikaalein.", "q1": "Friday ko cafe mein main hamesha kya order karti hoon?", "q2": "Doston se milte time Friday ke lunch mein mera usual order kya hota hai?"}}, "gold": {"answer_en": "the vegetarian wrap with sweet potato fries", "slots": [{"name": "wrap", "aliases": ["vegetarian wrap", "veggie wrap", "veg wrap", "वेजिटेरियन रैप", "वेज रैप"]}, {"name": "fries", "aliases": ["sweet potato fries", "sweet potato", "शकरकंद फ़्राइज़", "शकरकंद"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 9, "filler_seed": 935204485, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P044", "cand_id": "PX0822", "memory_type": "preference", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I prefer lightweight moisturisers with SPF and hyaluronic acid. I've just started a proper skincare routine. In what order should I apply things in the morning?", "assistant_reply": "Go from thinnest to thickest: cleanse, then any serum, then your moisturiser. If your moisturiser already protects from the sun well enough, that can be your last step; otherwise add sunscreen on top.", "followup_turn": "Do I need a different routine at night?", "followup_reply": "At night you can skip sun protection and use a slightly richer cream if your skin feels dry. Cleansing well at night is the most important step.", "q1": "What kind of moisturiser do I prefer?", "q2": "Which features do I look for in a moisturiser?", "q3": "I'm restocking at the pharmacy. What two things should I check on a face cream label so it matches what I like?", "distractor_sentence": "My wife prefers a thick night cream with retinol."}, "hi": {"memory_turn": "मुझे SPF और हयालूरोनिक एसिड वाले हल्के मॉइस्चराइज़र पसंद हैं। मैंने अभी-अभी त्वचा की देखभाल का सही रूटीन शुरू किया है। सुबह चीज़ें किस क्रम में लगानी चाहिए?", "assistant_reply": "सबसे पतली से सबसे गाढ़ी चीज़ की ओर चलें: पहले चेहरा धोएँ, फिर कोई सीरम, फिर मॉइस्चराइज़र। अगर आपका मॉइस्चराइज़र धूप से पूरी सुरक्षा देता है तो वही आख़िरी क़दम हो सकता है; वरना ऊपर से सनस्क्रीन लगाएँ।", "followup_turn": "क्या रात में अलग रूटीन चाहिए?", "followup_reply": "रात में धूप से सुरक्षा की ज़रूरत नहीं, और त्वचा रूखी लगे तो थोड़ी गाढ़ी क्रीम लगा सकते हैं। रात में अच्छे से चेहरा साफ़ करना सबसे ज़रूरी क़दम है।", "q1": "मुझे किस तरह का मॉइस्चराइज़र पसंद है?", "q2": "मॉइस्चराइज़र में मैं कौन-सी ख़ूबियाँ देखता हूँ?", "q3": "मैं दवा की दुकान से सामान दोबारा ले रहा हूँ। फ़ेस क्रीम के लेबल पर कौन-सी दो चीज़ें देखूँ ताकि वह मेरी पसंद से मेल खाए?", "distractor_sentence": "मेरी पत्नी को रेटिनॉल वाली गाढ़ी नाइट क्रीम पसंद है।"}, "hg": {"memory_turn": "Mujhe SPF aur hyaluronic acid wale lightweight moisturisers pasand hain. Maine abhi-abhi proper skincare routine shuru kiya hai. Subah cheezein kis order mein lagani chahiye?", "assistant_reply": "Sabse patli se sabse gaadhi cheez ki taraf chalein: pehle face wash, phir koi serum, phir moisturiser. Agar aapka moisturiser dhoop se poori protection deta hai toh wahi last step ho sakta hai; warna upar se sunscreen lagayein.", "followup_turn": "Kya raat mein alag routine chahiye?", "followup_reply": "Raat mein dhoop se protection ki zaroorat nahi, aur skin dry lage toh thodi gaadhi cream laga sakte hain. Raat mein acche se face saaf karna sabse zaroori step hai.", "q1": "Mujhe kis type ka moisturiser pasand hai?", "q2": "Moisturiser mein main kaunsi khoobiyan dekhta hoon?", "q3": "Main pharmacy se saamaan dobara le raha hoon. Face cream ke label pe kaunsi do cheezein dekhun taaki woh meri pasand se match kare?", "distractor_sentence": "Meri wife ko retinol wali gaadhi night cream pasand hai."}}, "gold": {"answer_en": "lightweight, with SPF and hyaluronic acid", "slots": [{"name": "spf", "aliases": ["SPF", "एसपीएफ़"]}, {"name": "hyaluronic", "aliases": ["hyaluronic", "हयालूरोनिक", "हायलूरोनिक"]}]}, "distractor": {"value_aliases": ["retinol", "रेटिनॉल"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 2, "filler_seed": 577508021, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "P045", "cand_id": "PX0861", "memory_type": "preference", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I think my favourite type of exhibit is Impressionist art; I loved the 'Impressionist Masters' show at the city art museum a few weeks ago. Can you suggest books to learn more about painting?", "assistant_reply": "E.H. Gombrich's 'The Story of Art' is a classic overview, and 'Ways of Seeing' by John Berger changes how you look at pictures. For something lighter, 'What Are You Looking At?' by Will Gompertz is fun.", "followup_turn": "Are there good online courses too?", "followup_reply": "Yes. Many museums post free lectures, and platforms like Smarthistory and Coursera have short art history courses you can follow at your own pace.", "q1": "Which type of art exhibit do I like most?", "q2": "What style of art is my favourite to see in exhibitions?", "distractor_sentence": "My friend only cares about contemporary installation art."}, "hi": {"memory_turn": "मुझे लगता है मेरी पसंदीदा प्रदर्शनी इम्प्रेशनिस्ट कला की होती है; कुछ हफ़्ते पहले शहर के आर्ट म्यूज़ियम में 'Impressionist Masters' शो मुझे बहुत अच्छा लगा। पेंटिंग के बारे में और जानने के लिए कुछ किताबें बताइए?", "assistant_reply": "ई.एच. गोम्ब्रिच की 'The Story of Art' एक क्लासिक परिचय है, और जॉन बर्जर की 'Ways of Seeing' तस्वीरें देखने का नज़रिया बदल देती है। कुछ हल्का चाहिए तो विल गोम्पर्ट्ज़ की 'What Are You Looking At?' मज़ेदार है।", "followup_turn": "क्या अच्छे ऑनलाइन कोर्स भी हैं?", "followup_reply": "हाँ। कई म्यूज़ियम मुफ़्त लेक्चर डालते हैं, और Smarthistory और Coursera जैसे प्लेटफ़ॉर्म पर कला इतिहास के छोटे कोर्स हैं जिन्हें आप अपनी रफ़्तार से कर सकती हैं।", "q1": "मुझे किस तरह की कला प्रदर्शनी सबसे ज़्यादा पसंद है?", "q2": "प्रदर्शनियों में मुझे कला की कौन-सी शैली देखना सबसे अच्छा लगता है?", "distractor_sentence": "मेरी दोस्त को सिर्फ़ कंटेम्पररी इंस्टॉलेशन आर्ट में दिलचस्पी है।"}, "hg": {"memory_turn": "Mujhe lagta hai meri favourite exhibition Impressionist art ki hoti hai; kuch hafte pehle sheher ke art museum mein 'Impressionist Masters' show mujhe bahut accha laga. Painting ke baare mein aur jaanne ke liye kuch books bataiye?", "assistant_reply": "E.H. Gombrich ki 'The Story of Art' ek classic overview hai, aur John Berger ki 'Ways of Seeing' pictures dekhne ka nazariya badal deti hai. Kuch halka chahiye toh Will Gompertz ki 'What Are You Looking At?' mazedaar hai.", "followup_turn": "Kya acche online courses bhi hain?", "followup_reply": "Haan. Kai museums free lectures daalte hain, aur Smarthistory aur Coursera jaise platforms pe art history ke chhote courses hain jinhe aap apni speed se kar sakti hain.", "q1": "Mujhe kis type ki art exhibition sabse zyaada pasand hai?", "q2": "Exhibitions mein mujhe art ka kaunsa style dekhna sabse accha lagta hai?", "distractor_sentence": "Meri friend ko sirf contemporary installation art mein interest hai."}}, "gold": {"answer_en": "Impressionist art", "slots": [{"name": "style", "aliases": ["Impressionist", "Impressionism", "इम्प्रेशनिस्ट", "इंप्रेशनिस्ट", "प्रभाववादी"]}]}, "distractor": {"value_aliases": ["contemporary", "कंटेम्पररी"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 5, "filler_seed": 90121988, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P046", "cand_id": "PX0927", "memory_type": "preference", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I've been learning R for data analysis, and one of my favourite libraries is ggplot2; I love how easy it makes good-looking plots. Which other R packages should I learn next?", "assistant_reply": "dplyr and tidyr for cleaning and reshaping data, readr for fast file import, and lubridate for dates. For reports, R Markdown or Quarto let you combine code, results and text in one document.", "followup_turn": "Is R or Python better for data analysis?", "followup_reply": "Both are strong. R shines for statistics and quick analysis, while Python is more general-purpose and common in machine learning. Many analysts use both.", "q1": "Which R plotting library do I like most?", "q2": "What is my favourite R package for making charts?", "q3": "I need to make charts for my report in R this afternoon. Which package should I reach for?"}, "hi": {"memory_turn": "मैं डेटा एनालिसिस के लिए R सीख रहा हूँ, और मेरी पसंदीदा लाइब्रेरी में से एक ggplot2 है; इससे सुंदर ग्राफ़ बनाना कितना आसान है, यह मुझे बहुत पसंद है। आगे मुझे R के कौन-से पैकेज सीखने चाहिए?", "assistant_reply": "डेटा साफ़ करने और ढाँचा बदलने के लिए dplyr और tidyr, तेज़ फ़ाइल इम्पोर्ट के लिए readr, और तारीख़ों के लिए lubridate सीखें। रिपोर्ट के लिए R Markdown या Quarto से कोड, नतीजे और टेक्स्ट एक ही दस्तावेज़ में आ जाते हैं।", "followup_turn": "डेटा एनालिसिस के लिए R बेहतर है या Python?", "followup_reply": "दोनों मज़बूत हैं। आँकड़ों और तेज़ विश्लेषण में R चमकता है, जबकि Python ज़्यादा आम-उपयोगी है और मशीन लर्निंग में ज़्यादा चलता है। कई एनालिस्ट दोनों इस्तेमाल करते हैं।", "q1": "R की कौन-सी प्लॉटिंग लाइब्रेरी मुझे सबसे ज़्यादा पसंद है?", "q2": "चार्ट बनाने के लिए R का मेरा पसंदीदा पैकेज कौन-सा है?", "q3": "आज दोपहर मुझे अपनी रिपोर्ट के लिए R में चार्ट बनाने हैं। मुझे कौन-सा पैकेज इस्तेमाल करना चाहिए?"}, "hg": {"memory_turn": "Main data analysis ke liye R seekh raha hoon, aur meri favourite libraries mein se ek ggplot2 hai; isse sundar plots banana kitna easy hai, yeh mujhe bahut pasand hai. Aage mujhe R ke kaunse packages seekhne chahiye?", "assistant_reply": "Data clean karne aur reshape karne ke liye dplyr aur tidyr, fast file import ke liye readr, aur dates ke liye lubridate seekhein. Reports ke liye R Markdown ya Quarto se code, results aur text ek hi document mein aa jaate hain.", "followup_turn": "Data analysis ke liye R better hai ya Python?", "followup_reply": "Dono strong hain. Statistics aur quick analysis mein R chamakta hai, jabki Python zyaada general-purpose hai aur machine learning mein zyaada chalta hai. Kai analysts dono use karte hain.", "q1": "R ki kaunsi plotting library mujhe sabse zyaada pasand hai?", "q2": "Charts banane ke liye R ka mera favourite package kaunsa hai?", "q3": "Aaj dopahar mujhe apni report ke liye R mein charts banane hain. Mujhe kaunsa package use karna chahiye?"}}, "gold": {"answer_en": "ggplot2", "slots": [{"name": "library", "aliases": ["ggplot2", "ggplot"]}]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 9, "filler_seed": 2666593875, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": false, "abl_q3": true, "abl_digits": false, "banned_fillers": []}
{"item_id": "P047", "cand_id": "PX1226", "memory_type": "preference", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "One of my favourite authors on meditation is Dan Harris; his book '10% Happier' explains mindfulness in such a relatable way. How can I build a daily meditation habit?", "assistant_reply": "Start with five minutes at the same time each day, right after something you already do, like brushing your teeth. Use a guided app at first, and don't worry about a busy mind; noticing it is the practice.", "followup_turn": "Is it better to meditate in the morning or at night?", "followup_reply": "Morning suits many people because the day hasn't got busy yet, but the best time is whichever you'll actually keep. Some like a short session at both times.", "q1": "Which author on meditation do I like?", "q2": "Who is one of my favourite writers about meditation?", "distractor_sentence": "My mom loves Thich Nhat Hanh's books and reads them every morning."}, "hi": {"memory_turn": "ध्यान पर लिखने वाले मेरे पसंदीदा लेखकों में से एक डैन हैरिस हैं; उनकी किताब '10% Happier' माइंडफ़ुलनेस को इतने अपनेपन से समझाती है। मैं रोज़ ध्यान की आदत कैसे बनाऊँ?", "assistant_reply": "हर दिन एक ही समय पर पाँच मिनट से शुरू करें, किसी ऐसे काम के ठीक बाद जो आप पहले से करती हैं, जैसे ब्रश करना। शुरू में गाइडेड ऐप इस्तेमाल करें, और भटकते मन की चिंता न करें; उसे पहचानना ही अभ्यास है।", "followup_turn": "ध्यान सुबह करना बेहतर है या रात में?", "followup_reply": "बहुत लोगों को सुबह ठीक लगती है क्योंकि दिन अभी व्यस्त नहीं हुआ होता, पर सबसे अच्छा समय वही है जिसे आप सच में निभा सकें। कुछ लोग दोनों समय छोटा सेशन करते हैं।", "q1": "ध्यान पर लिखने वाला कौन-सा लेखक मुझे पसंद है?", "q2": "ध्यान के बारे में लिखने वाले मेरे पसंदीदा लेखकों में कौन है?", "distractor_sentence": "मेरी माँ को थिक न्हात हान्ह की किताबें बहुत पसंद हैं और वे हर सुबह उन्हें पढ़ती हैं।"}, "hg": {"memory_turn": "Meditation pe likhne wale mere favourite authors mein se ek Dan Harris hain; unki book '10% Happier' mindfulness ko itne relatable tareeke se samjhati hai. Main roz meditation ki aadat kaise banaun?", "assistant_reply": "Har din ek hi time pe paanch minute se shuru karein, kisi aise kaam ke theek baad jo aap pehle se karti hain, jaise brush karna. Shuru mein guided app use karein, aur bhatakte mann ki chinta na karein; use pehchaanna hi practice hai.", "followup_turn": "Meditation subah karna better hai ya raat mein?", "followup_reply": "Bahut logon ko subah theek lagti hai kyunki din abhi busy nahi hua hota, par sabse accha time wahi hai jise aap sach mein nibha sakein. Kuch log dono time chhota session karte hain.", "q1": "Meditation pe likhne wala kaunsa author mujhe pasand hai?", "q2": "Meditation ke baare mein likhne wale mere favourite authors mein kaun hai?", "distractor_sentence": "Meri maa ko Thich Nhat Hanh ki books bahut pasand hain aur woh har subah unhe padhti hain."}}, "gold": {"answer_en": "Dan Harris", "slots": [{"name": "author", "aliases": ["Dan Harris", "Harris", "डैन हैरिस", "हैरिस"]}]}, "distractor": {"value_aliases": ["Thich Nhat Hanh", "थिक न्हात हान्ह"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 2, "filler_seed": 3989822862, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": true, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P048", "cand_id": "PX1259", "memory_type": "preference", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I'm really loving the music from 'Hadestown', and I think my favourite song is probably 'Way Down Hadestown'. Can you recommend other musicals with a similar folk or jazz sound?", "assistant_reply": "Try 'Once', built around folk songs played live by the cast, 'The Band's Visit' for gentle, jazzy music, and 'Natasha, Pierre and the Great Comet of 1812' for a mix of folk, rock and electronic.", "followup_turn": "Which of those has the best cast recording?", "followup_reply": "'Once' is often praised for its warm, intimate recording, and 'The Band's Visit' won awards for its music. Both work well just for listening.", "q1": "What is my favourite song from Hadestown?", "q2": "Which Hadestown song do I like the most?"}, "hi": {"memory_turn": "मुझे 'Hadestown' का संगीत बहुत पसंद आ रहा है, और मुझे लगता है मेरा पसंदीदा गाना शायद 'Way Down Hadestown' है। क्या आप ऐसे ही फ़ोक या जैज़ जैसी आवाज़ वाले और म्यूज़िकल बता सकते हैं?", "assistant_reply": "'Once' देखिए, जो कलाकारों के लाइव बजाए फ़ोक गानों पर बना है, हल्के जैज़ी संगीत के लिए 'The Band's Visit', और फ़ोक, रॉक और इलेक्ट्रॉनिक के मेल के लिए 'Natasha, Pierre and the Great Comet of 1812'।", "followup_turn": "इनमें से किसकी कास्ट रिकॉर्डिंग सबसे अच्छी है?", "followup_reply": "'Once' की गर्मजोशी भरी, निजी-सी रिकॉर्डिंग की अक्सर तारीफ़ होती है, और 'The Band's Visit' ने अपने संगीत के लिए पुरस्कार जीते। दोनों सिर्फ़ सुनने के लिए भी अच्छी हैं।", "q1": "Hadestown का मेरा पसंदीदा गाना कौन-सा है?", "q2": "Hadestown का कौन-सा गाना मुझे सबसे ज़्यादा पसंद है?"}, "hg": {"memory_turn": "Mujhe 'Hadestown' ka music bahut pasand aa raha hai, aur mujhe lagta hai mera favourite song shayad 'Way Down Hadestown' hai. Kya aap aise hi folk ya jazz jaisi awaaz wale aur musicals bata sakte hain?", "assistant_reply": "'Once' dekhiye, jo cast ke live bajaye folk songs pe bana hai, halke jazzy music ke liye 'The Band's Visit', aur folk, rock aur electronic ke mix ke liye 'Natasha, Pierre and the Great Comet of 1812'.", "followup_turn": "Inmein se kiski cast recording sabse acchi hai?", "followup_reply": "'Once' ki warm, intimate recording ki aksar taareef hoti hai, aur 'The Band's Visit' ne apne music ke liye awards jeete. Dono sirf sunne ke liye bhi acchi hain.", "q1": "Hadestown ka mera favourite song kaunsa hai?", "q2": "Hadestown ka kaunsa song mujhe sabse zyaada pasand hai?"}}, "gold": {"answer_en": "Way Down Hadestown", "slots": [{"name": "song", "aliases": ["Way Down Hadestown", "Way Down"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 5, "filler_seed": 3280279899, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": ["FL20"]}
{"item_id": "P049", "cand_id": "PX1262", "memory_type": "preference", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I finally watched the musical Wicked, and my favourite song from the show is probably 'Defying Gravity'; it's such an iconic, uplifting anthem. Are there singing classes for adults who are complete beginners?", "assistant_reply": "Yes, many music schools and community centres run beginner adult voice classes. Group classes are relaxed and affordable, while a few private lessons help you learn good breathing and avoid strain.", "followup_turn": "Can anyone learn to sing well?", "followup_reply": "Most people can improve a lot with practice. Pitch, breath control and tone are skills, so regular short practice makes a clear difference within a few months.", "q1": "What is my favourite song from Wicked?", "q2": "Which song from the musical Wicked do I like the most?", "distractor_sentence": "My sister's favourite song from that show is 'Popular', which I find too silly."}, "hi": {"memory_turn": "आख़िरकार मैंने म्यूज़िकल Wicked देख लिया, और इस शो का मेरा पसंदीदा गाना शायद 'Defying Gravity' है; यह बहुत ही मशहूर और जोश भरने वाला गीत है। क्या बिल्कुल नए वयस्कों के लिए गायन की क्लास होती हैं?", "assistant_reply": "हाँ, कई संगीत स्कूल और सामुदायिक केंद्र वयस्कों के लिए शुरुआती गायन क्लास चलाते हैं। ग्रुप क्लास आरामदायक और सस्ती होती हैं, जबकि कुछ निजी क्लास से सही साँस लेना सीखने और गले पर ज़ोर से बचने में मदद मिलती है।", "followup_turn": "क्या कोई भी अच्छा गाना सीख सकता है?", "followup_reply": "ज़्यादातर लोग अभ्यास से काफ़ी सुधार कर सकते हैं। सुर, साँस पर क़ाबू और आवाज़ का रंग हुनर हैं, इसलिए नियमित छोटे अभ्यास से कुछ महीनों में साफ़ फ़र्क दिखता है।", "q1": "Wicked का मेरा पसंदीदा गाना कौन-सा है?", "q2": "म्यूज़िकल Wicked का कौन-सा गाना मुझे सबसे ज़्यादा पसंद है?", "distractor_sentence": "मेरी बहन का उस शो का पसंदीदा गाना 'Popular' है, जो मुझे बहुत बचकाना लगता है।"}, "hg": {"memory_turn": "Aakhirkaar maine musical Wicked dekh liya, aur is show ka mera favourite song shayad 'Defying Gravity' hai; yeh bahut hi iconic aur josh bharne wala anthem hai. Kya bilkul naye adults ke liye singing classes hoti hain?", "assistant_reply": "Haan, kai music schools aur community centres adults ke liye beginner singing classes chalate hain. Group classes relaxed aur sasti hoti hain, jabki kuch private lessons se sahi breathing seekhne aur gale pe zor se bachne mein help milti hai.", "followup_turn": "Kya koi bhi accha gaana seekh sakta hai?", "followup_reply": "Zyaadatar log practice se kaafi improve kar sakte hain. Pitch, breath control aur tone skills hain, isliye regular chhoti practice se kuch mahinon mein clear fark dikhta hai.", "q1": "Wicked ka mera favourite song kaunsa hai?", "q2": "Musical Wicked ka kaunsa song mujhe sabse zyaada pasand hai?", "distractor_sentence": "Meri behen ka us show ka favourite song 'Popular' hai, jo mujhe bahut bachkana lagta hai."}}, "gold": {"answer_en": "Defying Gravity", "slots": [{"name": "song", "aliases": ["Defying Gravity", "डिफ़ाइंग ग्रैविटी"]}]}, "distractor": {"value_aliases": ["Popular"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 9, "filler_seed": 1038169147, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": false, "abl_q3": false, "abl_digits": false, "banned_fillers": []}
{"item_id": "P050", "cand_id": "PX1281", "memory_type": "preference", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I think my favourite thing to do in my craft room is quilting. The room is getting cluttered with fabric, though. How can I organise it better?", "assistant_reply": "Fold fabric around comic-book boards so it stands upright like books on a shelf, sort it by colour, and keep scraps in clear bins by size. A pegboard wall keeps rulers, scissors and thread within reach.", "followup_turn": "How do I stop fabric from fading on open shelves?", "followup_reply": "Keep shelves out of direct sunlight, or add a curtain or blinds to the window. Rotating fabric you display also helps it fade evenly if at all.", "q1": "What is my favourite thing to do in my craft room?", "q2": "Which craft do I enjoy most in my craft room?", "q3": "I have a whole free Saturday to spend in my hobby room. Which activity should I plan, based on what I enjoy most there?"}, "hi": {"memory_turn": "मुझे लगता है मेरे क्राफ़्ट रूम में मेरा सबसे पसंदीदा काम क्विल्टिंग है। पर कमरा कपड़ों से भरता जा रहा है। इसे बेहतर तरीक़े से कैसे व्यवस्थित करूँ?", "assistant_reply": "कपड़े को कॉमिक-बुक बोर्ड पर लपेटकर किताबों की तरह शेल्फ़ में खड़ा रखें, रंग के हिसाब से छाँटें, और कतरनों को आकार के हिसाब से पारदर्शी डिब्बों में रखें। पेगबोर्ड वाली दीवार पर फ़ीते, कैंची और धागा हाथ के पास रहते हैं।", "followup_turn": "खुली शेल्फ़ पर कपड़े का रंग फीका होने से कैसे रोकूँ?", "followup_reply": "शेल्फ़ को सीधी धूप से दूर रखें, या खिड़की पर पर्दा या ब्लाइंड लगाएँ। दिखाए जाने वाले कपड़े को बदलते रहने से अगर रंग उड़े भी तो बराबर उड़ता है।", "q1": "क्राफ़्ट रूम में मेरा सबसे पसंदीदा काम क्या है?", "q2": "अपने क्राफ़्ट रूम में मुझे कौन-सा हुनर सबसे ज़्यादा पसंद है?", "q3": "मेरे पास अपने शौक़ वाले कमरे में बिताने के लिए पूरा ख़ाली शनिवार है। वहाँ मुझे जो सबसे ज़्यादा पसंद है उसके हिसाब से कौन-सा काम प्लान करूँ?"}, "hg": {"memory_turn": "Mujhe lagta hai mere craft room mein mera sabse favourite kaam quilting hai. Par room kapdon se bharta ja raha hai. Ise better tareeke se kaise organise karoon?", "assistant_reply": "Kapde ko comic-book boards pe lapet ke books ki tarah shelf mein khada rakhein, colour ke hisaab se chhaantein, aur scraps ko size ke hisaab se clear bins mein rakhein. Pegboard wali deewar pe rulers, kainchi aur dhaaga haath ke paas rehte hain.", "followup_turn": "Khuli shelf pe kapde ka rang feeka hone se kaise rokun?", "followup_reply": "Shelf ko seedhi dhoop se door rakhein, ya khidki pe parda ya blinds lagayein. Display wale kapde ko badalte rehne se agar rang ude bhi toh barabar udta hai.", "q1": "Craft room mein mera sabse favourite kaam kya hai?", "q2": "Apne craft room mein mujhe kaunsa craft sabse zyaada pasand hai?", "q3": "Mere paas apne hobby room mein bitane ke liye poora free Saturday hai. Wahan mujhe jo sabse zyaada pasand hai uske hisaab se kaunsa kaam plan karoon?"}}, "gold": {"answer_en": "quilting", "slots": [{"name": "craft", "aliases": ["quilting", "quilt", "क्विल्टिंग", "क्विल्ट"]}]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": null, "lme_type": null}, "topic": null, "is_answerable": true, "evidence_position": 2, "filler_seed": 3728459596, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": true, "abl_q3": true, "abl_digits": false, "banned_fillers": ["FL29"]}
{"item_id": "E041", "cand_id": "EC057", "memory_type": "event", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I've been getting more into charity events lately: I helped raise $2,000 for a local animal shelter on January 20th, and it was an amazing experience! What other ways can I help the shelter?", "assistant_reply": "You could volunteer to walk dogs or socialise cats, foster an animal for a few weeks, collect blankets and food, or help with photos and posts that make adoptable pets more visible.", "followup_turn": "Is fostering hard for a first-timer?", "followup_reply": "Shelters usually give you supplies and support. Starting with an adult cat or a calm older dog is easier than puppies or kittens, which need a lot of attention.", "q1": "When did I help raise money for the local animal shelter?", "q2": "On what date did I take part in the fundraiser for the animal shelter?", "q3": "The shelter is thanking its supporters and wants the date of the fundraising event I helped with. What date should I give?", "distractor_sentence": "My friend ran a bake sale for the same shelter on February 3rd."}, "hi": {"memory_turn": "आजकल मुझे चैरिटी इवेंट में ज़्यादा दिलचस्पी हो रही है: मैंने 20 जनवरी को एक स्थानीय पशु आश्रय के लिए $2,000 जुटाने में मदद की, और यह कमाल का अनुभव था! मैं आश्रय की और किन तरीक़ों से मदद कर सकती हूँ?", "assistant_reply": "आप कुत्तों को घुमाने या बिल्लियों के साथ समय बिताने के लिए वॉलंटियर कर सकती हैं, कुछ हफ़्तों के लिए किसी जानवर को घर में पाल सकती हैं, कंबल और खाना इकट्ठा कर सकती हैं, या गोद लेने लायक़ जानवरों की फ़ोटो और पोस्ट में मदद कर सकती हैं।", "followup_turn": "क्या पहली बार किसी जानवर को अस्थायी रूप से पालना मुश्किल है?", "followup_reply": "आश्रय आमतौर पर सामान और मदद देते हैं। पिल्लों या बिल्ली के बच्चों से, जिन्हें बहुत ध्यान चाहिए, बड़ी बिल्ली या शांत उम्रदराज़ कुत्ते से शुरू करना आसान है।", "q1": "मैंने स्थानीय पशु आश्रय के लिए पैसे जुटाने में कब मदद की?", "q2": "पशु आश्रय के फ़ंडरेज़र में मैंने किस तारीख़ को हिस्सा लिया?", "q3": "आश्रय अपने समर्थकों को धन्यवाद दे रहा है और उसे उस फ़ंडरेज़िंग इवेंट की तारीख़ चाहिए जिसमें मैंने मदद की। कौन-सी तारीख़ बताऊँ?", "distractor_sentence": "मेरी दोस्त ने उसी आश्रय के लिए 3 फ़रवरी को बेक सेल लगाई थी।"}, "hg": {"memory_turn": "Aajkal mujhe charity events mein zyaada interest ho raha hai: maine 20 January ko ek local animal shelter ke liye $2,000 jutane mein help ki, aur yeh kamaal ka experience tha! Main shelter ki aur kin tareekon se help kar sakti hoon?", "assistant_reply": "Aap dogs ko ghumane ya cats ke saath time bitane ke liye volunteer kar sakti hain, kuch hafton ke liye kisi animal ko foster kar sakti hain, blankets aur khaana collect kar sakti hain, ya adopt karne laayak pets ki photos aur posts mein help kar sakti hain.", "followup_turn": "Kya pehli baar fostering karna mushkil hai?", "followup_reply": "Shelters usually saamaan aur support dete hain. Puppies ya kittens se, jinhe bahut attention chahiye, adult cat ya calm older dog se shuru karna easy hai.", "q1": "Maine local animal shelter ke liye paise jutane mein kab help ki?", "q2": "Animal shelter ke fundraiser mein maine kis date ko hissa liya?", "q3": "Shelter apne supporters ko thank you bol raha hai aur use us fundraising event ki date chahiye jismein maine help ki. Kaunsi date bataun?", "distractor_sentence": "Meri friend ne usi shelter ke liye 3 February ko bake sale lagayi thi."}}, "gold": {"answer_en": "January 20", "date": {"day": 20, "month": 1}, "slots": [{"name": "date", "aliases": ["January 20", "January 20th", "20 January", "20th January", "20th of January", "Jan 20", "20 Jan", "20 जनवरी"]}]}, "distractor": {"date": {"day": 3, "month": 2}, "value_aliases": ["February 3", "February 3rd", "3 February", "3rd February", "3rd of February", "Feb 3", "3 Feb", "3 फ़रवरी", "3 फरवरी"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "d851d5ba", "lme_type": "multi-session"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 4246940006, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": true, "abl_digits": true, "banned_fillers": []}
{"item_id": "E042", "cand_id": "EC062", "memory_type": "event", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I rode Space Mountain: Ghost Galaxy three times at Disneyland on September 24th during Mickey's Halloween Party, and it was a blast! I want to go back with friends. How can we skip the long queues?", "assistant_reply": "Arrive at opening and head straight to the most popular rides, use the park's paid express system for a few headliners, and try single-rider lines where they exist. Evenings during parades are often quieter.", "followup_turn": "Which days are least crowded?", "followup_reply": "Midweek days outside school holidays are usually quietest, especially Tuesdays and Wednesdays. Avoid weekends and the weeks around major holidays.", "q1": "When did I ride Space Mountain: Ghost Galaxy at Disneyland?", "q2": "On what date did I go on Space Mountain: Ghost Galaxy three times?"}, "hi": {"memory_turn": "मैंने 24 सितंबर को डिज़्नीलैंड में Mickey's Halloween Party के दौरान Space Mountain: Ghost Galaxy की तीन बार सवारी की, और बहुत मज़ा आया! मैं दोस्तों के साथ दोबारा जाना चाहता हूँ। लंबी लाइनों से कैसे बचें?", "assistant_reply": "पार्क खुलते ही पहुँचें और सीधे सबसे मशहूर राइड पर जाएँ, कुछ बड़ी राइड के लिए पार्क का पेड एक्सप्रेस सिस्टम इस्तेमाल करें, और जहाँ हो वहाँ सिंगल-राइडर लाइन आज़माएँ। परेड के समय शाम को अक्सर भीड़ कम होती है।", "followup_turn": "किन दिनों सबसे कम भीड़ होती है?", "followup_reply": "स्कूल की छुट्टियों के बाहर हफ़्ते के बीच के दिन, ख़ासकर मंगलवार और बुधवार, सबसे शांत रहते हैं। वीकेंड और बड़ी छुट्टियों के आसपास के हफ़्तों से बचें।", "q1": "मैंने डिज़्नीलैंड में Space Mountain: Ghost Galaxy की सवारी कब की?", "q2": "मैंने Space Mountain: Ghost Galaxy पर तीन बार किस तारीख़ को सवारी की?"}, "hg": {"memory_turn": "Maine 24 September ko Disneyland mein Mickey's Halloween Party ke dauraan Space Mountain: Ghost Galaxy ki teen baar ride ki, aur bahut maza aaya! Main doston ke saath dobara jaana chahta hoon. Lambi lines se kaise bachein?", "assistant_reply": "Park khulte hi pahunchein aur seedha sabse popular rides pe jaayein, kuch badi rides ke liye park ka paid express system use karein, aur jahan ho wahan single-rider line try karein. Parades ke time shaam ko aksar bheed kam hoti hai.", "followup_turn": "Kin dinon sabse kam bheed hoti hai?", "followup_reply": "School holidays ke bahar hafte ke beech ke din, especially Tuesday aur Wednesday, sabse shaant rehte hain. Weekends aur badi chhuttiyon ke aaspaas ke hafton se bachein.", "q1": "Maine Disneyland mein Space Mountain: Ghost Galaxy ki ride kab ki?", "q2": "Maine Space Mountain: Ghost Galaxy pe teen baar kis date ko ride ki?"}}, "gold": {"answer_en": "September 24", "date": {"day": 24, "month": 9}, "slots": [{"name": "date", "aliases": ["September 24", "September 24th", "24 September", "24th September", "24th of September", "Sep 24", "24 Sep", "24 सितंबर", "24 सितम्बर"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "gpt4_e05b82a6", "lme_type": "multi-session"}, "topic": "family/social", "is_answerable": true, "evidence_position": 5, "filler_seed": 3576084619, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": true, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E043", "cand_id": "EC063", "memory_type": "event", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I rode the Xcelerator rollercoaster at Knott's Berry Farm on October 8th, and it's still one of my favourite thrill rides. Which other coasters in California should I try?", "assistant_reply": "Try Twisted Colossus and X2 at Six Flags Magic Mountain, the Incredicoaster at Disney California Adventure, and GhostRider, the big wooden coaster also at Knott's.", "followup_turn": "Is X2 very intense?", "followup_reply": "Yes, it's one of the most intense: the seats rotate as the train moves, so you flip head over heels several times. Ride it on an empty stomach.", "q1": "When did I ride the Xcelerator rollercoaster?", "q2": "On what date did I go on Xcelerator at Knott's Berry Farm?", "q3": "I'm making a scrapbook page for the day I rode that launch coaster at the berry-farm theme park. Which date should I write?"}, "hi": {"memory_turn": "मैंने 8 अक्टूबर को Knott's Berry Farm में Xcelerator रोलरकोस्टर की सवारी की, और यह आज भी मेरी पसंदीदा रोमांचक राइड में से एक है। कैलिफ़ोर्निया में और कौन-से कोस्टर आज़माऊँ?", "assistant_reply": "Six Flags Magic Mountain में Twisted Colossus और X2, Disney California Adventure में Incredicoaster, और Knott's में ही लकड़ी का बड़ा कोस्टर GhostRider आज़माइए।", "followup_turn": "क्या X2 बहुत तेज़ और डरावना है?", "followup_reply": "हाँ, यह सबसे ज़बरदस्त में से है: ट्रेन चलते समय सीटें घूमती हैं, इसलिए आप कई बार सिर के बल पलटते हैं। ख़ाली पेट जाएँ।", "q1": "मैंने Xcelerator रोलरकोस्टर की सवारी कब की थी?", "q2": "Knott's Berry Farm में Xcelerator पर मैं किस तारीख़ को बैठी थी?", "q3": "मैं उस दिन के लिए स्क्रैपबुक का पन्ना बना रही हूँ जब मैंने बेरी-फ़ार्म वाले थीम पार्क में वह लॉन्च कोस्टर चलाया था। कौन-सी तारीख़ लिखूँ?"}, "hg": {"memory_turn": "Maine 8 October ko Knott's Berry Farm mein Xcelerator rollercoaster ki ride ki, aur yeh aaj bhi meri favourite thrill rides mein se ek hai. California mein aur kaunse coasters try karoon?", "assistant_reply": "Six Flags Magic Mountain mein Twisted Colossus aur X2, Disney California Adventure mein Incredicoaster, aur Knott's mein hi lakdi ka bada coaster GhostRider try kijiye.", "followup_turn": "Kya X2 bahut intense hai?", "followup_reply": "Haan, yeh sabse intense mein se hai: train chalte time seats ghoomti hain, isliye aap kai baar ulte palat jaate hain. Khaali pet jaayein.", "q1": "Maine Xcelerator rollercoaster ki ride kab ki thi?", "q2": "Knott's Berry Farm mein Xcelerator pe main kis date ko baithi thi?", "q3": "Main us din ke liye scrapbook ka page bana rahi hoon jab maine berry-farm wale theme park mein woh launch coaster chalaya tha. Kaunsi date likhun?"}}, "gold": {"answer_en": "October 8", "date": {"day": 8, "month": 10}, "slots": [{"name": "date", "aliases": ["October 8", "October 8th", "8 October", "8th October", "8th of October", "Oct 8", "8 Oct", "8 अक्टूबर", "8 अक्तूबर"]}]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "gpt4_e05b82a6", "lme_type": "multi-session"}, "topic": "other", "is_answerable": true, "evidence_position": 9, "filler_seed": 1375450510, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": true, "abl_digits": true, "banned_fillers": []}
{"item_id": "E044", "cand_id": "EC064", "memory_type": "event", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I rode the Revenge of the Mummy rollercoaster three times in a row at Universal Studios Hollywood on October 15th, and it was such a thrill! What other attractions there are worth it?", "assistant_reply": "The Studio Tour is a must, with real film sets and special effects. The Wizarding World of Harry Potter is great for theming, and Jurassic World: The Ride is fun if you don't mind getting wet.", "followup_turn": "How long does the Studio Tour take?", "followup_reply": "About an hour. Go early in the day or late in the afternoon, when the queue is usually shorter.", "q1": "When did I ride Revenge of the Mummy at Universal Studios Hollywood?", "q2": "On what date did I ride the Revenge of the Mummy coaster three times?", "distractor_sentence": "My cousin went to Universal Studios on October 29th for their Halloween nights."}, "hi": {"memory_turn": "मैंने 15 अक्टूबर को Universal Studios Hollywood में Revenge of the Mummy रोलरकोस्टर की लगातार तीन बार सवारी की, और बहुत रोमांच आया! वहाँ और कौन-से आकर्षण देखने लायक़ हैं?", "assistant_reply": "Studio Tour ज़रूर करें, जिसमें असली फ़िल्म सेट और स्पेशल इफ़ेक्ट हैं। सजावट के लिए The Wizarding World of Harry Potter बढ़िया है, और अगर भीगने से परहेज़ न हो तो Jurassic World: The Ride मज़ेदार है।", "followup_turn": "Studio Tour में कितना समय लगता है?", "followup_reply": "लगभग एक घंटा। दिन की शुरुआत में या दोपहर बाद जाएँ, जब आमतौर पर लाइन छोटी होती है।", "q1": "मैंने Universal Studios Hollywood में Revenge of the Mummy की सवारी कब की?", "q2": "मैंने Revenge of the Mummy कोस्टर पर तीन बार किस तारीख़ को सवारी की?", "distractor_sentence": "मेरा कज़िन 29 अक्टूबर को हैलोवीन नाइट्स के लिए Universal Studios गया था।"}, "hg": {"memory_turn": "Maine 15 October ko Universal Studios Hollywood mein Revenge of the Mummy rollercoaster ki lagataar teen baar ride ki, aur bahut thrill aaya! Wahan aur kaunse attractions dekhne laayak hain?", "assistant_reply": "Studio Tour zaroor karein, jismein asli film sets aur special effects hain. Theming ke liye The Wizarding World of Harry Potter badhiya hai, aur agar bheegne se parhez na ho toh Jurassic World: The Ride mazedaar hai.", "followup_turn": "Studio Tour mein kitna time lagta hai?", "followup_reply": "Lagbhag ek ghanta. Din ki shuruaat mein ya dopahar baad jaayein, jab usually line chhoti hoti hai.", "q1": "Maine Universal Studios Hollywood mein Revenge of the Mummy ki ride kab ki?", "q2": "Maine Revenge of the Mummy coaster pe teen baar kis date ko ride ki?", "distractor_sentence": "Mera cousin 29 October ko Halloween nights ke liye Universal Studios gaya tha."}}, "gold": {"answer_en": "October 15", "date": {"day": 15, "month": 10}, "slots": [{"name": "date", "aliases": ["October 15", "October 15th", "15 October", "15th October", "15th of October", "Oct 15", "15 Oct", "15 अक्टूबर", "15 अक्तूबर"]}]}, "distractor": {"date": {"day": 29, "month": 10}, "value_aliases": ["October 29", "October 29th", "29 October", "29th October", "29th of October", "Oct 29", "29 Oct", "29 अक्टूबर", "29 अक्तूबर"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "gpt4_e05b82a6", "lme_type": "multi-session"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 2548333759, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": true, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E045", "cand_id": "EC065", "memory_type": "event", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I attended a one-day photography workshop at a local studio on February 22nd, and it was really helpful. It was free, but I had to register online in advance. How can I keep improving on my own?", "assistant_reply": "Pick a weekly theme, like shadows or reflections, and shoot only that. Study photos you admire and note why they work, and share a few shots in an online group for honest feedback.", "followup_turn": "Should I shoot in RAW?", "followup_reply": "Yes, if you plan to edit. RAW files keep much more detail in highlights and shadows, so you can fix exposure and colour later without losing quality.", "q1": "When did I attend the one-day photography workshop?", "q2": "On what date was the free photography workshop I went to at a local studio?"}, "hi": {"memory_turn": "मैं 22 फ़रवरी को एक स्थानीय स्टूडियो में एक दिन की फ़ोटोग्राफ़ी वर्कशॉप में गई थी, और वह बहुत काम की रही। वह मुफ़्त थी, पर पहले से ऑनलाइन रजिस्टर करना पड़ा। मैं ख़ुद से कैसे सुधार करती रहूँ?", "assistant_reply": "हर हफ़्ते कोई एक विषय चुनें, जैसे परछाइयाँ या प्रतिबिंब, और सिर्फ़ उसी की फ़ोटो लें। जो फ़ोटो पसंद हों उन्हें ध्यान से देखें और लिखें कि वे क्यों अच्छी हैं, और ईमानदार राय के लिए कुछ फ़ोटो किसी ऑनलाइन ग्रुप में डालें।", "followup_turn": "क्या मुझे RAW में शूट करना चाहिए?", "followup_reply": "हाँ, अगर आप एडिट करने वाली हैं। RAW फ़ाइलों में रोशनी और परछाईं की कहीं ज़्यादा जानकारी रहती है, इसलिए बाद में क्वालिटी खोए बिना एक्सपोज़र और रंग ठीक कर सकती हैं।", "q1": "मैं एक दिन की फ़ोटोग्राफ़ी वर्कशॉप में कब गई थी?", "q2": "स्थानीय स्टूडियो में जिस मुफ़्त फ़ोटोग्राफ़ी वर्कशॉप में मैं गई, वह किस तारीख़ को थी?"}, "hg": {"memory_turn": "Main 22 February ko ek local studio mein ek din ki photography workshop mein gayi thi, aur woh bahut kaam ki rahi. Woh free thi, par pehle se online register karna pada. Main khud se kaise improve karti rahun?", "assistant_reply": "Har hafte koi ek theme chunein, jaise shadows ya reflections, aur sirf usi ki photos lein. Jo photos pasand hon unhe dhyaan se dekhein aur likhein ki woh kyun acchi hain, aur honest feedback ke liye kuch photos kisi online group mein daalein.", "followup_turn": "Kya mujhe RAW mein shoot karna chahiye?", "followup_reply": "Haan, agar aap edit karne wali hain. RAW files mein highlights aur shadows ki kahin zyaada detail rehti hai, isliye baad mein quality khoye bina exposure aur colour theek kar sakti hain.", "q1": "Main ek din ki photography workshop mein kab gayi thi?", "q2": "Local studio mein jis free photography workshop mein main gayi, woh kis date ko thi?"}}, "gold": {"answer_en": "February 22", "date": {"day": 22, "month": 2}, "slots": [{"name": "date", "aliases": ["February 22", "February 22nd", "22 February", "22nd February", "22nd of February", "Feb 22", "22 Feb", "22 फ़रवरी", "22 फरवरी"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "gpt4_731e37d7", "lme_type": "multi-session"}, "topic": "other", "is_answerable": true, "evidence_position": 5, "filler_seed": 1943737084, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "abl_short": true, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E046", "cand_id": "EC066", "memory_type": "event", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I attended a half-day mindfulness workshop at a studio near my home on December 12th, and it was really helpful. How can I keep practising mindfulness during a busy workday?", "assistant_reply": "Take three slow breaths before opening your inbox, eat lunch away from your screen, and use short walks between meetings to notice your surroundings. Even one minute of focused breathing resets your attention.", "followup_turn": "Are there good apps for this?", "followup_reply": "Headspace and Calm have short guided sessions, and Insight Timer has thousands of free ones. Many people find two or three minutes at a time easiest to keep up.", "q1": "When did I attend the half-day mindfulness workshop?", "q2": "On what date did I go to the mindfulness workshop near my home?", "distractor_sentence": "My wife went to a meditation retreat on January 9th."}, "hi": {"memory_turn": "मैं 12 दिसंबर को अपने घर के पास एक स्टूडियो में आधे दिन की माइंडफ़ुलनेस वर्कशॉप में गया था, और वह बहुत काम की रही। व्यस्त काम के दिन में माइंडफ़ुलनेस का अभ्यास कैसे जारी रखूँ?", "assistant_reply": "इनबॉक्स खोलने से पहले तीन धीमी साँसें लें, स्क्रीन से दूर दोपहर का खाना खाएँ, और मीटिंग के बीच छोटी सैर में अपने आसपास पर ध्यान दें। एक मिनट की ध्यान से ली गई साँस भी ध्यान को नया कर देती है।", "followup_turn": "क्या इसके लिए अच्छे ऐप हैं?", "followup_reply": "Headspace और Calm में छोटे गाइडेड सेशन हैं, और Insight Timer में हज़ारों मुफ़्त सेशन हैं। बहुत लोगों को एक बार में दो-तीन मिनट निभाना सबसे आसान लगता है।", "q1": "मैं आधे दिन की माइंडफ़ुलनेस वर्कशॉप में कब गया था?", "q2": "घर के पास वाली माइंडफ़ुलनेस वर्कशॉप में मैं किस तारीख़ को गया?", "distractor_sentence": "मेरी पत्नी 9 जनवरी को एक ध्यान शिविर में गई थी।"}, "hg": {"memory_turn": "Main 12 December ko apne ghar ke paas ek studio mein aadhe din ki mindfulness workshop mein gaya tha, aur woh bahut kaam ki rahi. Busy work day mein mindfulness ki practice kaise jaari rakhun?", "assistant_reply": "Inbox kholne se pehle teen dheemi saansein lein, screen se door lunch karein, aur meetings ke beech chhoti walk mein apne aaspaas pe dhyaan dein. Ek minute ki focused breathing bhi attention ko reset kar deti hai.", "followup_turn": "Kya iske liye acche apps hain?", "followup_reply": "Headspace aur Calm mein chhote guided sessions hain, aur Insight Timer mein hazaaron free sessions hain. Bahut logon ko ek baar mein do-teen minute nibhana sabse easy lagta hai.", "q1": "Main aadhe din ki mindfulness workshop mein kab gaya tha?", "q2": "Ghar ke paas wali mindfulness workshop mein main kis date ko gaya?", "distractor_sentence": "Meri wife 9 January ko ek meditation retreat mein gayi thi."}}, "gold": {"answer_en": "December 12", "date": {"day": 12, "month": 12}, "slots": [{"name": "date", "aliases": ["December 12", "December 12th", "12 December", "12th December", "12th of December", "Dec 12", "12 Dec", "12 दिसंबर", "12 दिसम्बर"]}]}, "distractor": {"date": {"day": 9, "month": 1}, "value_aliases": ["January 9", "January 9th", "9 January", "9th January", "9th of January", "Jan 9", "9 Jan", "9 जनवरी"]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "gpt4_731e37d7", "lme_type": "multi-session"}, "topic": "fitness/health", "is_answerable": true, "evidence_position": 9, "filler_seed": 3754320572, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": true, "abl_q3": false, "abl_digits": true, "banned_fillers": ["FL06", "FL31"]}
{"item_id": "E047", "cand_id": "EC069", "memory_type": "event", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I attended a lecture on sustainable development at the public library on April 10th, and it got me interested in learning more. Are there good free courses on the topic?", "assistant_reply": "The SDG Academy offers free courses on the UN Sustainable Development Goals, and Coursera and edX have introductory courses from universities that you can audit for free.", "followup_turn": "Can I make a difference at home too?", "followup_reply": "Yes: reduce food waste, choose public transport or cycling when you can, buy fewer but longer-lasting things, and support local repair shops and second-hand markets.", "q1": "When did I attend the lecture on sustainable development?", "q2": "On what date was the sustainable development lecture I went to at the public library?", "q3": "I'm writing a reflection for my course about the library talk that sparked my interest in green development. What date was it?", "distractor_sentence": "My friend is attending a climate talk on April 24th."}, "hi": {"memory_turn": "मैं 10 अप्रैल को सार्वजनिक पुस्तकालय में सतत विकास पर एक लेक्चर में गई थी, और उससे मेरी इस बारे में और जानने में दिलचस्पी जगी। क्या इस विषय पर अच्छे मुफ़्त कोर्स हैं?", "assistant_reply": "SDG Academy संयुक्त राष्ट्र के सतत विकास लक्ष्यों पर मुफ़्त कोर्स देती है, और Coursera व edX पर विश्वविद्यालयों के शुरुआती कोर्स हैं जिन्हें आप मुफ़्त में ऑडिट कर सकती हैं।", "followup_turn": "क्या मैं घर पर भी कुछ फ़र्क ला सकती हूँ?", "followup_reply": "हाँ: खाने की बर्बादी कम करें, जब हो सके सार्वजनिक परिवहन या साइकिल चुनें, कम पर टिकाऊ चीज़ें ख़रीदें, और स्थानीय मरम्मत की दुकानों और पुराने सामान के बाज़ारों का साथ दें।", "q1": "मैं सतत विकास वाले लेक्चर में कब गई थी?", "q2": "सार्वजनिक पुस्तकालय में जिस सतत विकास लेक्चर में मैं गई, वह किस तारीख़ को था?", "q3": "मैं अपने कोर्स के लिए पुस्तकालय की उस बातचीत पर एक चिंतन लिख रही हूँ जिससे हरित विकास में मेरी रुचि जगी। वह किस तारीख़ को थी?", "distractor_sentence": "मेरी दोस्त 24 अप्रैल को जलवायु पर एक बातचीत में जा रही है।"}, "hg": {"memory_turn": "Main 10 April ko public library mein sustainable development pe ek lecture mein gayi thi, aur usse meri is baare mein aur jaanne mein dilchaspi jagi. Kya is topic pe acche free courses hain?", "assistant_reply": "SDG Academy UN ke Sustainable Development Goals pe free courses deti hai, aur Coursera aur edX pe universities ke introductory courses hain jinhe aap free mein audit kar sakti hain.", "followup_turn": "Kya main ghar pe bhi kuch fark la sakti hoon?", "followup_reply": "Haan: khaane ki barbaadi kam karein, jab ho sake public transport ya cycle chunein, kam par tikaau cheezein khareedein, aur local repair shops aur second-hand markets ka saath dein.", "q1": "Main sustainable development wale lecture mein kab gayi thi?", "q2": "Public library mein jis sustainable development lecture mein main gayi, woh kis date ko tha?", "q3": "Main apne course ke liye library ki us talk pe ek reflection likh rahi hoon jisse green development mein meri interest jagi. Woh kis date ko thi?", "distractor_sentence": "Meri friend 24 April ko climate pe ek talk mein ja rahi hai."}}, "gold": {"answer_en": "April 10", "date": {"day": 10, "month": 4}, "slots": [{"name": "date", "aliases": ["April 10", "April 10th", "10 April", "10th April", "10th of April", "Apr 10", "10 Apr", "10 अप्रैल"]}]}, "distractor": {"date": {"day": 24, "month": 4}, "value_aliases": ["April 24", "April 24th", "24 April", "24th April", "24th of April", "Apr 24", "24 Apr", "24 अप्रैल"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "10d9b85a", "lme_type": "multi-session"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 2065274213, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": true, "abl_digits": true, "banned_fillers": []}
{"item_id": "E048", "cand_id": "EC071", "memory_type": "event", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I've had a pretty successful season so far: I sold 12 bunches of fresh organic herbs from my backyard garden at the farmers' market on May 15th. How can I make my stall more attractive?", "assistant_reply": "Use a clean tablecloth and different heights with crates or baskets, label every bunch with its name and price, and keep the herbs misted so they look fresh. Offering a small tasting or recipe card draws people in.", "followup_turn": "Which herbs sell best?", "followup_reply": "Basil, coriander, mint and parsley are reliable sellers because people cook with them often. Bundles for a specific dish, like a pesto pack, also do well.", "q1": "When did I sell my herbs at the farmers' market?", "q2": "On what date did I sell 12 bunches of herbs at the farmers' market?"}, "hi": {"memory_turn": "अब तक मेरा सीज़न काफ़ी अच्छा रहा है: 15 मई को किसान बाज़ार में मैंने अपने घर के बग़ीचे की ताज़ी ऑर्गेनिक जड़ी-बूटियों के 12 गुच्छे बेचे। अपनी दुकान को और आकर्षक कैसे बनाऊँ?", "assistant_reply": "साफ़ मेज़पोश बिछाएँ और टोकरियों या क्रेट से अलग-अलग ऊँचाई बनाएँ, हर गुच्छे पर नाम और दाम का लेबल लगाएँ, और जड़ी-बूटियों पर पानी छिड़कते रहें ताकि ताज़ी दिखें। थोड़ा चखाने या रेसिपी कार्ड देने से लोग खिंचे चले आते हैं।", "followup_turn": "कौन-सी जड़ी-बूटियाँ सबसे ज़्यादा बिकती हैं?", "followup_reply": "तुलसी, धनिया, पुदीना और अजमोद भरोसे से बिकते हैं क्योंकि लोग इन्हें अक्सर खाने में डालते हैं। किसी ख़ास डिश के लिए बने गुच्छे, जैसे पेस्टो पैक, भी अच्छे बिकते हैं।", "q1": "मैंने किसान बाज़ार में अपनी जड़ी-बूटियाँ कब बेचीं?", "q2": "मैंने किसान बाज़ार में जड़ी-बूटियों के 12 गुच्छे किस तारीख़ को बेचे?"}, "hg": {"memory_turn": "Ab tak mera season kaafi accha raha hai: 15 May ko farmers' market mein maine apne ghar ke garden ki fresh organic herbs ke 12 bunches beche. Apne stall ko aur attractive kaise banaun?", "assistant_reply": "Saaf tablecloth bichhayein aur baskets ya crates se alag-alag height banayein, har bunch pe naam aur price ka label lagayein, aur herbs pe paani chhidakte rahein taaki fresh dikhein. Thoda taste karane ya recipe card dene se log kheeche chale aate hain.", "followup_turn": "Kaunsi herbs sabse zyaada bikti hain?", "followup_reply": "Basil, dhaniya, pudina aur parsley bharose se bikte hain kyunki log inhe aksar khaane mein daalte hain. Kisi khaas dish ke liye bane bunches, jaise pesto pack, bhi acche bikte hain.", "q1": "Maine farmers' market mein apni herbs kab bechi?", "q2": "Maine farmers' market mein herbs ke 12 bunches kis date ko beche?"}}, "gold": {"answer_en": "May 15", "date": {"day": 15, "month": 5}, "slots": [{"name": "date", "aliases": ["May 15", "May 15th", "15 May", "15th May", "15th of May", "May 15", "15 May", "15 मई"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "2b8f3739", "lme_type": "multi-session"}, "topic": "hobbies", "is_answerable": true, "evidence_position": 5, "filler_seed": 1546062926, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E049", "cand_id": "EC072", "memory_type": "event", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I met Rachel, an old colleague from my previous company, at the TechConnect conference on February 10th. We talked about working together again. How do I follow up without sounding too pushy?", "assistant_reply": "Send a short message within a week saying you enjoyed catching up, mention one specific idea you discussed, and suggest a brief call. Keep it friendly and leave the timing open.", "followup_turn": "Should I send it on LinkedIn or by email?", "followup_reply": "Email feels more personal for someone you've worked with before, but LinkedIn is fine if that's how you usually talk. Use whichever she replies to faster.", "q1": "When did I meet my old colleague Rachel at the TechConnect conference?", "q2": "On what date did I run into Rachel at TechConnect?"}, "hi": {"memory_turn": "मैं 10 फ़रवरी को TechConnect कॉन्फ़्रेंस में अपनी पिछली कंपनी की पुरानी साथी रेचल से मिली। हमने फिर से साथ काम करने की बात की। बहुत ज़्यादा ज़ोर डाले बिना बात आगे कैसे बढ़ाऊँ?", "assistant_reply": "एक हफ़्ते के अंदर छोटा मैसेज भेजें कि मिलकर अच्छा लगा, जिस एक ख़ास आइडिया पर बात हुई उसका ज़िक्र करें, और एक छोटी कॉल का सुझाव दें। लहजा दोस्ताना रखें और समय उन पर छोड़ दें।", "followup_turn": "LinkedIn पर भेजूँ या ईमेल से?", "followup_reply": "जिसके साथ पहले काम किया हो, उसके लिए ईमेल ज़्यादा निजी लगता है, पर अगर आप आमतौर पर LinkedIn पर बात करती हैं तो वह भी ठीक है। जहाँ वह जल्दी जवाब देती हैं, वही इस्तेमाल करें।", "q1": "मैं TechConnect कॉन्फ़्रेंस में अपनी पुरानी साथी रेचल से कब मिली?", "q2": "TechConnect में मेरी रेचल से मुलाक़ात किस तारीख़ को हुई?"}, "hg": {"memory_turn": "Main 10 February ko TechConnect conference mein apni pichhli company ki purani colleague Rachel se mili. Humne phir se saath kaam karne ki baat ki. Bahut zyaada pushy lage bina follow up kaise karoon?", "assistant_reply": "Ek hafte ke andar chhota message bhejein ki milke accha laga, jis ek khaas idea pe baat hui uska zikr karein, aur ek chhoti call suggest karein. Tone friendly rakhein aur timing unpe chhod dein.", "followup_turn": "LinkedIn pe bhejun ya email se?", "followup_reply": "Jiske saath pehle kaam kiya ho, uske liye email zyaada personal lagta hai, par agar aap usually LinkedIn pe baat karti hain toh woh bhi theek hai. Jahan woh jaldi reply karti hain, wahi use karein.", "q1": "Main TechConnect conference mein apni purani colleague Rachel se kab mili?", "q2": "TechConnect mein meri Rachel se mulaqaat kis date ko hui?"}}, "gold": {"answer_en": "February 10", "date": {"day": 10, "month": 2}, "slots": [{"name": "date", "aliases": ["February 10", "February 10th", "10 February", "10th February", "10th of February", "Feb 10", "10 Feb", "10 फ़रवरी", "10 फरवरी"]}]}, "has_q3": false, "source": {"dataset": "LongMemEval", "question_id": "5a4f22c0", "lme_type": "knowledge-update"}, "topic": "other", "is_answerable": true, "evidence_position": 9, "filler_seed": 3429277154, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "abl_short": false, "abl_q3": false, "abl_digits": true, "banned_fillers": []}
{"item_id": "E050", "cand_id": "EC172", "memory_type": "event", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I don't know the exact mileage on my car, but I remember getting it serviced on February 10th, when I replaced the air filter. Now the engine light is on. What could that mean?", "assistant_reply": "It can mean many things, from a loose fuel cap to a sensor problem or misfire. A steady light usually isn't an emergency, but a flashing one means you should stop driving soon. A code reader can tell you the cause.", "followup_turn": "Can I check the code myself?", "followup_reply": "Yes. A cheap OBD-II scanner plugs in under the dashboard and shows the code on your phone. Search the code to see likely causes before visiting a mechanic.", "q1": "When did I get my car serviced?", "q2": "On what date was my car's last service, when the air filter was replaced?", "q3": "My mechanic suggests a check-up three months after the last service. From which date should I count?", "distractor_sentence": "My dad got his car serviced on February 28th."}, "hi": {"memory_turn": "मुझे अपनी कार की सही माइलेज नहीं पता, पर याद है कि 10 फ़रवरी को उसकी सर्विस करवाई थी, जब एयर फ़िल्टर बदला था। अब इंजन की लाइट जल रही है। इसका क्या मतलब हो सकता है?", "assistant_reply": "इसके कई मतलब हो सकते हैं, ढीले फ़्यूल कैप से लेकर सेंसर की दिक़्क़त या मिसफ़ायर तक। लगातार जलती लाइट आमतौर पर इमरजेंसी नहीं होती, पर टिमटिमाती लाइट का मतलब है जल्द गाड़ी रोकनी चाहिए। कोड रीडर से वजह पता चल सकती है।", "followup_turn": "क्या मैं ख़ुद कोड देख सकता हूँ?", "followup_reply": "हाँ। एक सस्ता OBD-II स्कैनर डैशबोर्ड के नीचे लगता है और फ़ोन पर कोड दिखाता है। मैकेनिक के पास जाने से पहले कोड खोजकर संभावित वजहें देख लें।", "q1": "मैंने अपनी कार की सर्विस कब करवाई थी?", "q2": "मेरी कार की पिछली सर्विस, जिसमें एयर फ़िल्टर बदला गया, किस तारीख़ को हुई थी?", "q3": "मेरे मैकेनिक पिछली सर्विस के तीन महीने बाद जाँच की सलाह देते हैं। मुझे किस तारीख़ से गिनना चाहिए?", "distractor_sentence": "मेरे पापा ने 28 फ़रवरी को अपनी कार की सर्विस करवाई थी।"}, "hg": {"memory_turn": "Mujhe apni car ki sahi mileage nahi pata, par yaad hai ki 10 February ko uski service karwayi thi, jab air filter badla tha. Ab engine light jal rahi hai. Iska kya matlab ho sakta hai?", "assistant_reply": "Iske kai matlab ho sakte hain, loose fuel cap se lekar sensor problem ya misfire tak. Lagataar jalti light usually emergency nahi hoti, par blink karti light ka matlab hai jaldi gaadi rokni chahiye. Code reader se wajah pata chal sakti hai.", "followup_turn": "Kya main khud code dekh sakta hoon?", "followup_reply": "Haan. Ek sasta OBD-II scanner dashboard ke neeche lagta hai aur phone pe code dikhata hai. Mechanic ke paas jaane se pehle code search karke possible wajah dekh lein.", "q1": "Maine apni car ki service kab karwayi thi?", "q2": "Meri car ki pichhli service, jismein air filter badla gaya, kis date ko hui thi?", "q3": "Mere mechanic pichhli service ke teen mahine baad check-up ki salaah dete hain. Mujhe kis date se ginna chahiye?", "distractor_sentence": "Mere papa ne 28 February ko apni car ki service karwayi thi."}}, "gold": {"answer_en": "February 10", "date": {"day": 10, "month": 2}, "slots": [{"name": "date", "aliases": ["February 10", "February 10th", "10 February", "10th February", "10th of February", "Feb 10", "10 Feb", "10 फ़रवरी", "10 फरवरी"]}]}, "distractor": {"date": {"day": 28, "month": 2}, "value_aliases": ["February 28", "February 28th", "28 February", "28th February", "28th of February", "Feb 28", "28 Feb", "28 फ़रवरी", "28 फरवरी"]}, "has_q3": true, "source": {"dataset": "LongMemEval", "question_id": "4bc144e2", "lme_type": "multi-session"}, "topic": "vehicles", "is_answerable": true, "evidence_position": 2, "filler_seed": 2744298547, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "abl_short": false, "abl_q3": true, "abl_digits": true, "banned_fillers": []}
{"item_id": "F019-ABS", "base_item": "F019", "memory_type": "fact", "is_answerable": false, "user_gender": "female", "text": {"en": {"q1": "How long did it take me to assemble my IKEA desk?", "q2": "How many hours did putting together my IKEA desk take?"}, "hi": {"q1": "मेरी IKEA डेस्क जोड़ने में मुझे कितना समय लगा?", "q2": "IKEA डेस्क असेंबल करने में मुझे कितने घंटे लगे?"}, "hg": {"q1": "Meri IKEA desk assemble karne mein mujhe kitna time laga?", "q2": "IKEA desk jodne mein mujhe kitne ghante lage?"}}, "gold": {"answer_en": "NO_ANSWER (only the bookshelf's assembly time, 4 hours, is stated; a desk is never mentioned)", "slots": []}, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "banned_fillers": [], "abl_short": false, "abl_q3": false, "abl_digits": false}
{"item_id": "F028-ABS", "base_item": "F028", "memory_type": "fact", "is_answerable": false, "user_gender": "male", "text": {"en": {"q1": "How much storage does my laptop have?", "q2": "What is the size of my work laptop's hard drive or SSD?"}, "hi": {"q1": "मेरे लैपटॉप में कितनी स्टोरेज है?", "q2": "मेरे काम वाले लैपटॉप की हार्ड ड्राइव या SSD कितनी बड़ी है?"}, "hg": {"q1": "Mere laptop mein kitni storage hai?", "q2": "Mere kaam wale laptop ki hard drive ya SSD kitni badi hai?"}}, "gold": {"answer_en": "NO_ANSWER (only the RAM, 16GB, is stated; storage is never mentioned)", "slots": []}, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "banned_fillers": ["FL28"], "abl_short": false, "abl_q3": false, "abl_digits": false}
{"item_id": "F030-ABS", "base_item": "F030", "memory_type": "fact", "is_answerable": false, "user_gender": "male", "text": {"en": {"q1": "Where did I do my Master's degree?", "q2": "From which university did I get my Master's?"}, "hi": {"q1": "मैंने अपनी मास्टर्स डिग्री कहाँ से की?", "q2": "मुझे मास्टर्स की डिग्री किस यूनिवर्सिटी से मिली?"}, "hg": {"q1": "Maine apni Master's degree kahan se ki?", "q2": "Mujhe Master's ki degree kis university se mili?"}}, "gold": {"answer_en": "NO_ANSWER (the user has not done a Master's; Stanford is only being considered, so 'Stanford' is a wrong answer)", "slots": []}, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "banned_fillers": [], "abl_short": false, "abl_q3": false, "abl_digits": false}
{"item_id": "F045-ABS", "base_item": "F045", "memory_type": "fact", "is_answerable": false, "user_gender": "female", "text": {"en": {"q1": "Which city does my brother live in?", "q2": "Where is my brother based?"}, "hi": {"q1": "मेरा भाई किस शहर में रहता है?", "q2": "मेरा भाई कहाँ रहता है?"}, "hg": {"q1": "Mera bhai kis city mein rehta hai?", "q2": "Mera bhai kahan rehta hai?"}}, "gold": {"answer_en": "NO_ANSWER (only sister Emily's city, Denver, and a cousin's city, Austin, are stated; no brother is mentioned)", "slots": []}, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "banned_fillers": [], "abl_short": false, "abl_q3": false, "abl_digits": false}
{"item_id": "P019-ABS", "base_item": "P019", "memory_type": "preference", "is_answerable": false, "user_gender": "female", "text": {"en": {"q1": "Which yoga pose do I like least?", "q2": "What is my least favourite pose in yoga?"}, "hi": {"q1": "मुझे कौन-सा योग आसन सबसे कम पसंद है?", "q2": "योग में मेरा सबसे नापसंद आसन कौन-सा है?"}, "hg": {"q1": "Mujhe kaunsa yoga pose sabse kam pasand hai?", "q2": "Yoga mein mera sabse naapasand pose kaunsa hai?"}}, "gold": {"answer_en": "NO_ANSWER (only the favourite pose, downward-facing dog, is stated)", "slots": []}, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "banned_fillers": ["FL06", "FL31"], "abl_short": false, "abl_q3": false, "abl_digits": false}
{"item_id": "P022-ABS", "base_item": "P022", "memory_type": "preference", "is_answerable": false, "user_gender": "male", "text": {"en": {"q1": "What drink do I always order at my favourite restaurant?", "q2": "At my favourite restaurant, which drink do I usually get?"}, "hi": {"q1": "अपने पसंदीदा रेस्टोरेंट में मैं हमेशा कौन-सा ड्रिंक मँगवाता हूँ?", "q2": "अपने पसंदीदा रेस्टोरेंट में मैं आमतौर पर कौन-सा पेय लेता हूँ?"}, "hg": {"q1": "Apne favourite restaurant mein main hamesha kaunsa drink order karta hoon?", "q2": "Apne favourite restaurant mein main usually kaunsa drink leta hoon?"}}, "gold": {"answer_en": "NO_ANSWER (only the restaurant, Ichiran, is stated; no drink is mentioned)", "slots": []}, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "banned_fillers": [], "abl_short": false, "abl_q3": false, "abl_digits": false}
{"item_id": "P026-ABS", "base_item": "P026", "memory_type": "preference", "is_answerable": false, "user_gender": "male", "text": {"en": {"q1": "Which cricket team do I support?", "q2": "What is my favourite cricket team?"}, "hi": {"q1": "मैं किस क्रिकेट टीम को सपोर्ट करता हूँ?", "q2": "मेरी पसंदीदा क्रिकेट टीम कौन-सी है?"}, "hg": {"q1": "Main kis cricket team ko support karta hoon?", "q2": "Meri favourite cricket team kaunsi hai?"}}, "gold": {"answer_en": "NO_ANSWER (only the football team, Manchester United, is stated)", "slots": []}, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "banned_fillers": [], "abl_short": false, "abl_q3": false, "abl_digits": false}
{"item_id": "P039-ABS", "base_item": "P039", "memory_type": "preference", "is_answerable": false, "user_gender": "female", "text": {"en": {"q1": "Which Depeche Mode song is my favourite?", "q2": "What is my favourite song by Depeche Mode?"}, "hi": {"q1": "डेपेश मोड का कौन-सा गाना मेरा पसंदीदा है?", "q2": "डेपेश मोड का मेरा पसंदीदा गाना कौन-सा है?"}, "hg": {"q1": "Depeche Mode ka kaunsa song mera favourite hai?", "q2": "Depeche Mode ka mera favourite song kaunsa hai?"}}, "gold": {"answer_en": "NO_ANSWER (only the favourite album, Violator, is stated; no song is mentioned)", "slots": []}, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "middle", "banned_fillers": [], "abl_short": false, "abl_q3": false, "abl_digits": false}
{"item_id": "E021-ABS", "base_item": "E021", "memory_type": "event", "is_answerable": false, "user_gender": "female", "text": {"en": {"q1": "When did I run my first half marathon?", "q2": "On what date did I complete my first half marathon?"}, "hi": {"q1": "मैंने अपनी पहली हाफ़ मैराथन कब दौड़ी?", "q2": "मैंने अपनी पहली हाफ़ मैराथन किस तारीख़ को पूरी की?"}, "hg": {"q1": "Maine apni pehli half marathon kab daudi?", "q2": "Maine apni pehli half marathon kis date ko complete ki?"}}, "gold": {"answer_en": "NO_ANSWER (only the date the running shoes were bought, January 10, is stated)", "slots": []}, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "banned_fillers": [], "abl_short": false, "abl_q3": false, "abl_digits": false}
{"item_id": "E032-ABS", "base_item": "E032", "memory_type": "event", "is_answerable": false, "user_gender": "male", "text": {"en": {"q1": "When did I move into my new townhouse?", "q2": "On what date did I move into the Brookside townhouse?"}, "hi": {"q1": "मैं अपने नए टाउनहाउस में कब शिफ़्ट हुआ?", "q2": "मैं ब्रुकसाइड वाले टाउनहाउस में किस तारीख़ को रहने आया?"}, "hg": {"q1": "Main apne naye townhouse mein kab shift hua?", "q2": "Main Brookside wale townhouse mein kis date ko rehne aaya?"}}, "gold": {"answer_en": "NO_ANSWER (only the offer date, February 25, is stated; the sale has not even closed)", "slots": []}, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "banned_fillers": [], "abl_short": false, "abl_q3": false, "abl_digits": false}
{"item_id": "E040-ABS", "base_item": "E040", "memory_type": "event", "is_answerable": false, "user_gender": "male", "text": {"en": {"q1": "When did I buy my road bike?", "q2": "On what date did I purchase my road bike?"}, "hi": {"q1": "मैंने अपनी रोड बाइक कब ख़रीदी?", "q2": "मैंने अपनी रोड बाइक किस तारीख़ को ख़रीदी थी?"}, "hg": {"q1": "Maine apni road bike kab khareedi?", "q2": "Maine apni road bike kis date ko khareedi thi?"}}, "gold": {"answer_en": "NO_ANSWER (only the service date at Pedal Power, March 10, is stated)", "slots": []}, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "late", "banned_fillers": [], "abl_short": false, "abl_q3": false, "abl_digits": false}
{"item_id": "E046-ABS", "base_item": "E046", "memory_type": "event", "is_answerable": false, "user_gender": "male", "text": {"en": {"q1": "When did I start going to yoga classes?", "q2": "On what date did I begin attending yoga classes?"}, "hi": {"q1": "मैंने योगा क्लास जाना कब शुरू किया?", "q2": "मैंने किस तारीख़ से योगा क्लास में जाना शुरू किया?"}, "hg": {"q1": "Maine yoga classes jaana kab shuru kiya?", "q2": "Maine kis date se yoga classes mein jaana shuru kiya?"}}, "gold": {"answer_en": "NO_ANSWER (only the mindfulness workshop date, December 12, is stated; yoga classes are never mentioned)", "slots": []}, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 19 Nov 2026", "evidence_class": "early", "banned_fillers": ["FL06", "FL31"], "abl_short": false, "abl_q3": false, "abl_digits": false}


In [ ]:
%%writefile /kaggle/working/clm/data/haystacks/filler_pool_en.jsonl
{"filler_id": "FL01", "session_id": "ultrachat_201585", "turns": [{"role": "user", "content": "In what ways did the Beatles' fame and exposure affect Lennon's songwriting?"}, {"role": "assistant", "content": "As an AI language model, I don't have personal beliefs, but I can give you a general answer. The Beatles' fame and exposure had a profound impact on Lennon's songwriting. With the band's massive success, Lennon was given a platform to share his music with the world, and he used it to write some of the most influential songs of all time."}, {"role": "user", "content": "Can you give me some examples of specific songs that were influenced by Lennon's fame and exposure?"}, {"role": "assistant", "content": "Sure, here are some examples of Lennon's songs that were influenced by his fame and exposure: 1. \"Help!\" - This song was written during a time when Lennon was feeling overwhelmed by the band's success and the constant demands placed on him as a public figure. 2. \"Strawberry Fields Forever\" - This song was inspired by Lennon's memories of growing up near Strawberry Field, a Salvation Army children's home."}]}
{"filler_id": "FL02", "session_id": "5fb7fd18_1", "turns": [{"role": "user", "content": "I'm looking for some new baking inspiration. I've been doing a lot of sweet treats lately, but I want to try something savory. Do you have any recommendations for a good savory bread recipe? By the way, I enjoyed my sourdough bread with a pat of butter the last time I made it, and I'm hoping to recreate that flavor."}, {"role": "assistant", "content": "A savory bread enthusiast in the making! I'd be delighted to help you explore the world of savory breads. Since you mentioned enjoying sourdough with butter, I'll suggest some recipes that share similar characteristics or flavor profiles. Here are a few ideas to get you started: 1. Rosemary and Olive Oil Focaccia: This Italian bread is perfect for sopping up olive oil, much like your sourdough with butter."}, {"role": "user", "content": "These recipes sound amazing, especially the Rosemary and Olive Oil Focaccia. I've never worked with rosemary in bread before, so I'm excited to try it out. Can you give me some tips on how to handle fresh rosemary and how much to use in the recipe?"}, {"role": "assistant", "content": "Working with fresh rosemary can be a bit tricky, but with some tips, you'll be a pro in no time! Handling Fresh Rosemary: 1. Choose fresh, fragrant sprigs: Select rosemary sprigs with a vibrant green color and a potent, piney aroma. Avoid wilted or brownish sprigs, as they can be bitter. 2. Store rosemary properly: Keep fresh rosemary in a plastic bag or wrap it in plastic wrap in the refrigerator to maintain its flavor and aroma."}]}
{"filler_id": "FL03", "session_id": "afbd7193_2", "turns": [{"role": "user", "content": "I'm looking for some tips on organizing my bookshelf. I just did a deep clean of my living room last Sunday and realized I need some ideas on how to maximize my bookshelf space. By the way, I commit to a 30-minute deep clean session every Sunday, so I want to make the most out of it."}, {"role": "assistant", "content": "Kudos to you for committing to a regular deep clean session! That's a great habit to maintain a tidy and organized living space. Now, let's dive into some tips to help you maximize your bookshelf space and make the most of your 30-minute Sunday deep clean session: 1. Purge and categorize: Start by removing all books from your shelf and sorting them into categories (e.g., fiction, non-fiction, biographies, self-help, etc.)."}, {"role": "user", "content": "I like the idea of categorizing and assigning a home for each category. But how do I decide what to keep and what to get rid of? I have a lot of books that I haven't read in years, but they hold sentimental value."}, {"role": "assistant", "content": "Deciding what to keep and what to let go of can be a challenging task, especially when sentimental value is involved. Here are some tips to help you make that decision: Sentimental value vs. practicality: 1. Ask yourself: When was the last time I looked at or read this book? Be honest. If it's been years, it's likely you won't miss it if you let it go. 2."}]}
{"filler_id": "FL04", "session_id": "ultrachat_391062", "turns": [{"role": "user", "content": "What are some common gaming terms and acronyms that I should be familiar with?"}, {"role": "assistant", "content": "Here are some common gaming terms and acronyms: 1. FPS - First-person shooter 2. MMO - Massively multiplayer online 3. PVP - Player versus player 4. NPC - Non-player character 5. RNG - Random number generator 6. MOBA - Multiplayer online battle arena 7. DLC - Downloadable content 8. LFG - Looking for group 9. AFK - Away from keyboard 10. DPS - Damage per second 11."}, {"role": "user", "content": "Dude, I'm so sick of the RNG in this game. I've been grinding for hours and still haven't gotten the drop I need. It's like the game is rigged against me."}, {"role": "assistant", "content": "I understand that the user is expressing frustration with a game due to rng (random number generator), meaning the outcome of certain events or drops is determined randomly rather than based on skill. the user is unhappy that after spending hours in the game and grinding, they still haven't obtained the desired drop which they believe to be due to poor luck or unfair odds."}]}
{"filler_id": "FL05", "session_id": "ultrachat_274962", "turns": [{"role": "user", "content": "How did Sulla's military victories and conquests shape his political ideology and influence his policies as a leader in Rome?"}, {"role": "assistant", "content": "Sulla's military victories and conquests played a significant role in shaping his political ideology and influencing his policies as a leader in Rome. As a military general, Sulla was known for his exceptional skills and tactical brilliance on the battlefield. Through his military successes, he gained immense popularity among the Roman people and established a power base that he would later use to seize control of the state."}, {"role": "user", "content": "It seems like Sulla was more interested in consolidating his own power than truly serving the Roman people. What impact did his policies have on the citizens of Rome?"}, {"role": "assistant", "content": "You are correct in stating that Sulla was more interested in consolidating his own power than serving the Roman people. His policies had a significant impact on the citizens of Rome that ultimately contributed to the decline of the Roman Republic. One of the most significant impacts of Sulla's policies was the centralization of power in the hands of a few individuals."}]}
{"filler_id": "FL06", "session_id": "f62c04c6", "turns": [{"role": "user", "content": "I'm thinking of creating a content calendar to plan out my social media posts for the next month. Can you help me with some ideas on what types of posts I should make and how often I should post?"}, {"role": "assistant", "content": "Creating a content calendar is a great way to plan and organize your social media content in advance. The type and frequency of posts will depend on your target audience, goals, and the platforms you're using, but here are some general ideas and guidelines to get you started: Post Types: 1. Educational/Informative: Share tips, tutorials, industry news, or how-to guides related to your niche. 2. Promotional: Share updates about your products, services, or events. 3."}, {"role": "user", "content": "That's a lot of great info! I think I'll start by creating a template for each platform and then filling it in with ideas. Can you give me some ideas for Instagram Reels? I recently posted one about my morning yoga routine and got 50 views and 10 likes, so I'm thinking of doing a series on yoga or meditation."}, {"role": "assistant", "content": "Instagram Reels are a fantastic way to engage with your audience and increase visibility. Congratulations on your first Reel's success! A yoga or meditation series is an excellent idea, especially since you've already seen interest in your morning yoga routine Reel. Here are some ideas to get you started: Yoga Series: 1. Morning Routine Variations: Share different morning yoga routines, each focusing on a specific area, like flexibility, strength, or relaxation. 2."}]}
{"filler_id": "FL07", "session_id": "8969b3ca_2", "turns": [{"role": "user", "content": "I'm looking to get an estimate of the total costs involved in buying a house. Can you help me with that?"}, {"role": "assistant", "content": "Estimating the total costs involved in buying a house can help you plan and budget for this significant investment. Here's a breakdown of the typical costs you can expect to incur: 1. Purchase Price: This is the biggest cost of buying a house, and it's the amount you'll pay to the seller for the property. 2. Down Payment: This is a percentage of the purchase price that you'll pay upfront, typically ranging from 3.5% to 20% of the purchase price."}, {"role": "user", "content": "I'm actually getting close to closing on a house, and I'd like to get a more accurate estimate of my closing costs. The sellers agreed to give me a credit of $2,000 towards closing costs, so I'd like to factor that in. Can you help me with that?"}, {"role": "assistant", "content": "Congratulations on getting close to closing on your new home! I'd be happy to help you estimate your closing costs and factor in the $2,000 credit from the sellers. To get an accurate estimate, I'll need some more information from you. Please provide the following details: 1. Purchase price: What is the final purchase price of the house? 2. Loan amount: How much are you borrowing to finance the purchase? 3. Loan type: What type of loan are you getting?"}]}
{"filler_id": "FL08", "session_id": "50d66391_7", "turns": [{"role": "user", "content": "I'm looking for some advice on bedroom lighting. I've been thinking about getting some floor lamps for my bedroom to create a relaxing reading nook."}, {"role": "assistant", "content": "What a lovely idea! Floor lamps can be a great addition to a bedroom, creating a cozy and inviting atmosphere perfect for a relaxing reading nook. Here are some tips to consider when choosing the right floor lamps for your bedroom: 1. Consider the purpose: Since you mentioned a reading nook, you'll want to prioritize task lighting. Look for lamps with adjustable arms or heads that can direct light exactly where you need it. 2."}, {"role": "user", "content": "I'm currently using a bedside lamp with a new bulb that I got from Home Depot for around $15, and I really like its warm tone. Do you think I should look for floor lamps with similar bulb options or color temperature?"}, {"role": "assistant", "content": "You've already found a bulb you like, which is a great starting point! Yes, it's a good idea to look for floor lamps that can accommodate similar bulb options or have a similar color temperature to the one you're currently using. Since you mentioned you like the warm tone of the bulb you got from Home Depot, I'm assuming it's likely in the range of 2700K-3000K, which is a warm white or soft white color temperature."}]}
{"filler_id": "FL09", "session_id": "ultrachat_436464", "turns": [{"role": "user", "content": "What are the key initiatives taken by World Health Organization (WHO) to improve public health across the globe?"}, {"role": "assistant", "content": "The World Health Organization (WHO) has undertaken numerous key initiatives to improve public health across the globe, including: 1. Universal Health Coverage: WHO aims to ensure that everyone can access the health services they need without facing financial hardship. This includes expanding access to essential health services and addressing health inequalities. 2. Global Vaccine Action Plan: WHO works to ensure that all people, regardless of where they live, have access to life-saving vaccines."}, {"role": "user", "content": "It's great to see all these initiatives by WHO. But do you think they are effective in improving public health?"}, {"role": "assistant", "content": "I do not have personal opinions. however, who's initiatives have shown significant progress in improving public health worldwide. for example, the global vaccine action plan has contributed to increasing vaccination coverage in many countries, and the tobacco free initiative has been successful in reducing the demand for tobacco products. similarly, who's efforts in disease control, emergency response, and mental health promotion have also led to positive outcomes."}]}
{"filler_id": "FL10", "session_id": "429be85c_4", "turns": [{"role": "user", "content": "I'm looking to try out some new breakfast treats to pair with my coffee on weekends. I like to indulge in a bigger coffee break with some breakfast treats on weekends, and I'm getting a bit tired of my usual cinnamon rolls. Do you have any recipe suggestions or recommendations?"}, {"role": "assistant", "content": "I'm happy to help! Indulging in a delicious breakfast treat with a warm cup of coffee on weekends is one of life's greatest pleasures. I've got some scrumptious recipe suggestions to help you mix things up and find a new favorite breakfast companion. Here are a few ideas to get you started: 1. Cranberry Orange Scones: Flaky, buttery scones infused with dried cranberries and orange zest are a perfect match for a rich cup of coffee. 2."}, {"role": "user", "content": "What's the difference between using unsalted butter and salted butter in these recipes? Does it really make a difference in the final taste and texture?"}, {"role": "assistant", "content": "Using unsalted butter versus salted butter can indeed make a difference in the final taste and texture of your baked goods."}]}
{"filler_id": "FL11", "session_id": "33994682_3", "turns": [{"role": "user", "content": "I'm considering freezing my eggs soon and I've been doing some research, but I'm not sure about the process and costs involved. Can you walk me through the general procedure and give me an idea of how much it typically costs? By the way, my sister, who's only three years older than me, is 35 now, and it's crazy to think about how fast time has flown by."}, {"role": "assistant", "content": "I'd be happy to help you understand the process and costs involved in egg freezing! The Process: Egg freezing, also known as oocyte cryopreservation, is a multi-step process that typically takes around 2-3 weeks. Here's an overview: 1. Initial Consultation: You'll meet with a fertility specialist (reproductive endocrinologist) to discuss your medical history, fertility goals, and any concerns you may have. 2."}, {"role": "user", "content": "I'm also curious to know more about the success rates of egg freezing. What are the chances of a frozen egg resulting in a healthy pregnancy and birth, and does my age play a significant role in this?"}, {"role": "assistant", "content": "Success Rates of Egg Freezing: The success rates of egg freezing are improving with advancements in technology and vitrification methods. However, it's essential to understand that egg freezing is not a guarantee of a successful pregnancy or birth. The chances of a frozen egg resulting in a healthy pregnancy and birth depend on several factors, including: 1."}]}
{"filler_id": "FL12", "session_id": "f75238a5_2", "turns": [{"role": "user", "content": "I've been thinking about exploring AI tools for my project, and I was wondering if you could recommend some resources for learning about the latest developments in AI. By the way, I recently had a great conversation with someone about this topic - we discussed the latest developments in AI for about 10 minutes, and it really sparked my interest."}, {"role": "assistant", "content": "I'm thrilled to hear that you're interested in exploring AI tools for your project! There are indeed many exciting developments in the field of AI, and staying up-to-date with the latest advancements can be incredibly valuable. As for resources, I'd be happy to recommend some top-notch ones to get you started: 1. The AI Alignment Podcast: A fantastic podcast that delves into the latest AI research, applications, and implications."}, {"role": "user", "content": "I'll definitely check out those resources, especially the podcast and the blog by Emerj. Can you tell me more about the current state of AI applications in industries like healthcare and finance?"}, {"role": "assistant", "content": "AI is transforming industries across the board, and healthcare and finance are no exceptions. Here's a brief overview of the current state of AI applications in these sectors: Healthcare: 1. Disease diagnosis: AI-powered systems can analyze medical images, such as X-rays and MRIs, to detect diseases like cancer, diabetes, and cardiovascular disease with high accuracy. 2. Personalized medicine: AI helps tailor treatment plans to individual patients based on their genetic profiles, medical histories, and lifestyle factors. 3."}]}
{"filler_id": "FL13", "session_id": "ultrachat_364297", "turns": [{"role": "user", "content": "Can you compare and contrast the breeding habits of ducks and geese?"}, {"role": "assistant", "content": "I do not have personal experiences, emotions, or beliefs. however, i can provide you the information you need. ducks and geese are both waterfowl and have some similarities in breeding habits, but there are also some differences. here are some comparisons and contrasts: similarities: - both ducks and geese form pair bonds during the breeding season. - both species typically mate for life or long-term, although some will switch partners."}, {"role": "user", "content": "Can you tell me more about the courtship displays of ducks? It sounds interesting."}, {"role": "assistant", "content": "Sure, I'd be happy to! Ducks are known for their elaborate courtship displays, which are basically a series of behaviors used to attract a mate. Here are some examples of courtship displays that you might see in ducks: - Head-throws: Some species of ducks will toss their heads back and forth and make various sounds to show off their physical prowess and vocal abilities."}]}
{"filler_id": "FL14", "session_id": "08ca1f31_2", "turns": [{"role": "user", "content": "I'm thinking of planning a longer bike trip on my Trek Emonda this weekend, maybe 80-100 miles. Can you help me with some route suggestions or tips for preparing my bike for a longer ride? By the way, I recently had a tune-up done on it about 3 weeks ago, and the bike shop replaced the chain and adjusted the derailleurs, which cost me $80."}, {"role": "assistant", "content": "A longer bike trip sounds like a great adventure! I'd be happy to help you with route suggestions and tips for preparing your bike. Route Suggestions: Before we dive into bike prep, let's talk routes. Since you didn't specify a location, I'll provide some general suggestions. You can use online tools like MapMyRide, Ride with GPS, or Strava to find routes in your area. 1."}, {"role": "user", "content": "I was thinking of doing a route with some hills, as I've been doing a lot of flat commutes on my Trek Emonda lately. Do you have any specific hill-climbing tips or recommendations for gearing and cadence?"}, {"role": "assistant", "content": "Hill-climbing is a great way to challenge yourself and improve your overall cycling fitness. Here are some hill-climbing tips, gearing recommendations, and cadence advice to help you tackle those inclines on your Trek Emonda: Hill-Climbing Tips: 1. Pace yourself: Start with a steady, moderate pace, and try to maintain it throughout the climb. Avoid surging or sprinting, as this can lead to exhaustion. 2."}]}
{"filler_id": "FL15", "session_id": "1c1f5ccc_6", "turns": [{"role": "user", "content": "I'm interested in learning more about the cultural significance of Indian clothing, particularly the scarf I recently bought at a festival. Can you tell me more about the history and symbolism behind traditional Indian scarves? By the way, I've been trying to learn more about my own cultural heritage, including the history and customs of India, and this seems like a great place to start."}, {"role": "assistant", "content": "What a wonderful topic! Indian clothing, particularly the scarf you're interested in, holds a rich cultural significance that dates back centuries. I'm more than happy to delve into the history and symbolism behind traditional Indian scarves, and I'm thrilled to hear that you're exploring your cultural heritage. In India, scarves are an integral part of traditional attire, and they're known by different names depending on the region. Some common types of Indian scarves include: 1."}, {"role": "user", "content": "I'd like to know more about the different types of fabrics used in traditional Indian scarves. Are there any specific fabrics that are considered more symbolic or prestigious than others?"}, {"role": "assistant", "content": "Fabrics play a vital role in traditional Indian scarves, and each type has its own unique characteristics, significance, and cultural associations. Let me introduce you to some of the most common and revered fabrics used in Indian scarves: 1. Silk: Silk is considered the most prestigious and luxurious fabric in Indian textiles. It's often associated with royalty, wealth, and grandeur. Silk scarves are highly prized for their softness, luster, and durability."}]}
{"filler_id": "FL16", "session_id": "0651f7bd_2", "turns": [{"role": "user", "content": "I'm looking for some tips on how to properly store and display my vintage camera collection. I've got a few rare ones and I want to make sure they're protected from dust and damage. By the way, I got a rare card of Cal Ripken Jr. from 1982 in the mail today, but that's a whole other collection of mine."}, {"role": "assistant", "content": "Congratulations on your rare Cal Ripken Jr. card! That's exciting! Now, let's focus on storing and displaying your vintage camera collection. Proper storage and display are crucial to preserving the condition and value of your cameras. Here are some tips to help you keep your vintage cameras protected from dust and damage: Storage: 1. Climate control: Store your cameras in a room with a consistent temperature between 60°F to 70°F (15°C to 21°C) and humidity below 60%."}, {"role": "user", "content": "Can you recommend some good resources for learning about camera restoration and maintenance? I've got a few cameras that need some TLC, and I'm not sure where to start."}, {"role": "assistant", "content": "Camera restoration and maintenance are essential skills for any camera enthusiast. Here are some excellent resources to help you learn and get started: Online Forums and Communities: 1. The Camera Repair Forum: A dedicated forum with a vast community of experts and enthusiasts sharing knowledge and experiences on camera repair and maintenance. 2. Photography Stack Exchange: A Q&A platform where you can ask questions and get answers from experienced photographers and camera enthusiasts. 3."}]}
{"filler_id": "FL17", "session_id": "01d4b26b_1", "turns": [{"role": "user", "content": "I'm looking for some gift ideas for a coworker's baby shower. I want something eco-friendly and cute. By the way, I just gave my sister a silver necklace with a small pendant for her birthday last month, and she loved it."}, {"role": "assistant", "content": "What a lovely gift for your sister! I'm happy to help you with some eco-friendly and adorable gift ideas for your coworker's baby shower. Here are some suggestions: 1. Organic Cotton Onesie and Baby Blanket Set: A soft, eco-friendly onesie and blanket set made from organic cotton. You can even opt for a set with a cute design or a personalized message. 2. Bamboo Baby Toys: Bamboo is a sustainable, eco-friendly material that's perfect for baby toys."}, {"role": "user", "content": "I like the organic cotton onesie and baby blanket set idea. Can you tell me more about the brands that offer customization options?"}, {"role": "assistant", "content": "Organic cotton onesie and baby blanket sets make for a thoughtful and eco-friendly gift. Several brands offer customization options for onesies and baby blankets. Here are a few popular ones: 1. Hanna Andersson: Known for their soft, organic cotton clothing, Hanna Andersson offers customization options for onesies, sleepers, and blankets. You can choose from various fonts, colors, and designs to create a personalized gift. 2."}]}
{"filler_id": "FL18", "session_id": "e2691068", "turns": [{"role": "user", "content": "I'm looking for some new breakfast ideas that can give me a energy boost. Do you have any suggestions?"}, {"role": "assistant", "content": "A energizing breakfast is just what you need to start your day off right! I've got some delicious and nutritious breakfast ideas that'll give you a boost of energy to tackle your day: 1. Avocado Toast with Poached Eggs: Whole grain toast topped with mashed avocado, a poached egg, and a sprinkle of red pepper flakes. The healthy fats from avocado and protein from the egg will keep you full and focused. 2."}, {"role": "user", "content": "I like the sound of the Avocado Toast with Poached Eggs and the Smoothie Bowl. Do you have any tips on how to prepare them quickly in the morning?"}, {"role": "assistant", "content": "Great choices! Both Avocado Toast with Poached Eggs and Smoothie Bowl are delicious and nutritious breakfast options that can be prepared quickly in the morning. Here are some tips to help you prepare them quickly: Avocado Toast with Poached Eggs: 1. Prep ahead: Mash the avocado and store it in an airtight container in the fridge overnight. This will save you about 5 minutes in the morning. 2."}]}
{"filler_id": "FL19", "session_id": "ultrachat_415674", "turns": [{"role": "user", "content": "How have advancements in neuroscience and psychology changed our understanding of the human mind and behavior?"}, {"role": "assistant", "content": "Advancements in neuroscience and psychology have revolutionized our understanding of the human mind and behavior in a number of ways: 1. Understanding of the Brain: Advancements in neuroimaging technology, such as functional magnetic resonance imaging (fMRI) and electroencephalogram (EEG), have allowed us to measure and better understand brain activity. This technology has allowed researchers to investigate how different parts of the brain are involved in various cognitive functions, such as perception, attention, emotion, decision-making, and social cognition. 2."}, {"role": "user", "content": "It's fascinating to see how developments in neuroscience and psychology have led to new insights on the human mind and behavior. What do you think will be the next breakthrough in these fields?"}, {"role": "assistant", "content": "I do not have personal opinions or the ability to predict future events. however, based on ongoing research and recent developments, there are several areas where we might expect to see significant breakthroughs in neuroscience and psychology in the near future. one of the areas of research that is receiving increasing attention is the study of the gut-brain axis."}]}
{"filler_id": "FL20", "session_id": "ultrachat_93661", "turns": [{"role": "user", "content": "What are some best practices for staying organized and managing time effectively in a fast-paced work environment?"}, {"role": "assistant", "content": "1. Use a planner or calendar: Write down all of your tasks and deadlines in one place to help you stay on top of your schedule and prevent things from slipping through the cracks. 2. Prioritize your tasks: Prioritize your work by importance and deadline. Start with the most critical work first, and then work your way down to less important tasks. 3."}, {"role": "user", "content": "I especially struggle with eliminating distractions, so I'll definitely work on that. Do you have any recommendations for specific productivity apps or tools?"}, {"role": "assistant", "content": "Yes, there are many productivity apps and tools available that can help you stay organized and manage your time effectively. Here are some recommendations: 1. Trello: A visual project management tool that allows you to create boards, cards, and lists to organize your tasks and workflows. 2. Focus@will: A music streaming service that plays background music designed to help you stay focused and productive. 3."}]}
{"filler_id": "FL21", "session_id": "b7c5c12f_2", "turns": [{"role": "user", "content": "I'm looking for some recommendations on sustainable living. I've been reading a lot about it lately, especially in my weekly magazine that I've been receiving every week. Anyway, can you give me some tips on how to reduce my carbon footprint in daily life?"}, {"role": "assistant", "content": "Sustainable living is such an important topic! I'm more than happy to help you with that. Here are some practical tips to reduce your carbon footprint in daily life: 1. Reduce Energy Consumption Turn off lights, electronics, and appliances when not in use to avoid standby power consumption. Switch to energy-efficient LED bulbs. Adjust your thermostat to use less energy for heating and cooling. 2. Conserve Water Take shorter showers (5-10 minutes) to save water and energy."}, {"role": "user", "content": "I'm actually quite interested in the sustainable fashion aspect, which I recently read about in my magazine. Can you give me some tips on how to make my wardrobe more eco-friendly?"}, {"role": "assistant", "content": "Sustainable fashion is a wonderful area to focus on! The fashion industry has a significant environmental impact, but by making a few conscious choices, you can reduce your fashion footprint. Here are some tips to help you create a more eco-friendly wardrobe: 1. Buy Second-Hand First Thrift stores, consignment shops, and online marketplaces like eBay, ThredUp, or Poshmark are great resources for second-hand shopping. You can find unique, high-quality pieces while reducing the demand for new, resource-intensive clothing. 2."}]}
{"filler_id": "FL22", "session_id": "df6970d8", "turns": [{"role": "user", "content": "I'm looking for recommendations on new board games to play with my friends and family. Any suggestions?"}, {"role": "assistant", "content": "New board games to play with friends and family? I've got a treasure trove of suggestions for you! Here are some modern classics, hidden gems, and popular games across various genres: Strategy and Competition 1. Gloomhaven: A cooperative campaign-based game with modular board and Legacy-style gameplay. 2. Scythe: A worker placement game set in an alternate history of the 1920s, with engine-building and area control. 3."}, {"role": "user", "content": "I've played Ticket to Ride and Settlers of Catan before, and they're great games. Have you got any recommendations for games that are similar to Scattergories?"}, {"role": "assistant", "content": "Scattergories is a fantastic game! If you enjoy the creative, word-based gameplay, here are some recommendations that might scratch that same itch: 1. Codenames: While not exactly like Scattergories, Codenames involves word-guessing and creative thinking. Players try to identify words based on one-word clues given by their teammate. 2. Quelf: A party game that challenges players to come up with unique words or phrases that fit certain criteria, often with hilarious results. 3."}]}
{"filler_id": "FL23", "session_id": "ultrachat_518652", "turns": [{"role": "user", "content": "What was the significance of the Trojan War in Greek mythology?"}, {"role": "assistant", "content": "The Trojan War was one of the most important events in Greek mythology. It was a war fought between the Greeks and the Trojans, and it lasted for ten years. The war was fought over Helen of Troy, who was the wife of Menelaus, the king of Sparta. Helen had been abducted by Paris, the prince of Troy, and taken to Troy. The war was significant because it was said to have been fought for honor and glory."}, {"role": "user", "content": "Which side ultimately won the Trojan War?"}, {"role": "assistant", "content": "The Greeks ultimately won the Trojan War by using the Trojan Horse, a wooden horse filled with Greek warriors, to invade and conquer the city of Troy. The Trojan Horse was left outside the gates of Troy as a supposed offering to the gods, and the Trojans brought it inside the city. That night, the Greek soldiers emerged from the horse and opened the gates of Troy, allowing the Greek army to enter and conquer the city."}]}
{"filler_id": "FL24", "session_id": "ultrachat_106352", "turns": [{"role": "user", "content": "What is the recommended amount of sleep per night for a healthy adult?"}, {"role": "assistant", "content": "The recommended amount of sleep per night for a healthy adult is 7-9 hours."}, {"role": "user", "content": "That sounds like a lot of time to spend sleeping. Can't I just get by with 5-6 hours?"}, {"role": "assistant", "content": "As an AI language model, I cannot give medical advice or suggest making changes to health routines. However, studies have shown that getting less than 7 hours of sleep consistently can lead to negative health effects, including decreased cognitive function, fatigue, and a weakened immune system. It is recommended that adults aim for at least 7 hours of sleep per night to maintain good health."}]}
{"filler_id": "FL25", "session_id": "fb793c87_2", "turns": [{"role": "user", "content": "I'm looking for some new shows to watch on HBO Max. I've been meaning to check out \"Game of Thrones\" for a while, and I finally started it about a month ago. I've finished the first four seasons so far, and I'm really enjoying it. Do you have any other recommendations similar to \"Game of Thrones\"?"}, {"role": "assistant", "content": "\"Game of Thrones\" is an epic show! You're halfway through, and I'm sure you're hooked. If you're looking for similar shows on HBO Max, I've got some fantastic recommendations for you: 1. Westworld: Another HBO epic, set in a futuristic theme park where guests can interact with lifelike robots. It's a complex, thought-provoking series with a similar blend of action, drama, and politics. 2. Vikings: A historical drama that follows the adventures of Ragnar Lothbrok, a legendary Viking warrior."}, {"role": "user", "content": "That's a great list! I'll definitely check some of them out. I've also been watching some documentaries on Hulu, particularly the ones on true crime and science. Do you have any recommendations for documentaries on HBO Max?"}, {"role": "assistant", "content": "HBO Max has an impressive collection of documentaries, and I've got some fantastic recommendations for you: True Crime: 1. I'll Be Gone in the Dark: A documentary series based on Michelle McNamara's book about the Golden State Killer. It's a gripping and thought-provoking exploration of the case and its impact on the victims and their families. 2."}]}
{"filler_id": "FL26", "session_id": "b0b73826", "turns": [{"role": "user", "content": "I'm trying to plan a get-together with my friends this weekend. Do you have any recommendations for a good brunch spot?"}, {"role": "assistant", "content": "I'd be happy to help! However, I'm a large language model, I don't have personal experiences or real-time information about specific locations. But I can provide you with some general tips and questions to help you find a great brunch spot for your gathering: 1. Consider your friends' preferences: Ask your friends about their dietary restrictions, favorite cuisines, and preferred ambiance. This will help you narrow down options. 2."}, {"role": "user", "content": "I was thinking of trying out a new place, but now I'm considering going back to a spot I've been to recently with my parents. It had a really great atmosphere and delicious food."}, {"role": "assistant", "content": "That's a great idea! Going back to a place you've been to before can be a great option, especially if you know and love the atmosphere and food. It's always nice to have a reliable spot that you can count on to deliver a great experience. Plus, since you've been there with your parents, you might have a special connection to the place, and it could be fun to share that experience with your friends."}]}
{"filler_id": "FL27", "session_id": "ultrachat_576167", "turns": [{"role": "user", "content": "How do the types of lenses used in photography affect the final image captured?"}, {"role": "assistant", "content": "The types of lenses used in photography affect the final image in several ways: 1. Focal length: The focal length of a lens determines its angle of view and the magnification of the subject. A longer focal length lens (telephoto) will compress the perspective and create a shallower depth of field, while a shorter focal length lens (wide-angle) will exaggerate the perspective and have a wider depth of field. 2."}, {"role": "user", "content": "How can I choose the right lens for a particular subject or situation in photography?"}, {"role": "assistant", "content": "Choosing the right lens for a particular subject or situation in photography can greatly impact the final image. Here are some considerations to help you choose the right lens: 1. Focal length: Consider the distance between you and the subject. If you are shooting a subject from afar, then consider using a longer focal length lens (telephoto). If you are closer to your subject, then consider using a shorter focal length lens (wide-angle). 2. Aperture: Consider the lighting conditions."}]}
{"filler_id": "FL28", "session_id": "ultrachat_182518", "turns": [{"role": "user", "content": "How has the mining industry impacted the economy and social structure of San Luis Potosí throughout history?"}, {"role": "assistant", "content": "The mining industry has had a significant impact on the economy and social structure of San Luis Potosí throughout history. The discovery of silver deposits in the region in the 16th century led to the establishment of numerous mines and mining towns, which in turn attracted a large population of migrants seeking work and other opportunities."}, {"role": "user", "content": "It's really interesting to learn about the impact of mining on San Luis Potosí. Have there been any efforts to address the negative consequences of the industry?"}, {"role": "assistant", "content": "Yes, there have been efforts to address the negative consequences of the mining industry in San Luis Potosí. In recent years, there has been increased awareness of the environmental impacts of mining on local communities and ecosystems, leading to the adoption of stricter regulations and the development of more sustainable mining practices."}]}
{"filler_id": "FL29", "session_id": "98cd882c_1", "turns": [{"role": "user", "content": "I'm looking for some ideas for a new sewing project. I just finished making a quilt for my sister's birthday and I'm itching to start something new. By the way, it took me around 3 weeks to complete the quilt, and I'm really happy with how it turned out."}, {"role": "assistant", "content": "Congratulations on completing the quilt for your sister's birthday! That's a wonderful accomplishment, and I'm sure she'll treasure it. Three weeks is a great timeframe to complete a quilt, considering the amount of time and effort that goes into it. Now, let's get started on brainstorming ideas for your next sewing project! Here are a few suggestions to get you going: 1. Tote Bag or Purse: A functional and practical project that can be completed quickly."}, {"role": "user", "content": "I've been thinking about making a sundress for summer, maybe something with a floral pattern. I've never made a garment before, so I'm a bit nervous about it. Do you have any tips or patterns that you would recommend for a beginner like me?"}, {"role": "assistant", "content": "Making a sundress is a great project for a beginner! With a little guidance, you'll be able to create a beautiful and comfortable garment for the summer. I'm happy to help you with tips and pattern recommendations. Tips for a Beginner: 1. Choose a simple pattern: Look for patterns labeled \"beginner-friendly\" or \"easy.\" Avoid patterns with too many pieces, complex construction, or intricate details. 2. Select a forgiving fabric: Opt for a lightweight, breathable fabric like cotton, linen, or rayon."}]}
{"filler_id": "FL30", "session_id": "f5f1ff92_1", "turns": [{"role": "user", "content": "I'm looking to get some advice on the mortgage application process. I've already submitted all the required documents to the lender, including my pay stubs, bank statements, and tax returns, but I'm not sure what to expect next. Can you walk me through the typical steps after submitting the application?"}, {"role": "assistant", "content": "Congratulations on submitting your mortgage application! After submitting your documents, the lender will review and process your application. Here's a general outline of the typical steps that follow: 1. Initial Review: The lender's loan processor will review your application to ensure all required documents are complete and in order. This step usually takes 1-3 business days. 2. Loan Processing: The loan processor will review your credit report, verify your income, and evaluate your debt-to-income ratio."}, {"role": "user", "content": "I'm a bit concerned about the appraisal step. Can you tell me more about what happens during an appraisal and how it affects the loan approval process?"}, {"role": "assistant", "content": "The appraisal step is a crucial part of the mortgage process, and it's natural to have questions about it. Here's a detailed explanation of what happens during an appraisal and how it affects the loan approval process: What is an appraisal? A real estate appraisal is an independent evaluation of a property's value, conducted by a licensed appraiser. The appraiser provides an objective opinion of the property's market value, based on various factors such as: 1."}]}
{"filler_id": "FL31", "session_id": "39948bcf_4", "turns": [{"role": "user", "content": "I'm looking for some recommendations on yoga classes in my area. I've been interested in continuing my practice since I attended a weekend-long yoga retreat last year in November, which included several workshops on meditation, pranayama, and asana practice."}, {"role": "assistant", "content": "Wonderful! It's great to hear that you're inspired to continue your yoga practice after the retreat. Here are some tips to help you find a yoga class that suits your needs: 1. Online Search: Type in \"yoga classes near me\" or \"yoga studios in [your city/zip code]\" in a search engine like Google. You can also use online directories like YogaFinder or Yoga Alliance to find studios and classes in your area. 2."}, {"role": "user", "content": "I'm actually interested in finding a studio that focuses on restorative yoga, as I really enjoyed that particular workshop during the retreat. Do you know of any studios in my area that specialize in restorative yoga?"}, {"role": "assistant", "content": "Wonderful! Restorative yoga is a great focus for a studio, and it's excellent that you're looking to deepen your practice in that area. Unfortunately, I'm a large language model, I don't have access to your location or real-time information about specific studios in your area. However, I can guide you on how to find a restorative yoga studio near you: 1."}]}
{"filler_id": "FL32", "session_id": "ultrachat_82739", "turns": [{"role": "user", "content": "Are backstage passes usually limited to a certain number of people? If so, how can I increase my chances of securing one?"}, {"role": "assistant", "content": "Yes, backstage passes are usually limited to a certain number of people. To increase your chances of securing one, here are some tips: 1. Join fan clubs or mailing lists: Many artists and bands offer exclusive access to backstage passes or pre-sales of tickets to their fan clubs or mailing lists. 2. Follow them on social media: Artists and bands often announce giveaways or promotions for backstage passes on their social media platforms. 3."}, {"role": "user", "content": "I'll definitely start following them on social media and enter some contests. Do you know if there are any upcoming concerts with backstage passes available?"}, {"role": "assistant", "content": "As an AI language model, I do not have access to real-time information about upcoming concerts with backstage passes available. However, you can check the artist or band's official website or social media pages for information about their upcoming concerts and any backstage pass opportunities they may have available. You can also check with the concert venue or ticketing agency for any special VIP or backstage packages they offer."}]}
{"filler_id": "FL33", "session_id": "f340a3db", "turns": [{"role": "user", "content": "I'm planning to launch a social media campaign on May 1st to raise awareness about mental health, can you help me come up with some engaging post ideas and hashtags?"}, {"role": "assistant", "content": "What a wonderful initiative! I'd be happy to help you with engaging post ideas and hashtags for your social media campaign to raise awareness about mental health. Here are some suggestions: Post Ideas: 1. Personal Stories: Share personal stories of struggle and triumph over mental health issues. Encourage your followers to share their own stories using a branded hashtag. 2."}, {"role": "user", "content": "Can you suggest some popular mental health advocates or influencers I can reach out to for collaborations or Q&A sessions?"}, {"role": "assistant", "content": "Here are some popular mental health advocates and influencers across various platforms who you may consider reaching out to for collaborations or Q&A sessions: Mental Health Advocates: 1. Lady Gaga (@ladygaga) - Singer-songwriter and founder of the Born This Way Foundation, which focuses on mental health and wellness. 2. Demi Lovato (@ddlovato) - Singer-songwriter and mental health advocate who has spoken openly about her struggles with depression and addiction. 3."}]}
{"filler_id": "FL34", "session_id": "ultrachat_245735", "turns": [{"role": "user", "content": "What cultural barriers have hindered the preservation of the Kiowa language, and how have they been overcome?"}, {"role": "assistant", "content": "Several cultural barriers have hindered the preservation of the Kiowa language, including historical trauma, forced assimilation, and the acculturation of younger generations. In the late 1800s, the U.S. government forced indigenous peoples onto reservations and established boarding schools where they were not allowed to speak their native languages. This caused Kiowa children to be removed from their families and communities, leading to intergenerational trauma and a disconnection from their language and culture."}, {"role": "user", "content": "That's really interesting. Do you know how successful these language revitalization programs have been so far?"}, {"role": "assistant", "content": "Yes, there have been some successes in Kiowa language revitalization efforts. For example, the Kiowa Tribe's language immersion program in Anadarko, Oklahoma, has seen positive results in terms of language acquisition and usage. The program focuses on teaching the language to young children and providing cultural exposure to help them develop a strong sense of identity and connection to their community."}]}
{"filler_id": "FL35", "session_id": "ultrachat_20447", "turns": [{"role": "user", "content": "How has remote work affected the social and emotional wellbeing of employees?"}, {"role": "assistant", "content": "As an AI language model, I do not have personal experiences, opinions, or emotions. However, studies and research have shown that remote work can have both positive and negative impacts on the social and emotional wellbeing of employees. Positive impacts of remote work on social and emotional wellbeing: 1. Flexible work arrangements can reduce stress and burnout associated with long commutes and demanding schedules. 2."}, {"role": "user", "content": "Can employers implement any strategies to prevent remote workers from feeling lonely and isolated while working from home?"}, {"role": "assistant", "content": "Yes, employers can implement various strategies to help remote workers feel less lonely and isolated while working from home. Here are a few strategies: 1. Encourage regular virtual meetings to keep remote workers connected. Schedule regular video check-ins and team calls to foster a sense of belonging and collaboration. 2. Create virtual coffee breaks, lunches or happy hours. These breaks help employees to take a break, socialize with colleagues and rejuvenate. 3. Facilitate team building activities."}]}
{"filler_id": "FL36", "session_id": "9f35ac68", "turns": [{"role": "user", "content": "I'm thinking of switching to a monthly bus pass, can you tell me the cost and where I can buy one?"}, {"role": "assistant", "content": "I'd be happy to help! However, I need a bit more information from you. Could you please provide me with the following details: 1. The city or region you live in (or plan to use the bus pass in)? 2. The type of bus pass you're interested in (e.g. adult, student, senior, disabled)? 3. The duration of the pass you're looking for (e.g. 1 month, 3 months, 12 months)?"}, {"role": "user", "content": "I live in the city, and I'm interested in an adult monthly pass. I'm looking for a 1-month pass."}, {"role": "assistant", "content": "I'd be happy to help! However, I need to clarify that I'm a large language model, I don't have access to real-time information about specific cities or transportation systems. But I can give you some general guidance on how to find the information you're looking for. To get the exact cost and details on where to buy a monthly bus pass in your city, I recommend the following steps: 1."}]}
{"filler_id": "FL37", "session_id": "285a29e7_2", "turns": [{"role": "user", "content": "I'm looking for some book recommendations. I've been reading a lot lately, and I'm always up for discovering new authors. By the way, I attended a poetry reading at a local coffee shop today, where several local poets read from their recent collections - it was amazing to see so many talented people share their work. Anyway, what are some popular books or authors you'd suggest?"}, {"role": "assistant", "content": "That sounds like a wonderful experience! Attending a poetry reading at a local coffee shop is a great way to discover new talent and connect with the literary community. As for book recommendations, I'd be happy to suggest some popular books and authors across various genres. Keep in mind that personal tastes may vary, but these are some widely acclaimed and popular choices: Fiction: 1."}, {"role": "user", "content": "I'd love to explore some poetry recommendations more. The poetry reading I attended today really opened my eyes to the talent out there. Do you have any suggestions for contemporary poetry collections or poets that I might enjoy, given my interest in discovering new voices?"}, {"role": "assistant", "content": "I'm thrilled to hear that the poetry reading inspired you to explore more poetry! Contemporary poetry is a vibrant and diverse field, and there are many incredible poets and collections worth discovering. Here are some recommendations that might resonate with you: Contemporary Poetry Collections: 1. When I Grow Up I Want to Be a List of Further Possibilities by Chen Chen: A debut collection that explores identity, family, love, and the search for meaning in a chaotic world. 2."}]}
{"filler_id": "FL38", "session_id": "sharegpt_NRqjZJS_0", "turns": [{"role": "user", "content": "Write an essay comparing and contrasting Ernest Hemmingway and John Steinbeck's writing styles"}, {"role": "assistant", "content": "Ernest Hemingway and John Steinbeck are two of the most renowned authors of the 20th century. Both men wrote about the struggles of the common man and the human condition, but their writing styles were vastly different. Hemingway is known for his concise, direct, and simple writing style. He uses short sentences and a minimalist approach to convey his message."}, {"role": "user", "content": "Please include specific examples by referencing some of the books they have written"}, {"role": "assistant", "content": "Ernest Hemingway and John Steinbeck are two of the most renowned authors of the 20th century. Both men wrote about the struggles of the common man and the human condition, but their writing styles were vastly different. Hemingway is known for his concise, direct, and simple writing style."}]}
{"filler_id": "FL39", "session_id": "0daaefa7_3", "turns": [{"role": "user", "content": "I'm trying to get into web development, and I was wondering if you could recommend some online resources for learning JavaScript. By the way, I just attended a 2-day coding bootcamp in mid-February, where I learned the basics of JavaScript, but I want to continue improving my skills."}, {"role": "assistant", "content": "Congratulations on taking the first step into web development and attending a coding bootcamp! With a solid foundation in JavaScript basics, you're ready to dive deeper and improve your skills. Here are some online resources to help you continue learning JavaScript: Tutorials and Courses: 1. Codecademy's JavaScript Course: A comprehensive and interactive course that covers JavaScript fundamentals, intermediate topics, and advanced concepts. 2. FreeCodeCamp: A non-profit platform that offers a range of JavaScript challenges, projects, and certifications."}, {"role": "user", "content": "I'll definitely check out these resources. I'm especially interested in practicing with JSFiddle and HackerRank's challenges. Do you have any tips on how to stay motivated and avoid burnout while learning to code?"}, {"role": "assistant", "content": "Practicing with JSFiddle and HackerRank's challenges is an excellent way to improve your JavaScript skills. Staying motivated and avoiding burnout are crucial aspects of learning to code. Here are some tips to help you: 1. Set achievable goals and milestones: Break down your learning journey into smaller, manageable goals. Celebrate your achievements, no matter how small they may seem. This will help you stay motivated and track your progress. 2."}]}
{"filler_id": "FL40", "session_id": "ultrachat_134317", "turns": [{"role": "user", "content": "How did the Emperor's role as head of the religion of the Roman state contribute to their authority?"}, {"role": "assistant", "content": "The Emperor's role as head of the religion of the Roman state was significant in contributing to their authority because religion played a significant role in the lives of the Roman people. The Emperors had control over religious institutions, rituals, and ceremonies, which allowed them to present themselves as the guardians of the traditional Roman values and customs."}, {"role": "user", "content": "Interesting, so did the Emperors have to follow certain religious practices or was their position more of a symbolic one?"}, {"role": "assistant", "content": "The Emperor's position as head of the religion of the Roman state was not just symbolic; rather, they were expected to follow certain religious practices and participate in religious ceremonies. The Emperor's religious duties included performing sacrifices, offering prayers, and officiating at important religious ceremonies. The Emperor was believed to have a special relationship with the gods and was regarded as the high priest of the Roman religion."}]}


In [ ]:
%%writefile /kaggle/working/clm/data/haystacks/filler_pool_hi.jsonl
{"filler_id": "FL01", "session_id": "ultrachat_201585", "turns": [{"role": "user", "content": "बीटल्स के प्रसिद्धि और प्रसार के कौन से तरीके लेनन के गीत लेखन पर प्रभाव डाले?"}, {"role": "assistant", "content": "एक AI भाषा मॉडल के रूप में, मुझको व्यक्तिगत विश्वास नहीं हैं, लेकिन मैं आपको एक सामान्य उत्तर दे सकता हूँ। बीटल्स के प्रसिद्धि और प्रसार लेनन के गीत लेखन पर गहरा प्रभाव डाले। बैंड के विशाल सफलता के कारण, लेनन को दुनिया के साथ अपने गाने साझा करने के लिए एक मंच दिया गया था, और उन्होंने इसका उपयोग करके सभी समय के सबसे प्रभावशाली गीत लिखे।"}, {"role": "user", "content": "क्या आप मुझे लेनन के प्रसिद्धता और प्रभाव के कारण बने हुए कुछ विशिष्ट गीतों के उदाहरण दे सकते हैं?"}, {"role": "assistant", "content": "हां, यहां लेनन के गीतों के कुछ उदाहरण हैं जो उनके प्रसिद्धि और प्रभाव के कारण लिखे गए थे: 1. \"हेल्प!\" - यह गीत लेनन के बैंड के सफलता के कारण उनके दबाव और एक सार्वजनिक व्यक्ति के रूप में निरंतर मांगों के कारण उनके द्वारा लिखा गया था। 2. \"स्ट्रॉबेरी फिल्ड्स फॉरेवर\" - यह गीत लेनन के बचपन के स्मृतियों के प्रभाव से लिखा गया था जब वे स्ट्रॉबेरी फिल्ड के पास रहते थे, जो एक सेवा सेना बच्चों के आश्रय भवन था।"}]}
{"filler_id": "FL02", "session_id": "5fb7fd18_1", "turns": [{"role": "user", "content": "मैं कुछ नए बेकिंग के विचारों की तलाश में हूँ। मैं अपने अंतिम समय में कई मीठे खानों को कर रहा हूँ, लेकिन मैं कुछ खारा चीज को आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज"}, {"role": "assistant", "content": "एक स्वादिष्ट ब्रेड के प्रेमी के बन रहे हैं! मैं आपकी स्वादिष्ट ब्रेड के दुनिया की खोज में मदद करने के लिए खुश होऊंगा। चूंकि आप बर्तन के साथ सॉर्डोउ खाना पसंद करते हैं, इसलिए मैं कुछ रेसिपी सुझाऊंगा जो आपके सॉर्डोउ बर्तन के समान गुण या स्वाद के होंगे। आपके शुरुआत के लिए कुछ विचार यहां हैं: 1. रोजमेरी और ओलीव तेल फोकासिया: यह इतालवी ब्रेड ओलीव तेल के साथ लुभावना होता है, जैसे आपके सॉर्डोउ बर्तन के साथ।"}, {"role": "user", "content": "इन रेसिपीज बहुत अद्भुत लग रही हैं, विशेषकर रोजमेरी और ओलीव ऑइल फोकासिया के लिए। मैंने कभी ब्रेड में रोजमेरी का उपयोग नहीं किया है, इसलिए मैं इसे आजमाने के लिए उत्सुक हूं। आप मुझे कुछ टिप्स दे सकते हैं कि नए रोजमेरी के साथ कैसे निपटें और रेसिपी में कितनी मात्रा उपयोग करें?"}, {"role": "assistant", "content": "नए रोजमेरी के साथ काम करना थोड़ा मुश्किल हो सकता है, लेकिन कुछ टिप्स के साथ आप बहुत जल्द एक प्रोफेशनल बन जाएंगे! नए रोजमेरी के साथ काम करें: 1. ताजा, महकते छोटे खंड चुनें: एक ताजा और महकते रोजमेरी के छोटे खंड चुनें जिनका बहुत तेज बूंदी वाला रंग हो और जो लहजा रखते हों। बर्बाद हो गए या भूरे रंग के खंड बचाएं, क्योंकि वे खट्टे स्वाद के हो सकते हैं। 2. रोजमेरी को ठीक तरह से संग्रहित करें: ताजा रोजमेरी को प्लास्टिक बैग में रखें या इसे"}]}
{"filler_id": "FL03", "session_id": "afbd7193_2", "turns": [{"role": "user", "content": "मैं अपने किताबों के शेल्फ को आयोजित करने के बारे में कुछ टिप्स खोज रहा हूँ। मैं अपने लाइविंग रूम की गहरी सफाई शनिवार को कर चुका हूँ और मुझे लगता है कि मुझे कुछ विचार चाहिए जिससे मैं अपने किताबों के शेल्फ के स्थान को अधिकतम कर सकूँ। बताओ कि, मैं हर शनिवार को 30 मिनट की गहरी सफाई शेड्यूल करता हूँ, इसलिए मैं इसे अच्छी तरह से उपयोग करना चाहता हूँ।"}, {"role": "assistant", "content": "तुम्हारे लिए एक नियमित गहरी सफाई के सत्र के लिए बांधे रखने के लिए बधाई! यह एक अच्छी आदत है जो तुम्हारे घर के साफ और आगे रखे जाने वाले जगह को बरकरार रखने में मदद करती है। अब, चलो कुछ टिप्स के बारे में जाने के लिए डूब जाएं जो तुम्हारे किताबों के जगह के अधिकतम उपयोग और तुम्हारे 30 मिनट के रविवार के गहरी सफाई के सत्र के अधिकतम उपयोग के लिए मदद करेंगे: 1. छोड़ दो और वर्गीकृत करो: शुरू करो अपने शेल्फ से सभी किताबों को हटा दो और उन्हें"}, {"role": "user", "content": "मुझे विभिन्न श्रेणियों के अनुसार वस्तुओं को वर्गीकृत करके उन्हें अपने घर में रखने की विचारधारा पसंद है। लेकिन मैं कैसे निर्णय ले सकती हूँ कि क्या रखना है और क्या छोड़ देना है? मुझको कई किताबें हैं जिन्हें मैं बहुत सालों से पढ़ नहीं रही हूँ, लेकिन वे मेरे भावुक मूल्य के हैं।"}, {"role": "assistant", "content": "निर्णय लेना कि क्या रखें और क्या छोड़ दें, विशेष रूप से जब भावुक मूल्य के बारे में बात की जाए, एक चुनौतीपूर्ण कार्य हो सकता है। आपके लिए इस निर्णय लेने में मदद करने के लिए कुछ सलाह यहां दी गई है: भावुक मूल्य और व्यावहारिकता के बीच अंतर: 1. अपने आप से पूछें: यह किताब आखिर कब देखी गई या पढ़ी गई? ईमानदार हों। यदि यह सालों से नहीं देखी गई है, तो आपको इसे छोड़ देने से बहुत बुरा लगेगा। 2."}]}
{"filler_id": "FL04", "session_id": "ultrachat_391062", "turns": [{"role": "user", "content": "कुछ सामान्य गेमिंग शब्द और अकृतियाँ कौन सी हैं जिनके बारे में मुझे जानकारी होनी चाहिए?"}, {"role": "assistant", "content": "यहां कुछ सामान्य गेमिंग शब्द और संक्षिप्त रूप हैं: 1. FPS - फर्स्ट पर्सन शूटर 2. MMO - मैसिवली मल्टीप्लेर ऑनलाइन 3. PVP - प्लेयर वर्सस प्लेयर 4. NPC - नॉन-प्लेयर चरित्र 5. RNG - रैंडम नंबर जेनरेटर 6. MOBA - मल्टीप्लेर ऑनलाइन बैटल एरिया 7. DLC - डाउनलोडेबल कंटेंट 8. LFG - लूकिंग फॉर ग्रुप 9. AFK - एवे फ्रॉम कीबोर्ड 10. DPS - डेमेज पर सेकंड 11."}, {"role": "user", "content": "दुद, मैं इस खेल में आरएनजी के लिए बहुत थक गया हूँ। मैं घंटों तक लड़ रहा हूँ और अभी तक मुझे जरूरत के अनुसार ड्रॉप नहीं मिला है। यह लगता है कि खेल मेरे खिलाफ तैयार किया गया है।"}, {"role": "assistant", "content": "मैं समझता हूँ कि उपयोगकर्ता खेल में एक रैंडम नंबर जेनरेटर (rng) के कारण असंतुष्ट है, जिसका अर्थ यह है कि कुछ घटनाओं या ड्रॉप के परिणाम यादृच्छिक तौर पर निर्धारित होते हैं बल्कि कौशल पर निर्भर नहीं होते। उपयोगकर्ता चिंतित है कि वे खेल में घंटों बिताए हुए और ग्रिंड करते रहे हैं लेकिन अभी तक अपने चाहे ड्रॉप को प्राप्त नहीं कर सके हैं, जिसके लिए उनका विश्वास है कि यह बुरी लकी या अनुचित अवसरों के कारण है।"}]}
{"filler_id": "FL05", "session_id": "ultrachat_274962", "turns": [{"role": "user", "content": "सुल्ला के सैन्य विजय और आक्रमण कैसे उनके राजनीतिक विचारधारा को बनाए रखे और रोम के एक नेता के रूप में उनकी नीतियों को प्रभावित करे?"}, {"role": "assistant", "content": "सुल्ला के सैन्य विजय और विजयों ने उनके राजनीतिक विचारधारा के निर्माण में महत्वपूर्ण भूमिका निभाई और रोम के एक नेता के रूप में उनकी नीतियों पर प्रभाव डाला। सैन्य सामग्री के रूप में सुल्ला के लिए अपने युद्ध क्षेत्र में अद्वितीय कौशल और टैक्टिकल दिग्गजता के लिए जाने जाते थे। अपनी सैन्य सफलताओं के माध्यम से वह रोम के लोगों में बहुत लोकप्रिय हो गए और एक शक्ति केंद्र की स्थापना कर ली, जिसका उपयोग बाद में राज्य के नियंत्रण के लिए किया ग"}, {"role": "user", "content": "लगता है कि सुल्ला अपनी शक्ति को मजबूत करने में अधिक रुचि रखता था, वास्तव में रोम के लोगों के लिए सेवा करने से अधिक। उसकी नीतियों ने रोम के नागरिकों पर क्या प्रभाव डाला?"}, {"role": "assistant", "content": "आपका कहना सही है कि सुल्ला रोमन लोगों के सेवा करने की बजाए अपनी शक्ति के विस्तार पर अधिक ध्यान देता था। उनकी नीतियों ने रोम के नागरिकों पर गहरा प्रभाव डाला जो अंततः रोमन गणतंत्र के गिरने में योगदान दिया। सुल्ला की नीतियों का सबसे महत्वपूर्ण प्रभाव एक ही व्यक्ति के हाथों शक्ति के केंद्रीकरण में था।"}]}
{"filler_id": "FL06", "session_id": "f62c04c6", "turns": [{"role": "user", "content": "मैं अगले महीने के लिए मेरे सोशल मीडिया पोस्ट के लिए एक कंटेंट कैलेंडर बनाने के बारे में सोच रहा हूँ। मुझे कुछ विचार देने में आप मदद कर सकते हैं कि मैं किस प्रकार के पोस्ट बनाऊ और कितनी आवश्यकता है कि मैं इतने पोस्ट करूँ?"}, {"role": "assistant", "content": "एक कंटेंट कैलेंडर बनाना आपके सोशल मीडिया कंटेंट को आगे बढ़ाने और आगे आने वाले समय में आयोजित करने के एक अच्छा तरीका है। पोस्ट के प्रकार और आवृत्ति आपके लक्ष्य उपभोक्ता, लक्ष्य और आप उपयोग कर रहे हैं उन प्लेटफॉर्म पर निर्भर करते हैं, लेकिन यहां कुछ सामान्य विचार और निर्देश हैं जिनसे आप शुरू कर सकते हैं: पोस्ट के प्रकार: 1. शिक्षात्मक/सूचनात्मक: आपके विषय से संबंधित टिप्स, ट्यूटोरियल, उद्योग समाचार या कैसे करें गाइड शेयर क"}, {"role": "user", "content": "वह काफी अच्छी जानकारी है! मुझे लगता है कि मैं पहले प्रत्येक प्लेटफॉर्म के लिए एक टेम्पलेट बनाऊंगा और फिर उसमें विचार भर दूंगा। आप मुझे इंस्टाग्राम रील्स के लिए कुछ विचार दे सकते हैं? मैं अपने सुबह के योग रूटीन के बारे में एक पोस्ट शेयर कर चुका हूं और उसे 50 व्यूज और 10 लाइक मिले हैं, इसलिए मैं योग या मेडिटेशन के एक सीरीज करने के विचार में हूं।"}, {"role": "assistant", "content": "इंस्टाग्राम रील्स अपने पब्लिक के साथ इंटरैक्ट करने और अपनी विशिष्टता को बढ़ाने के लिए एक अद्भुत तरीका है। आपके पहले रील के सफलता पर बधाई! योग या मनोमन्त्रण की एक सीरीज एक अच्छा विचार है, विशेष रूप से आपके सुबह के योग रूटीन रील में देखे गए रुचि के कारण। आपके शुरुआत के लिए कुछ विचार यहां हैं: योग सीरीज: 1. सुबह के रूटीन के विविधताएं: विभिन्न सुबह के योग रूटीन शेयर करें, जिनमें से प्रत्येक एक विशिष्ट क्षेत्र पर ध्यान केंद्रित करता ह"}]}
{"filler_id": "FL07", "session_id": "8969b3ca_2", "turns": [{"role": "user", "content": "मुझे घर खरीदने में शामिल होने वाले कुल खर्च का अनुमान लगाने की आवश्यकता है। आप मुझे इसकी सहायता कर सकते हैं?"}, {"role": "assistant", "content": "घर खरीदने में शामिल होने वाले कुल खर्च का अनुमान लगाना आपको इस महत्वपूर्ण निवेश के लिए योजना बनाने और बजट बनाने में मदद कर सकता है। यहां आपको अपेक्षित आम खर्च का विस्तार से विवरण दिया गया है: 1. खरीद मूल्य: यह घर खरीदने के लिए सबसे बड़ा खर्च होता है, और यह आपके द्वारा विक्रेता को घर के लिए भुगतान किया जाने वाला राशि होती है। 2. आगे भुगतान: यह खरीद मूल्य का एक प्रतिशत होता है जिसे आप आगे से भुगतान करते हैं, आमतौर पर खरीद म"}, {"role": "user", "content": "मैं वास्तव में घर के बंद होने के निकट आ गया हूँ, और मुझे मेरे बंद होने के खर्च के एक अधिक सटीक अनुमान की आवश्यकता है। बिक्री कर्ता मुझे बंद होने के खर्च के लिए $2,000 क्रेडिट देने पर सहमत हुए हैं, इसलिए मैं उसे शामिल करना चाहता हूँ। आप मुझे इसकी सहायता कर सकते हैं?"}, {"role": "assistant", "content": "आपको अपने नए घर के बंद होने के निकट आ जाने पर बधाई! मैं आपकी समाप्ति लागत का अनुमान लगाने और विक्रेताओं के $2,000 क्रेडिट को शामिल करने में खुश होऊंगा। सटीक अनुमान लेने के लिए मुझे आपसे कुछ अतिरिक्त जानकारी की आवश्यकता होगी। कृपया निम्नलिखित विवरण प्रदान करें: 1. खरीद मूल्य: घर का अंतिम खरीद मूल्य क्या है? 2. ऋण राशि: आप खरीद के लिए कितना ऋण ले रहे हैं? 3. ऋण प्रकार: आप किस प्रकार का ऋण ले रहे हैं?"}]}
{"filler_id": "FL08", "session_id": "50d66391_7", "turns": [{"role": "user", "content": "मैं बेडरूम में रोशनी के बारे में कुछ सलाह ढूंढ रहा हूँ। मैं अपने बेडरूम में कुछ फर्श पर रखे जा सकने वाले लैंप खरीदने के बारे में सोच रहा हूँ ताकि एक आरामदायक पठन कोना बन सके।"}, {"role": "assistant", "content": "कितना अच्छा विचार है! फ्लोर लैंप बेडरूम में एक अच्छा अंतर्गत बन सकते हैं, जो एक गर्म और आमंत्रक वातावरण बनाते हैं, जो आराम करते समय पठन के कोने के लिए आदेश देते हैं। अपने बेडरूम के लिए सही फ्लोर लैंप चुनते समय विचार करने के लिए यहां कुछ सलाह हैं: 1. उद्देश्य के अनुसार विचार करें: आपने पठन के कोने के बारे में बताया है, इसलिए आपको गतिविधि विशेष रूप से चाहिए। लैंप जिनमें अनुकूलित बाहु या सिर होते हैं, जो आपके आवश्यकतान"}, {"role": "user", "content": "मैं अभी एक बिस्तर के ओर लैंप का उपयोग कर रहा हूँ जिसमें मैंने होम डेपॉट से लगभग 15 डॉलर में खरीदा नए बल्ब का उपयोग कर रहा हूँ, और मुझे इसका गर्म टोन बहुत पसंद है। क्या आप मेरे लिए फ्लोर लैंप खोजने के लिए ऐसे बल्ब के विकल्प या रंग तापमान के समान खोजने की सलाह देंगे?"}, {"role": "assistant", "content": "आपने पहले ही एक बल्ब चुन लिया है जिसे आप पसंद करते हैं, जो एक अच्छा शुरुआती बिंदु है! हां, फर्श पर रखे जाने वाले लैंप की तलाश करना एक अच्छा विचार है जो आपके वर्तमान बल्ब के समान विकल्पों को स्वीकार कर सके या जिसका रंग तापक्रम आपके वर्तमान बल्ब के समान हो। चूंकि आपने घर के शॉपिंग सेंटर से खरीदे गए बल्ब के गर्म रंग को पसंद करने का उल्लेख किया है, मैं यह मान रहा हूं कि इसका रंग तापक्रम 2700K-3000K के बीच होगा, जो एक गर्म सफेद या म"}]}
{"filler_id": "FL09", "session_id": "ultrachat_436464", "turns": [{"role": "user", "content": "विश्व स्वास्थ्य संगठन (WHO) ने वैश्विक सार्वजनिक स्वास्थ्य को सुधारने के लिए कौन से प्रमुख प्रयोजन शुरू किए हैं?"}, {"role": "assistant", "content": "विश्व स्वास्थ्य संगठन (WHO) ने विश्व भर में सार्वजनिक स्वास्थ्य को सुधारने के लिए कई महत्वपूर्ण पहलें की हैं, जिनमें शामिल हैं: 1. सार्वभौमिक स्वास्थ्य आवास: WHO का उद्देश्य यह है कि हर कोई अपनी आवश्यक स्वास्थ्य सेवाओं के लिए पहुँच सके बिना आर्थिक कठिनाइयों के सामने न खड़ा हो। इसमें महत्वपूर्ण स्वास्थ्य सेवाओं के लिए पहुँच को बढ़ावा देना और स्वास्थ्य असमानताओं को संभालना शामिल है। 2. वैक्सीन कार्य योजना: WHO का उद्देश्य यह है कि सभी लोग,"}, {"role": "user", "content": "संस्था के इन सभी प्रयासों को देखना बहुत अच्छा है। लेकिन आपकी राय है कि ये सार्वजनिक स्वास्थ्य में सुधार के लिए प्रभावी हैं कि नहीं?"}, {"role": "assistant", "content": "मैं व्यक्तिगत राय नहीं रखता। हालांकि, जीवन रक्षा के कार्यक्रम ने वैश्विक स्तर पर सार्वजनिक स्वास्थ्य में महत्वपूर्ण प्रगति की है। उदाहरण के लिए, वैश्विक टीकाकरण कार्य योजना ने कई देशों में टीकाकरण कवरेज को बढ़ावा दिया है, और टॉबैको रहित योजना ने टॉबैको उत्पादों की मांग को कम करने में सफलता प्राप्त की है। इसी तरह, जीवन रक्षा के प्रयास बीमारी नियंत्रण, आपातकालीन प्रतिक्रिया और मानसिक स्वास्थ्य के प्रोत्साहन में भी धन्यवाद धन्यवाद ध"}]}
{"filler_id": "FL10", "session_id": "429be85c_4", "turns": [{"role": "user", "content": "मेरे पास अपने कॉफी के साथ शनिवार के दिन नए ब्रेकफास्ट के ट्रीट्स ट्राई करने की इच्छा है। मैं शनिवार के दिन अपने कॉफी ब्रेक के साथ बड़ा आनंद लेना पसंद करता हूँ और मेरे सामान्य किन्नौर के ट्रीट्स के लिए थोड़ा थक गया हूँ। क्या आपके पास कोई रेसिपी सुझाव या अनुशंसा है?"}, {"role": "assistant", "content": "मुझे आपकी मदद करने के लिए खुशी हो रही है! अपने छुट्टी के दिन एक स्वादिष्ट उठाव के साथ गर्म कप कॉफी के साथ आनंद लेना जीवन के सबसे बड़े आनंदों में से एक है। मैं आपके लिए कुछ शानदार रेसिपी के सुझाव ले आया हूँ जो आपको अपने नए पसंदीदा उठाव के साथ खुश करेंगे। आपके शुरुआत के लिए कुछ विचार यहां हैं: 1. क्रैंबरी अरंग बिस्कुट: गोल्डन बिस्कुट जो क्रैंबरी और अरंग के जूस से भरे हुए हैं, एक गहरी कॉफी के साथ बहुत अच्छा मेल खाते हैं। 2."}, {"role": "user", "content": "इन रेसिपी में अल्प नमक वाले बटर और नमक वाले बटर के उपयोग में क्या अंतर होता है? अंतिम स्वाद और संरचना में वास्तव में कोई अंतर होता है कि नहीं?"}, {"role": "assistant", "content": "अपने बेक किए गए खाद्य पदार्थों के अंतिम स्वाद और संरचना में असल्टेड बटर के उपयोग और सल्टेड बटर के उपयोग के बीच अंतर हो सकता है।"}]}
{"filler_id": "FL11", "session_id": "33994682_3", "turns": [{"role": "user", "content": "मैं जल्द ही अपने अंडे ठंडा करने के बारे में सोच रही हूँ और मैं कुछ शोध कर रही हूँ, लेकिन प्रक्रिया और खर्च के बारे में मुझे निश्चित नहीं है। आप मुझे सामान्य प्रक्रिया के बारे में बता सकते हैं और मुझे यह बताएं कि इसका आम खर्च कितना होता है। बता दें, मेरी बहन, जो मेरे से केवल तीन साल बड़ी है, अब 35 साल की है, और विचार करना भी असामान्य है कि समय कितनी तेजी से बीत गया है।"}, {"role": "assistant", "content": "मैं आपकी मदद करने के लिए खुश होऊंगा जिसके माध्यम से आप अंडा ठंडा करने की प्रक्रिया और खर्च के बारे में जान सकें! प्रक्रिया: अंडा ठंडा करना, जिसे आमतौर पर अंडाणु शीतलन भी कहा जाता है, एक बहुत स्टेप वाली प्रक्रिया है जो आमतौर पर 2-3 सप्ताह के लिए लगभग लगती है। यहां एक सारांश है: 1. प्रारंभिक सलाह: आप एक निर्धारण विशेषज्ञ (प्रजनन एंडोक्रिनोलॉजिस्ट) के साथ मिलेंगे जहां आपके चिकित्सा इतिहास, गर्भावस्था लक्ष्य और आपके पास के कोई भी चिंता"}, {"role": "user", "content": "मैं अपने बारे में अधिक जानना भी इच्छुक हूँ कि अंडा ठंडा करने के सफलता के दर क्या है। एक ठंडे अंडे से स्वस्थ गर्भावस्था और जन्म के कितने अवसर हो सकते हैं, और मेरी आयु इसमें महत्वपूर्ण भूमिका निभाती है या नहीं?"}, {"role": "assistant", "content": "अंडा ठंडा करने के सफलता दर: अंडा ठंडा करने की सफलता तकनीक और वित्रिफिकेशन विधियों में प्रगति के साथ सुधर रही है। हालांकि, यह समझना महत्वपूर्ण है कि अंडा ठंडा करना एक सफल गर्भावस्था या जन्म की गारंटी नहीं होता। ठंडे अंडे से स्वस्थ गर्भावस्था और जन्म की संभावना कई कारकों पर निर्भर करती है, जिनमें से: 1."}]}
{"filler_id": "FL12", "session_id": "f75238a5_2", "turns": [{"role": "user", "content": "मेरे परियोजना के लिए AI टूल्स के बारे में विचार कर रहा हूँ, और मैं आपसे बात करके इस विषय में शिक्षा के लिए कुछ संसाधनों की सिफारिश करना चाहता हूँ। बताओ कि आप क्या सोचते हैं। बताओ कि आप क्या सोचते हैं। बताओ कि आप क्या सोचते हैं। बताओ कि आप क्या सोचते हैं। बताओ कि आप क्या सोचते हैं। बताओ कि आप क्या सोचते हैं। बताओ कि आप क्या सोचते हैं। बताओ कि आप क्या सोचते हैं। बताओ कि आप क्या सोचते हैं। बताओ कि आप क्या सोचते हैं। बताओ कि आप क्या सोचते हैं। ब"}, {"role": "assistant", "content": "मैं आपके प्रोजेक्ट के लिए AI टूल्स के बारे में जानने के लिए आपकी रुचि के बारे में सुनकर बहुत खुश हूँ! वास्तव में AI के क्षेत्र में कई उत्साहजनक विकास हुए हैं, और नवीनतम उन्नतियों के साथ अपडेट रहना बहुत मूल्यवान हो सकता है। आपके लिए संसाधनों के बारे में, मैं कुछ श्रेष्ठ संसाधनों की सिफारिश करने के लिए खुश होऊंगा: 1. AI Alignment पॉडकास्ट: एक अद्भुत पॉडकास्ट जो नवीनतम AI अनुसंधान, अनुप्रयोग और परिणामों के बारे में गहराई से चर्चा करता है।"}, {"role": "user", "content": "मैं निश्चित रूप से उन संसाधनों को देख लूंगा, विशेषकर एमर्ज के पॉडकास्ट और ब्लॉग। आप मुझे चिकित्सा और वित्त जैसे क्षेत्रों में एआई के वर्तमान स्थिति के बारे में अधिक बता सकते हैं?"}, {"role": "assistant", "content": "AI उद्योगों के सभी क्षेत्रों में परिवर्तन कर रही है, और स्वास्थ्य और वित्त भी छोड़े बिना अपवाद नहीं हैं। इन क्षेत्रों में वर्तमान स्थिति में AI के अनुप्रयोगों का एक संक्षिप्त विवरण नीचे दिया गया है: स्वास्थ्य: 1. बीमारी निदान: AI आधारित प्रणालियाँ चिकित्सा छवियों, जैसे एक्स-रे और एमआरआई के विश्लेषण कर सकती हैं ताकि कैंसर, मधुमेह और दिल की बीमारियों जैसी बीमारियों का उच्च दक्षता के साथ पता लगाया जा सके। 2. व्यक्तिगत चिकित्सा: AI व"}]}
{"filler_id": "FL13", "session_id": "ultrachat_364297", "turns": [{"role": "user", "content": "क्या आप कबूतर और गैल के प्रजनन आचरण की तुलना और विरोधाभास कर सकते हैं?"}, {"role": "assistant", "content": "मैं व्यक्तिगत अनुभव, भावनाएँ या विश्वास नहीं रखता। हालांकि, मैं आपको आवश्यक जानकारी प्रदान कर सकता हूँ। मूंग और गौरैया दोनों ही पानी के पक्षी हैं और उनके जनन आचरण में कुछ समानताएँ हैं, लेकिन उनमें कुछ अंतर भी हैं। यहाँ कुछ तुलना और विरोधाभास हैं: समानताएँ: - मूंग और गौरैया दोनों जनन के मौसम में जोड़े बनाते हैं। - दोनों प्रजातियाँ आमतौर पर जीवन या लंबे समय तक एक दूसरे के साथ विवाह करते हैं, हालांकि कुछ विवाहित व्य"}, {"role": "user", "content": "क्या आप मुझे डकों के प्रेम दिखावों के बारे में अधिक बता सकते हैं? यह सुनने लायक है।"}, {"role": "assistant", "content": "हां, मैं खुश होऊंगा! बत्तखों के लिए अपने विवाह संगीत के लिए जटिल दिखावट के लिए जाने जाते हैं, जो एक श्रृंखला व्यवहार होते हैं जो एक गर्लफ्रेंड को आकर्षित करने के लिए होते हैं। आप बत्तखों में देख सकते हैं इन विवाह संगीत के कुछ उदाहरण: - सिर फेंकना: कुछ बत्तखों के प्रजातियां अपने सिर को आगे और पीछे फेंकते हैं और विभिन्न ध्वनियां बनाते हैं ताकि अपनी शारीरिक क्षमता और ध्वनि क्षमता को दिखाएं।"}]}
{"filler_id": "FL14", "session_id": "08ca1f31_2", "turns": [{"role": "user", "content": "मैं अपने ट्रेक ईमोंडा पर इस शनिवार के अवकाश पर एक लंबी बाइक यात्रा योजना करने के बारे में सोच रहा हूँ, लगभग 80-100 मील की। क्या आप मुझे कुछ रूट सुझाव या लंबी यात्रा के लिए मेरी बाइक के तैयारी के लिए टिप्स दे सकते हैं? बताओ कि मैं इस बाइक पर लगभग तीन सप्ताह पहले एक ट्यून-अप करवाया था, और बाइक दुकान ने चेन को बदल दिया और डिरेलियर्स को समायोजित कर दिया, जिसके लिए मुझे 80 डॉलर खर्च करने पड़े।"}, {"role": "assistant", "content": "एक लंबी साइकिल यात्रा के बारे में सुनकर एक अद्भुत यात्रा के बारे में लगता है! मैं आपकी साइकिल के तैयारी के लिए मार्ग सुझाव और टिप्स के लिए आपकी मदद करने के लिए खुश होऊंगा। मार्ग सुझाव: हम बाइक तैयारी में ले जाने से पहले, मार्गों के बारे में बात करते हैं। क्योंकि आपने कोई स्थान निर्दिष्ट नहीं किया है, इसलिए मैं आपके क्षेत्र में मार्ग खोजने के लिए ऑनलाइन टूल्स जैसे मैप माइ राइड, राइड विथ जीपीएस या स्ट्रावा का उपयोग कर सकते हैं। 1."}, {"role": "user", "content": "मैं एक जगह जहां कुछ पहाड़ियां हों वाला रूट करने के बारे में सोच रहा हूँ, क्योंकि मैं अपने Trek Emonda के साथ अक्सर समतल यात्राएं कर रहा हूँ। क्या आपके पास कोई विशेष पहाड़ियों पर चढ़ाई के टिप्स या गियर और कैडेंस के लिए सुझाव हैं?"}, {"role": "assistant", "content": "हिल क्लाइम्बिंग अपने आप को चुनौती देने और अपनी सामान्य साइकिलिंग फिटनेस को सुधारने के एक अच्छा तरीका है। यहां कुछ हिल क्लाइम्बिंग के टिप्स, गियरिंग के सुझाव और केडेंस के बारे में बताया गया है जो आपको अपने Trek Emonda पर उन ढलानों को छेड़न में मदद करेंगे: हिल क्लाइम्बिंग टिप्स: 1. आपको अपनी गति को नियंत्रित रखें: शुरू करें एक स्थिर और मध्यम गति से और ढलान के दौरान इसे बनाए रखें। अतिरिक्त तेजी से चलना या दौड़ना बचाएं, क्योंकि इससे थ"}]}
{"filler_id": "FL15", "session_id": "1c1f5ccc_6", "turns": [{"role": "user", "content": "मैं भारतीय वेशभूषा के सांस्कृतिक महत्व के बारे में अधिक जानना चाहता हूँ, विशेष रूप से अतीत में एक त्यौहार में मुझे खरीदे गए शाल के बारे में। क्या आप मुझे भारतीय पारंपरिक शाल के इतिहास और प्रतीकात्मक अर्थ के बारे में अधिक बता सकते हैं? बता दूँ कि मैं अपनी सांस्कृतिक विरासत के बारे में अधिक जानने की कोशिश कर रहा हूँ, भारत के इतिहास और परंपराओं के बारे में भी, और यह एक अच्छा शुरुआत के बिंदु लगता है।"}, {"role": "assistant", "content": "कैसा अद्भुत विषय! भारतीय वस्त्र, विशेषकर आपके द्वारा रुचि रखे गले बांधने वाले शाल, एक गहरी सांस्कृतिक महत्व रखते हैं जो सदियों से चलते आ रहे हैं। मैं इसके इतिहास और प्रतीकात्मक अर्थ में गहराई से जाने के लिए खुश हूं, और मुझे खुशी है कि आप अपनी सांस्कृतिक विरासत की खोज में लगे हुए हैं। भारत में शाल पारंपरिक वस्त्र के एक अभिन्न हिस्सा हैं, और उनके नाम विभिन्न क्षेत्रों के अनुसार अलग-अलग होते हैं। कुछ सामान्य भारतीय शालों के प"}, {"role": "user", "content": "मुझे विभिन्न प्रकार के कपड़ों के बारे में अधिक जानकारी चाहिए जो पारंपरिक भारतीय ढोलक में उपयोग किए जाते हैं। क्या कोई विशेष कपड़ा अन्य कपड़ों की तुलना में अधिक प्रतीकात्मक या प्रतिष्ठित माना जाता है?"}, {"role": "assistant", "content": "फैब्रिक भारतीय पारंपरिक ढोलकों में एक महत्वपूर्ण भूमिका निभाते हैं, और प्रत्येक प्रकार के अपने विशिष्ट गुण, महत्व और सांस्कृतिक संबंध होते हैं। मुझे आपको भारतीय ढोलकों में सबसे आम और पवित्र फैब्रिकों के बारे में जानाकर खुशी होगी: 1. सूत: सूत भारतीय वस्त्रों में सबसे श्रेष्ठ और लक्जरी फैब्रिक माना जाता है। यह राजतंत्र, समृद्धि और विशालता के साथ जुड़ा होता है। सूत ढोलक अपनी नरमता, चमक और टिकाऊता के कारण बहुत मूल्यवान माने जाते"}]}
{"filler_id": "FL16", "session_id": "0651f7bd_2", "turns": [{"role": "user", "content": "मैं अपने वेंटेज कैमरा कलेक्शन के सही तरीके से संग्रह और प्रदर्शन के बारे में कुछ टिप्स ढूंढ रहा हूँ। मुझको कुछ दुर्लभ कैमरे हैं और मैं चाहता हूँ कि वे धूल और नुकसान से बचे रहें। बता दूँ कि मुझे आज ईमेल के जरिए 1982 के से एक दुर्लभ कार्ड मिला है, जो कैल रिप्किन जूनियर का है, लेकिन वह मेरी एक अलग कलेक्शन है।"}, {"role": "assistant", "content": "आपको अपने दुर्लभ कैल रिप्किन जूनियर कार्ड पर बधाई! यह बहुत उत्सुक कर देता है! अब, हम अपने वेंटेज कैमरा संग्रह के संग्रह और प्रदर्शन पर ध्यान केंद्रित करेंगे। सही संग्रह और प्रदर्शन आपके कैमरों की स्थिति और मूल्य के संरक्षण के लिए आवश्यक है। आपके वेंटेज कैमरों को धूल और नुकसान से बचाने के लिए निम्नलिखित टिप्स आपके लिए मददगार हो सकते हैं: संग्रह: 1. जलवायु नियंत्रण: अपने कैमरों को 60 डिग्री फारेनहाइट से 70 डिग्री फारेनहाइट (15 डिग्री सेल"}, {"role": "user", "content": "क्या आप कैमरा रिस्टोरेशन और रखरखाव के बारे में सीखने के लिए कुछ अच्छे संसाधन बता सकते हैं? मुझको कुछ कैमरे हैं जिन्हें कुछ देखभाल की जरूरत है, और मैं नहीं जानता कि कहां से शुरू करू।"}, {"role": "assistant", "content": "कैमरा बहाली और रखरखाव के लिए आवश्यक कौशल हैं, जो किसी भी कैमरा प्रेमी के लिए महत्वपूर्ण हैं। यहां कुछ अद्भुत संसाधन हैं जो आपको सीखने और शुरू करने में मदद करेंगे: ऑनलाइन फॉरम और समुदाय: 1. द एंटरप्राइज रिपेयर फॉरम: एक निर्धारित फॉरम जहां एक विशाल समुदाय के विशेषज्ञ और प्रेमी एक दूसरे के साथ कैमरा रिपेयर और रखरखाव के ज्ञान और अनुभव साझा करते हैं। 2. फोटोग्राफी स्टैक एक्सचेंज: एक प्रश्न-उत्तर प्लेटफॉर्म जहां आप प्रश्न पू"}]}
{"filler_id": "FL17", "session_id": "01d4b26b_1", "turns": [{"role": "user", "content": "मैं अपने सहकर्मी के बच्चे के शोउअर के लिए कुछ उपहार के विचार खोज रहा हूँ। मैं कुछ वातावरण अनुकूल और मस्त चीज चाहता हूँ। बता दूँ कि मैं अपनी बहन को पिछले महीने उसके जन्मदिन के लिए एक चांदी के गले के जेवर दिया था, जिसमें छोटा चिन्ह भी था, और वह उसे बहुत पसंद कर गई।"}, {"role": "assistant", "content": "आपकी बहन के लिए कितना खास उपहार है! मैं आपकी सहकर्मी के बच्चे के लिए एक वातावरण दोस्त और सुंदर उपहार के विचारों के लिए मदद करने में खुश हूं। यहां कुछ सुझाव हैं: 1. आकर्षक आकार के आकर्षक आकार के आकर्षक आकार के आकर्षक आकार के आकर्षक आकार के आकर्षक आकार के आकर्षक आकार के आकर्षक आकार के आकर्षक आकार के आकर्षक आकार के आकर्षक आकार के आकर्षक आकार के आकर्षक आकार के आकर्षक आकार के आकर्षक आकार के आकर्षक आकार के आकर्षक आकार के आकर्षक आकार"}, {"role": "user", "content": "मुझे आकर्षित करता है ऑर्गेनिक कॉटन के ओनीसी और बच्चे के बेड सेट की विचार। क्या आप मुझे अधिक बता सकते हैं कि कौन से ब्रांड व्यक्तिगत विकल्पों के लिए कस्टमाइजेशन के विकल्प प्रदान करते हैं?"}, {"role": "assistant", "content": "संगठित कपास के एकसाथ बच्चे के कपड़े और बच्चे के ढांचे के सेट एक सोचे-समझे और पर्यावरण अनुकूल उपहार के रूप में बनाए जा सकते हैं। कई ब्रांड एकसाथ बच्चे के कपड़े और बच्चे के ढांचे के लिए कस्टमाइजेशन विकल्प प्रदान करते हैं। यहां कुछ प्रसिद्ध ब्रांड हैं: 1. हना एंडर्सन: अपने नरम, संगठित कपास कपड़ों के लिए जाने जाने वाली हना एंडर्सन एकसाथ बच्चे के कपड़े, सोने के कपड़े और ढांचे के लिए कस्टमाइजेशन विकल्प प्रदान करती है। आप विभिन्न फॉंट, र"}]}
{"filler_id": "FL18", "session_id": "e2691068", "turns": [{"role": "user", "content": "मैं कुछ नए उत्तेजना देने वाले उन्नत उपभोक्ता विचारों की तलाश में हूँ। क्या आपके पास कोई सुझाव हैं?"}, {"role": "assistant", "content": "एक ऊर्जा भरपूर उठाने के लिए ब्रेकफास्ट आपके दिन की शुरुआत के लिए बहुत अच्छा होता है! मुझको कुछ लुभावने और पोषक तत्व भरपूर ब्रेकफास्ट के विचार हैं जो आपको अपने दिन के काम के लिए ऊर्जा देंगे: 1. एवोकेडो टॉस्ट से पकाए अंडे: पूर्ण गेहूं के टॉस्ट पर मसालेदार एवोकेडो, एक पकाए अंडे और लाल चिप्पी के छोटे छिड़काव के साथ। एवोकेडो के स्वास्थ्य पूर्ण वसा और अंडे के प्रोटीन आपको भरपूर और ध्यान केंद्रित रखेंगे। 2."}, {"role": "user", "content": "मुझे अवकाडो टॉस्ट और पूरी तरह से पके हुए अंडे के साथ अवकाडो टॉस्ट की ध्वनि पसंद है और स्मूथी बॉल की ध्वनि भी। क्या आपके पास सुबह में इन्हें तेजी से तैयार करने के लिए कोई टिप्स हैं?"}, {"role": "assistant", "content": "अच्छे चुनाव हैं! अवकाडो टॉस्ट से पोच्ड अंडे और स्मूथी बॉल दोनों ब्रेकफास्ट के नामी और पोषक वाले विकल्प हैं जो सुबह तेजी से तैयार किए जा सकते हैं। आप इन्हें तेजी से तैयार करने के लिए कुछ टिप्स यहां दिए गए हैं: अवकाडो टॉस्ट से पोच्ड अंडे: 1. आगे तैयार करें: अवकाडो को मसाला करें और एक बंद बर्तन में ठंड के बर्तन में रात भर रख दें। इससे आपको सुबह लगभग 5 मिनट बच जाएंगे। 2."}]}
{"filler_id": "FL19", "session_id": "ultrachat_415674", "turns": [{"role": "user", "content": "कैसे नॉर्सियोलॉजी और मनोविज्ञान में प्रगति हमारे ज्ञान को मानव मन और व्यवहार के समझ में बदल दिया है?"}, {"role": "assistant", "content": "मनोविज्ञान और मनोविद्या में प्रगति हमारे ज्ञान के लिए मानसिक चेतना और व्यवहार के बारे में बहुत तरह से बदल दिया है: 1. मस्तिष्क के बारे में ज्ञान: मनोविज्ञान तकनीकी उपकरणों में प्रगति, जैसे कि कार्यात्मक चुम्बकीय रेजोनेंस छवि (fMRI) और इलेक्ट्रोएंस्फालोग्राम (EEG), हमें मस्तिष्क के गतिविधियों को मापने और बेहतर ढंग से समझने की अनुमति देते हैं। इस तकनीक ने शोधकर्ताओं को विभिन्न मस्तिष्क क्षेत्रों के विभिन्न मानसिक कार्यों, जैसे कि अनुभव, ध"}, {"role": "user", "content": "दिमाग और व्यवहार के बारे में मनुष्य के नए अनुभव कैसे न्यूरोसाइंस और मनोविज्ञान में विकास के कारण हुए हैं, इसको देखना बहुत दिलचस्प है। आप क्या सोचते हैं कि इन क्षेत्रों में अगला ब्रेकथ्रू क्या होगा?"}, {"role": "assistant", "content": "मैं व्यक्तिगत राय या भविष्य के घटनाओं की भविष्यवाणी करने की क्षमता नहीं रखता। हालांकि, चल रहे अनुसंधान और हाल के विकास के आधार पर, भूतिशास्त्र और मनोविज्ञान में भविष्य में कई क्षेत्रों में महत्वपूर्ण उपलब्धियां देखने की उम्मीद है। एक ऐसा अनुसंधान क्षेत्र जिस पर बढ़ती ध्यान दिया जा रहा है, गैस्ट्रोइंटेस्टीनल अक्ष के अध्ययन है।"}]}
{"filler_id": "FL20", "session_id": "ultrachat_93661", "turns": [{"role": "user", "content": "किसी तेज गति के काम के वातावरण में आगे बढ़े रहने और समय के प्रभावी तरीके से प्रबंध करने के लिए कुछ सर्वोत्तम अभ्यास क्या हैं?"}, {"role": "assistant", "content": "1. एक प्लेनर या कैलेंडर का उपयोग करें: अपने सभी कार्य और मैदानी तारीखों को एक स्थान पर लिख दें ताकि आप अपने अपने अनुसूची के ऊपर नियंत्रण रख सकें और कुछ बातें छूट न जाएं। 2. अपने कार्य को प्राथमिकता दें: आपके कार्य को महत्व और मैदानी तारीख के आधार पर प्राथमिकता दें। सबसे महत्वपूर्ण कार्य से शुरू करें और फिर कम महत्वपूर्ण कार्य तक अपने आप को बढ़ाएं। 3."}, {"role": "user", "content": "मुझे विशेष रूप से अव्यवस्था के बारे में लड़ना कठिन होता है, इसलिए मैं निश्चित रूप से उस पर काम करूंगा। क्या आपके पास कोई विशिष्ट उत्पादकता एप्स या टूल्स के लिए सिफारिश है?"}, {"role": "assistant", "content": "हां, कई उत्पादकता एप्लिकेशन और टूल्स उपलब्ध हैं जो आपको आगे बढ़े रहने और अपना समय प्रभावी तरीके से प्रबंधित करने में मदद कर सकते हैं। यहां कुछ सिफारिशें हैं: 1. ट्रेलो: एक दृश्य प्रोजेक्ट प्रबंधन टूल जो आपको बोर्ड, कार्ड और सूची बनाने की अनुमति देता है ताकि आप अपने कार्य और कार्यप्रवाह को आगे बढ़ा सकें। 2. फोकस@विल: एक संगीत स्ट्रीमिंग सेवा जो पीछे की गाना बजाती है जो आपको ध्यान रखे रहने और उत्पादकता बनाए रखने में मदद करती है।"}]}
{"filler_id": "FL21", "session_id": "b7c5c12f_2", "turns": [{"role": "user", "content": "मैं कुछ सुस्तीनेबल लाइफ के लिए सिफारिश चाहता हूँ। मैं अपने हफ्तेवार मैगजीन में इस बारे में बहुत पढ़ रहा हूँ जो मुझे हर हफ्ते प्राप्त होता है। कोई भी, क्या आप मुझे दैनिक जीवन में मेरे कार्बन फुटप्रिंट को कम करने के लिए कुछ टिप्स दे सकते हैं?"}, {"role": "assistant", "content": "सुस्थिर जीवन एक बहुत महत्वपूर्ण विषय है! मैं इस बारे में आपकी मदद करने में बिल्कुल खुश हूं। यहां कुछ व्यावहारिक टिप्स हैं जो आपके दैनिक जीवन में आपके कार्बन प्रभाव को कम करने में मदद कर सकते हैं: 1. ऊर्जा उपभोग कम करें जब आप उपकरणों और उपकरणों का उपयोग नहीं कर रहे हों तो लाइट्स, इलेक्ट्रॉनिक्स और उपकरण बंद कर दें ताकि स्टैंडबाई ऊर्जा उपभोग को रोका जा सके। ऊर्जा-कुशल LED बल्बों में स्विच करें। आपके थरमोस्टेट को सं调整 करें ताकि गर्म"}, {"role": "user", "content": "मैं वास्तव में स्थायी वस्त्र विषय में बहुत रुचि रखता हूँ, जिसके बारे में मैं अपने मासिक पत्रिका में हाल ही में पढ़ा हूँ। क्या आप मुझे बता सकते हैं कि मेरे कपड़े कैसे अधिक पर्यावरण अनुकूल बनाए जा सकते हैं?"}, {"role": "assistant", "content": "स्थायी वस्त्र एक अद्भुत क्षेत्र है जिस पर ध्यान केंद्रित किया जा सकता है! वस्त्र उद्योग के पर्यावरण पर बहुत बड़ा प्रभाव होता है, लेकिन कुछ जागरूक चयन करके आप अपने वस्त्र के प्रभाव को कम कर सकते हैं। यहां कुछ सलाह हैं जो आपको एक अधिक पर्यावरण अनुकूल वस्त्र भंडार बनान में मदद कर सकती हैं: 1. पहले से बेचे गए वस्तुओं को खरीदें दुकानों, विनिमय दुकानों और ईबे, थ्रेडअप या पॉशमार्क जैसे ऑनलाइन बाजारों के माध्यम से दूसरे हाथ के खरीदारी के लिए"}]}
{"filler_id": "FL22", "session_id": "df6970d8", "turns": [{"role": "user", "content": "मेरे द्वारा मित्रों और परिवार के साथ खेलने वाले नए बोर्ड गेम के लिए सुझाव चाहिए। कोई सुझाव हैं?"}, {"role": "assistant", "content": "नए बोर्ड गेम अपने दोस्तों और परिवार के साथ खेलने के लिए? मैं आपके लिए एक धनी भंडार के सुझाव ले आया हूँ! यहाँ कुछ आधुनिक क्लासिक, छिपे हुए रत्न और विभिन्न श्रेणियों में लोकप्रिय गेम हैं: रणनीति और प्रतियोगिता 1. ग्लूमहेवन: सहयोगी कैंपेन आधारित गेम है, जिसमें मॉड्यूलर बोर्ड और लेजेंड स्टाइल गेमप्ले होता है। 2. स्काइथ: 1920 के एक वैकल्पिक इतिहास में सेट किया गया एक वर्कर प्लेसमेंट गेम है, जिसमें इंजन बिल्डिंग और क्षेत्र"}, {"role": "user", "content": "मैंने पहले टिकेट टू राइड और सेटलर्स ऑफ़ कैटन खेले हैं, और वे बहुत अच्छे खेल हैं। क्या आपको बर्बर गेटरीज के समान खेलों के लिए सिफारिश करनी है?"}, {"role": "assistant", "content": "स्कैटरगेटरीज एक अद्भुत खेल है! अगर आप क利एटिव, शब्द आधारित खेल के लिए रोमांचित हैं, तो यहां कुछ सिफारिशें हैं जो उसी चाह लगाने में मदद कर सकती हैं: 1. कोडनेम्स: यह स्कैटरगेटरीज के ठीक वैसे नहीं है, लेकिन कोडनेम्स में शब्द अनुमान लगाने और क्रिएटिव सोच के लिए खेल खेला जाता है। खिलाड़ी अपने साथी द्वारा दिए गए एक शब्द के आधार पर शब्दों की पहचान करने की कोशिश करते हैं। 2. क्वेल्फ़: एक पार्टी खेल जो खिलाड़ियों को निश्चित मानदंड"}]}
{"filler_id": "FL23", "session_id": "ultrachat_518652", "turns": [{"role": "user", "content": "त्रोजन युद्ध के ग्रीक मिथकोलोजी में क्या महत्व था?"}, {"role": "assistant", "content": "त्रोआन के युद्ध ग्रीक मिथकोलोजी के सबसे महत्वपूर्ण घटनाओं में से एक था। यह एक युद्ध था जो ग्रीक और त्रोआन के बीच लड़ा गया था और इसने दस साल तक चला। युद्ध त्रोआ की हेलन के लिए लड़ा गया था, जो स्पार्टा के राजा मेनेलाउस की पत्नी थी। हेलन को त्रोआ के प्रिंस पेरिस द्वारा लूट लिया गया था और त्रोआ में ले जाया गया था। युद्ध का महत्व यह था कि इसे धर्म और गौरव के लिए लड़े जाने के बारे में कहा जाता था।"}, {"role": "user", "content": "त्रोआन युद्ध के अंत में कौन सा पक्ष जीत गया?"}, {"role": "assistant", "content": "ग्रीक अंततः ट्रोइ के युद्ध में विजयी हो गए, जिसमें उन्होंने ट्रोइयन घोड़ा का उपयोग किया, जो एक लकड़ी के घोड़े में ग्रीक सैनिकों के भरे हुए थे, ताकि वे ट्रोइ शहर को आक्रमण कर सकें और जीत ले सकें। ट्रोइयन घोड़ा ट्रोइ के द्वारों के बाहर छोड़ दिया गया था, जिसे ईश्वरों के लिए एक बर्बादी के रूप में माना जाता था, और ट्रोजन इसे शहर के अंदर ले आए। उस रात, ग्रीक सैनिक घोड़े से बाहर निकले और ट्रोइ के द्वार खोल दिए, जिससे ग्रीक सेना शह"}]}
{"filler_id": "FL24", "session_id": "ultrachat_106352", "turns": [{"role": "user", "content": "एक स्वस्थ वयस्क के लिए प्रतिदिन कितने घंटे सोना अनुशंसित है?"}, {"role": "assistant", "content": "एक स्वस्थ वयस्क के लिए प्रतिदिन नींद के अनुशोधन अवधि 7-9 घंटे होनी चाहिए।"}, {"role": "user", "content": "उसमें सोने के लिए बहुत समय बर्बाद करना लगता है। मुझे क्यों नहीं 5-6 घंटे से भी ज्यादा नहीं हो जाता?"}, {"role": "assistant", "content": "एक AI भाषा मॉडल के रूप में, मैं चिकित्सा सलाह देने या स्वास्थ्य व्यवस्था में परिवर्तन करने की सलाह नहीं दे सकता। हालांकि, अध्ययनों ने दिखाया है कि सतत रूप से 7 घंटे से कम सोना नकारात्मक स्वास्थ्य प्रभाव का कारण बन सकता है, जिसमें समावेश है बुद्धिमत्ता कार्य कम हो जाना, थकावट और ताकतवर इम्यून सिस्टम कमजोर हो जाना। यह सिफारिश की जाती है कि वयस्क लोगों को कम से कम प्रतिदिन 7 घंटे सोना चाहिए ताकि अच्छा स्वास्थ्य बनाए रखा जा सके।"}]}
{"filler_id": "FL25", "session_id": "fb793c87_2", "turns": [{"role": "user", "content": "मैं एचबीओ मैक्स पर कुछ नए शो देखने की तलाश में हूँ। मैं लंबे समय से \"गेम ऑफ थ्रोंस\" को देखना चाहता आ रहा हूँ, और लगभग एक महीने पहले इसे शुरू कर दिया। अब तक मैं पहले चार सीज़न पूरे कर चुका हूँ, और मैं इसे बहुत अच्छी तरह से ले रहा हूँ। क्या आपको \"गेम ऑफ थ्रोंस\" जैसे कुछ अन्य सुचारु रूप से अनुशंसा करने के लिए कोई बात है?"}, {"role": "assistant", "content": "\"Game of Thrones\" एक महान शो है! आप आधे रास्ते पर हैं, और मुझे आपको लगता है कि आप इसमें फंस गए हैं। अगर आप HBO Max पर इस तरह के अन्य शो खोज रहे हैं, तो मैं आपके लिए कुछ अद्भुत सुझाव ले आया हूँ: 1. वेस्टवर्ल्ड: एक अन्य HBO महाकाव्य, जो भविष्य के एक थीम पार्क में सेट है, जहां ग्राहक जीवंत रोबोट के साथ अंतराल बना सकते हैं। यह एक जटिल और विचार उत्प्रेरक श्रृंखला है, जिसमें कार्यकलाप, ड्रामा और राजनीति के मिश्रण के समान गुण हैं। 2. विंग्स: एक ऐत"}, {"role": "user", "content": "वह एक अच्छी सूची है! मैं निश्चित रूप से उनमें से कुछ देखने जा रहा हूँ। मैंने अपने अपने समय में हुलू पर कुछ दॉक्यूमेंटरी देखी हैं, विशेष रूप से वे जो वास्तविक अपराध और विज्ञान पर हैं। क्या आप हूबॉ मैक्स पर दॉक्यूमेंटरी के लिए कोई सिफारिश कर सकते हैं?"}, {"role": "assistant", "content": "एचबीओ मैक्स में दस्तावेजी फिल्मों के एक अद्भुत संग्रह है, और मुझको आपके लिए कुछ अद्भुत सुझाव हैं: सच्चा अपराध: 1. मैं अंधकार में जाऊंगा: गोल्डन स्टेट किलर के बारे में मिचेल मैकनामरा की किताब पर आधारित एक दस्तावेजी श्रृंखला। यह एक बेहद रोमांचक और चिंतन प्रेरक अध्ययन है, जो मामले और इसके शिकारों और उनके परिवारों पर प्रभाव के बारे में है। 2."}]}
{"filler_id": "FL26", "session_id": "b0b73826", "turns": [{"role": "user", "content": "मैं अपने दोस्तों के साथ इस छुट्टी के दिन एक जमकर इकट्ठा होने की योजना बना रहा हूँ। क्या आपको कोई अच्छा ब्रंच स्थान के बारे में सुझाव है?"}, {"role": "assistant", "content": "मैं आपकी मदद करने के लिए खुश होऊंगा! हालांकि, मैं एक बड़े भाषा मॉडल हूं, मुझको व्यक्तिगत अनुभव या विशिष्ट स्थानों के लेटेस्ट जानकारी नहीं है। लेकिन मैं आपको कुछ सामान्य सलाह और प्रश्न दे सकता हूं जो आपके समूह के लिए एक अच्छा ब्रंच स्थान खोजन में मदद करेंगे: 1. अपने दोस्तों की पसंदों को ध्यान में रखें: अपने दोस्तों से उनके आहार के विशेषाधिकार, पसंदीदा खाना और पसंदीदा वातावरण के बारे में पूछें। इससे आपके विकल्पों को सीमित करने में"}, {"role": "user", "content": "मैं एक नए स्थान को आज तक आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आज आ"}, {"role": "assistant", "content": "वह एक अच्छा विचार है! एक स्थान पर वापस जाना एक अच्छा विकल्प हो सकता है, विशेष रूप से आप उस स्थान के माहौल और खाना जानते और पसंद करते हों तो। एक विश्वसनीय स्थान के पास होना हमेशा अच्छा लगता है जिस पर आप एक अच्छा अनुभव अपेक्षा कर सकते हैं। और इसके अलावा, आप अपने माता-पिता के साथ वहां गए हैं, इसलिए आपके लिए वहां एक विशेष संबंध हो सकता है, और अपने दोस्तों के साथ उस अनुभव को साझा करना भी मजेदार हो सकता है।"}]}
{"filler_id": "FL27", "session_id": "ultrachat_576167", "turns": [{"role": "user", "content": "फोटोग्राफी में प्रयोग किए जाने वाले लेंस के प्रकार कैसे अंतिम चित्र को प्रभावित करते हैं?"}, {"role": "assistant", "content": "फोटोग्राफी में प्रयोग किए जाने वाले लेंस के प्रकार अंतिम छवि को कई तरह से प्रभावित करते हैं: 1. फोकल लंबाई: एक लेंस की फोकल लंबाई इसके दृश्य कोण और वस्तु के आवर्धन को निर्धारित करती है। एक लंबी फोकल लंबाई वाला लेंस (टेलीफोटो) परिपेक्ष्य को संकुचित करेगा और गहराई के क्षेत्र को कम करेगा, जबकि एक छोटी फोकल लंबाई वाला लेंस (वाइड-एंगल) परिपेक्ष्य को बढ़ाएगा और गहराई के क्षेत्र को विस्तारित करेगा। 2."}, {"role": "user", "content": "फोटोग्राफी में एक विशेष विषय या स्थिति के लिए सही लेंस कैसे चुने?"}, {"role": "assistant", "content": "फोटोग्राफी में किसी विशेष वस्तु या स्थिति के लिए सही लेंस का चयन अंतिम छवि पर बहुत बड़ा प्रभाव डाल सकता है। आपके लिए सही लेंस चुनने में मदद करने वाले कुछ बिंदु यहां दिए गए हैं: 1. फोकल लंबाई: आप और वस्तु के बीच की दूरी को ध्यान में रखें। अगर आप दूर से वस्तु को छाया रहे हैं, तो लंबी फोकल लंबाई के लेंस (टेलीफोटो) का उपयोग करें। अगर आप अपनी वस्तु से निकट हैं, तो छोटी फोकल लंबाई के लेंस (वाइड-एंगल) का उपयोग करें। 2. एपर्चर: प्रकाश"}]}
{"filler_id": "FL28", "session_id": "ultrachat_182518", "turns": [{"role": "user", "content": "सां लुइस पोटोसी के अर्थव्यवस्था और सामाजिक संरचना के इतिहास में मिनिंग उद्योग किस प्रकार प्रभाव डाला है?"}, {"role": "assistant", "content": "सान लुइस पोटोसी के अर्थव्यवस्था और सामाजिक संरचना पर खनन उद्योग का बहुत बड़ा प्रभाव रहा है। 16 वीं सदी में इस क्षेत्र में चांदी के भंडार के खोज के कारण कई खान और खनन शहरों की स्थापना हुई, जिसके परिणामस्वरूप अपना काम और अन्य अवसरों की खोज में आकर बड़ी संख्या में प्रवासी लोग आए।"}, {"role": "user", "content": "सैन लुइस पोटोसी पर मिनिंग के प्रभाव के बारे में सीखना वास्तव में दिलचस्प है। क्या इस उद्योग के नकारात्मक परिणामों के खिलाफ कोई प्रयास किए गए हैं?"}, {"role": "assistant", "content": "हां, सैन लुइस पोटोसी में मिनिंग उद्योग के नकारात्मक परिणामों के बारे में अधिक जागरूकता के कारण उपाय किए गए हैं। हाल के वर्षों में, मिनिंग के स्थानीय समुदायों और पारिस्थितिक तंत्र पर पर्यावरणीय प्रभावों के बारे में अधिक जागरूकता हो गई है, जिसके कारण सख्त नियमों के अपनाने और अधिक सुस्थिर मिनिंग अभ्यासों के विकास के लिए कदम उठाए गए हैं।"}]}
{"filler_id": "FL29", "session_id": "98cd882c_1", "turns": [{"role": "user", "content": "मैं एक नए सिलाई के प्रोजेक्ट के लिए कुछ विचार खोज रहा हूँ। मेरी बहन के जन्मदिन के लिए मैंने एक किट को बना दिया है और अब मैं कुछ नए काम करने के लिए बहुत उत्सुक हूँ। बता दूँ कि मुझे किट को पूरा करने में लगभग 3 सप्ताह लगे और मैं इसके अपने रूप में बहुत खुश हूँ।"}, {"role": "assistant", "content": "आपको आपकी बहन के जन्मदिन के लिए काटा बनाने पर बधाई! यह एक अद्भुत साधन है, और मुझे खुशी हो रही है कि वह इसे बहुत चाहेगी। तीन सप्ताह में एक काटा बनाना एक बहुत अच्छा समय है, विचार करें कि इसमें कितना समय और मेहनत लगती है। अब, आइए अपने अगले सिलाई परियोजना के लिए विचारों के बारे में ब्रेंड बर्निंग शुरू करते हैं! आपके लिए कुछ सुझाव यहां दिए गए हैं: 1. टोट बैग या बैग: एक कार्यशील और व्यावहारिक परियोजना जो तेजी से पूरा की जा सकती है"}, {"role": "user", "content": "मैं गर्मी के लिए एक संडूल बनाने के बारे में सोच रही हूँ, संभवतः कोई फूलों के छाप वाला चीज। मैंने कभी भी कोई वस्त्र बनाया ही नहीं है, इसलिए मुझे इसके बारे में थोड़ा डर लग रहा है। क्या आप ऐसे कुछ सलाह या पैटर्न बता सकते हैं जो एक शुरुआती व्यक्ति जैसे मुझे उपयोगी हो सकें?"}, {"role": "assistant", "content": "सूडेन बनाना शुरूआती व्यक्ति के लिए एक अच्छा प्रोजेक्ट है! थोड़ी सी मार्गदर्शन के साथ आप गर्मी के लिए एक सुंदर और सुविधाजनक कपड़ा बना सकते हैं। मैं आपके लिए सलाह और पैटर्न के चयन के बारे में मदद करने के लिए खुश हूं। शुरूआती के लिए सलाह: 1. एक सरल पैटर्न चुनें: \"शुरूआती अनुकूल\" या \"आसान\" लेबल वाले पैटर्न की तलाश करें। बहुत सारे टुकड़ों, जटिल निर्माण या जटिल विवरण वाले पैटर्न से बचें। 2. एक बर्दाश्त करने योग्य कपड़ा चुन"}]}
{"filler_id": "FL30", "session_id": "f5f1ff92_1", "turns": [{"role": "user", "content": "मैं ऋण आवेदन प्रक्रिया के बारे में कुछ सलाह लेना चाहता हूँ। मैंने लेन्डर को आवेदन के लिए सभी आवश्यक दस्तावेज जमा कर दिए हैं, जिसमें मेरे वेतन चेक, बैंक स्टेटमेंट और कर वापसी शामिल हैं, लेकिन मुझे आगे क्या अपेक्षित है इस बारे में निश्चित नहीं है। आवेदन जमा करने के बाद सामान्य चरण क्या होते हैं, इसके बारे में मुझे बताएं।"}, {"role": "assistant", "content": "आपके ऋण आवेदन के आवेदन के लिए बधाई! आपके दस्तावेज जमा करने के बाद, ऋणदाता आपके आवेदन की जांच करेगा और आवेदन की प्रक्रिया शुरू करेगा। आपके आवेदन के बाद आमतौर पर होने वाले कदमों का एक सामान्य ढांचा नीचे दिया गया है: 1. प्रारंभिक जांच: ऋणदाता के ऋण प्रक्रियाकार आपके आवेदन की जांच करेगा ताकि सभी आवश्यक दस्तावेज पूर्ण और ठीक हों। यह कदम आमतौर पर 1-3 कार्य दिन लेता है। 2. ऋण प्रक्रिया: ऋण प्रक्रियाकार आपके क्रेडिट रिपोर्ट की जांच करेगा, आप"}, {"role": "user", "content": "मैं अंकन के चरण के बारे में थोड़ा चिंतित हूँ। आप मुझे बता सकते हैं कि अंकन के दौरान क्या होता है और यह ऋण मंजूरी प्रक्रिया पर कैसे प्रभाव डालता है?"}, {"role": "assistant", "content": "मूल्यांकन कदम मकान ऋण प्रक्रिया का एक महत्वपूर्ण हिस्सा होता है, और इसके बारे में सवाल करना स्वाभाविक है। यहां एक विस्तृत समझौता है कि मूल्यांकन के दौरान क्या होता है और यह ऋण मंजूरी प्रक्रिया पर कैसे प्रभाव डालता है: क्या मूल्यांकन है? एक आवासीय मूल्यांकन एक संपत्ति के मूल्य का स्वतंत्र आकलन होता है, जो एक प्रमाणित मूल्यांकनकर्ता द्वारा किया जाता है। मूल्यांकनकर्ता संपत्ति के बाजार मूल्य के वैज्ञानिक अनुमान देता है, जो निम्नल"}]}
{"filler_id": "FL31", "session_id": "39948bcf_4", "turns": [{"role": "user", "content": "मेरे क्षेत्र में कुछ योग क्लास के सुझाव चाहिए। मैं अपने अभ्यास को जारी रखना चाहता हूँ क्योंकि पिछले नवंबर में मैं एक छह दिवसीय योग रिट्रीट में शामिल हुआ था, जिसमें ध्यान, प्राणायाम और आसन के कई कार्यशालाएँ शामिल थीं।"}, {"role": "assistant", "content": "अद्भुत! आपको रिट्रीट के बाद अपने योग के अभ्यास को जारी रखने के लिए प्रेरित महसूस करना बहुत अच्छा है। आपके आवश्यकतानुसार एक योग क्लास खोजने के लिए निम्नलिखित टिप्स आपके लिए मददगार हो सकते हैं: 1. ऑनलाइन खोज: गूगल जैसे खोज इंजन में \"योग क्लासेज नेयर मी\" या \"योग स्टूडियो इन [आपका शहर/पिन कोड]\" टाइप करें। आप ऑनलाइन डायरेक्टरीज जैसे योग फाइंडर या योग एलियंस का उपयोग भी कर सकते हैं जिससे आपके क्षेत्र में स्टूडियो और क्लास खोजना स"}, {"role": "user", "content": "मैं वास्तव में एक स्टूडियो खोज रहा हूँ जो रेस्टॉरेटिव योग पर केंद्रित होता है, क्योंकि मैं उस विशेष कार्यशाला को रिट्रीट के दौरान बहुत अच्छा लगा। क्या आपको मेरे क्षेत्र में रेस्टॉरेटिव योग पर विशेषज्ञ रहे ऐसे कोई स्टूडियो जाने हैं?"}, {"role": "assistant", "content": "अद्भुत! रिस्टोरेटिव योग एक स्टूडियो के लिए एक अच्छा ध्यान केंद्र है, और आपके इस क्षेत्र में अपने अभ्यास को गहरा करने के लिए आपकी खोज करना बहुत अच्छा है। बुराई बात यह है कि मैं एक बड़े भाषा मॉडल हूँ, इसलिए मुझको आपके स्थान या आपके क्षेत्र में विशिष्ट स्टूडियो के बारे में वास्तविक समय के डेटा के लिए पहुँच नहीं है। हालांकि, मैं आपको अपने पास रिस्टोरेटिव योग स्टूडियो कैसे ढूंढ़ें इस बारे में मार्गदर्शन कर सकता हूँ: 1."}]}
{"filler_id": "FL32", "session_id": "ultrachat_82739", "turns": [{"role": "user", "content": "क्या बैकस्टेज पास आमतौर पर कुछ व्यक्तियों के लिए ही सीमित होते हैं? यदि हां, तो मैं एक पास प्राप्त करने के अपनी संभावना कैसे बढ़ा सकता हूं?"}, {"role": "assistant", "content": "हाँ, बैकस्टेज पास आमतौर पर कुछ विशिष्ट लोगों के लिए ही उपलब्ध होते हैं। आपके लिए एक पास प्राप्त करने के अवसर बढ़ाने के लिए यहां कुछ टिप्स हैं: 1. फैन क्लब या ईमेल सूची में शामिल हो जाएं: कई कलाकार और बैंड अपने फैन क्लब या ईमेल सूची के सदस्यों के लिए बैकस्टेज पास या टिकट के प्री-बिक्री के लिए विशेष पहुंच प्रदान करते हैं। 2. सोशल मीडिया पर उनका फॉलो करें: कलाकार और बैंड अक्सर अपने सोशल मीडिया प्लेटफॉर्म पर बैकस्टेज पास के गिफ्ट या"}, {"role": "user", "content": "मैं निश्चित रूप से सोशल मीडिया पर उन्हें फॉलो करना शुरू करूंगा और कुछ कंपटिशन में भाग लूंगा। क्या आप जानते हैं कि क्या कोई आगामी कंसर्ट है जिसमें बैकस्टेज पास उपलब्ध हैं?"}, {"role": "assistant", "content": "एक AI भाषा मॉडल के रूप में, मुझको आगामी कन्सर्ट के बारे में वास्तविक समय की जानकारी नहीं है जिनमें बैकस्टेज पास उपलब्ध हों। हालांकि, आप आर्टिस्ट या बैंड की आधिकारिक वेबसाइट या सोशल मीडिया पृष्ठों के माध्यम से अपने आगामी कन्सर्ट के बारे में जानकारी ले सकते हैं और उनके द्वारा उपलब्ध बैकस्टेज पास के अवसरों के बारे में जान सकते हैं। आप निश्चित रूप से कन्सर्ट वेन्यू या टिकटिंग एजेंसी से भी चेक कर सकते हैं कि वे कोई विशेष VIP या बैकस"}]}
{"filler_id": "FL33", "session_id": "f340a3db", "turns": [{"role": "user", "content": "मैं में अपने सामाजिक संचार अभियान की योजना बना रहा हूँ 1 मई को मानसिक स्वास्थ्य के बारे में जागरूकता फैलाने के लिए, क्या आप मुझे कुछ रोचक पोस्ट विचार और हैशटैग बनाने में मदद कर सकते हैं?"}, {"role": "assistant", "content": "कैसा अद्भुत प्रयोग! मैं आपकी सामाजिक मीडिया कैम्पेन के लिए ध्यान आकर्षित करने वाले पोस्ट आइडिया और हैशटैग की मदद करने के लिए खुश होऊंगा। यहां कुछ सुझाव हैं: पोस्ट आइडिया: 1. व्यक्तिगत कहानियां: मानसिक स्वास्थ्य संबंधी चुनौतियों और उनके विजय के व्यक्तिगत कहानियां साझा करें। अपने फॉलोअर्स को अपनी कहानियों को साझा करने के लिए एक ब्रांडेड हैशटैग का उपयोग करने के लिए प्रोत्साहित करें। 2."}, {"role": "user", "content": "क्या आप कुछ लोकप्रिय मानसिक स्वास्थ्य समर्थक या प्रभावशाली लोगों के नाम सुझा सकते हैं जिनसे मैं सहयोग या प्रश्न-उत्तर बैठक के लिए संपर्क कर सकूं?"}, {"role": "assistant", "content": "यहां कुछ लोकप्रिय मानसिक स्वास्थ्य संचारक और प्रभावशाली व्यक्ति हैं जो विभिन्न प्लेटफॉर्म पर हैं जिनसे आप सहयोग या प्रश्न-उत्तर बैठक के लिए संपर्क कर सकते हैं: मानसिक स्वास्थ्य संचारक: 1. लेडी गागा (@ladygaga) - गायक-गीतक और जन्म इस तरह हुआ फाउंडेशन के संस्थापक, जो मानसिक स्वास्थ्य और स्वास्थ्य के बारे में काम करता है। 2. डेमी लॉवेटो (@ddlovato) - गायक-गीतक और मानसिक स्वास्थ्य संचारक जो अपने डिप्रेशन और नशा से जुड़े अपने तकलीफों के बारे में खु"}]}
{"filler_id": "FL34", "session_id": "ultrachat_245735", "turns": [{"role": "user", "content": "किउआ भाषा के संरक्षण को रोकने वाले सांस्कृतिक बाधाएँ क्या हैं, और उन्हें कैसे पार किया गया है?"}, {"role": "assistant", "content": "कई सांस्कृतिक बाधाएँ निवासी लोगों की भाषा के संरक्षण को रोकती रही हैं, जिसमें ऐतिहासिक चोट, बलपूर्वक समाजीकरण और छोटे लोगों के सांस्कृतिक अनुकरण शामिल हैं। 1800 के दशक के अंत में, अमेरिकी सरकार अपने निवासी लोगों को रिजर्वेशन पर बलपूर्वक ले जाकर बोर्डिंग स्कूल बनाये जहाँ वे अपनी मातृभाषा बोलने की अनुमति नहीं दी गई थी। इससे किओवा बच्चों को अपने परिवार और समुदाय से अलग कर दिया गया था, जिसके परिणामस्वरूप पीढ़ी दर पीढ"}, {"role": "user", "content": "वाकई दिलचस्प है। क्या आप जानते हैं कि इन भाषा पुनर्जीवन कार्यक्रमों तक अब तक कितनी सफलता हुई है?"}, {"role": "assistant", "content": "हां, किओवा भाषा के पुनर्जीवन प्रयास में कुछ कामयाबी हुई हैं। उदाहरण के लिए, ओकलाहोमा के एनाडार्को में किओवा जनजाति के भाषा डूब कार्यक्रम ने भाषा अधिग्रहण और उपयोग के अंतर्गत धन्यवाद लाभदायक परिणाम देखे गए हैं। यह कार्यक्रम छोटे बच्चों को भाषा सिखाने और सांस्कृतिक संपर्क प्रदान करके उनमें एक मजबूत पहचान और समुदाय से जुड़े रहने की भावना के विकास में मदद करता है।"}]}
{"filler_id": "FL35", "session_id": "ultrachat_20447", "turns": [{"role": "user", "content": "रिमोट वर्क ने कर्मचारियों के सामाजिक और आभावी स्वास्थ्य पर कैसा प्रभाव डाला है?"}, {"role": "assistant", "content": "एक AI भाषा मॉडल के रूप में, मैं व्यक्तिगत अनुभव, राय या भावनाओं के साथ नहीं हूँ। हालांकि, अध्ययन और शोध दिखाते हैं कि दूरस्थ कार्य करना कर्मचारियों के सामाजिक और भावनात्मक स्वास्थ्य पर दोनों धनात्मक और नकारात्मक प्रभाव डाल सकता है। दूरस्थ कार्य के सामाजिक और भावनात्मक स्वास्थ्य पर धनात्मक प्रभाव: 1. लचीले कार्य व्यवस्थाएं लंबे यात्रा और भारी योजनाओं से संबंधित तनाव और खोई हुई ऊर्जा को कम कर सकती हैं। 2."}, {"role": "user", "content": "क्या नौकरी करने वाले लोग घर से काम करते समय अकेले और अलग लगने से बचने के लिए कोई रणनीति अपना सकते हैं?"}, {"role": "assistant", "content": "हाँ, नौकरी करने वाले लोग घर से काम करते समय अकेले और अलग लगने से बचे रह सकते हैं। यहां कुछ रणनीतियां हैं: 1. नियमित आनलाइन मीटिंग के लिए प्रोत्साहित करें ताकि दूरस्थ कर्मचारी संबंधित रह सकें। नियमित वीडियो कॉल और टीम कॉल के लिए निर्धारित करें ताकि सदस्यता और सहयोग की भावना बनी रहे। 2. आनलाइन कॉफी ब्रेक, लंच या हैपी अवधि बनाएं। इन ब्रेक के मदद से कर्मचारी आराम कर सकते हैं, सहकर्मियों के साथ सामाजिक रह सकते हैं और ऊर्जा फिर से प्राप"}]}
{"filler_id": "FL36", "session_id": "9f35ac68", "turns": [{"role": "user", "content": "मैं मासिक बस पास के बदले आने के बारे में सोच रहा हूँ, आप मुझे लागत बता सकते हैं और जहां से मैं इसकी खरीद कर सकता हूँ?"}, {"role": "assistant", "content": "मुझे आपकी मदद करने के लिए खुशी होगी! हालांकि, मुझे आपसे थोड़ी अधिक जानकारी चाहिए। कृपया मुझे निम्नलिखित विवरण प्रदान कर सकते हैं: 1. आप कहाँ रहते हैं (या बस पास का उपयोग करने के लिए योजना बना रहे हैं)? 2. आप किस प्रकार के बस पास के बारे में रुचि रखते हैं (उदाहरण के लिए: वयस्क, छात्र, वृद्ध, अवहेलित)? 3. आप खोज रहे हैं जिसके लिए पास की अवधि (उदाहरण के लिए: 1 महीना, 3 महीना, 12 महीना)?"}, {"role": "user", "content": "मैं शहर में रहता हूँ, और मैं एक वयस्क मासिक पास के लिए रुचि रखता हूँ। मैं एक महीने के पास की खोज कर रहा हूँ।"}, {"role": "assistant", "content": "मुझे आपकी मदद करने के लिए खुशी होगी! हालांकि, मैं एक बड़े भाषा मॉडल हूं, मुझको विशिष्ट शहरों या परिवहन प्रणालियों के लेटेस्ट जानकारी के लिए एक्सेस नहीं है। लेकिन मैं आपको वह जानकारी खोजने के लिए कुछ सामान्य सलाह दे सकता हूं। आपके शहर में मासिक बस पास के ठीक लागत और खरीद के बारे में जानकारी प्राप्त करने के लिए, मैं आपको निम्नलिखित कदम अनुशंसा करता हूं: 1."}]}
{"filler_id": "FL37", "session_id": "285a29e7_2", "turns": [{"role": "user", "content": "मैं कुछ किताबों के सुझाव चाहता हूँ। हाल ही में मैं बहुत पढ़ रहा हूँ, और मैं हमेशा नए लेखकों की खोज में रहता हूँ। बता दूँ कि मैं आज एक स्थानीय कॉफी शॉप में एक कविता पाठ के आयोजन में शामिल हुआ, जहां कई स्थानीय कवि अपने हालिया संग्रहों से पाठ देते रहे - ऐसे बहुत सारे दिग्गज लोगों के काम देखना बहुत अद्भुत लगा। कोई बात नहीं, आपको कुछ लोकप्रिय किताबें या लेखक बताएं जो आप अनुशंसा करोगे?"}, {"role": "assistant", "content": "उस बात के बारे में सुनकर एक अद्भुत अनुभव लगता है! स्थानीय कॉफी शॉप में कविता पाठ के लिए जाना नए तालंत खोजने और साहित्यिक समुदाय से जुड़ने के एक अच्छा तरीका है। अब किताबों के सुझाव के बारे में, मैं विभिन्न शैलियों में कुछ लोकप्रिय किताबों और लेखकों के सुझाव देने के लिए खुश होऊंगा। ध्यान रखें कि व्यक्तिगत पसंद भिन्न हो सकती हैं, लेकिन ये कुछ व्यापक रूप से सराहे गए और लोकप्रिय चयन हैं: कथाकाव्य: 1."}, {"role": "user", "content": "मुझे कुछ कविता के सुझाव अधिक खोजने में खुशी होगी। आज मैंने जो कविता पाठ सुना वह मेरी आंखों को खोल दिया और बाहर के कलाकारों की योग्यता के बारे में मुझे जानकारी मिली। क्या आप मेरे लिए आधुनिक कविता संग्रह या कवि के कुछ सुझाव दे सकते हैं, जिन्हें मैं खासकर खोज रहा हूं और नए आवाजों की खोज में लगा हुआ हूं?"}, {"role": "assistant", "content": "मुझे बहुत खुशी हो रही है कि कविता पाठ ने आपको अधिक कविता के अन्वेषण के लिए प्रेरित किया है! आधुनिक कविता एक जीवंत और विविध क्षेत्र है, और कई अद्भुत कवि और संग्रह खोजने लायक हैं। यहां कुछ सिफारिशें हैं जो आपके लिए अहम हो सकती हैं: आधुनिक कविता संग्रह: 1. जब मैं बड़ा होऊंगा तो मैं एक सूची के अतिरिक्त संभावनाओं के रूप में चाहता हूं द्वारा चेन चेन: एक प्रथम आउटपुट संग्रह जो पहचान, परिवार, प्यार और अस्थिर दुनिया में अर्थ की खोज के बारे में"}]}
{"filler_id": "FL38", "session_id": "sharegpt_NRqjZJS_0", "turns": [{"role": "user", "content": "लेख लिखें जिसमें एर्नेस्ट हेमिंगवे और जॉन स्टीनबेक के लेखन शैलियों की तुलना और विरोधाभास की जाए।"}, {"role": "assistant", "content": "एर्नेस्ट हेमिंगवे और जॉन स्टीनबेक 20 वीं सदी के सबसे प्रसिद्ध लेखकों में से दो हैं। दोनों व्यक्ति आम आदमी और मनुष्य के जीवन की दशा के बारे में लिखते रहे, लेकिन उनके लेखन के शैली बहुत अलग थे। हेमिंगवे के लिए उनके संक्षिप्त, सीधे और सरल लेखन शैली के लिए जाने जाते हैं। वे छोटे वाक्य और न्यूनतमिस्त अप्रत्यक्ष तरीके से अपनी संदेश व्यक्त करते हैं।"}, {"role": "user", "content": "कृपया कुछ उदाहरण शामिल करें, जिनमें उनके द्वारा लिखे गए कुछ किताबों का उल्लेख करके स्पष्ट रूप से संदर्भ दें।"}, {"role": "assistant", "content": "एर्नेस्ट हेमिंगवे और जॉन स्टीनबेक 20 वीं सदी के सबसे प्रसिद्ध लेखकों में से दो हैं। दोनों व्यक्ति आम आदमी और मनुष्य के जीवन की दशा के तकलीफों के बारे में लिखते रहे, लेकिन उनके लेखन के शैली बहुत अलग थे। हेमिंगवे के लिए उनके संक्षिप्त, सीधे और सरल लेखन शैली के लिए जाने जाते हैं।"}]}
{"filler_id": "FL39", "session_id": "0daaefa7_3", "turns": [{"role": "user", "content": "मैं वेब डेवलपमेंट में प्रवेश करने की कोशिश कर रहा हूँ, और मैं आपसे बात कर रहा हूँ कि क्या आप मुझे कुछ ऑनलाइन संसाधनों की सिफारिश कर सकते हैं जहां मैं जावास्क्रिप्ट सीख सकूँ। बता दें कि मैं अप्रैल के मध्य में एक दो दिवसीय कोडिंग बूटकैम्प में शामिल हुआ था, जहां मैंने जावास्क्रिप्ट के मूल बातों के बारे में सीखा, लेकिन मैं अपने कौशल को और बेहतर बनाना चाहता हूँ।"}, {"role": "assistant", "content": "आपको वेब विकास में पहला कदम उठाने और कोडिंग बूटकैंप में शामिल होने पर बधाई! जावास्क्रिप्ट के मूल आधार के साथ आप अपने कौशल को बेहतर बनाने और गहरा जानने के लिए तैयार हैं। यहां कुछ ऑनलाइन संसाधन हैं जो आपको जावास्क्रिप्ट के अध्ययन के लिए मदद करेंगे: संसाधन और कोर्स: 1. Codecademy के जावास्क्रिप्ट कोर्स: एक व्यापक और अंतरक्रियाशील कोर्स जो जावास्क्रिप्ट के मूल आधार, मध्यम विषय और उन्नत अवधारणाओं को शामिल करता है। 2. FreeCodeCamp: एक गैर-लाभकारी प"}, {"role": "user", "content": "मैं निश्चित रूप से इन संसाधनों को देखूंगा। मैं विशेष रूप से JSFiddle और HackerRank के चुनौतियों के साथ अभ्यास करने में रुचि रखता हूं। कोड सीखते समय मोटिवेट रहने और बर्नआउट से बचने के लिए क्या सलाह देनी चाहिए?"}, {"role": "assistant", "content": "जेसीएफ़िल्ड और हैकररैंक के प्रश्नों के साथ अभ्यास करना आपके जावास्क्रिप्ट कौशल को बेहतर बनाने के लिए एक अद्भुत तरीका है। निरंतर प्रेरित रहना और थकावट से बचना कोड लर्निंग के लिए महत्वपूर्ण अंग है। आपके लिए कुछ सलाह यहां दी गई है: 1. प्राप्त करने योग्य लक्ष्य और मीलपत्थर निर्धारित करें: अपने शिक्षा के यात्रा को छोटे, प्रबंधन करने योग्य लक्ष्यों में विभाजित करें। चाहे वे कितने छोटे लगे भी, अपने सफलताओं का जश्न मनाएं। इससे आप निरंत"}]}
{"filler_id": "FL40", "session_id": "ultrachat_134317", "turns": [{"role": "user", "content": "रोमन राज्य के धर्म के प्रमुख के रूप में सम्राट के भूमिका ने उनकी शक्ति में कैसे योगदान दिया?"}, {"role": "assistant", "content": "रोमन राज्य के धर्म के संस्थापक के रूप में ईम्पीरर की भूमिका उनकी शक्ति के निर्माण में महत्वपूर्ण भूमिका निभाई क्योंकि धर्म रोमन लोगों के जीवन में महत्वपूर्ण भूमिका निभाता था। ईम्पीरर धार्मिक संस्थानों, रिटुअल और समारोहों पर नियंत्रण रखते थे, जिससे वे आपने को रोमन पारंपरिक मूल्यों और प्रथाओं के संरक्षक के रूप में प्रस्तुत कर सकते थे।"}, {"role": "user", "content": "हां, तो साम्राज्य के शासकों को कुछ धार्मिक अनुष्ठान करने पड़ते थे या उनकी स्थिति अधिक चिह्नात्मक थी?"}, {"role": "assistant", "content": "रोमन राज्य के धर्म के संस्थापक के रूप में सम्राट की स्थिति केवल चिह्नात्मक नहीं थी; बल्कि उन्हें निश्चित धार्मिक अभ्यासों का पालन करने और महत्वपूर्ण धार्मिक समारोहों में भाग लेने की उम्मीद रखी जाती थी। सम्राट के धार्मिक कर्तव्यों में बलिदान करना, प्रार्थना करना और महत्वपूर्ण धार्मिक समारोहों के अध्यक्ष बने रहना शामिल था। सम्राट को ईश्वरों से विशेष रूप से संबंधित होने का विश्वास किया जाता था और उन्हें रोमन धर्म के उच्च पितृपाल के रू"}]}


In [ ]:
%%writefile /kaggle/working/clm/data/haystacks/filler_pool_hg.jsonl
{"filler_id": "FL01", "session_id": "ultrachat_201585", "turns": [{"role": "user", "content": "Beatles ki fame aur exposure Lennon ki songwriting me kaise impact kar thi?"}, {"role": "assistant", "content": "Aapke liye ek general answer de sakti hoon. Beatles ki fame aur exposure Lennon ki songwriting par deep impact kar thi. Band ki massive success ke karan, Lennon ko world ke saath apne music share karne ka platform mila, aur uske zariye wo sabse influential songs mein se kuch likh paye."}, {"role": "user", "content": "Kya aap mujhe kuch specific songs ka example de sakte hain jo Lennon ki fame aur exposure se influence hui hain?"}, {"role": "assistant", "content": "Haan, yahan kuch Lennon ka geet hain jo uski fame aur exposure se influenced hain: 1. \"Help!\" - Yeh geet ek waqt likha gaya tha jab Lennon band ka success aur public figure hone wala constant demands usse overwhelm karte hue tha. 2. \"Strawberry Fields Forever\" - Yeh geet Lennon ka Strawberry Field, Salvation Army children's home ke near growing up ke memories se inspired hai."}]}
{"filler_id": "FL02", "session_id": "5fb7fd18_1", "turns": [{"role": "user", "content": "Main kuch new baking inspiration chaiye hoon. Main aajkal bahut sweet treats banti hoon, lekin main kuch savory ki koshish karne ka chah rahi hoon. Kya aap kuch acchi savory bread recipe ka suggestion kar sakte hain? Taraf se, main last baar jab main sourdough bread banayi thi, toh usme ek pat of butter lagaye thi aur main usi flavor ko phir se banane ka icheej kar rahi hoon."}, {"role": "assistant", "content": "Ek swadist bread ke premi ke ban rahi hain! Main aapki swadist bread ke duniya ki khoj me madad karnay ke liye khush hongaa. Chuunki aap bartan ke saath sardoow khana pasand karte hain, isliye main kuch recipes sujhauongaa jo aapke sardoow bartan ke saman gun ya swad ke honge. Aapke shuruaat ke liye kuch vichar yahan hain: 1. Rozmeeri aur olive tel focacia: yeh italiya bread olive tel ke saath luvawan hata hai, jaise aapke sardoow bartan ke saath."}, {"role": "user", "content": "Yeh recipes bahut acche lag rahe hain, khushboo aur olie oil focaccia khusus se. Main rosemary ka use bread mei kisi bhi tarah se nahi kiya hai, isliye main usse try karne mei excited hai. Kya aap mujhe kuch tips de sakte hain fresh rosemary ka use kaise karein aur recipe mei kitanा use karna chahiye?"}, {"role": "assistant", "content": "Fresh rosemary ke saath kam karna ek little tricky hote hai, lekin kuch tips ke saath aap ko pro ban jayenge! Handling Fresh Rosemary: 1. Choose fresh, fragrant sprigs: Select rosemary sprigs jo vibrant green color aur potent, piney aroma rakhenge. Wilted ya brownish sprigs avoid karein, kyunki unme bitter taste ho sakti hai. 2. Store rosemary properly: Fresh rosemary ko plastic bag me rakhke ya plastic wrap me wrap karein aur fridge me rakhein taaki uska flavor aur aroma rakhe."}]}
{"filler_id": "FL03", "session_id": "afbd7193_2", "turns": [{"role": "user", "content": "Maine bookshelf organize karne ke liye kuch tips chahiye. Pehle Sunday ko maine apni living room ko deep clean kar diya aur pata chala ki bookshelf space ko maximum kaise kiya jaye. Taraf se bata do, main har Sunday 30 minute deep clean session ki commit hai, toh main usse maximum benefit le paune chahta hoon."}, {"role": "assistant", "content": "Regular deep clean session ke liye commit karne ke liye kudos! Ghar ko tidy aur organized rakhne ki yeh bahut acchi habit hai. Ab kuch tips dekhte hain jinse aap apne bookshelf space ko maximize kar sakein aur apne 30-minute ke Sunday deep clean session ka poora fayda utha sakein: 1. Purge aur categorize karein: pehle shelf se saari books nikaal lijiye aur unhe categories mein sort kijiye (jaise fiction, non-fiction, biographies, self-help, etc.)."}, {"role": "user", "content": "Main category karte hue aur har category ko ek home assign karte hue ka idea pasand karta hoon. Lekin kaise pata chalega ki kya rakhna hai aur kya chhodna hai? Main kuch pustakon ka possession hai jo saalों से nahi padhi hai, lekin unme sentimenta value hai."}, {"role": "assistant", "content": "Bataane ki kya rakhna hai aur kya chhodna hai ek mushkil kaam hai, especially jab sentimenta value ka matlab aaye. Yahan kuch tips hai jo aapko is decision mein madad karein: Sentimental value vs. practicality: 1. Apne aap ko puche: ye book kisi baar kya dekha ya padha? Sach mehnat karein. Agar saal baad ho gaya hai, toh aisa lagta hai ki aap usse chhodne ke baad kuch miss nahi karein. 2."}]}
{"filler_id": "FL04", "session_id": "ultrachat_391062", "turns": [{"role": "user", "content": "Kya kuch common gaming terms aur acronyms hain jo maine jaane chahiye?"}, {"role": "assistant", "content": "Yahan kuch common gaming terms aur acronyms hain: 1. FPS - First-person shooter 2. MMO - Massively multiplayer online 3. PVP - Player versus player 4. NPC - Non-player character 5. RNG - Random number generator 6. MOBA - Multiplayer online battle arena 7. DLC - Downloadable content 8. LFG - Looking for group 9. AFK - Away from keyboard 10. DPS - Damage per second 11."}, {"role": "user", "content": "Bhai, main yeh game ke RNG se bahut bohot kiya hoon. Hours lage hue hain aur mera need wala drop aaj tak nahi aaya. Yeh game mera against hai jaisa lag raha hai."}, {"role": "assistant", "content": "Main samajh gaya hai ki user ek game par rng (random number generator) ke karan frustation kar raha hai, jo kisi event ya drop ke outcome ko randomly decide kar deta hai, skill par nahi. User unke hours ka game play aur grinding ke baad bhi chahie jaane wala drop nahi mila hai, jo unki baat me poor luck ya unfair odds ke karan hoga."}]}
{"filler_id": "FL05", "session_id": "ultrachat_274962", "turns": [{"role": "user", "content": "Kaise Sulla ke military victories aur conquests uske political ideology ko shape karne mein madad kari aur uske policies ko influence karne mein?"}, {"role": "assistant", "content": "Sulla ke military victories aur conquests ne apne political ideology ke formation mein mahaan roop se kaam kiya aur uske policies ko bhi influence kiya. Jabki Sulla ek military general thi, uska battlefield par exceptional skills aur tactical brilliance famous thi. Apne military successes ke karan, uska Roman people mein immense popularity ho gayi aur usne ek power base bana li jo later state ke control lene mein use help ki."}, {"role": "user", "content": "Lag raha hai ki Sulla apne own power ko consolidate karna zyada interested thi, jabki Roman logon ke liye truly service karna nahi. Apne policies ne Rome ke citizens par kya impact pado?"}, {"role": "assistant", "content": "Aapne sahi kaha hai ki Sulla Rome ke logon ke liye service karne se zyada apne authority ko consolidate karne mein interested thi. Uske policies Rome ke citizens par big impact pade jo ultimately Roman Republic ke decline mein contribute karne mein aayi. Sulla ke policies ke sabse zyada impact mein power few individuals ke hands mein centralize hone ka tha."}]}
{"filler_id": "FL06", "session_id": "f62c04c6", "turns": [{"role": "user", "content": "Main aajkal ek content calendar banane ke baare mein soch raha hoon jo mere liye next month ke social media posts ke plan kiya jaye. Kya aap mujhe kuch ideas de sakte hain ki kya type ke posts karte raho aur kitni frequency se?"}, {"role": "assistant", "content": "Content calendar banane ka ek accha tarika hai jo aapko social media content ko advance plan aur organize karne mein madad karta hai. Post ka type aur frequency aapke target audience, goals aur use kar rahe hoon platforms par depend karta hai, lekin yahan kuch general ideas aur guidelines diye gaye hain jo aapko shuru karne mein madad kare: Post Types: 1. Educational/Informative: Apne niche se related tips, tutorials, industry news ya how-to guides share kare. 2. Promotional: Apne products, services ya events ke updates share kare. 3."}, {"role": "user", "content": "Woh ek aurat ki baat hai! Mai phir se har platform ke liye ek template banane ka start karein aur phir usme ideas daal denge. Kuch ideas Instagram Reels ke liye de sakte hain? Mai naye se ek post kiya tha morning yoga routine ke bare mein aur 50 views aur 10 likes mil gaye, isliye mai yoga ya meditation ke series kiya karna chahta hoon."}, {"role": "assistant", "content": "Instagram Reels ek aadhaarik tarika hai audience ke sath engage hone aur visibility badhane ke liye. Aapke pehle Reel ka success par congratulations! Yoga ya meditation series ek aadhaarik idea hai, khushiyon ke liye aapke pehle morning yoga routine Reel ke interest ko dekh kar. Yahan kuch ideas diye jate hain jo aapko shuru karne me madad karein: Yoga Series: 1. Morning Routine Variations: Alag-alag morning yoga routines share karein, har ek specific area par focus kare, jaise flexibility, strength ya relaxation. 2."}]}
{"filler_id": "FL07", "session_id": "8969b3ca_2", "turns": [{"role": "user", "content": "Maina ghar khareedi ke liye total cost ka estimate pata karna chahta hoon. Aap mera help kar sakte hain?"}, {"role": "assistant", "content": "Ghar khareedi ke liye total cost estimate karna aapko is big investment ke liye plan aur budget karna samajhne me madad karta hai. Yahan ek breakdown hai aapko expect hone wale typical costs ke bare me: 1. Purchase Price: Yeh ghar khareedi ke liye sabse bada cost hai, aur yeh aapko property ke seller se lete wala amount hai. 2. Down Payment: Yeh purchase price ka ek percentage hai jo aapne upfront, yaani phir se lete wala hai, jo 3.5% se 20% tak purchase price ka hota hai."}, {"role": "user", "content": "Main aksar house closing ke liye close ho raha hoon, aur main apni closing cost ke liye zyada accurate estimate chaahta hoon. Sellers ne $2,000 ka credit closing cost ke liye de diya hai, to main usko factor karna chahta hoon. Aap mera help kar sakte hain?"}, {"role": "assistant", "content": "Nayi ghar ki closing ke close hone par congratulations! Aapko closing cost estimate karne me mera help karna accha lagayega aur sellers se milane wala $2000 credit bhi consider karna. Accurate estimate karne ke liye aapko kuch aur information chaiye hai. Please provide the following details: 1. Purchase price: House ka final purchase price kitna hai? 2. Loan amount: Aap kitna loan le rahe hain? 3. Loan type: Aap kis type ka loan le rahe hain?"}]}
{"filler_id": "FL08", "session_id": "50d66391_7", "turns": [{"role": "user", "content": "Main bedroom lighting ke baare mein kuch advice chah raha hoon. Main floor lamps leke aane ke baare mein soch raha hoon jo ki reading nook banane mein help kare."}, {"role": "assistant", "content": "Yeh kya achcha idea hai! Floor lamps bedroom mein ek accha addition ho sakti hai, cozy aur inviting atmosphere banate hue jo relax karne wale reading nook ke liye perfect hai. Yahan kuch tips diye jate hai jo aap apni bedroom ke liye suitable floor lamps choose karne me consider kar sakte hain: 1. Purpose ko consider karo: Kyunki aapne reading nook mention kiya hai, to task lighting ko priority dijye. Lamps jo adjustable arms ya heads wale ho unhe choose karo jo light ko aapki zaroorat ke hisab se exact place par direct kar sake. 2."}, {"role": "user", "content": "Main aajkal ek bedside lamp use kar raha hoon jo ek naya bulb se lagaya hua hai jo main Home Depot se chhota sa 15$ me le aaya tha, aur main uske warm tone ko bahut pasand karta hoon. Kya aap mera mana karte hain ki main floor lamps ke liye aise hi bulb options ya color temperature wale dekhun?"}, {"role": "assistant", "content": "Aapne pahle hi ek bulb pichhli hui jo aapko pasand hai, jo ek acchi shuruaati hai! Haan, floor lamps ke liye dekhein jo aapas mei jaisi bulb options le saken ya jo aapke paas use kar rahi bulb ke jaisa color temperature rakhenge. Kyunki aapne mention kiya ki aap Home Depot se le li hui bulb ka warm tone pasand hai, toh main assume kar raha hu ki uska range 2700K-3000K hai, jo ek warm white ya soft white color temperature hai."}]}
{"filler_id": "FL09", "session_id": "ultrachat_436464", "turns": [{"role": "user", "content": "Kya WHO ne global health mein public health ko improve karne ke liye kya key initiatives liye hain?"}, {"role": "assistant", "content": "Duniya sabse bada health organization (WHO) ne global health mei improvement ke liye kuch important initiatives start kiye hain, jaise: 1. Universal Health Coverage: WHO ka target hai ki sab log apne health services ki zaroorat ko paise ki koshish ki bina pao sake. Ismein essential health services ki access ko badhaya ja raha hai aur health inequalities ko samajhaya ja raha hai. 2. Global Vaccine Action Plan: WHO ka kaam hai ki sab log, woh kahan bhi rahe, life-saving vaccines ki access pao sake."}, {"role": "user", "content": "WHO ke in saare initiatives dekhne par achha lagta hai. Lekin aap kya soch rahe hain ki yeh public health mei improvement karne mei effective hain?"}, {"role": "assistant", "content": "Main koi personal opinion nahi rakhata hoon. However, WHO ke initiatives jo global health mei significant progress kiya hai, unka example global vaccine action plan hai jo vaccination coverage ko increase karne mei contribute kiya hai, aur tobacco free initiative jo tobacco products ke demand ko kam karne mei successful raha hai. Similarly, WHO ke efforts in disease control, emergency response, aur mental health promotion ne bhi positive outcomes kiya hai."}]}
{"filler_id": "FL10", "session_id": "429be85c_4", "turns": [{"role": "user", "content": "Main weekend ke liye kuch new breakfast treats try karne ke liye hai jo coffee ke saath pair ho saken. Main weekend ke liye bada coffee break lete hoon aur kuch breakfast treats lete hoon, aur main apni usual cinnamon rolls se kuch tired ho rahi hoon. Kya aap kuch recipe suggestions ya recommendations ho sakte hain?"}, {"role": "assistant", "content": "Main help karne ke liye happy hoon! Shab sab ke liye ek acha breakfast treat karein aur ek garam coffee ke saath weekend bhar pasand karte hain, yeh zindagi ke sabse acche khushiyon mein se ek hai. Mai kuch scrumptious recipe suggestions leke aayi hoon jo aapko kuch new ka try karne me help kare. Yahan kuch ideas diye hain jo aapko shuru karne me help kare: 1. Cranberry Orange Scones: Flaky, buttery scones jo dried cranberries aur orange zest se baarish hoti hai, yeh ek rich cup of coffee ke perfect match hote hain. 2."}, {"role": "user", "content": "Unsalted butter aur salted butter ke beech kya fark hai in recipes me? Kya ye real me final taste aur texture me fark karata hai?"}, {"role": "assistant", "content": "Unsalted butter ka use salted butter ke jaise karne se aapke baked goods ke final taste aur texture me fark a sakta hai."}]}
{"filler_id": "FL11", "session_id": "33994682_3", "turns": [{"role": "user", "content": "Main apni andheri kholne ke liye freeze karne wala hoon aur kuch research bhi kar raha hoon, lekin process aur cost ke baare mei nahi pata. Aap mujhe general procedure explain kar sakte hain aur uske saath cost ke bare mei bhi idea de sakte hain? Taraf se, meri behen jo meri tarah hi 3 saal bada hai, ab 35 saal ki hai aur sochne mei bhi kafi ghat gaya hai ki kitna fast time badal gaya."}, {"role": "assistant", "content": "Main aapko process aur cost ke baare mein samajhne mein madad karne ke liye happy hoon! Process: Egg freezing, jo oocyte cryopreservation bhi kaha jata hai, ek multi-step process hai jo 2-3 weeks tak le sakti hai. Yahan overview hai: 1. Initial consultation: Aap ek fertility specialist (reproductive endocrinologist) ke saath meet karega, jo aapke medical history, fertility goals aur aapke paas kya concerns hai, us baare mein discuss karega. 2."}, {"role": "user", "content": "Main bhi egg freezing ka success rate janne ki koshish kar raha hoon. Frozen egg se healthy pregnancy aur child ki kya probability hai, aur mera age isme kya role play karta hai?"}, {"role": "assistant", "content": "Egg Freezing Ka Sucess Rate: Egg freezing ka success rate technology aur vitrification methods mein improvement ke saath badh raha hai. Lekin, yeh samajhne ka ek important hai ki egg freezing kisi bhi successful pregnancy ya child birth ka guarantee nahi hota. Frozen egg ke healthy pregnancy aur child birth hone ka chance kuch factors par depend karta hai, jaise: 1."}]}
{"filler_id": "FL12", "session_id": "f75238a5_2", "turns": [{"role": "user", "content": "Main aajkal apni project ke liye AI tools explore karne ke baare mein soch raha hoon, aur main soch raha hoon ki aap kuch resources suggest kar sakte hain AI ke latest developments ke baare mein. Taraf se, main ne aapko is topic ke baare mein ek acchi baat ki thi - humne is topic par 10 minute tak discuss kiya, aur isne meri interest ko bahut jorda kiya."}, {"role": "assistant", "content": "Main baarish se pata chala ke aap apne project ke liye AI tools ke baare mein explore karne ke liye interested hain! AI ke field mein kuch bhi aage badh raha hai aur sabse latest advancements ke saath up to date rehna bahut mehnat ki baat hai. Resources ke liye, main aapko kuch top notch ones recommend kar sakte hain: 1. AI Alignment Podcast: Ek acha podcast jo latest AI research, applications aur implications ke baare mein deep dive karta hai."}, {"role": "user", "content": "Main definitely un resources check karega, especially podcast aur blog se Emerj. Kya aap AI ka current state healthcare aur finance jaise industries mein application ke baare mein aur bata sakte hain?"}, {"role": "assistant", "content": "AI industries mein khatron ke saamne aayi hai, aur healthcare aur finance bhi exception nahi hain. Yahan ek brief overview hai AI ka current state in these sectors: Healthcare: 1. Disease diagnosis: AI powered systems medical images jaise X-rays aur MRIs analyze karke cancer, diabetes aur cardiovascular disease jese diseases detect kar sakte hain high accuracy se. 2. Personalized medicine: AI patients ke genetic profiles, medical histories aur lifestyle factors ke basis par treatment plans tailor karne mei help karta hai. 3."}]}
{"filler_id": "FL13", "session_id": "ultrachat_364297", "turns": [{"role": "user", "content": "Aap ducks aur geese ke breeding habits ko compare aur contrast kar sakte hain?"}, {"role": "assistant", "content": "Main koi personal experiences, emotions ya beliefs nahi rakhata hoon. Lekin main aapko jo information chahiye, usse provide kar sakta hoon. Ducks aur geese dono waterfowl hote hain aur breeding habits mein kuch similarities hain, lekin kuch differences bhi hain. Yahan kuch comparisons aur contrasts diye gaye hain: Similarities: - Ducks aur geese dono breeding season mein pair bonds bana leti hain. - Dono species ko mostly life ya long-term mating hota hai, although kuch cases mein partners change ho jate hain."}, {"role": "user", "content": "Kya aap mujhe duck ke courtship display ke baare mein zyada bat sakte hain? Iska sound interesting lag raha hai."}, {"role": "assistant", "content": "Haan, main accha hoon! Mausam ke liye kuch aur karna hai, lekin aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko kuch aur samajhna hai. Aapko k"}]}
{"filler_id": "FL14", "session_id": "08ca1f31_2", "turns": [{"role": "user", "content": "Main is weekend par meri Trek Emonda par ek longer bike trip plan karna chahta hoon, maybe 80-100 miles. Kya aap mujhe route suggestions ya tips provide kar sakte hain bike ko long ride ke liye prepare karna? Taraf se, main 3 weeks pahle us bike par tune-up karwaya tha aur shop ne chain replace kar di aur derailleur adjust kar diya jo $80 ka laga."}, {"role": "assistant", "content": "Ek long bike trip ka idea bahut accha lag raha hai! Mai aapke liye route ke suggestion aur bike prepare ke tips madad karne ke liye happy hoon. Route Suggestions: Pehle bike prep ke baare mein jao, lekin route ke baare mein baat karte hain. Kyunki aapne location nahi bataya, toh mai kuch general suggestions dedeta hoon. Aap online tools jaise MapMyRide, Ride with GPS ya Strava use karke apne area mein routes find kar sakte hain. 1."}, {"role": "user", "content": "Main ek route baar baar karte hoon jo kuch hills le jaye, kyunki aajkal main apni Trek Emonda se aadha time flat commutes karte hoon. Kya aap kuch specific hill climbing tips ya gearing aur cadence ke liye recommendations rakhate hain?"}, {"role": "assistant", "content": "Pahade climber karne ka ek accha tareeka aapke aapko challenge karna aur overall cycling fitness badhane ka hai. Yahan kuch hill-climbing tips, gearing ke liye recommend karte hain aur cadence ke baare mein advice diya gaya hai jo aapko Trek Emonda par un inclines ko tackle karne me madad kare: Hill-Climbing Tips: 1. Aapko pace kijiye: Shuru me ek steady aur moderate pace lein aur usko climb ke through rakhein. Surging ya sprinting avoid kijiye, kyunki yeh exhaustion ka karan ho sakta hai. 2."}]}
{"filler_id": "FL15", "session_id": "1c1f5ccc_6", "turns": [{"role": "user", "content": "Main Indian vastra ke cultural significance ke baare mein jankari lete hoon, jismein mera naya festival se leke aaya scarf bhi shamil hai. Kya aap mujhe traditional Indian scarf ke history aur symbolism ke baare mein jyada bat sakte hain? Taraf se, main apni apni cultural heritage ke baare mein jyada jankari le raha hoon, jismein India ke history aur customs bhi shamil hain, aur yeh ek acchi shuruaat lag rahi hai."}, {"role": "assistant", "content": "Yeh kya acha topic hai! Indian clothing, jismein aapko interest raha hai scarf, ek bohot rich cultural significance rakhta hai jo centuries tak chal raha hai. Mai aapke liye history aur symbolism ke baare mein baat karne ke liye more than happy hoon, aur aapko apne cultural heritage ko explore karne ke liye thrilled hoon. India mein scarves traditional attire ka ek integral part hote hain, aur unki names regions par depend karte hain. Kuch common types of Indian scarves hai: 1."}, {"role": "user", "content": "Maine traditional Indian scarves mein use hone wale different fabrics ke baare mein jankari lekhni chaahiye. Kya kuch specific fabrics aise hain jo dusre se zyada symbolic ya prestigious maana jata hai?"}, {"role": "assistant", "content": "Fabrics ek mahatvapurn role khelte hain traditonal Indian shawl mein, aur har type ka apna apna unique characteristics, importance aur cultural associations hote hain. Mai aapko kuch sabse common aur revered fabrics ke bare mein introduce karte hoon jo Indian shawls mein use hoti hain: 1. Silk: Silk Indian textiles mein sabse prestigious aur luxurious fabric maana jata hai. Yeh royal, wealth aur grandeur ke saath associate hota hai. Silk shawls unke softness, luster aur durability ke liye bahut cherish ki jaati hain."}]}
{"filler_id": "FL16", "session_id": "0651f7bd_2", "turns": [{"role": "user", "content": "Maine apni vintage camera collection ko kaise store aur display karein wahi tips chahiye. Mai few rare ones lagbhag rakhi hoon aur unhe dust aur damage se bachane ki zaroorat hai. Taraf se, aaj mai 1982 ka Cal Ripken Jr. ka ek rare card post mein mila hai, lekin wo meri ek alag collection hai."}, {"role": "assistant", "content": "Tumhare rare Cal Ripken Jr. card par congratulations! Yeh badi baat hai! Ab, let's focus on storing aur displaying tumhare vintage camera collection. Proper storage aur display tumhare cameras ke condition aur value ko preserve karne ke liye crucial hai. Yahan kuch tips hai jo tumhare vintage cameras ko dust aur damage se protect karne me help karein: Storage: 1. Climate control: Tumhare cameras ko 60°F se 70°F (15°C se 21°C) ke beech ek consistent temperature wale room mein store karein aur humidity 60% se kam rakhie."}, {"role": "user", "content": "Kya aap kuch acchi resources suggest kar sakte hain camera restoration aur maintenance ke baare mein? Mai chhoti si cameras lag rahi hoon jo TLC chahiye, aur mai nahi pata kahan se shuru karo."}, {"role": "assistant", "content": "Camera restoration aur maintenance ek kamniyaat skill hai jo kisi bhi camera enthusiast ke liye zaroori hai. Yahan kuch acha resources hai jo aapko learn karne aur shurु karna me help karein: Online Forums aur Communities: 1. The Camera Repair Forum: ek dedicated forum jo ek bada community of experts aur enthusiasts ka hai jo camera repair aur maintenance ke bare me knowledge aur experiences share karte hain. 2. Photography Stack Exchange: ek Q&A platform jo aapko questions puchne aur experienced photographers aur camera enthusiasts se answers milane ka maafat deta hai. 3."}]}
{"filler_id": "FL17", "session_id": "01d4b26b_1", "turns": [{"role": "user", "content": "Main ek coworker ka baby shower ke liye gift idea seekh rahi hoon. Kuch eco-friendly aur cute chizo chahiye. Taraf se, main apni baaji ko pichle maah birthday par ek silver necklace sath me ek chhoti si pendant di thi aur woh usko bahut pasand ki thi."}, {"role": "assistant", "content": "Tumhari behen ke liye kya achcha gift! Main tumhe kuch eco-friendly aur cute gift ideas help kar sakti hoon jo tumhare coworker ke baby shower ke liye hain. Yahan kuch suggestions hain: 1. Organic Cotton Onesie aur Baby Blanket Set: Ek soft, eco-friendly onesie aur blanket set jo organic cotton se banaya gaya hai. Aap ek cute design ya personalized message wala set bhi choose kar sakte hain. 2. Bamboo Baby Toys: Bamboo ek sustainable, eco-friendly material hai jo baby toys ke liye perfect hai."}, {"role": "user", "content": "Main organic cotton onesie aur baby blanket set ka idea pasand karte hain. Kya aap mujhe us brands ke baare mein bata sakte hain jo customization options dete hain?"}, {"role": "assistant", "content": "Organic cotton onesie aur baby blanket sets sabse acha gift hote hain, thoughtful aur eco-friendly. Kuch brands onesies aur baby blankets ke liye customization options pesh karte hain. Yahan kuch popular brands hain: 1. Hanna Andersson: Unka soft, organic cotton clothing famous hai, Hanna Andersson onesies, sleepers aur blankets ke liye customization options pesh karte hain. Aap fonts, colors aur designs ke various options se personalized gift banaye sakte hain. 2."}]}
{"filler_id": "FL18", "session_id": "e2691068", "turns": [{"role": "user", "content": "Main kuch new breakfast ideas chaiye jo meine energy boost degi. Kya aap koi suggestion rakh sakte hain?"}, {"role": "assistant", "content": "Ek jagaar badhao wala breakfast aapke din ka shuru hone ke liye thoda sa acha hai! Mai kuch delicious aur nutritious breakfast ka idea leke aayi hoon jo aapko din ka kaam karne ke liye energy dedege: 1. Avocado Toast with Poached Eggs: Whole grain toast par mashed avocado, ek poached egg aur ek chhooti red pepper flakes. Avocado ke healthy fats aur egg ke protein aapko full aur focused rakhege. 2."}, {"role": "user", "content": "Main Avocado Toast with Poached Eggs aur Smoothie Bowl ka sound pasand karte hain. Kya aapko shorba ke liye kuch tips hain jo sabse pehle waqt mein tayyar karte hue mil sake?"}, {"role": "assistant", "content": "Acha choices! Dono Avocado Toast with Poached Eggs aur Smoothie Bowl acchi aur sundar shakhar hai jo saboot me fast prepare kar sakti hain. Yahan kuch tips hai jo aapko saboot me fast prepare karne me help karein: Avocado Toast with Poached Eggs: 1. Prep ahead: Avocado mash karo aur ek airtight container me fridge me rakh do jis me aapko saboot me 5 minute save ho jaye. 2."}]}
{"filler_id": "FL19", "session_id": "ultrachat_415674", "turns": [{"role": "user", "content": "Kaise neuroscience aur psychology mein advancements humare human mind aur behavior ke baare mein samajh parivartan kar liye hain?"}, {"role": "assistant", "content": "Neuroscience aur psychology mein advancements ne human mind aur behavior ke baare mein hamare understanding ko ek number of tareeqa se revolution kiya hai: 1. Brain Ka Samajh: Neuroimaging technology ke advancements, jaise ki functional magnetic resonance imaging (fMRI) aur electroencephalogram (EEG), ne hamare ko brain activity ko measure aur better samajhne ki facility di hai. Yeh technology researchers ko different brain parts ko various cognitive functions, jaise ki perception, attention, emotion, decision-making aur social cognition mein kaise involved hoti hai, investigate karne ki permission di hai. 2."}, {"role": "user", "content": "Itne aasani se dekhne wala hai ki neuroscience aur psychology mein developments kaise human mind aur behavior ke baare mein new insights pade hain. Aap kya soch rahe hain ki ye fields mein next breakthrough kya hogi?"}, {"role": "assistant", "content": "Main koi personal opinion nahi rakh paati hoon ya future events predict karne ka ability nahi hain. However, ongoing research aur recent developments ke base par, neuroscience aur psychology mein near future mein kuch areas mein significant breakthroughs ki expectation ho sakti hai. Ek research area jo increasing attention ko li rahi hai, gut-brain axis ka study hai."}]}
{"filler_id": "FL20", "session_id": "ultrachat_93661", "turns": [{"role": "user", "content": "Kya kuch best practices hain jo fast-paced work environment mein organize raha aur time effectively manage kare?"}, {"role": "assistant", "content": "1. Planner ya calendar use karo: sab tasks aur deadlines ek jagah likho taki aap apne schedule ko manage kar sakein aur cheezon ka kuch nahi miss ho. 2. Tasks ko prioritize karo: work ko importance aur deadline ke hisab se prioritize karo. sabse critical work pe aage badhna shuru karo, phir less important tasks tak pahuncho. 3."}, {"role": "user", "content": "Main especially distractions se free hone me struggle karta hoon, isliye woh definitely work karega. Kya aap kisi specific productivity apps ya tools ke liye recommend karte hain?"}, {"role": "assistant", "content": "Haan, kuch productivity apps aur tools available hain jo aapko organize rakhne aur time effectively manage karne me help karte hain. Yahan kuch recommendations hain: 1. Trello: Ek visual project management tool jo aapko boards, cards aur lists banane ke liye allow karta hai jo aapke tasks aur workflows ko organize karte hain. 2. Focus@will: Ek music streaming service jo background music play karta hai jo aapko focused aur productive rakhne me help karta hai. 3."}]}
{"filler_id": "FL21", "session_id": "b7c5c12f_2", "turns": [{"role": "user", "content": "Main sustainable living ke liye kuch suggestions chah raha hoon. Aajkal main uske baare mein bahut padh raha hoon, jisme mera har hafte pahunchane wala weekly magazine bhi shamil hai. Kisi bhi case me, aap mujhe daily life me carbon footprint kam kaise karein uske baare mein kuch tips de sakte hain?"}, {"role": "assistant", "content": "Sustainable living ek bahut important topic hai! Mai aapko iska help karne mei aapko zyada than happy hoon. Yahan kuch practical tips hai jo aapke daily life mei carbon footprint ko kam karne mei madad karein: 1. Energy Consumption ko kam karo: Lights, electronics aur appliances ko jab use nahi karte ho tab off karo taki standby power consumption nahi ho. LED bulbs use karo jo energy efficient hote hain. Thermostat ko adjust karo taki heating aur cooling ke liye less energy use ho. 2. Water ko conserve karo: Shorter showers (5-10 minutes) leke water aur energy ko save karo."}, {"role": "user", "content": "Main actual ek tarah se sustainable fashion ke baare mein interested hoon, jo main naya magazine mein pichle din read kiya tha. Kya aap mujhe kuch tips de sakte hain jo maine wardrobe ko more eco-friendly karna sakte hain?"}, {"role": "assistant", "content": "Sustainable fashion ek acchi area hai jisme focus karna chahiye! Fashion industry ki ek big environmental impact hoti hai, lekin kuch conscious choices karke aap apni fashion footprint ko kam kar sakte hain. Yahan kuch tips hai jo aapke eco-friendly wardrobe banane me help karein: 1. Pehle Second-Hand buy karo: Thrift stores, consignment shops aur online marketplaces jaise eBay, ThredUp ya Poshmark second-hand shopping ke liye acche resources hain. Aap unique aur high-quality items find kar sakte hain, samayev aap new, resource-intensive clothing ke demand ko kam kar sakte hain. 2."}]}
{"filler_id": "FL22", "session_id": "df6970d8", "turns": [{"role": "user", "content": "Main new board games ke liye recommendations chah raha hoon jo mere doston aur parrtton ke saath khelne ke liye ho. Koi suggestion hai?"}, {"role": "assistant", "content": "Naya board games kahan se mil sakte hain friends aur family ke saath? Mai apko ek treasure trove suggestions leke aayi hoon! Yahan kuch modern classics, hidden gems aur popular games various genres ke liye hain: Strategy aur Competition 1. Gloomhaven: Ek cooperative campaign-based game jo modular board aur Legacy-style gameplay ka use karta hai. 2. Scythe: Ek worker placement game jo 1920s ka ek alternate history par based hai, aur engine-building aur area control ka include karta hai. 3."}, {"role": "user", "content": "Main pahle Ticket to Ride aur Settlers of Catan play kar chuke hoon, un acche games hain. Kya aap Scattergories jaisi games ke liye kuch recommendations hain?"}, {"role": "assistant", "content": "Scattergories ek badi acchi game hai! Agar aap creative, word-based gameplay enjoy karte hain, to yahan kuch recommendations hain jo aapke same feel kiya karein: 1. Codenames: Chahe woh Scattergories ke exactly aise nahi ho, Codenames me word guessing aur creative thinking ka use hota hai. Players ek word clue ke basis par words identify karte hain jo unke teammate deta hai. 2. Quelf: Ek party game jo players ko kuch criteria ke basis par unique words ya phrases banane ki challenge deti hai, jo baar baar hilarious results dekhne wale hote hain. 3."}]}
{"filler_id": "FL23", "session_id": "ultrachat_518652", "turns": [{"role": "user", "content": "Trojan war ke Greek mythology mein kya mahatva thi?"}, {"role": "assistant", "content": "Trojan war Greek mythology ke sabse important events mein se ek tha. Yeh ek yudd tha jo Greeks aur Trojans ke beech hua tha aur yeh 10 saal tak chala. Yeh yudd Helen of Troy ke liye hua tha jo Menelaus, Sparta ka raja, ka pati thi. Helen Paris, Troy ka prince, ne abduct karke Troy le li thi. Yeh yudd important tha kyunki yeh honor aur glory ke liye hua hua tha ke kaha jaata tha."}, {"role": "user", "content": "Trojan war me kis taraf ultimately jeet gayi?"}, {"role": "assistant", "content": "Greeks ne Trojan war ko jeet liya, kyunki unhone Trojan horse ka istemal kiya, jo ek laud ki gadi thi jo Greek soldiers ke saath bhari thi. Yeh horse Troy ki darwaze ke bahar rakha gaya tha, jo ek dharma parivish (offering) ke taur par godon ke liye thi. Trojans ne usko city ke andar le aaya. Us raat, Greek soldiers horse se nikal ke Troy ki darwaze kholiye aur Greek army ko city meh enter karne ki permission di."}]}
{"filler_id": "FL24", "session_id": "ultrachat_106352", "turns": [{"role": "user", "content": "Kya healthy adult ke liye raste me kitna sleep recommend hain?"}, {"role": "assistant", "content": "A healthy adult ko har raat 7-9 ghante sleep kiya jaye."}, {"role": "user", "content": "Woh sirf sleep karne ka bahut time lag raha hai. Mai 5-6 ghante se kaise nahi chal sakti?"}, {"role": "assistant", "content": "Aapke liye ek AI language model hoon, isliye main health routines mein changes karne ke liye advice nahi de sakta. Lekin studies ke according, 7 ghante se kam sleep consistently lete hue negative health effects ho sakte hain, jaise cognitive function me kam, fatigue aur weak immune system. Adults ko 7 ghante sleep per night karne chahiye taki good health rakh sake."}]}
{"filler_id": "FL25", "session_id": "fb793c87_2", "turns": [{"role": "user", "content": "Main HBO Max par kuch new shows dekhne ke liye search kar raha hoon. Main \"Game of Thrones\" ko ek dosti ke liye check karne ka plan kar raha tha, aur ek maah pahle usko shuru kar diya. Ab tak main pehle do paanch seasons complete kar chuke hoon, aur main bahut enjoy kar raha hoon. Kya aap \"Game of Thrones\" jaisa kuch aur recommend kar sakte hain?"}, {"role": "assistant", "content": "\"Game of Thrones\" ek epic show hai! Aap halfway hai aur main surahat karta hoon aap hook ho gaye hain. Agar aap HBO Max par similar shows dekhna chahte hain, toh main aapke liye kuch fantastic recommendations lagate hain: 1. Westworld: Ek aur HBO epic, jo ek futuristic theme park par set hai jaha guests lifelike robots ke saath interact kar sakte hain. Yeh ek complex aur thought-provoking series hai jo action, drama aur politics ka similar blend rakhata hai. 2. Vikings: Ek historical drama jo Ragnar Lothbrok, ek legendary Viking warrior ke adventures ko follow karta hai."}, {"role": "user", "content": "Woh ek acchi list hai! Main definitely kuch dekhte hoon. Main aajkal Hulu par kuch documentaries dekhta hoon, jisme true crime aur science ke special series sabse zyada pasand hain. Kya aap HBO Max par documentaries ke liye kuch suggestions kar sakte hain?"}, {"role": "assistant", "content": "HBO Max par documentaries ka ek impressive collection hai, aur main tumhe kuch badi acchi suggestions leke aaya hoon: True Crime: 1. I'll Be Gone in the Dark: ek documentary series jo Michelle McNamara ki book \"The Golden State Killer\" par based hai. Yeh case aur uska victims aur unke paryavaran par impact ka ek gripping aur thought-provoking exploration hai. 2."}]}
{"filler_id": "FL26", "session_id": "b0b73826", "turns": [{"role": "user", "content": "Main is weekend ke liye friends ke saath ek get together plan karte ho. Kya aap kisi acchi brunch place ke liye suggestion dena chahenge?"}, {"role": "assistant", "content": "Main help karne ke liye happy hoon! Lekin main ek large language model hoon, main personal experiences ya specific locations ke liye real-time information nahi rakh sakta. Lekin main aapko kuch general tips aur questions provide kar sakta hoon jo aapko acha brunch spot find karne me help kare: 1. Aapke friends ke preferences consider kare: Puchh lo aapke friends ko unke dietary restrictions, favorite cuisines aur preferred ambiance ke baare me. Yeh aapko options ko narrow karne me help karega. 2."}, {"role": "user", "content": "Main ek naya jagah try karne ke baare mein soch raha tha, lekin ab main usi jagah jayein ke baare mein soch raha hu jo meri ma ke saath naya se naya gaya tha. Waha accha vibe thi aur mithai makan."}, {"role": "assistant", "content": "Woh ek acchi idea hai! Pehle se ghusa hue jagah pata karna ek acchi option hai, especially agar aap uske atmosphere aur khana ke baare me pata hain aur pasand bhi karte hain. Ek reliable jagah rakhna hamesha accha lagta hai jo aapko ek acchi experience deti ho. Plus, aap apne parents ke saath waha gaye hain toh aapko us jagah ek special connection ho sakti hai, aur aap apne friends ke saath us experience share karne ka fun ho sakti hai."}]}
{"filler_id": "FL27", "session_id": "ultrachat_576167", "turns": [{"role": "user", "content": "Kaise photography mein use hone wale lens ke types final image ko affect karte hain?"}, {"role": "assistant", "content": "Photography mein use hone wale lenses ke types final image ko kuch tarah se affect karte hain: 1. Focal length: Lens ke focal length uska angle of view aur subject ka magnification determine karta hai. Ek longer focal length lens (telephoto) perspective ko compress karta hai aur depth of field ko shallow banata hai, jabki ek shorter focal length lens (wide-angle) perspective ko exaggerate karta hai aur depth of field ko wider banata hai. 2."}, {"role": "user", "content": "Photo mein kisi subject ya situation ke liye sahi lens kaise choose kare?"}, {"role": "assistant", "content": "Saarey subject ya situation ke liye sahi lens choose karna photography mein final image ko bahut prabhavit karta hai. Yahan kuch considerations diye gaye hain jo aapko sahi lens choose karne mei madad karein: 1. Focal length: Aapke subject se distance consider karein. Agar aap ek dost se farak hai toh longer focal length lens (telephoto) use karein. Agar aap subject se close hai toh shorter focal length lens (wide-angle) use karein. 2. Aperture: Lighting conditions consider karein."}]}
{"filler_id": "FL28", "session_id": "ultrachat_182518", "turns": [{"role": "user", "content": "Kaise mining industry ne San Luis Potosí ki economy aur social structure ko history mei impact kiya hai?"}, {"role": "assistant", "content": "Mining industry ne San Luis Potosí ki economy aur social structure par aapna ek mahatvapurn impact rakha hai. 16th century mein region mein silver deposits ka discovery ho gaya, jo multiple mines aur mining towns ka establishment karne mein madad karaya, jo ki work aur aur opportunities ke liye aadmiyon ka ek bada population attract karne mein madad karayi."}, {"role": "user", "content": "Mining ke San Luis Potosí par impact janne ka real interest hai. Kya industry ke negative consequences ke liye koi efforts kiye gaye hain?"}, {"role": "assistant", "content": "Haan, koshish ki gayi hai negative consequences of mining industry in San Luis Potosí ko samalne ke liye. Pasand se, local communities aur ecosystems par mining ke environmental impacts ke baare me awareness badh gayi hai, jo strict regulations ka adoption aur more sustainable mining practices ke development mein help kare."}]}
{"filler_id": "FL29", "session_id": "98cd882c_1", "turns": [{"role": "user", "content": "Maine apni bhabhi ke janam ka 3 weeks mei ek quilt banaya hai aur ab kuch naya start karne ke liye ready hoon. Kya kuch new sewing project ke liye ideas mil sakti hain? Taraf se, quilt ke liye 3 weeks lagaye, aur main uska result par bahut happy hoon."}, {"role": "assistant", "content": "Bhaiya, aapne sister ke birthday ke liye quilt complete kar liya! Yeh ek acchi achievement hai, aur main suraj hai ki woh isko cherish karegi. Teen hafte ek quilt complete karne ka time frame bahut accha hai, jo ki iske liye lagne wale time aur effort ko dekh kar. Ab, aapke next sewing project ke liye brainstorming shuru karte hain! Yahan kuch suggestions hai jo aapko start karne me help kare: 1. Tote Bag ya Purse: Ek functional aur practical project jo ki tazakar se complete ho sakta hai."}, {"role": "user", "content": "Main summer ke liye ek sundress banane par soch rahi hoon, phir se kuch floral pattern wala kuch naya ho. Main kabhi garment nahi banai hai, isliye main uske baare mein kuch nervous hoon. Kya aap kuch tips ya patterns suggest kar sakte hain jo ek shuruwati jaise mein ke liye help karein?"}, {"role": "assistant", "content": "Sundress banane ka ek accha project hai beginner ke liye! Kuch guide ke saath aap ek beautiful aur comfortable garment summer ke liye banaye sakte hain. Mere pas tips aur pattern suggest karne ke liye help karne ka mauka hai. Tips for a Beginner: 1. Choose a simple pattern: \"beginner-friendly\" ya \"easy\" label wale patterns dekhenge. Too many pieces, complex construction ya intricate details wale patterns avoid karein. 2. Select a forgiving fabric: Cotton, linen ya rayon jaise lightweight aur breathable fabrics choose karein."}]}
{"filler_id": "FL30", "session_id": "f5f1ff92_1", "turns": [{"role": "user", "content": "Main mortgage application process ke baare mein kuch advice chahiye. Main pahle hi lender ko sab required documents bhej chuke hoon, jaise mein pay stubs, bank statements aur tax returns. Lekin main next ke liye kya expect karein is baare mein nahi pata. Aap mujhe application bhejne ke baad typical steps ke bare mein walk through kar sakte hain?"}, {"role": "assistant", "content": "Mortgage application submit karne par congratulations! Document submit karne ke baad lender aapke application ko review aur process karega. Yahan ek general outline hai jo typical steps ke baare mein hai: 1. Initial Review: Lender ka loan processor aapke application ko review karega taki sab required documents complete aur sahi hote dekhe. Is step me 1-3 business days lagte hain. 2. Loan Processing: Loan processor aapke credit report ko review karega, income ko verify karega aur debt-to-income ratio ko evaluate karega."}, {"role": "user", "content": "Main appraisal step ke baare me ek tarah se dar gaya hoon. Aap mujhe kuch aur bataye ki appraisal ke waqt kya hota hai aur iska loan approval process par kaise impact padta hai?"}, {"role": "assistant", "content": "Appraisal step mortgage process ke ek crucial part hai, aur iske baare me questions hone natural hai. Yahan ek detailed explanation hai ki appraisal ke time kya hota hai aur iska loan approval process par kaise impact padta hai: Kya hai appraisal? Real estate appraisal ek independent evaluation hai property ka value, jo licensed appraiser ke dwara kiya jata hai. Appraiser property ka market value ke objective opinion provide karta hai, jo various factors par based hota hai jaise: 1."}]}
{"filler_id": "FL31", "session_id": "39948bcf_4", "turns": [{"role": "user", "content": "Main apne area me yoga class ke liye kuch suggestions chah raha hoon. Main pass on ke liye practice continue karne me interested hoon kyunki last year November me main ek akhanda yoga retreat me gaya tha jo meditation, pranayama aur asana practice ke kuch workshops bhi shamil the."}, {"role": "assistant", "content": "Wohi! Achha hai k ke aap retreat ke baad yoga practice ke liye inspired hain. Yahan kuch tips hai jo aapke needs ke according yoga class find karne me madad karein: 1. Online Search: Google ya kisi aur search engine me \"yoga classes near me\" ya \"yoga studios in [your city/zip code]\" type karein. Aap bhi online directories jaise YogaFinder ya Yoga Alliance use kar sakte hain jo aapke area me studios aur classes find karne me madad karte hain. 2."}, {"role": "user", "content": "Main aksar restorative yoga par focus karte hue studio find karne mein interested hoon, kyunki main retreat ke durn usi workshop ko bahut pasand kiya tha. Kya aap meri area me kisi bhi studio ke bare me jaante hain jo restorative yoga me specialise ho?"}, {"role": "assistant", "content": "Adhut! Restorative yoga ek studio ke liye ek accha dhyan kendr hai, aur aapke is kshetra mein aapke abhyas ko gharaa karna ke liye aapke khaj karna bahut accha hai. Burai baaat yeh hai ki main ek bade bhasha model hoon, isliye mujko aapke sthaan ya aapke kshetra mein vishisht studio ke bare me vaastavik samay ke data ke liye pahuch nahi hai. Haalankar, main aapko apne pas restorative yoga studio kaise dhundhe is bare me margadarshan kar sakta hoon: 1."}]}
{"filler_id": "FL32", "session_id": "ultrachat_82739", "turns": [{"role": "user", "content": "Backstage passes kya kuch logon tak hi maante hain? Agar hai to, kaise main ek paku karein?"}, {"role": "assistant", "content": "Haan, backstage passes kai aadat se kuch logon ke liye hi milte hain. Apne chances badhane ke liye yahan kuch tips hai: 1. Fan clubs ya mailing lists mein shamil ho jao: Kuch artist aur bands apne fan clubs ya mailing lists ke liye backstage passes ya tickets ke pre-sales ka exclusive access provide karte hain. 2. Social media par follow karo: Artists aur bands apne social media platforms par backstage passes ke giveaways ya promotions announce karte hain. 3."}, {"role": "user", "content": "Main definitely unko social media par follow karoonga aur kuch contests enter karunga. Aap kya jaante hain ki koi upcoming concerts hain jo backstage passes bhi mil sake?"}, {"role": "assistant", "content": "Aapke liye real-time information milne se main sab help nahi kar sakta hoon upcoming concerts ke bare me jo backstage passes available hote hain. Lekin aap artist ya band ka official website ya social media page check kar sakte hain jo aapke upcoming concerts ke bare me aur kya backstage pass opportunities hain unke paas. Aap concert venue ya ticketing agency se bhi puch sakte hain jo special VIP ya backstage packages pesh karte hain."}]}
{"filler_id": "FL33", "session_id": "f340a3db", "turns": [{"role": "user", "content": "Main May 1st ko ek social media campaign start karne ke liye plan karte hoon, mental health ke baare mein awareness badhane ke liye, aap mujhe kuch engaging post ideas aur hashtags banaane me help karsakte hain?"}, {"role": "assistant", "content": "Yeh kya acha initiative hai! Mai aapke liye social media campaign ke liye mental health par awarness badhane ke liye engaging post ideas aur hashtags help karne ke liye happy hun. Yahan kuch suggestions hai: Post Ideas: 1. Personal Stories: Mental health issues ke struggle aur triumph ke personal stories share karein. Apne followers ko apni stories share karne ke liye ek branded hashtag use karne ka motivate karein. 2."}, {"role": "user", "content": "Kya aap kuch popular mental health advocates ya influencers suggest kar sakte hain jo main collaboration ya Q&A session ke liye contact kar sake?"}, {"role": "assistant", "content": "Yahan kuch popular mental health advocates aur influencers hai jo various platforms par hai jo aap ke liye collaborations ya Q&A sessions ke liye contact kar sakte hain: Mental Health Advocates: 1. Lady Gaga (@ladygaga) - Singer-songwriter aur Born This Way Foundation ka founder jo mental health aur wellness par focus karta hai. 2. Demi Lovato (@ddlovato) - Singer-songwriter aur mental health advocate jo apni depression aur addiction ke struggles ko open karke bata chuki hai. 3."}]}
{"filler_id": "FL34", "session_id": "ultrachat_245735", "turns": [{"role": "user", "content": "Kya culture ke barriers Kiowa bhasha ka saaksharata mein rokte hain, aur unhe kaise surmount kiya gaya hai?"}, {"role": "assistant", "content": "Kuch cultural barriers ne Kiowa bhasha ka saaksharataan khatm karne me rokava diya, jaise historical trauma, forced assimilation, aur young generation ke acculturation. 1800s ke late me, US government ne indigenous logon ko reservations par marr diya aur boarding schools banaye jo ki unke native bhasha ke use par band kar diya. Isne Kiowa bachhon ko unke families aur communities se nikal diya jo ki intergenerational trauma aur unke bhasha aur culture se disconnect hone me help kara."}, {"role": "user", "content": "Woh kafi interesting hai. Aap kya jaante hain ki yeh language revitalization programs ab tak kitna successful huye hain?"}, {"role": "assistant", "content": "Haan, Kiowa bhasha prachar yatra mein kuch samarjane hui hain. Jaise ki, Anadarko, Oklahoma mein Kiowa Jati ka bhasha upayog program, bhasha parichay aur upyog mein positive result de rakha hai. Yeh program chhoti baccheon mein bhasha parikshan karne aur unke samaj ke saath identity aur samaj se sambandh banane ke liye sanskriti upayog karta hai."}]}
{"filler_id": "FL35", "session_id": "ultrachat_20447", "turns": [{"role": "user", "content": "Kaise remote work employees ke social aur emotional wellbeing par impact kar raha hai?"}, {"role": "assistant", "content": "Aapke liye AI language model hoon, isliye main personal experiences, opinions ya emotions nahi rakh sakta. Lekin studies aur research ke according, remote work employees ke social aur emotional wellbeing par dono positive aur negative impacts ho sakte hain. Remote work ke positive impacts social aur emotional wellbeing par: 1. Flexible work arrangements long commutes aur demanding schedules se associate hone wale stress aur burnout ko kam kar sakte hain. 2."}, {"role": "user", "content": "Kya employers koi strategies implement kar sakte hain taaki remote workers home se kaam karne wala lonley aur isolated feel na kare?"}, {"role": "assistant", "content": "Haan, employers various strategies implement kar sakte hain jo remote workers ko home se kaam karte hue less lonely aur isolated feel karne me help karte hain. Yahan kuch strategies hain: 1. Regular virtual meetings encourage karein jo remote workers ko connect rakhe. Regular video check-ins aur team calls schedule karein jo belonging aur collaboration ka feeling foster kare. 2. Virtual coffee breaks, lunches ya happy hours create karein. Yeh breaks employees ko break lene, colleagues ke saath socialize karene aur rejuvenate hone me help karte hain. 3. Team building activities facilitate karein."}]}
{"filler_id": "FL36", "session_id": "9f35ac68", "turns": [{"role": "user", "content": "Main monthly bus pass karna soch raha hoon, aap mujhe cost aur kahan se le sakte hain bataye sakte hain?"}, {"role": "assistant", "content": "Main help karne ke liye happy hoon! Lekin main a bit more information chahiye. Kripya aapko neeche diye gaye details provide karein: 1. Aap kis city ya region me rahi hain (ya bus pass use karne ke liye plan karte hain)? 2. Aap kis type ka bus pass interested hain (jaise: adult, student, senior, disabled)? 3. Aap kis duration ka pass chahiye (jaise: 1 month, 3 months, 12 months)?"}, {"role": "user", "content": "Main city me rahi hoon aur main ek adult monthly pass ke liye interested hoon. Main 1-month pass seekh rahi hoon."}, {"role": "assistant", "content": "Main help karne ke liye happy hoon! Lekin main clarify karte hu ke main ek large language model hoon, main specific cities ya transportation systems ke real-time information ki access nahi rakhata hoon. Lekin main aapko kuch general guidance de sakte hu jo aapko information milane ke liye help kare. Aap apni city mein monthly bus pass kaise le sakte hain aur uska exact cost kya hai uske baare mein information pachhane ke liye aapko following steps follow karna chahiye: 1."}]}
{"filler_id": "FL37", "session_id": "285a29e7_2", "turns": [{"role": "user", "content": "Main kuch book suggestions chah raha hoon. Main aajkal bahut padh raha hoon, aur mere liye new authors discover karna hi sabse accha hota hai. Taraf se, main aaj ek local coffee shop mein poetry reading ke liye gaya tha, jahan kuch local poets apne recent collections se padhaye the - aisa dekhne ke liye bahut accha lagaya. Kuch bhi, kya aap kuch popular books ya authors suggest kar sakte hain?"}, {"role": "assistant", "content": "Woh sunne wala ek acha experience lag raha hai! Local coffee shop par poetry reading attend karna ek accha tareeka hai new talent ke saath connect hone aur literary community me connect hone ke liye. Book recommendations ke liye, main kuch popular books aur authors ke saath suggest kar sakti hoon jo different genres me popular hain. Yaad rakhna ki personal taste vary kar sakti hai, lekin yeh kuch widely acclaimed aur popular choices hain: Fiction: 1."}, {"role": "user", "content": "Main kuch poetry recommendations ke baare mein jyada explore karne ke liye khush hoon. Aaj mai jisme attend kiya poetry reading mera dil badal gaya hai, yahan talent ka ek aisa chakkar hai. Kya aap mujhe kuch contemporary poetry collections ya poets ka suggestion de sakte hain jo main enjoy kar payoon, jo ki main new voices ke discovery ke liye interested hoon?"}, {"role": "assistant", "content": "Main baahut khush hoon ki shairiyan ke baad aapne aur shairiyat ko explore karne ki koshish ki hai! Shairiyat ek jivanmukh aur prakar ke kshetra hai, aur bahut acche shair aur kashis ko seekhne wale hai. Yahan kuch behtar shairiyat kashis ke liye suggestions hai jo aapke liye resonates karein: Shairiyat Kashis: 1. \"When I Grow Up I Want to Be a List of Further Possibilities\" by Chen Chen: Ek premi kashis jo identiy, paryavaran, prem aur chaos ke duniya mein maanavta ka search ko explore karte hue hai. 2."}]}
{"filler_id": "FL38", "session_id": "sharegpt_NRqjZJS_0", "turns": [{"role": "user", "content": "Essay likhein jo Ernest Hemmingway aur John Steinbeck ke writing styles ko compare aur contrast kare."}, {"role": "assistant", "content": "Ernest Hemingway aur John Steinbeck 20th century ka do sabse famous writers hain. Dono log common man aur human condition ke struggles ko likhte the, lekin unka writing style bahut alag tha. Hemingway ka style concise, direct aur simple hai. Uska use short sentences aur minimalist approach hai message convey karne ke liye."}, {"role": "user", "content": "Kuch books ke reference se specific examples include karein jo unhone likhe hain."}, {"role": "assistant", "content": "Ernest Hemingway aur John Steinbeck 20th century ka do sabse famous writers hain. Dono log common man aur human condition ke struggles ko likhte the, lekin unka writing style bahut alag tha. Hemingway ka writing style concise, direct aur simple tha."}]}
{"filler_id": "FL39", "session_id": "0daaefa7_3", "turns": [{"role": "user", "content": "Main web development me entry karna chah raha hu, aur main soch raha hu ki aap kuch online resources suggest kar sakte hain JavaScript learn karne ke liye. Taraf se, main mid-February me ek 2 din ka coding bootcamp ke liye ja chuka hu, jaha main JavaScript ke basic ko seekh chuka hu, lekin main apne skills ko further improve karna chahta hu."}, {"role": "assistant", "content": "Bada achha hai ki aapne web development mei pasand ki shuruat ki aur coding bootcamp mei join kiya! JavaScript ke basic mei strong foundation hai to aap further dive kar sakte hain aur apne skills ko improve kar sakte hain. Yahan aapke JavaScript learn karne ke liye online resources diye gaye hain: Tutorials aur Courses: 1. Codecademy ka JavaScript Course: JavaScript ke basic, intermediate aur advanced concepts ko cover karte hue comprehensive aur interactive course hai. 2. FreeCodeCamp: Ek non-profit platform jo JavaScript ke challenges, projects aur certifications provide karte hain."}, {"role": "user", "content": "Main definitely yeh resources check karein ge. Mai especially JSFiddle aur HackerRank ke challenges par practice karne me interested hoon. Kya aapko koi tips hai jo code learn karne waqt motivation rakhne aur burnout se bachne me madad kare?"}, {"role": "assistant", "content": "JSFiddle aur HackerRank ke challenges ke saath practice karne se aapke JavaScript skills mei aadhi tarif hui jayegi. Motivation rakhne aur burnout se bachne ek code learn karne ke liye important aspects hain. Yahan kuch tips diye gaye hain: 1. Pachanewali goals aur milestones set karein: Apni learning journey ko chhoti, manage karne wali goals mei break karein. Kisi bhi chhoti achievement ko celebrate karein, yeh aapko motivation rakhne mei help karega aur apni progress track karne mei madad karega. 2."}]}
{"filler_id": "FL40", "session_id": "ultrachat_134317", "turns": [{"role": "user", "content": "Imperator ka religion ka head hone se Roman state ke authority me kaise kontribut ki?"}, {"role": "assistant", "content": "Imperator ka role religion ka head rakhna Roman state ke authority mei aapna hissa diya kyun ki religion Roman logon ke zindagi mei ek mahatvapurn role khata tha. Emperors ne religious institutions, rituals aur ceremonies par control rakha jo unhe traditional Roman values aur customs ke guardians banate rakha."}, {"role": "user", "content": "Chhoti chhoti baatein, to emperors ko koi specific religious practices karni padti thi ya unka position sirf symbolic thi?"}, {"role": "assistant", "content": "Chakravarti ka post religion ka head raja tha, yeh sirf symbolic nathi thi; balki unko certain religious practices follow karne aur religious ceremonies me shasthriya hone ki icheyat thi. Chakravarti ka religious duty shamil tha sacrifices karna, prayers pesh karna aur important religious ceremonies me officiate karna. Chakravarti ko god ke special relationship hone ki meethi thi aur unhe Roman religion ka high priest maana jata tha."}]}


## 4. Restore earlier parts and define the stage runner

In [ ]:
RUN, KROOT, ITEMS = "main_r1", "/kaggle/working/clm", "/kaggle/working/clm/data/items/items_main.jsonl"
RESULTS = f"{KROOT}/results/{RUN}"
NEEDS = ['1_prep']

# Earlier parts arrive under /kaggle/input/<notebook>/clm/results/<run>/ (or as results_<run>_*.zip).
found = []
for src in sorted(glob.glob(f"/kaggle/input/**/clm/results/{RUN}", recursive=True)):
    shutil.copytree(src, RESULTS, dirs_exist_ok=True); found.append(src)
for z in glob.glob(f"/kaggle/input/**/results_{RUN}_*.zip", recursive=True):
    shutil.unpack_archive(z, RESULTS); found.append(z)
print("restored from:", found or "nothing")
need_files = {"1_prep": "histories.jsonl", "2_qwen": "generations_qwen3-8b.jsonl", "3_llama": "generations_llama3.1-8b.jsonl"}
for n in NEEDS:
    f = need_files[n]
    missing = [b for b in ('main', 'short', 'q3', 'digits') if not os.path.exists(f"{RESULTS}/{'' if b == 'main' else 'abl_' + b + '/'}{f}")]
    if missing:
        raise RuntimeError(f"STOP: output of part {RUN}_{n} not found for blocks {missing}. "
                           "Add it with Add Input -> Your Work, then run again.")

def stage(name, block, extra=""):
    ip = get_ipython()
    ip.system(f"cd {KROOT}/src && CLM_ROOT={KROOT} CLM_RUN={RUN} CLM_BLOCK={block} "
              f"CLM_BACKEND={os.environ['CLM_BACKEND']} python -m clm.stages {name} --items {ITEMS} {extra}")
    if ip.user_ns.get("_exit_code", 0) != 0:
        raise RuntimeError(f"Stage '{name}' (block {block}) failed - copy the error above to Claude.")
    print(f"stage {name} / {block}: done")

## 5. Reader llama3.1-8b: main block first, then the ablations

In [ ]:
for b in ('main', 'short', 'q3', 'digits'):
    stage('generate', b, '--model llama3.1-8b')
!rm -rf ~/.cache/huggingface/hub/models--meta-llama--Llama-3.1-8B-Instruct ~/.cache/huggingface/xet ~/.cache/pip /tmp/*.whl 2>/dev/null; df -h /kaggle/working | tail -1

## Save outputs

In [ ]:
# The whole results folder stays in /kaggle/working and becomes this notebook's output (input for the next part).
shutil.make_archive(f"/kaggle/working/results_{RUN}_3_llama", "zip", RESULTS)
print("Output folder:", RESULTS)
print(f"Download zip: /kaggle/working/results_{RUN}_3_llama.zip  (right panel -> Output)")